In [ ]:
# ==============================================================================
# CELL 0: INSTALLATIONS
# (Run this section, THEN STOP, MANUALLY RESTART RUNTIME, then run THE ENTIRE CELL AGAIN)
# ==============================================================================
print("Starting installations (if this is the first run of the cell in a new session)...\n")
# Step 1: Install specific PyTorch ecosystem versions
print("Installing PyTorch, Torchvision, Torchaudio...")
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
print("PyTorch ecosystem installation attempt finished.")

# Step 2: Install other necessary packages
print("\nInstalling sentence-transformers and other libraries...")
!pip install sentence-transformers==2.7.0 nltk tqdm scikit-learn pandas # Added pandas here
print("Other packages installation attempt finished.")

print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
print("CELL 0 (INSTALLATIONS) HAS COMPLETED BOTH INSTALLATION STEPS.")
print("YOU **MUST** NOW MANUALLY RESTART THE COLAB RUNTIME FOR ALL CHANGES TO TAKE EFFECT.")
print("GO TO THE MENU: Runtime -> Restart runtime...")
print("AFTER RESTARTING, RUN THIS ENTIRE CELL AGAIN FROM THE TOP.")
print("If you skip this restart, you will likely see import errors or version conflicts.")
print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")

# The script will continue from here AFTER the manual runtime restart.
# The pip install commands above will effectively be skipped on the second run if packages are already installed.

# ==============================================================================
# CELL 1: IMPORTS, DRIVE MOUNT, PATH DEFINITIONS, NLTK, CLEANING UTILITIES
# (This part runs after the restart)
# ==============================================================================
print("\n--- SECTION 1: Initializing Setup ---")
from google.colab import drive
import os
import gc
from collections import OrderedDict, defaultdict
import zipfile
import json
import re
import nltk # NLTK should be imported before trying to access its submodules
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm.notebook import tqdm # Use notebook version for better display in Colab
import numpy as np
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd # Ensure pandas is imported

try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Google Drive mounting error: {e}. Please ensure Drive is accessible.")

# --- Base Paths ---
DRIVE_BASE_FOLDER = '/content/drive/MyDrive/colab-4/' # ENSURE THIS IS YOUR CORRECT BASE FOLDER

# --- Common File Names (Training & Testing) ---
# Ensure these filenames exactly match what you have in DRIVE_BASE_FOLDER
ABSTRACT_TRAIN_ZIP_FILENAME = 'longeval_sci_training_2025_abstract.zip'
FULLTEXT_TRAIN_ZIP_FILENAME = 'longeval_sci_training_2025_fulltext.zip'
ABSTRACT_TEST_ZIP_FILENAME  = 'longeval_sci_testing_2025_abstract.zip'
FULLTEXT_TEST_ZIP_FILENAME  = 'longeval_sci_testing_2025_fulltext.zip'

# --- Paths for A1 (Abstract Training) ---
ABSTRACT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, ABSTRACT_TRAIN_ZIP_FILENAME)
PIPELINE_A1_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A1_abstract_tfidf_train/')
os.makedirs(PIPELINE_A1_CACHE_DIR, exist_ok=True)
loaded_data_cache_A1_path = os.path.join(PIPELINE_A1_CACHE_DIR, 'loaded_cleaned_abstract_data_A1.pkl')
tfidf_vectorizer_A1_path = os.path.join(PIPELINE_A1_CACHE_DIR, 'tfidf_vectorizer_A1.pkl')
tfidf_matrix_A1_path = os.path.join(PIPELINE_A1_CACHE_DIR, 'tfidf_matrix_A1.pkl')
tfidf_results_A1_path = os.path.join(PIPELINE_A1_CACHE_DIR, 'tfidf_results_A1.pkl')
tfidf_predictions_A1_path = os.path.join(PIPELINE_A1_CACHE_DIR, 'tfidf_predictions_A1.pkl')

# --- Paths for A2 (Full-Text Training) ---
FULLTEXT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, FULLTEXT_TRAIN_ZIP_FILENAME)
PIPELINE_A2_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A2_fulltext_tfidf_train/')
os.makedirs(PIPELINE_A2_CACHE_DIR, exist_ok=True)
loaded_data_cache_A2_path = os.path.join(PIPELINE_A2_CACHE_DIR, 'loaded_cleaned_fulltext_data_A2.pkl')
tfidf_vectorizer_A2_path = os.path.join(PIPELINE_A2_CACHE_DIR, 'tfidf_vectorizer_A2.pkl')
tfidf_matrix_A2_path = os.path.join(PIPELINE_A2_CACHE_DIR, 'tfidf_matrix_A2.pkl')
tfidf_results_A2_path = os.path.join(PIPELINE_A2_CACHE_DIR, 'tfidf_results_A2.pkl')
tfidf_predictions_A2_path = os.path.join(PIPELINE_A2_CACHE_DIR, 'tfidf_predictions_A2.pkl')

# --- Paths for A3 (Abstract Testing) ---
ABSTRACT_TEST_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, ABSTRACT_TEST_ZIP_FILENAME)
PIPELINE_A3_TEST_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A3_abstract_tfidf_test/')
os.makedirs(PIPELINE_A3_TEST_CACHE_DIR, exist_ok=True)

# --- Paths for A4 (Full-Text Testing) ---
FULLTEXT_TEST_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, FULLTEXT_TEST_ZIP_FILENAME)
PIPELINE_A4_TEST_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A4_fulltext_tfidf_test/')
os.makedirs(PIPELINE_A4_TEST_CACHE_DIR, exist_ok=True)

print("Path definitions complete.")
# Verify essential ZIP paths
for zip_path_var_name, zip_file_path in [ # Renamed for clarity
    ("ABSTRACT_TRAIN_ZIP_PATH", ABSTRACT_TRAIN_ZIP_PATH),
    ("FULLTEXT_TRAIN_ZIP_PATH", FULLTEXT_TRAIN_ZIP_PATH),
    ("ABSTRACT_TEST_ZIP_PATH", ABSTRACT_TEST_ZIP_PATH),
    ("FULLTEXT_TEST_ZIP_PATH", FULLTEXT_TEST_ZIP_PATH)
]:
    if not os.path.exists(zip_file_path): # Use the full path variable
        print(f"CRITICAL WARNING: {zip_path_var_name} not found at: {zip_file_path}. Subsequent steps will likely fail.")

# NLTK Resources
nltk_resources_ready = False
try:
    stopwords.words('english'); WordNetLemmatizer().lemmatize('cats')
    print("NLTK resources seem available.")
    nltk_resources_ready = True
except LookupError:
    print("Attempting to download NLTK resources (stopwords, wordnet)...")
    try:
        nltk.download('stopwords', quiet=True); nltk.download('wordnet', quiet=True)
        nltk_resources_ready = True # Assume successful if no error
    except Exception as e_nltk_download:
        print(f"ERROR: Failed to download NLTK resources: {e_nltk_download}")
except Exception as e_nltk_check:
    print(f"An error occurred while checking NLTK resources: {e_nltk_check}. Attempting download.")
    try:
        nltk.download('stopwords', quiet=True); nltk.download('wordnet', quiet=True)
        nltk_resources_ready = True
    except Exception as e_nltk_download_fallback:
        print(f"ERROR: Failed to download NLTK resources even after fallback: {e_nltk_download_fallback}")

if nltk_resources_ready:
    STOP = set(stopwords.words("english"))
    LEM = WordNetLemmatizer()
    _token_pat = re.compile(r"[A-Za-z]{2,}") # Require at least 2 characters
    print("Text cleaning utilities ready.")
else:
    print("ERROR: NLTK resources not properly initialized. Text cleaning may fail.")
    STOP, LEM, _token_pat = set(), None, re.compile(r"[A-Za-z]{2,}") # Fallbacks

def clean_text_for_tfidf(text: str) -> str:
    if not isinstance(text, str): return ""
    if not LEM: return " ".join(text.lower().strip().split()) # Basic fallback
    toks = _token_pat.findall(text.lower())
    toks = [t for t in toks if t not in STOP and len(t) > 1]
    return " ".join([LEM.lemmatize(t) for t in toks])

def extract_text_from_abstract_json(record_dict):
    title = record_dict.get("title", "") or ""
    abstract_content = record_dict.get("abstract", "") or ""
    return f"{title} {abstract_content}".strip()

MAX_WORDS_FOR_TFIDF_FULLTEXT = 400
def extract_and_truncate_full_text_content_for_tfidf(record_dict):
    title = record_dict.get("title", "") or ""
    abstract = record_dict.get("abstract", "") or ""
    main_text = ""; possible_fields = ["text", "full_text", "body", "body_text", "content", "fulltext"]
    for field in possible_fields:
        content = record_dict.get(field)
        if content and isinstance(content, str): main_text = content; break
    combined = f"{title} {abstract} {main_text}".strip().split()
    return " ".join(combined[:MAX_WORDS_FOR_TFIDF_FULLTEXT])

print("Setup and helper functions defined.")

# ==============================================================================
# CELL 2: A1 - TF-IDF MODEL TRAINING & EVALUATION (ABSTRACTS)
# ==============================================================================
print("\n\n--- SECTION A1: TF-IDF on Training Abstracts ---")
TOP_K_EVAL = 5 # Define TOP_K_EVAL at the beginning of the section
all_ids_A1, all_cleaned_texts_A1, queries_A1, qrels_A1 = [], [], OrderedDict(), defaultdict(set)
data_loaded_A1 = False

if os.path.exists(loaded_data_cache_A1_path):
    print(f"Attempting to load cached A1 data from: {loaded_data_cache_A1_path}")
    try:
        with open(loaded_data_cache_A1_path, 'rb') as f: cached_data = pickle.load(f)
        all_ids_A1 = cached_data.get('all_ids_A1', [])
        all_cleaned_texts_A1 = cached_data.get('all_cleaned_texts_A1', [])
        queries_A1 = cached_data.get('queries_A1', OrderedDict())
        qrels_A1 = cached_data.get('qrels_A1', defaultdict(set))
        if all_ids_A1 and all_cleaned_texts_A1 and queries_A1 and (len(all_ids_A1) == len(all_cleaned_texts_A1)): # Basic content check
            print(f"Successfully loaded {len(all_ids_A1)} cleaned docs, {len(queries_A1)} queries for A1 from cache."); data_loaded_A1 = True
        else: print("A1 cached data incomplete/invalid. Reloading."); all_ids_A1,all_cleaned_texts_A1,queries_A1,qrels_A1 = [],[],OrderedDict(),defaultdict(set) # Reset
    except Exception as e: print(f"Error loading A1 cache: {e}. Reloading."); all_ids_A1,all_cleaned_texts_A1,queries_A1,qrels_A1 = [],[],OrderedDict(),defaultdict(set) # Reset

if not data_loaded_A1:
    if os.path.exists(ABSTRACT_TRAIN_ZIP_PATH):
        print(f"Loading & Cleaning A1 Data from: {ABSTRACT_TRAIN_ZIP_PATH}")
        with zipfile.ZipFile(ABSTRACT_TRAIN_ZIP_PATH, 'r') as z:
            entries = z.namelist()
            nested_root_A1 = ""
            if entries : nested_root_A1 = entries[0].split('/')[0] if '/' in entries[0] else ""

            jsonl_prefix_A1 = os.path.join(nested_root_A1, 'documents/') if nested_root_A1 else 'documents/'
            q_path_A1 = os.path.join(nested_root_A1, 'queries.txt') if nested_root_A1 else 'queries.txt'
            r_path_A1 = os.path.join(nested_root_A1, 'qrels.txt') if nested_root_A1 else 'qrels.txt'

            jsonl_files_A1 = [info for info in z.infolist() if info.filename.startswith(jsonl_prefix_A1) and info.filename.endswith(".jsonl") and not info.is_dir()]
            print(f"  Found {len(jsonl_files_A1)} JSONL files in '{jsonl_prefix_A1}' for A1.")
            for info in tqdm(jsonl_files_A1, desc="Reading A1 JSONLs", unit="file"):
                with z.open(info) as f:
                    for line_bytes in f:
                        try:
                            rec = json.loads(line_bytes.decode('utf-8'))
                            doc_id = rec.get("id"); raw_text = extract_text_from_abstract_json(rec)
                            if doc_id and raw_text: all_ids_A1.append(str(doc_id)); all_cleaned_texts_A1.append(clean_text_for_tfidf(raw_text))
                        except Exception: pass # Optionally log line processing errors
            print(f"  Total A1 docs processed: {len(all_cleaned_texts_A1)}")
            if q_path_A1 in z.namelist():
                 with z.open(q_path_A1) as f_q: queries_A1.update(OrderedDict(l.decode('utf-8',errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8',errors='ignore').strip()))
                 print(f"  → {len(queries_A1)} A1 queries loaded.")
            else: print(f"  ERROR: A1 queries.txt not found at '{q_path_A1}'")
            if r_path_A1 in z.namelist():
                with z.open(r_path_A1) as f_r:
                    for l in f_r:
                        parts=l.decode('utf-8',errors='ignore').strip().split()
                        if len(parts) >= 3 and parts[-1].isdigit() and int(parts[-1])>0 : qrels_A1[str(parts[0])].add(str(parts[2]))
                print(f"  → {len(qrels_A1)} A1 queries with qrels.")
            else: print(f"  ERROR: A1 qrels.txt not found at '{r_path_A1}'")

            if all_cleaned_texts_A1 and queries_A1: # Qrels might be empty for some datasets, still save others
                data_to_cache_A1 = {'all_ids_A1':all_ids_A1, 'all_cleaned_texts_A1':all_cleaned_texts_A1, 'queries_A1':queries_A1, 'qrels_A1':qrels_A1}
                with open(loaded_data_cache_A1_path,'wb') as f_co: pickle.dump(data_to_cache_A1,f_co, protocol=pickle.HIGHEST_PROTOCOL)
                print("A1 data (cleaned texts, queries, qrels) saved to cache."); data_loaded_A1 = True
            else: print("Warning: A1 data loading incomplete, not caching.")
    else: print(f"A1 Data ZIP not found at {ABSTRACT_TRAIN_ZIP_PATH}. Cannot load A1 data.")

tfidf_results_A1, tfidf_predictions_A1 = {}, OrderedDict()
tfidf_vectorizer_A1, doc_term_matrix_A1 = None, None
a1_model_evaluated = False

if data_loaded_A1:
    if os.path.exists(tfidf_results_A1_path) and os.path.exists(tfidf_predictions_A1_path) and \
       os.path.exists(tfidf_vectorizer_A1_path) and os.path.exists(tfidf_matrix_A1_path):
        try:
            print("Attempting to load all cached A1 TF-IDF components & results...")
            with open(tfidf_results_A1_path, 'rb') as f: tfidf_results_A1 = pickle.load(f)
            with open(tfidf_predictions_A1_path, 'rb') as f: tfidf_predictions_A1 = pickle.load(f)
            with open(tfidf_vectorizer_A1_path, 'rb') as f: tfidf_vectorizer_A1 = pickle.load(f)
            with open(tfidf_matrix_A1_path, 'rb') as f: doc_term_matrix_A1 = pickle.load(f)
            if tfidf_results_A1 and tfidf_predictions_A1 and tfidf_vectorizer_A1 is not None and doc_term_matrix_A1 is not None:
                 print("Loaded all TF-IDF components & results for A1 from cache.")
                 a1_model_evaluated = True
            else: print("A1 TF-IDF cached data incomplete. Regenerating."); tfidf_results_A1, tfidf_predictions_A1, tfidf_vectorizer_A1, doc_term_matrix_A1 = {}, OrderedDict(), None, None
        except Exception as e_load_a1_all: print(f"Error loading cached A1 TF-IDF components: {e_load_a1_all}. Regenerating."); tfidf_results_A1, tfidf_predictions_A1, tfidf_vectorizer_A1, doc_term_matrix_A1 = {}, OrderedDict(), None, None

    if not a1_model_evaluated: # If results or model parts were not fully loaded
        if tfidf_vectorizer_A1 is None or doc_term_matrix_A1 is None : # If model parts need to be (re)computed
            print("Training TF-IDF Vectorizer for A1 (Abstracts)...")
            tfidf_vectorizer_A1 = TfidfVectorizer(input='content', analyzer='word', token_pattern=r"(?u)\b\w\w+\b", lowercase=True, max_df=0.9, min_df=5, norm='l2')
            if not all_cleaned_texts_A1: print("ERROR: A1 cleaned texts missing for TF-IDF training.")
            else:
                doc_term_matrix_A1 = tfidf_vectorizer_A1.fit_transform(all_cleaned_texts_A1)
                print(f"A1 TF-IDF matrix created. Shape: {doc_term_matrix_A1.shape}")
                try:
                    with open(tfidf_vectorizer_A1_path, 'wb') as f: pickle.dump(tfidf_vectorizer_A1, f, protocol=pickle.HIGHEST_PROTOCOL)
                    with open(tfidf_matrix_A1_path, 'wb') as f: pickle.dump(doc_term_matrix_A1, f, protocol=pickle.HIGHEST_PROTOCOL)
                    print("A1 TF-IDF vectorizer and matrix saved.")
                except Exception as e_save_vm_A1: print(f"Error saving A1 TF-IDF vectorizer/matrix: {e_save_vm_A1}")
        else: print("A1 TF-IDF vectorizer and matrix were already loaded.")

        if tfidf_vectorizer_A1 and doc_term_matrix_A1 is not None: # Ensure model is ready before eval
            print(f"Evaluating A1 TF-IDF Model (Top {TOP_K_EVAL})...")
            temp_p_A1, temp_r_A1, temp_ap_A1 = [], [], []
            for qid, q_txt in tqdm(queries_A1.items(), desc="A1 TF-IDF Retrieval (Abstracts)"):
                cleaned_q_txt = clean_text_for_tfidf(q_txt)
                if not cleaned_q_txt.strip(): # Handle empty queries after cleaning
                    preds = []
                else:
                    query_vec = tfidf_vectorizer_A1.transform([cleaned_q_txt])
                    scores = cosine_similarity(query_vec, doc_term_matrix_A1)[0]
                    top_idx = np.argsort(scores)[::-1][:TOP_K_EVAL]; preds = [all_ids_A1[i] for i in top_idx]
                tfidf_predictions_A1[qid] = preds
                gold,n_gold=qrels_A1.get(qid,set()),len(qrels_A1.get(qid,set()))
                hits=[1 if d in gold else 0 for d in preds]; n_hits=sum(hits)
                temp_p_A1.append(n_hits/TOP_K_EVAL if TOP_K_EVAL > 0 else 0.0)
                temp_r_A1.append(n_hits/n_gold if n_gold > 0 else (0.0 if n_hits == 0 else 1.0)) # Avoid division by zero if n_gold is 0
                if n_gold > 0:
                    ap_q,h_q=0.0,0; [(h_q:=h_q+1, ap_q:=ap_q+h_q/(i+1)) for i,h_val in enumerate(hits) if h_val] # Python 3.8+ walrus operator
                    den_ap=min(n_gold,TOP_K_EVAL); temp_ap_A1.append(ap_q/den_ap if den_ap>0 else 0.0)
                else: temp_ap_A1.append(0.0)
            tfidf_results_A1={'P@5':np.mean(temp_p_A1) if temp_p_A1 else 0.0,'R@5':np.mean(temp_r_A1) if temp_r_A1 else 0.0,'MAP@5':np.mean(temp_ap_A1) if temp_ap_A1 else 0.0}
            with open(tfidf_results_A1_path, 'wb') as f: pickle.dump(tfidf_results_A1, f, protocol=pickle.HIGHEST_PROTOCOL)
            with open(tfidf_predictions_A1_path, 'wb') as f: pickle.dump(tfidf_predictions_A1, f, protocol=pickle.HIGHEST_PROTOCOL)
            print("A1 TF-IDF results and predictions saved.")
            a1_model_evaluated = True
        else: print("Could not evaluate A1 TF-IDF: model or matrix not available.")

    if tfidf_results_A1: print(f"A1 TF-IDF Results (Top {TOP_K_EVAL}): {tfidf_results_A1}")
    if tfidf_predictions_A1: [print(f"A1 Sample Query {list(tfidf_predictions_A1.keys())[i]}: {list(tfidf_predictions_A1.values())[i][:3]}") for i in range(min(3, len(tfidf_predictions_A1)))]
else: print("Skipping A1 TF-IDF: data loading failed.")
gc.collect()

# ==============================================================================
# CELL 3: A2 - TF-IDF MODEL TRAINING & EVALUATION (FULL-TEXT)
# ==============================================================================
print("\n\n--- SECTION A2: TF-IDF on Training Full-Text ---")
TOP_K_EVAL = 5 # Define for A2 section as well
all_ids_A2, all_cleaned_texts_A2, queries_A2, qrels_A2 = [], [], OrderedDict(), defaultdict(set)
data_loaded_A2 = False

if os.path.exists(loaded_data_cache_A2_path):
    print(f"Attempting to load cached A2 data from: {loaded_data_cache_A2_path}")
    try:
        with open(loaded_data_cache_A2_path, 'rb') as f: cached_data = pickle.load(f)
        all_ids_A2 = cached_data.get('all_ids_A2', [])
        all_cleaned_texts_A2 = cached_data.get('all_cleaned_texts_A2', [])
        queries_A2 = cached_data.get('queries_A2', OrderedDict())
        qrels_A2 = cached_data.get('qrels_A2', defaultdict(set))
        if all_ids_A2 and all_cleaned_texts_A2 and queries_A2 and (len(all_ids_A2) == len(all_cleaned_texts_A2)):
            print(f"Successfully loaded {len(all_ids_A2)} cleaned docs for A2 from cache."); data_loaded_A2 = True
        else: print("A2 cached data incomplete. Reloading."); all_ids_A2,all_cleaned_texts_A2,queries_A2,qrels_A2 = [],[],OrderedDict(),defaultdict(set)
    except Exception as e: print(f"Error loading A2 cache: {e}. Reloading."); all_ids_A2,all_cleaned_texts_A2,queries_A2,qrels_A2 = [],[],OrderedDict(),defaultdict(set)

if not data_loaded_A2:
    if os.path.exists(FULLTEXT_TRAIN_ZIP_PATH):
        print(f"Loading & Cleaning A2 Data from: {FULLTEXT_TRAIN_ZIP_PATH}")
        with zipfile.ZipFile(FULLTEXT_TRAIN_ZIP_PATH, 'r') as z:
            entries = z.namelist()
            nested_root_A2 = entries[0].split('/')[0] if entries and '/' in entries[0] else ""
            jsonl_prefix_A2 = os.path.join(nested_root_A2, 'documents/') if nested_root_A2 else 'documents/'
            q_path_A2 = os.path.join(nested_root_A2, 'queries.txt') if nested_root_A2 else 'queries.txt'
            r_path_A2 = os.path.join(nested_root_A2, 'qrels.txt') if nested_root_A2 else 'qrels.txt'

            jsonl_files_A2 = [info for info in z.infolist() if info.filename.startswith(jsonl_prefix_A2) and info.filename.endswith(".jsonl") and not info.is_dir()]
            print(f"  Found {len(jsonl_files_A2)} JSONL files in '{jsonl_prefix_A2}' for A2.")
            for info in tqdm(jsonl_files_A2, desc="Reading A2 JSONLs", unit="file"):
                with z.open(info) as f:
                    for line_bytes in f:
                        try:
                            rec=json.loads(line_bytes.decode('utf-8'))
                            doc_id=rec.get("id"); raw_text=extract_and_truncate_full_text_content_for_tfidf(rec)
                            if doc_id and raw_text: all_ids_A2.append(str(doc_id)); all_cleaned_texts_A2.append(clean_text_for_tfidf(raw_text))
                        except Exception: pass
            print(f"  Total A2 docs processed: {len(all_cleaned_texts_A2)}")
            if q_path_A2 in z.namelist():
                 with z.open(q_path_A2) as f_q: queries_A2.update(OrderedDict(l.decode('utf-8',errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8',errors='ignore').strip()))
                 print(f"  → {len(queries_A2)} A2 queries loaded.")
            else: print(f"  ERROR: A2 queries.txt not found at '{q_path_A2}'")
            if r_path_A2 in z.namelist():
                with z.open(r_path_A2) as f_r:
                    for l in f_r: parts=l.decode('utf-8',errors='ignore').strip().split(); qrels_A2[str(parts[0])].add(str(parts[2])) if len(parts)>=3 and parts[-1].isdigit() and int(parts[-1])>0 else None
                print(f"  → {len(qrels_A2)} A2 queries with qrels.")
            else: print(f"  ERROR: A2 qrels.txt not found at '{r_path_A2}'")

            if all_cleaned_texts_A2 and queries_A2:
                with open(loaded_data_cache_A2_path,'wb') as f_co: pickle.dump({'all_ids_A2':all_ids_A2, 'all_cleaned_texts_A2':all_cleaned_texts_A2, 'queries_A2':queries_A2, 'qrels_A2':qrels_A2},f_co, protocol=pickle.HIGHEST_PROTOCOL)
                print("A2 data saved to cache."); data_loaded_A2 = True
            else: print("Warning: A2 data loading incomplete, not caching.")
    else: print(f"A2 Data ZIP not found at {FULLTEXT_TRAIN_ZIP_PATH}. Cannot load A2 data.")

tfidf_results_A2, tfidf_predictions_A2 = {}, OrderedDict()
tfidf_vectorizer_A2, doc_term_matrix_A2 = None, None
a2_model_evaluated = False

if data_loaded_A2:
    if os.path.exists(tfidf_results_A2_path) and os.path.exists(tfidf_predictions_A2_path) and \
       os.path.exists(tfidf_vectorizer_A2_path) and os.path.exists(tfidf_matrix_A2_path):
        try:
            print("Attempting to load all cached A2 TF-IDF components & results...")
            with open(tfidf_results_A2_path, 'rb') as f: tfidf_results_A2 = pickle.load(f)
            with open(tfidf_predictions_A2_path, 'rb') as f: tfidf_predictions_A2 = pickle.load(f)
            with open(tfidf_vectorizer_A2_path, 'rb') as f: tfidf_vectorizer_A2 = pickle.load(f)
            with open(tfidf_matrix_A2_path, 'rb') as f: doc_term_matrix_A2 = pickle.load(f)
            if tfidf_results_A2 and tfidf_predictions_A2 and tfidf_vectorizer_A2 is not None and doc_term_matrix_A2 is not None:
                print("Loaded all TF-IDF components & results for A2 from cache.")
                a2_model_evaluated = True
            else: print("A2 TF-IDF cached data incomplete. Regenerating."); tfidf_results_A2, tfidf_predictions_A2, tfidf_vectorizer_A2, doc_term_matrix_A2 = {}, OrderedDict(), None, None
        except Exception as e_load_a2_all: print(f"Error loading cached A2 TF-IDF components: {e_load_a2_all}. Regenerating."); tfidf_results_A2, tfidf_predictions_A2, tfidf_vectorizer_A2, doc_term_matrix_A2 = {}, OrderedDict(), None, None

    if not a2_model_evaluated:
        if tfidf_vectorizer_A2 is None or doc_term_matrix_A2 is None:
            print("Training TF-IDF Vectorizer for A2 (Full-Text)...")
            tfidf_vectorizer_A2 = TfidfVectorizer(input='content', analyzer='word', token_pattern=r"(?u)\b\w\w+\b", lowercase=True, max_df=0.9, min_df=5, norm='l2')
            if not all_cleaned_texts_A2: print("ERROR: A2 cleaned texts missing for TF-IDF training.")
            else:
                doc_term_matrix_A2 = tfidf_vectorizer_A2.fit_transform(all_cleaned_texts_A2)
                print(f"A2 TF-IDF matrix created. Shape: {doc_term_matrix_A2.shape}")
                try:
                    with open(tfidf_vectorizer_A2_path, 'wb') as f: pickle.dump(tfidf_vectorizer_A2, f, protocol=pickle.HIGHEST_PROTOCOL)
                    with open(tfidf_matrix_A2_path, 'wb') as f: pickle.dump(doc_term_matrix_A2, f, protocol=pickle.HIGHEST_PROTOCOL)
                    print("A2 TF-IDF vectorizer and matrix saved.")
                except Exception as e_save_vm_A2: print(f"Error saving A2 TF-IDF vectorizer/matrix: {e_save_vm_A2}")
        else: print("A2 TF-IDF vectorizer and matrix were already loaded.")

        if tfidf_vectorizer_A2 and doc_term_matrix_A2 is not None:
            print(f"Evaluating A2 TF-IDF Model (Top {TOP_K_EVAL})...")
            temp_p_A2, temp_r_A2, temp_ap_A2 = [], [], []
            for qid, q_txt in tqdm(queries_A2.items(), desc="A2 TF-IDF Retrieval (Full-Text)"):
                cleaned_q_txt = clean_text_for_tfidf(q_txt)
                if not cleaned_q_txt.strip(): preds = []
                else:
                    query_vec = tfidf_vectorizer_A2.transform([cleaned_q_txt])
                    scores = cosine_similarity(query_vec, doc_term_matrix_A2)[0]
                    top_idx = np.argsort(scores)[::-1][:TOP_K_EVAL]; preds = [all_ids_A2[i] for i in top_idx]
                tfidf_predictions_A2[qid] = preds
                gold,n_gold=qrels_A2.get(qid,set()),len(qrels_A2.get(qid,set()))
                hits=[1 if d in gold else 0 for d in preds]; n_hits=sum(hits)
                temp_p_A2.append(n_hits/TOP_K_EVAL if TOP_K_EVAL > 0 else 0.0)
                temp_r_A2.append(n_hits/n_gold if n_gold > 0 else (0.0 if n_hits == 0 else 1.0))
                if n_gold > 0:
                    ap_q,h_q=0.0,0; [(h_q:=h_q+1, ap_q:=ap_q+h_q/(i+1)) for i,h_val in enumerate(hits) if h_val]
                    den_ap=min(n_gold,TOP_K_EVAL); temp_ap_A2.append(ap_q/den_ap if den_ap > 0 else 0.0)
                else: temp_ap_A2.append(0.0)
            tfidf_results_A2={'P@5':np.mean(temp_p_A2) if temp_p_A2 else 0.0,'R@5':np.mean(temp_r_A2) if temp_r_A2 else 0.0,'MAP@5':np.mean(temp_ap_A2) if temp_ap_A2 else 0.0}
            with open(tfidf_results_A2_path, 'wb') as f: pickle.dump(tfidf_results_A2, f, protocol=pickle.HIGHEST_PROTOCOL)
            with open(tfidf_predictions_A2_path, 'wb') as f: pickle.dump(tfidf_predictions_A2, f, protocol=pickle.HIGHEST_PROTOCOL)
            print("A2 TF-IDF results and predictions saved.")
            a2_model_evaluated = True
        else: print("Could not evaluate A2 TF-IDF: model or matrix not available.")

    if tfidf_results_A2: print(f"A2 TF-IDF Results (Top {TOP_K_EVAL}, Truncated ~{MAX_WORDS_FOR_TFIDF_FULLTEXT} words): {tfidf_results_A2}")
    if tfidf_predictions_A2: [print(f"A2 Sample Query {list(tfidf_predictions_A2.keys())[i]}: {list(tfidf_predictions_A2.values())[i][:3]}") for i in range(min(3, len(tfidf_predictions_A2)))]
else: print("Skipping A2 TF-IDF: data loading failed.")
gc.collect()

# ==============================================================================
# CELL 4: HELPER FUNCTIONS FOR TEST DATA LOADING (A3 & A4)
# ==============================================================================
print("\n\n--- SECTION Helper Functions for Test Data (A3 & A4) ---")
def load_test_queries_from_zip_corrected(zip_path, query_file_name_in_zip):
    queries_dict = OrderedDict()
    if not os.path.exists(zip_path): print(f"ERROR: Query ZIP {zip_path} not found"); return queries_dict
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            actual_query_path = next((name for name in z.namelist() if name.endswith(query_file_name_in_zip)), None)
            if not actual_query_path:
                 potential_root = z.namelist()[0].split('/')[0] if z.namelist() and '/' in z.namelist()[0] else ""
                 if potential_root: actual_query_path = next((name for name in z.namelist() if name == f"{potential_root}/{query_file_name_in_zip}" or name == query_file_name_in_zip), None) # Also check root of zip
                 if not actual_query_path and query_file_name_in_zip in z.namelist(): # Check root as a last resort
                     actual_query_path = query_file_name_in_zip

            if actual_query_path and actual_query_path in z.namelist():
                print(f"  Found query file: '{actual_query_path}' in {os.path.basename(zip_path)}")
                with z.open(actual_query_path) as f_q:
                    queries_dict.update(OrderedDict(l.decode('utf-8', errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8', errors='ignore').strip()))
                print(f"    → Loaded {len(queries_dict)} queries from {query_file_name_in_zip}.")
            else:
                print(f"  ERROR: '{query_file_name_in_zip}' not found in {os.path.basename(zip_path)} at common paths. Available entries: {z.namelist()[:20]}") # Show some available files
    except Exception as e: print(f"  Error loading queries from {query_file_name_in_zip} in {zip_path}: {e}")
    return queries_dict

def load_and_clean_test_documents_corrected(test_zip_path, text_extraction_func, desc_prefix=""):
    all_test_ids, all_cleaned_test_texts = [], []
    if not os.path.exists(test_zip_path): print(f"ERROR: Test doc ZIP {test_zip_path} not found"); return [], []
    print(f"--- Loading & Cleaning {desc_prefix} TEST Docs from: {os.path.basename(test_zip_path)} ---")
    try:
        with zipfile.ZipFile(test_zip_path, 'r') as z:
            entries = z.namelist()
            nested_root_test = entries[0].split('/')[0] if entries and '/' in entries[0] else ""
            jsonl_prefix_test = os.path.join(nested_root_test, 'documents/') if nested_root_test else 'documents/'
            jsonl_files_test = [info for info in z.infolist() if info.filename.startswith(jsonl_prefix_test) and info.filename.endswith(".jsonl") and not info.is_dir()]
            if not jsonl_files_test and nested_root_test: # Fallback for structure like "zip_root/documents_000001.jsonl"
                 jsonl_prefix_test = nested_root_test + "/" # if documents folder is the root archive name itself
                 jsonl_files_test = [info for info in z.infolist() if info.filename.startswith(jsonl_prefix_test) and info.filename.endswith(".jsonl") and not info.is_dir()]
            if not jsonl_files_test: # Final fallback if documents are at absolute root of zip
                 jsonl_prefix_test = 'documents/' # Check for documents/ at root
                 jsonl_files_test = [info for info in z.infolist() if info.filename.startswith(jsonl_prefix_test) and info.filename.endswith(".jsonl") and not info.is_dir()]
                 if not jsonl_files_test: # check if files are directly at root (no 'documents/' folder)
                    jsonl_files_test = [info for info in z.infolist() if info.filename.endswith(".jsonl") and not info.is_dir() and '/' not in info.filename]
                    if jsonl_files_test: print(f"  Note: Found JSONL files directly at ZIP root (no 'documents/' folder).")


            print(f"  Looking for JSONLs starting with '{jsonl_prefix_test}'. Found {len(jsonl_files_test)} files.")
            for info in tqdm(jsonl_files_test, desc=f"Reading {desc_prefix} Test JSONLs", unit="file"):
                with z.open(info) as f:
                    for line_bytes in f:
                        try:
                            rec=json.loads(line_bytes.decode('utf-8'))
                            doc_id=rec.get("id"); raw_text=text_extraction_func(rec)
                            if doc_id and raw_text: all_test_ids.append(str(doc_id)); all_cleaned_test_texts.append(clean_text_for_tfidf(raw_text))
                        except Exception: pass
            print(f"  Total {desc_prefix} TEST docs processed: {len(all_cleaned_test_texts)}")
    except Exception as e: print(f"  Error loading/cleaning {desc_prefix} TEST documents: {e}")
    return all_test_ids, all_cleaned_test_texts

TEST_QUERY_FILES = ["queries_2024-11_test.txt", "queries_2025-01_test.txt"]
TOP_K_TEST_PREDICTIONS = 100
print("Test data loading helper functions defined.")

# ==============================================================================
# CELL 5: A3 - TF-IDF TESTING (ABSTRACTS)
# ==============================================================================
print("\n\n--- SECTION A3: TF-IDF Testing (Abstracts Test Data using A1 Model) ---")
if tfidf_vectorizer_A1 is None :
    if os.path.exists(tfidf_vectorizer_A1_path):
        print("A1 vectorizer not in memory. Loading for A3...")
        try:
            with open(tfidf_vectorizer_A1_path, 'rb') as f: tfidf_vectorizer_A1 = pickle.load(f)
            print("A1 TF-IDF vectorizer loaded successfully for A3.")
        except Exception as e: print(f"ERROR: Could not load A1 TF-IDF vectorizer for A3: {e}")
    else: print(f"ERROR: A1 TF-IDF vectorizer file not found at {tfidf_vectorizer_A1_path}. Cannot perform A3.")

if tfidf_vectorizer_A1:
    ids_A3_test, cleaned_texts_A3_test = load_and_clean_test_documents_corrected(
        ABSTRACT_TEST_ZIP_PATH, extract_text_from_abstract_json, desc_prefix="A3 Abstract"
    )
    if ids_A3_test and cleaned_texts_A3_test:
        print("Transforming A3 test documents using A1 vectorizer...")
        try:
            test_doc_matrix_A3 = tfidf_vectorizer_A1.transform(cleaned_texts_A3_test)
            print(f"A3 test doc matrix created. Shape: {test_doc_matrix_A3.shape}")
            for query_file_name in TEST_QUERY_FILES:
                tfidf_predictions_A3_path_specific = os.path.join(PIPELINE_A3_TEST_CACHE_DIR, f'tfidf_predictions_A3_{os.path.splitext(query_file_name)[0]}.pkl')
                if os.path.exists(tfidf_predictions_A3_path_specific):
                    try:
                        with open(tfidf_predictions_A3_path_specific, 'rb') as f: temp_preds = pickle.load(f)
                        print(f"Loaded A3 predictions for {query_file_name} from cache ({len(temp_preds)} queries).")
                        if temp_preds: [print(f"A3 Sample Query ({query_file_name}) {list(temp_preds.keys())[i]}: {list(temp_preds.values())[i][:3]}") for i in range(min(2, len(temp_preds)))]
                        continue
                    except Exception as e_load_pred_a3: print(f"Error loading A3 predictions for {query_file_name}: {e_load_pred_a3}. Regenerating.")

                queries_A3_test_set = load_test_queries_from_zip_corrected(ABSTRACT_TEST_ZIP_PATH, query_file_name)
                if queries_A3_test_set:
                    current_tfidf_predictions_A3 = OrderedDict()
                    for qid, q_txt in tqdm(queries_A3_test_set.items(), desc=f"A3 TF-IDF Testing ({query_file_name})"):
                        cleaned_q_txt_test = clean_text_for_tfidf(q_txt)
                        if not cleaned_q_txt_test.strip(): preds_test = []
                        else:
                            q_vec = tfidf_vectorizer_A1.transform([cleaned_q_txt_test])
                            scores = cosine_similarity(q_vec, test_doc_matrix_A3)[0]
                            top_idx = np.argsort(scores)[::-1][:TOP_K_TEST_PREDICTIONS]
                            preds_test = [ids_A3_test[i] for i in top_idx]
                        current_tfidf_predictions_A3[qid] = preds_test
                    with open(tfidf_predictions_A3_path_specific, 'wb') as f: pickle.dump(current_tfidf_predictions_A3, f, protocol=pickle.HIGHEST_PROTOCOL)
                    print(f"A3 TF-IDF test predictions for {query_file_name} saved.")
                    if current_tfidf_predictions_A3: [print(f"A3 Sample Query ({query_file_name}) {list(current_tfidf_predictions_A3.keys())[i]}: {list(current_tfidf_predictions_A3.values())[i][:3]}") for i in range(min(2, len(current_tfidf_predictions_A3)))]
                else: print(f"No test queries loaded for {query_file_name} in A3. Skipping retrieval for this set.")
        except AttributeError as e_attr: print(f"Error during A3 testing (likely vectorizer not fitted): {e_attr}")
        except Exception as e: print(f"Generic error during A3 testing: {e}")
    else: print("A3 testing skipped: No test documents loaded or processed for A3.")
else: print("A3 testing skipped: A1 TF-IDF vectorizer was not loaded or trained successfully.")
gc.collect()

# ==============================================================================
# CELL 6: A4 - TF-IDF TESTING (FULL-TEXT)
# ==============================================================================
print("\n\n--- SECTION A4: TF-IDF Testing (Full-Text Test Data using A2 Model) ---")
if tfidf_vectorizer_A2 is None:
    if os.path.exists(tfidf_vectorizer_A2_path):
        print("A2 vectorizer not in memory. Loading for A4...")
        try:
            with open(tfidf_vectorizer_A2_path, 'rb') as f: tfidf_vectorizer_A2 = pickle.load(f)
            print("A2 TF-IDF vectorizer loaded successfully for A4.")
        except Exception as e: print(f"ERROR: Could not load A2 TF-IDF vectorizer for A4: {e}")
    else: print(f"ERROR: A2 TF-IDF vectorizer file not found at {tfidf_vectorizer_A2_path}. Cannot perform A4.")

if tfidf_vectorizer_A2:
    ids_A4_test, cleaned_texts_A4_test = load_and_clean_test_documents_corrected(
        FULLTEXT_TEST_ZIP_PATH, extract_and_truncate_full_text_content_for_tfidf, desc_prefix="A4 Full-Text"
    )
    if ids_A4_test and cleaned_texts_A4_test:
        print("Transforming A4 test documents using A2 vectorizer...")
        try:
            test_doc_matrix_A4 = tfidf_vectorizer_A2.transform(cleaned_texts_A4_test)
            print(f"A4 test doc matrix created. Shape: {test_doc_matrix_A4.shape}")
            for query_file_name in TEST_QUERY_FILES:
                tfidf_predictions_A4_path_specific = os.path.join(PIPELINE_A4_TEST_CACHE_DIR, f'tfidf_predictions_A4_{os.path.splitext(query_file_name)[0]}.pkl')
                if os.path.exists(tfidf_predictions_A4_path_specific):
                    try:
                        with open(tfidf_predictions_A4_path_specific, 'rb') as f: temp_preds = pickle.load(f)
                        print(f"Loaded A4 predictions for {query_file_name} from cache ({len(temp_preds)} queries).")
                        if temp_preds: [print(f"A4 Sample Query ({query_file_name}) {list(temp_preds.keys())[i]}: {list(temp_preds.values())[i][:3]}") for i in range(min(2, len(temp_preds)))]
                        continue
                    except Exception as e_load_pred_a4: print(f"Error loading A4 predictions for {query_file_name}: {e_load_pred_a4}. Regenerating.")

                queries_A4_test_set = load_test_queries_from_zip_corrected(FULLTEXT_TEST_ZIP_PATH, query_file_name)
                if queries_A4_test_set:
                    current_tfidf_predictions_A4 = OrderedDict()
                    for qid, q_txt in tqdm(queries_A4_test_set.items(), desc=f"A4 TF-IDF Testing ({query_file_name})"):
                        cleaned_q_txt_test_a4 = clean_text_for_tfidf(q_txt)
                        if not cleaned_q_txt_test_a4.strip(): preds_test_a4 = []
                        else:
                            q_vec = tfidf_vectorizer_A2.transform([cleaned_q_txt_test_a4])
                            scores = cosine_similarity(q_vec, test_doc_matrix_A4)[0]
                            top_idx = np.argsort(scores)[::-1][:TOP_K_TEST_PREDICTIONS]
                            preds_test_a4 = [ids_A4_test[i] for i in top_idx]
                        current_tfidf_predictions_A4[qid] = preds_test_a4
                    with open(tfidf_predictions_A4_path_specific, 'wb') as f: pickle.dump(current_tfidf_predictions_A4, f, protocol=pickle.HIGHEST_PROTOCOL)
                    print(f"A4 TF-IDF test predictions for {query_file_name} saved.")
                    if current_tfidf_predictions_A4: [print(f"A4 Sample Query ({query_file_name}) {list(current_tfidf_predictions_A4.keys())[i]}: {list(current_tfidf_predictions_A4.values())[i][:3]}") for i in range(min(2, len(current_tfidf_predictions_A4)))]
                else: print(f"No test queries loaded for {query_file_name} in A4. Skipping retrieval for this set.")
        except AttributeError as e_attr_a4: print(f"Error during A4 testing (likely vectorizer not fitted): {e_attr_a4}")
        except Exception as e_a4: print(f"Generic error during A4 testing: {e_a4}")
    else: print("A4 testing skipped: No test documents loaded or processed for A4.")
else: print("A4 testing skipped: A2 TF-IDF vectorizer was not loaded or trained successfully.")
gc.collect()

print("\n\n--- TF-IDF A1, A2, A3, A4 Processing Complete ---")

Starting installations (if this is the first run of the cell in a new session)...

Installing PyTorch, Torchvision, Torchaudio...
Looking in indexes: https://download.pytorch.org/whl/cu121
PyTorch ecosystem installation attempt finished.

Installing sentence-transformers and other libraries...
Other packages installation attempt finished.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CELL 0 (INSTALLATIONS) HAS COMPLETED BOTH INSTALLATION STEPS.
YOU **MUST** NOW MANUALLY RESTART THE COLAB RUNTIME FOR ALL CHANGES TO TAKE EFFECT.
GO TO THE MENU: Runtime -> Restart runtime...
AFTER RESTARTING, RUN THIS ENTIRE CELL AGAIN FROM THE TOP.
If you skip this restart, you will likely see import errors or version conflicts.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


--- SECTION 1: Initializing Setup ---
Mounted at /content/drive
Google Drive mounted successfully.
Path definitions complete.
NLTK resources seem available.
Text cle

Reading A2 JSONLs:   0%|          | 0/21 [00:00<?, ?file/s]

  Total A2 docs processed: 2014257
  → 393 A2 queries loaded.
  → 393 A2 queries with qrels.
A2 data saved to cache.
Attempting to load all cached A2 TF-IDF components & results...
Loaded all TF-IDF components & results for A2 from cache.
A2 TF-IDF Results (Top 5, Truncated ~400 words): {'P@5': np.float64(0.1282442748091603), 'R@5': np.float64(0.08019866954583944), 'MAP@5': np.float64(0.08781382527565734)}
A2 Sample Query ce5bfacf-8652-4bc1-a5b0-6144a917fb1c: ['114488858', '135918841', '138360709']
A2 Sample Query 71657c3b-4112-49d5-86cb-705325aeb06e: ['158245958', '143423861', '145303924']
A2 Sample Query f60d94de-b903-48da-b13f-a8334c70f949: ['5599422', '49614213', '48055161']


--- SECTION Helper Functions for Test Data (A3 & A4) ---
Test data loading helper functions defined.


--- SECTION A3: TF-IDF Testing (Abstracts Test Data using A1 Model) ---
--- Loading & Cleaning A3 Abstract TEST Docs from: longeval_sci_testing_2025_abstract.zip ---
  Looking for JSONLs starting with 'longe

Reading A3 Abstract Test JSONLs:   0%|          | 0/16 [00:00<?, ?file/s]

  Total A3 Abstract TEST docs processed: 1524039
Transforming A3 test documents using A1 vectorizer...
A3 test doc matrix created. Shape: (1524039, 489203)
Loaded A3 predictions for queries_2024-11_test.txt from cache (99 queries).
A3 Sample Query (queries_2024-11_test.txt) d585f080-4519-4952-8278-0d13fcd03fec: ['70339957', '117725589', '70339957']
A3 Sample Query (queries_2024-11_test.txt) 254ecdb5-45e5-45ce-9b3f-492d1e1c085e: ['42822119', '30394313', '63171506']
Loaded A3 predictions for queries_2025-01_test.txt from cache (492 queries).
A3 Sample Query (queries_2025-01_test.txt) 6bdc12c2-05cf-4dec-a005-8bd0b695b47f: ['8866276', '8866276', '118169450']
A3 Sample Query (queries_2025-01_test.txt) a3915e30-a219-4be5-9681-3f0e3f9a3449: ['65009040', '45718540', '152433080']


--- SECTION A4: TF-IDF Testing (Full-Text Test Data using A2 Model) ---
--- Loading & Cleaning A4 Full-Text TEST Docs from: longeval_sci_testing_2025_fulltext.zip ---
  Looking for JSONLs starting with 'longeval_sci_

Reading A4 Full-Text Test JSONLs:   0%|          | 0/16 [00:00<?, ?file/s]

  Total A4 Full-Text TEST docs processed: 1524039
Transforming A4 test documents using A2 vectorizer...
A4 test doc matrix created. Shape: (1524039, 463609)
Loaded A4 predictions for queries_2024-11_test.txt from cache (99 queries).
A4 Sample Query (queries_2024-11_test.txt) d585f080-4519-4952-8278-0d13fcd03fec: ['117725589', '70339957', '70339957']
A4 Sample Query (queries_2024-11_test.txt) 254ecdb5-45e5-45ce-9b3f-492d1e1c085e: ['42822119', '30394313', '63171506']
Loaded A4 predictions for queries_2025-01_test.txt from cache (492 queries).
A4 Sample Query (queries_2025-01_test.txt) 6bdc12c2-05cf-4dec-a005-8bd0b695b47f: ['118169450', '8866276', '8866276']
A4 Sample Query (queries_2025-01_test.txt) a3915e30-a219-4be5-9681-3f0e3f9a3449: ['65009040', '45718540', '152432939']


--- TF-IDF A1, A2, A3, A4 Processing Complete ---


In [ ]:
# ==============================================================================
# CELL 0: INSTALLATIONS FOR A5
# (Run this section, THEN STOP, MANUALLY RESTART RUNTIME, then run THE ENTIRE CELL AGAIN)
# ==============================================================================
print("Starting installations for A5 (if this is the first run of the cell in a new session)...\n")
# rank_bm25 for BM25, nltk for tokenization/stopwords, tqdm for progress
!pip install rank_bm25 nltk tqdm pandas scikit-learn -q

print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
print("A5 INSTALLATIONS SECTION COMPLETE (or requirements already satisfied).")
print("1. IF PIP INSTALLS RAN ABOVE (AND IT WAS THE FIRST TIME FOR THESE PACKAGES IN THIS SESSION):")
print("   YOU **MUST** HAVE MANUALLY RESTARTED THE COLAB RUNTIME BEFORE THIS RUN.")
print("   (Menu: Runtime -> Restart runtime...)")
print("2. AFTER RESTARTING, RUN THIS ENTIRE CELL AGAIN FROM THE TOP.")
print("3. **BEFORE THIS RUN (especially the second time after restart):**")
print("   Consider manually deleting the following file from the A5 cache directory")
print("   '/content/drive/MyDrive/colab-4/cache_A5_abstract_bm25_train/' (if it exists from a previous problematic run):")
print("   - 'ids_queries_qrels_A5_abstract.pkl' (to force correct qrels re-parsing).")
print("   - Optionally, delete 'tokenized_corpus_A5_abstract.pkl' and 'bm25_model_A5_abstract.pkl' for a full A5 re-run.")
print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")

# The script will continue from here AFTER the manual runtime restart.

# ==============================================================================
# SECTION 1: IMPORTS, DRIVE MOUNT, PATHS, NLTK, CLEANING/EXTRACTION UTILITIES
# ==============================================================================
print("\n--- SECTION 1: Initializing Setup for A5 ---")
from google.colab import drive
import os
import gc
from collections import OrderedDict, defaultdict, Counter
import zipfile
import json
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm.notebook import tqdm
import numpy as np
import pickle
from rank_bm25 import BM25Okapi # For BM25
import pandas as pd

try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Google Drive mounting error: {e}. Please ensure Drive is accessible and authorized.")

# --- Path Definitions for A5 ---
DRIVE_BASE_FOLDER = '/content/drive/MyDrive/colab-4/'
ABSTRACT_TRAIN_ZIP_FILENAME = 'longeval_sci_training_2025_abstract.zip'
ABSTRACT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, ABSTRACT_TRAIN_ZIP_FILENAME)

PIPELINE_A5_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A5_abstract_bm25_train/')
os.makedirs(PIPELINE_A5_CACHE_DIR, exist_ok=True)
print(f"A5 Cache Directory: {PIPELINE_A5_CACHE_DIR}")

if not os.path.exists(ABSTRACT_TRAIN_ZIP_PATH):
    print(f"CRITICAL WARNING: ABSTRACT_TRAIN_ZIP_PATH not found at: {ABSTRACT_TRAIN_ZIP_PATH}. A5 will fail.")

# NLTK Resources
nltk_resources_ready_A5 = False
try:
    stopwords.words('english'); WordNetLemmatizer().lemmatize('cats'); nltk.word_tokenize("test")
    print("NLTK resources (stopwords, wordnet, punkt) found. Utilities ready.")
    nltk_resources_ready_A5 = True
except LookupError:
    print("NLTK resources not found. Downloading stopwords, wordnet, punkt...")
    try:
        nltk.download('stopwords', quiet=True); nltk.download('wordnet', quiet=True); nltk.download('punkt', quiet=True)
        nltk_resources_ready_A5 = True
    except Exception as e_nltk_download: print(f"ERROR: Failed to download NLTK resources for A5: {e_nltk_download}")
except Exception as e_nltk_check:
    print(f"An error occurred while checking NLTK resources for A5: {e_nltk_check}. Attempting download.")
    try:
        nltk.download('stopwords', quiet=True); nltk.download('wordnet', quiet=True); nltk.download('punkt', quiet=True)
        nltk_resources_ready_A5 = True
    except Exception as e_nltk_download_fallback: print(f"ERROR: Failed to download NLTK resources for A5 (fallback): {e_nltk_download_fallback}")

if nltk_resources_ready_A5:
    STOP = set(stopwords.words("english")); LEM = WordNetLemmatizer()
    _token_pat = re.compile(r"[A-Za-z]{2,}")
    print("Text cleaning utilities for A5 initialized.")
else:
    print("ERROR: NLTK resources not properly initialized for A5. Text cleaning may fail.")
    STOP, LEM, _token_pat = set(), None, re.compile(r"[A-Za-z]{2,}")

def clean_text_for_bm25(text: str) -> list:
    if not isinstance(text, str): return []
    if not LEM or not nltk_resources_ready_A5: return text.lower().strip().split()
    try: toks = nltk.word_tokenize(text.lower())
    except Exception: toks = text.lower().split()
    toks = [t for t in toks if t.isalpha() and t not in STOP and len(t) > 1]
    return [LEM.lemmatize(t) for t in toks]

def extract_text_from_abstract_json(record_dict):
    title = record_dict.get("title", "") or ""
    abstract_content = record_dict.get("abstract", "") or ""
    return f"{title} {abstract_content}".strip()

print("Initial setup and helper functions for A5 defined.")

BM25_K1 = 1.5; BM25_B = 0.75; RM3_FB_DOCS = 5; RM3_FB_TERMS = 5
TOP_K_EVAL_A5 = 5; TOP_K_CANDIDATES_FOR_RERANKING_A5 = 100
print(f"A5 BM25/RM3 Parameters: K1={BM25_K1}, B={BM25_B}, FB_DOCS={RM3_FB_DOCS}, FB_TERMS={RM3_FB_TERMS}, EVAL_K={TOP_K_EVAL_A5}, PREDS_K={TOP_K_CANDIDATES_FOR_RERANKING_A5}")

# ==============================================================================
# SECTION 2: HELPER FUNCTION FOR LOADING & TOKENIZING TRAINING DATA FOR BM25 (A5)
# (Corrected Qrels Parsing)
# ==============================================================================
print("\n--- SECTION 2: Defining Training Data Loading Helper for BM25 (A5) ---")
def load_and_tokenize_training_data_for_bm25_A5(
    data_zip_path, text_extraction_func,
    tokenized_corpus_cache_path, ids_queries_qrels_cache_path,
    data_description_prefix="A5 Abstract"
):
    all_ids, tokenized_corpus, queries, qrels = [], [], OrderedDict(), defaultdict(set)
    data_fully_loaded, loaded_corpus_from_cache, loaded_meta_from_cache = False, False, False

    if os.path.exists(tokenized_corpus_cache_path):
        print(f"  Attempting to load cached tokenized corpus for {data_description_prefix} from: {tokenized_corpus_cache_path}")
        try:
            with open(tokenized_corpus_cache_path, 'rb') as f: tokenized_corpus = pickle.load(f)
            if isinstance(tokenized_corpus, list) and (not tokenized_corpus or isinstance(tokenized_corpus[0], list)):
                loaded_corpus_from_cache = True; print(f"    Successfully loaded tokenized corpus for {data_description_prefix} ({len(tokenized_corpus)} docs).")
            else: tokenized_corpus = []
        except Exception as e: print(f"    Error loading tokenized corpus cache for {data_description_prefix}: {e}"); tokenized_corpus = []
    if os.path.exists(ids_queries_qrels_cache_path):
        print(f"  Attempting to load cached IDs/queries/qrels for {data_description_prefix} from: {ids_queries_qrels_cache_path}")
        try:
            with open(ids_queries_qrels_cache_path, 'rb') as f: meta_data = pickle.load(f)
            all_ids = meta_data.get('all_ids', [])
            queries = meta_data.get('queries', OrderedDict())
            qrels_temp = meta_data.get('qrels', defaultdict(set))
            qrels = defaultdict(set, qrels_temp)
            if all_ids and queries:
                loaded_meta_from_cache = True; print(f"    Successfully loaded IDs ({len(all_ids)}), queries ({len(queries)}), qrels ({len(qrels)}) for {data_description_prefix}.")
            else: all_ids, queries, qrels = [], OrderedDict(), defaultdict(set)
        except Exception as e: print(f"    Error loading IDs/queries/qrels cache for {data_description_prefix}: {e}"); all_ids, queries, qrels = [], OrderedDict(), defaultdict(set)

    if loaded_corpus_from_cache and loaded_meta_from_cache and (len(all_ids) == len(tokenized_corpus)):
        data_fully_loaded = True; print(f"  {data_description_prefix} training data (corpus & meta) successfully loaded from cache.")
    else:
        print(f"  Cache miss or mismatch for {data_description_prefix}. Processing from ZIP.")
        all_ids, tokenized_corpus, queries, qrels = [], [], OrderedDict(), defaultdict(set)
        if not os.path.exists(data_zip_path): print(f"  ERROR: {data_description_prefix} Data ZIP not found: {data_zip_path}"); return [], [], OrderedDict(), defaultdict(set), False
        print(f"  --- Loading & Tokenizing {data_description_prefix} Training Data from: {os.path.basename(data_zip_path)} ---")
        temp_raw_texts_for_tokenization, temp_ids_for_tokenization = [], []
        try:
            with zipfile.ZipFile(data_zip_path, 'r') as z:
                entries = z.namelist()
                nested_root_zip = entries[0].split('/')[0] if entries and '/' in entries[0] else ""
                jsonl_prefix_zip = os.path.join(nested_root_zip, 'documents/') if nested_root_zip else 'documents/'
                q_path_zip = os.path.join(nested_root_zip, 'queries.txt') if nested_root_zip else 'queries.txt'
                r_path_zip = os.path.join(nested_root_zip, 'qrels.txt') if nested_root_zip else 'qrels.txt'
                print(f"    Using jsonl_prefix: '{jsonl_prefix_zip}', query_path: '{q_path_zip}', qrels_path: '{r_path_zip}'")

                jsonl_files_zip = [info for info in z.infolist() if info.filename.lower().startswith(jsonl_prefix_zip.lower()) and info.filename.lower().endswith(".jsonl") and not info.is_dir()]
                print(f"    Found {len(jsonl_files_zip)} JSONL files.")
                for info in tqdm(jsonl_files_zip, desc=f"Reading {data_description_prefix} JSONLs", unit="file"):
                    with z.open(info) as f_jsonl:
                        for line_bytes in f_jsonl:
                            try:
                                record = json.loads(line_bytes.decode('utf-8'))
                                doc_id_str = str(record.get("id"))
                                raw_text = text_extraction_func(record)
                                if doc_id_str and raw_text:
                                    temp_ids_for_tokenization.append(doc_id_str)
                                    temp_raw_texts_for_tokenization.append(raw_text)
                            except Exception: pass
                all_ids = temp_ids_for_tokenization

                if not loaded_corpus_from_cache or len(tokenized_corpus) != len(all_ids):
                    print(f"    Tokenizing {len(temp_raw_texts_for_tokenization)} {data_description_prefix} documents for BM25...")
                    tokenized_corpus = [clean_text_for_bm25(text) for text in tqdm(temp_raw_texts_for_tokenization, desc=f"Tokenizing {data_description_prefix}")]
                    if tokenized_corpus and (len(all_ids) == len(tokenized_corpus)):
                        with open(tokenized_corpus_cache_path, 'wb') as f_tc_cache: pickle.dump(tokenized_corpus, f_tc_cache, protocol=pickle.HIGHEST_PROTOCOL)
                        print(f"    Saved tokenized corpus ({len(tokenized_corpus)} docs) for {data_description_prefix} to cache.")
                    else: print(f"    Warning: Tokenized corpus empty or ID/corpus length mismatch for {data_description_prefix}. IDs: {len(all_ids)}, Corpus: {len(tokenized_corpus if tokenized_corpus else [])}")

                if not loaded_meta_from_cache:
                    if q_path_zip in z.namelist():
                        with z.open(q_path_zip) as f_q: queries.update(OrderedDict(l.decode('utf-8',errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8',errors='ignore').strip()))
                        print(f"    Loaded {len(queries)} queries for {data_description_prefix}.")
                    else: print(f"  ERROR: {data_description_prefix} queries.txt not found at '{q_path_zip}'")

                    if r_path_zip in z.namelist():
                        print(f"    Processing qrels file: {r_path_zip}")
                        lines_in_qrels, qrels_added = 0, 0
                        with z.open(r_path_zip) as f_r:
                            for i, l_bytes in enumerate(f_r):
                                lines_in_qrels +=1
                                line_str = l_bytes.decode('utf-8', errors='ignore').strip()
                                parts = line_str.split()
                                qid_r, did_r, rel_r_str = None, None, None # Initialize
                                if len(parts) == 4: # qid iter docid rel
                                    qid_r, _, did_r, rel_r_str = parts[0], parts[1], parts[2], parts[3]
                                elif len(parts) == 3: # qid docid rel
                                    qid_r, did_r, rel_r_str = parts[0], parts[1], parts[2]
                                else:
                                    if i < 10: print(f"      Skipping qrels line {i+1} (unexpected parts {len(parts)}): '{line_str}'")
                                    continue
                                try:
                                    if int(rel_r_str) > 0: qrels[str(qid_r)].add(str(did_r)); qrels_added+=1
                                except ValueError:
                                    if i < 10: print(f"      Skipping qrels line {i+1} (ValueError for rel '{rel_r_str}'): '{line_str}'")
                        print(f"    Finished qrels. Lines read: {lines_in_qrels}. Entries added: {qrels_added}. Unique QIDs in qrels: {len(qrels)}.")
                    else: print(f"    ERROR: Qrels file '{r_path_zip}' not found in ZIP.")

                    if all_ids and queries:
                        with open(ids_queries_qrels_cache_path, 'wb') as f_meta_cache: pickle.dump({'all_ids': all_ids, 'queries': queries, 'qrels': qrels}, f_meta_cache, protocol=pickle.HIGHEST_PROTOCOL)
                        print(f"    Saved IDs, queries, qrels for {data_description_prefix} to cache.")
                data_fully_loaded = bool(all_ids and tokenized_corpus and queries and (len(all_ids) == len(tokenized_corpus)))
        except Exception as e_load_zip: print(f"    Error during full load/tokenize for {data_description_prefix} data from ZIP: {e_load_zip}")
    if data_fully_loaded : print(f"  {data_description_prefix} training data successfully processed and/or loaded from cache.")
    else: print(f"  ERROR: {data_description_prefix} training data loading/tokenization failed or mismatch in counts.")
    return all_ids, tokenized_corpus, queries, qrels, data_fully_loaded

# ==============================================================================
# SECTION 3: HELPER FUNCTION FOR BM25+RM3 EVALUATION & PREDICTION (WITH ENHANCED DEBUG)
# ==============================================================================
print("\n--- SECTION 3: Defining BM25+RM3 Evaluation Helper (with ENHANCED DEBUG prints) ---")
def evaluate_bm25_rm3_A5(
    bm25_model, tokenized_corpus_docs, all_doc_ids, queries_dict, qrels_dict,
    desc_prefix="A5", fb_docs_param=5, fb_terms_param=5,
    top_k_eval_param=5, top_k_preds_to_return_param=100,
    QIDs_TO_DEBUG = None # Expects a list of QIDs or None
):
    results = {}; predictions = OrderedDict(); temp_p, temp_r, temp_ap = [], [], []
    if not all_doc_ids or not tokenized_corpus_docs or bm25_model is None:
        print(f"Error in evaluate_bm25_rm3_A5 ({desc_prefix}): Missing critical inputs.")
        return results, predictions

    print(f"Starting evaluation for {desc_prefix} with {len(queries_dict)} queries.")
    if not qrels_dict: print(f"WARNING ({desc_prefix}): qrels_dict is empty. Metrics will be zero.")

    for qid_idx, (qid, query_text_raw) in enumerate(tqdm(queries_dict.items(), desc=f"BM25+RM3 ({desc_prefix}) Evaluation")):
        IS_DEBUG_QID = (QIDs_TO_DEBUG is not None and qid in QIDs_TO_DEBUG)

        if IS_DEBUG_QID: print(f"\n\n==================== DEBUGGING QID: {qid} ====================")
        if IS_DEBUG_QID: print(f"  Raw Query: '{query_text_raw}'")

        tokenized_query = clean_text_for_bm25(query_text_raw)
        if IS_DEBUG_QID: print(f"  Tokenized Query ({len(tokenized_query)} terms): {tokenized_query[:20]}") # Show first 20 terms

        if not tokenized_query:
            if IS_DEBUG_QID: print("  Query is empty after cleaning. Skipping.")
            predictions[qid] = []
            if qrels_dict and qid in qrels_dict and qrels_dict[qid]: temp_p.append(0.0); temp_r.append(0.0); temp_ap.append(0.0)
            continue

        initial_scores = bm25_model.get_scores(tokenized_query)
        valid_initial_indices = np.argsort(initial_scores)[::-1]
        feedback_doc_indices = [idx for idx in valid_initial_indices if idx < len(tokenized_corpus_docs)][:fb_docs_param]

        if IS_DEBUG_QID:
            print(f"  Initial BM25 Scores (Top 5 values): {initial_scores[feedback_doc_indices] if len(feedback_doc_indices) > 0 else 'N/A'}")
            print(f"  Feedback Doc Indices (corpus indices): {feedback_doc_indices}")
            for i, fb_idx in enumerate(feedback_doc_indices):
                 print(f"    Fb Doc {i+1} (ID: {all_doc_ids[fb_idx]}, Score: {initial_scores[fb_idx]:.4f}): {' '.join(tokenized_corpus_docs[fb_idx][:15])}...")

        term_scores_expansion = Counter()
        if feedback_doc_indices:
            feedback_doc_actual_scores = initial_scores[feedback_doc_indices]
            scores_sum = np.sum(feedback_doc_actual_scores)
            doc_weights = feedback_doc_actual_scores / scores_sum if scores_sum > 1e-9 else np.ones(len(feedback_doc_actual_scores)) / (len(feedback_doc_actual_scores) if len(feedback_doc_actual_scores)>0 else 1.0)
            for i, doc_idx in enumerate(feedback_doc_indices):
                doc_tokens = tokenized_corpus_docs[doc_idx]; term_freq_in_doc = Counter(doc_tokens); doc_len = len(doc_tokens)
                if doc_len > 0:
                    current_doc_weight = doc_weights[i] if i < len(doc_weights) else (1.0/len(feedback_doc_indices) if len(feedback_doc_indices)>0 else 0)
                    for term, freq in term_freq_in_doc.items(): term_scores_expansion[term] += current_doc_weight * (freq / doc_len)

        if IS_DEBUG_QID: print(f"  Term Scores for Expansion (Top 10 before removing query terms): {term_scores_expansion.most_common(10)}")
        for t_original in tokenized_query: term_scores_expansion.pop(t_original, None)
        expansion_terms = [t for t, s_exp in term_scores_expansion.most_common(fb_terms_param)] # Renamed s to s_exp
        if IS_DEBUG_QID: print(f"  Expansion Terms (Top {fb_terms_param}): {expansion_terms}")

        final_query_tokens = tokenized_query + expansion_terms
        if not final_query_tokens and tokenized_query: final_query_tokens = tokenized_query # Fallback if expansion is empty
        if IS_DEBUG_QID: print(f"  Final Query Tokens for Re-Retrieval ({len(final_query_tokens)} terms): {final_query_tokens[:20]}")

        if not final_query_tokens: # If still empty, cannot proceed
            predictions[qid] = []
            if qrels_dict and qid in qrels_dict and qrels_dict[qid]: temp_p.append(0.0); temp_r.append(0.0); temp_ap.append(0.0)
            if IS_DEBUG_QID: print("  Final query tokens list is empty. Assigning empty predictions.")
            continue

        final_scores = bm25_model.get_scores(final_query_tokens)
        top_all_indices = [i for i in np.argsort(final_scores)[::-1] if i < len(all_doc_ids)]

        current_preds_doc_ids_full = [str(all_doc_ids[i]) for i in top_all_indices[:top_k_preds_to_return_param]]
        predictions[qid] = current_preds_doc_ids_full
        if IS_DEBUG_QID:
            print(f"  Final Scores (Top 5 values): {final_scores[top_all_indices[:5]] if len(top_all_indices) > 0 else 'N/A'}")
            print(f"  Top {top_k_preds_to_return_param} Predicted Doc IDs (Full List): {current_preds_doc_ids_full[:10]}...")

        current_preds_ids_eval = current_preds_doc_ids_full[:top_k_eval_param]
        gold_relevant_docs = qrels_dict.get(qid)

        if IS_DEBUG_QID:
            print(f"  Gold Relevant IDs for QID {qid} (from qrels): {gold_relevant_docs}")
            if gold_relevant_docs: print(f"    Type of first gold ID: {type(list(gold_relevant_docs)[0]) if gold_relevant_docs else 'N/A'}")
            print(f"  Predicted IDs for Eval (Top {top_k_eval_param}): {current_preds_ids_eval}")
            if current_preds_ids_eval : print(f"    Type of first predicted ID: {type(current_preds_ids_eval[0]) if current_preds_ids_eval else 'N/A'}")

        n_gold = len(gold_relevant_docs) if gold_relevant_docs is not None else 0
        hits = [1 if doc_id in gold_relevant_docs else 0 for doc_id in current_preds_ids_eval] if gold_relevant_docs is not None else [0]*len(current_preds_ids_eval)
        n_hits = sum(hits)

        if IS_DEBUG_QID: print(f"  Hits array for Top {top_k_eval_param}: {hits}, Num Hits: {n_hits}")

        temp_p.append(n_hits / top_k_eval_param if top_k_eval_param > 0 else 0.0)
        temp_r.append(n_hits / n_gold if n_gold > 0 else (0.0 if n_hits == 0 else 1.0))

        if n_gold > 0:
            ap_q, h_q_count = 0.0, 0
            for rank_idx, hit_flag_val in enumerate(hits):
                if hit_flag_val: h_q_count += 1; ap_q += h_q_count / (rank_idx + 1)
            den_ap = min(n_gold, top_k_eval_param)
            current_ap = ap_q / den_ap if den_ap > 0 else 0.0
            temp_ap.append(current_ap)
            if IS_DEBUG_QID: print(f"    P@{top_k_eval_param}: {n_hits / top_k_eval_param if top_k_eval_param > 0 else 0.0:.4f}, R@{top_k_eval_param}: {n_hits / n_gold if n_gold > 0 else (0.0 if n_hits == 0 else 1.0):.4f}, AP: {current_ap:.4f}")
        else:
            temp_ap.append(0.0)
            if IS_DEBUG_QID: print(f"    P@{top_k_eval_param}: {n_hits / top_k_eval_param if top_k_eval_param > 0 else 0.0:.4f}, R@{top_k_eval_param}: {0.0 if n_hits == 0 else 1.0:.4f}, AP: 0.0 (no gold docs)")
        if IS_DEBUG_QID: print(f"================== END DEBUG QID: {qid} ==================\n")

    results[f'P@{top_k_eval_param}'] = np.mean(temp_p) if temp_p else 0.0
    results[f'R@{top_k_eval_param}'] = np.mean(temp_r) if temp_r else 0.0
    results[f'MAP@{top_k_eval_param}'] = np.mean(temp_ap) if temp_ap else 0.0
    print(f"Final Metrics ({desc_prefix}): P@{top_k_eval_param}={results[f'P@{top_k_eval_param}']:.4f}, R@{top_k_eval_param}={results[f'R@{top_k_eval_param}']:.4f}, MAP@{top_k_eval_param}={results[f'MAP@{top_k_eval_param}']:.4f}")
    return results, predictions

# ==============================================================================
# SECTION A5: BM25+RM3 ON TRAINING ABSTRACTS - EXECUTION
# ==============================================================================
print("\n\n--- SECTION A5: BM25+RM3 on Training Abstracts ---")
tokenized_corpus_A5_cache_path = os.path.join(PIPELINE_A5_CACHE_DIR, 'tokenized_corpus_A5_abstract.pkl')
ids_queries_qrels_A5_cache_path = os.path.join(PIPELINE_A5_CACHE_DIR, 'ids_queries_qrels_A5_abstract.pkl')
bm25_model_A5_cache_path = os.path.join(PIPELINE_A5_CACHE_DIR, 'bm25_model_A5_abstract.pkl')
bm25_results_A5_cache_path = os.path.join(PIPELINE_A5_CACHE_DIR, f'bm25_rm3_results_A5_abstract_eval{TOP_K_EVAL_A5}.pkl')
bm25_predictions_A5_cache_path = os.path.join(PIPELINE_A5_CACHE_DIR, f'bm25_rm3_predictions_A5_abstract_top{TOP_K_CANDIDATES_FOR_RERANKING_A5}.pkl')

ids_A5, corpus_A5_tokenized, queries_A5, qrels_A5, loaded_A5_data_fully = load_and_tokenize_training_data_for_bm25_A5(
    ABSTRACT_TRAIN_ZIP_PATH, extract_text_from_abstract_json,
    tokenized_corpus_A5_cache_path, ids_queries_qrels_A5_cache_path, "A5 Abstract"
)

bm25_A5_model = None
bm25_rm3_results_A5 = {}
bm25_rm3_predictions_A5 = OrderedDict()
A5_fully_processed_and_evaluated = False

if loaded_A5_data_fully:
    if os.path.exists(bm25_model_A5_cache_path) and \
       os.path.exists(bm25_results_A5_cache_path) and \
       os.path.exists(bm25_predictions_A5_cache_path):
        print("  Attempting to load all cached A5 BM25+RM3 components & results...")
        try:
            with open(bm25_model_A5_cache_path, 'rb') as f: bm25_A5_model = pickle.load(f)
            with open(bm25_results_A5_cache_path, 'rb') as f: bm25_rm3_results_A5 = pickle.load(f)
            with open(bm25_predictions_A5_cache_path, 'rb') as f: bm25_rm3_predictions_A5 = pickle.load(f)
            if bm25_A5_model and isinstance(bm25_rm3_results_A5, dict) and \
               isinstance(bm25_rm3_predictions_A5, OrderedDict) and \
               (len(bm25_rm3_predictions_A5) == len(queries_A5) if queries_A5 else True):
                A5_fully_processed_and_evaluated = True
                print("    Successfully loaded all A5 BM25+RM3 components and results from cache.")
        except Exception as e: print(f"    Error loading full A5 BM25+RM3 cache: {e}. Will regenerate."); bm25_A5_model, bm25_rm3_results_A5, bm25_rm3_predictions_A5 = None, {}, OrderedDict()

    if not A5_fully_processed_and_evaluated:
        if bm25_A5_model is None:
            if os.path.exists(bm25_model_A5_cache_path):
                print(f"  Attempting to load only A5 BM25 model from {bm25_model_A5_cache_path}")
                try:
                    with open(bm25_model_A5_cache_path, 'rb') as f: bm25_A5_model = pickle.load(f)
                    print("    Successfully loaded A5 BM25 model from cache.")
                except Exception as e_load_bm25_model: print(f"    Error loading A5 BM25 model from cache: {e_load_bm25_model}"); bm25_A5_model = None
            else:
                print("  Building BM25 index for A5 (Abstracts)...")
                if corpus_A5_tokenized and len(corpus_A5_tokenized) == len(ids_A5):
                    bm25_A5_model = BM25Okapi(corpus_A5_tokenized, k1=BM25_K1, b=BM25_B)
                    print("    A5 BM25 model built.")
                    try:
                        with open(bm25_model_A5_cache_path, 'wb') as f_bm25_save: pickle.dump(bm25_A5_model, f_bm25_save, protocol=pickle.HIGHEST_PROTOCOL)
                        print("    A5 BM25 model saved to cache.")
                    except Exception as e_save_bm25: print(f"    Error saving A5 BM25 model: {e_save_bm25}")
                else: print(f"  ERROR: A5 tokenized corpus empty/mismatched. Cannot build BM25 model.")

        if bm25_A5_model:
            print("  Evaluating BM25+RM3 for A5 (Abstracts) and generating predictions...")
            # *************************************************************************************
            # >>> IMPORTANT: For targeted debugging, modify QIDs_TO_DEBUG here <<<
            # Example: QIDs_TO_DEBUG_LIST = ['your_qid1_from_qrels', 'your_qid2_from_qrels']
            QIDs_TO_DEBUG_LIST = None # Set to None to run all, or list of strings for specific QIDs
            # *************************************************************************************
            bm25_rm3_results_A5, bm25_rm3_predictions_A5 = evaluate_bm25_rm3_A5(
                bm25_A5_model, corpus_A5_tokenized, ids_A5, queries_A5, qrels_A5, "A5 Train",
                RM3_FB_DOCS, RM3_FB_TERMS, TOP_K_EVAL_A5, TOP_K_CANDIDATES_FOR_RERANKING_A5,
                QIDs_TO_DEBUG = QIDs_TO_DEBUG_LIST
            )
            if bm25_rm3_results_A5:
                with open(bm25_results_A5_cache_path, 'wb') as f_res_save: pickle.dump(bm25_rm3_results_A5, f_res_save, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    A5 results saved to {bm25_results_A5_cache_path}.")
            if bm25_rm3_predictions_A5:
                with open(bm25_predictions_A5_cache_path, 'wb') as f_preds_save: pickle.dump(bm25_rm3_predictions_A5, f_preds_save, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    A5 predictions saved to {bm25_predictions_A5_cache_path}.")
            A5_fully_processed_and_evaluated = True
        else: print("  Skipping A5 evaluation: BM25 model for A5 not available.")

    if bm25_rm3_results_A5:
        print(f"\nA5: BM25+RM3 (Abstracts Train) Final Results (Top {TOP_K_EVAL_A5}):")
        for m,v in bm25_rm3_results_A5.items(): print(f"  {m}: {v:.4f}")
    if bm25_rm3_predictions_A5:
        count_a5=0; print(f"\n--- Sample A5 Predictions (Top {TOP_K_CANDIDATES_FOR_RERANKING_A5}, showing 3) ---")
        for qid_a5,preds_a5 in bm25_rm3_predictions_A5.items():
            print(f"QID {qid_a5} Top 3 (from {len(preds_a5)}): {preds_a5[:3]}"); count_a5+=1
            if count_a5>=3: break
    elif loaded_A5_data : print("A5 BM25+RM3 processing not fully completed or results not generated/loaded.")
else:
    print("Skipping A5: Data loading/tokenization failed.")

print("\n\n--- A5 (BM25+RM3 Training Abstracts) Processing Complete ---")
gc.collect()

Starting installations for A5 (if this is the first run of the cell in a new session)...


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
A5 INSTALLATIONS SECTION COMPLETE (or requirements already satisfied).
1. IF PIP INSTALLS RAN ABOVE (AND IT WAS THE FIRST TIME FOR THESE PACKAGES IN THIS SESSION):
   YOU **MUST** HAVE MANUALLY RESTARTED THE COLAB RUNTIME BEFORE THIS RUN.
   (Menu: Runtime -> Restart runtime...)
2. AFTER RESTARTING, RUN THIS ENTIRE CELL AGAIN FROM THE TOP.
3. **BEFORE THIS RUN (especially the second time after restart):**
   Consider manually deleting the following file from the A5 cache directory
   '/content/drive/MyDrive/colab-4/cache_A5_abstract_bm25_train/' (if it exists from a previous problematic run):
   - 'ids_queries_qrels_A5_abstract.pkl' (to force correct qrels re-parsing).
   - Optionally, delete 'tokenized_corpus_A5_abstract.pkl' and 'bm25_model_A5_abstract.pkl' for a full A5 re-run.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

BM25+RM3 (A5 Train) Evaluation:   0%|          | 0/393 [00:00<?, ?it/s]

Final Metrics (A5 Train): P@5=0.1318, R@5=0.0809, MAP@5=0.0946
    A5 results saved to /content/drive/MyDrive/colab-4/cache_A5_abstract_bm25_train/bm25_rm3_results_A5_abstract_eval5.pkl.
    A5 predictions saved to /content/drive/MyDrive/colab-4/cache_A5_abstract_bm25_train/bm25_rm3_predictions_A5_abstract_top100.pkl.

A5: BM25+RM3 (Abstracts Train) Final Results (Top 5):
  P@5: 0.1318
  R@5: 0.0809
  MAP@5: 0.0946

--- Sample A5 Predictions (Top 100, showing 3) ---
QID ce5bfacf-8652-4bc1-a5b0-6144a917fb1c Top 3 (from 100): ['24162972', '120154901', '112515106']
QID 71657c3b-4112-49d5-86cb-705325aeb06e Top 3 (from 100): ['50639875', '145156217', '135126299']
QID f60d94de-b903-48da-b13f-a8334c70f949 Top 3 (from 100): ['74798737', '9362554', '156394500']


--- A5 (BM25+RM3 Training Abstracts) Processing Complete ---


18

In [ ]:
# ==============================================================================
# CELL 0: INSTALLATIONS
# (Run this section, THEN STOP, MANUALLY RESTART RUNTIME, then run Cell 1 and subsequent task cells)
# ==============================================================================
print("Starting installations (if this is the first run of the cell in a new session)...\n")
# Step 1: Install specific PyTorch ecosystem versions
print("Installing PyTorch, Torchvision, Torchaudio...")
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
print("PyTorch ecosystem installation attempt finished.")

# Step 2: Install other necessary packages
print("\nInstalling rank_bm25, sentence-transformers and other libraries...")
!pip install rank_bm25 sentence-transformers==2.7.0 nltk tqdm scikit-learn pandas -q
print("Other packages installation attempt finished.")

print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
print("CELL 0 (INSTALLATIONS) HAS COMPLETED BOTH INSTALLATION STEPS.")
print("YOU **MUST** NOW MANUALLY RESTART THE COLAB RUNTIME FOR ALL CHANGES TO TAKE EFFECT.")
print("GO TO THE MENU: Runtime -> Restart runtime...")
print("AFTER RESTARTING, PROCEED TO RUN CELL 1 (Common Setup & Helper Functions).")
print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")

Starting installations (if this is the first run of the cell in a new session)...

Installing PyTorch, Torchvision, Torchaudio...
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 96.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 42.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 98.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 17.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 35.9 MB/s eta 0:00:00
     

In [ ]:
# ==============================================================================
# CELL 1: COMMON SETUP & HELPER FUNCTIONS
# (Run this cell AFTER Cell 0 and AFTER restarting the Colab Runtime)
# ==============================================================================
print("\n--- SECTION 1: Initializing Common Setup & Helper Functions ---")
from google.colab import drive
import os
import gc
from collections import OrderedDict, defaultdict, Counter
import zipfile
import json
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm.notebook import tqdm
import numpy as np
import pickle
from rank_bm25 import BM25Okapi
import pandas as pd

try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Google Drive mounting error: {e}. Please ensure Drive is accessible and authorized.")

# --- Path Definitions ---
DRIVE_BASE_FOLDER = '/content/drive/MyDrive/colab-4/'
ABSTRACT_TRAIN_ZIP_FILENAME = 'longeval_sci_training_2025_abstract.zip'
FULLTEXT_TRAIN_ZIP_FILENAME = 'longeval_sci_training_2025_fulltext.zip'
ABSTRACT_TEST_ZIP_FILENAME  = 'longeval_sci_testing_2025_abstract.zip'
FULLTEXT_TEST_ZIP_FILENAME  = 'longeval_sci_testing_2025_fulltext.zip'
ABSTRACT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, ABSTRACT_TRAIN_ZIP_FILENAME)
FULLTEXT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, FULLTEXT_TRAIN_ZIP_FILENAME)
ABSTRACT_TEST_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, ABSTRACT_TEST_ZIP_FILENAME)
FULLTEXT_TEST_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, FULLTEXT_TEST_ZIP_FILENAME)

PIPELINE_A5_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A5_abstract_bm25_train/')
PIPELINE_A6_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A6_fulltext_bm25_train/')
PIPELINE_A7_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A7_abstract_bm25_test/')
PIPELINE_A8_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_A8_fulltext_bm25_test/')
os.makedirs(PIPELINE_A5_CACHE_DIR, exist_ok=True)
os.makedirs(PIPELINE_A6_CACHE_DIR, exist_ok=True)
os.makedirs(PIPELINE_A7_CACHE_DIR, exist_ok=True)
os.makedirs(PIPELINE_A8_CACHE_DIR, exist_ok=True)
print("Path definitions complete.")

# Verify essential ZIP paths
for zip_path_var_name, zip_file_path in [
    ("ABSTRACT_TRAIN_ZIP_PATH", ABSTRACT_TRAIN_ZIP_PATH), ("FULLTEXT_TRAIN_ZIP_PATH", FULLTEXT_TRAIN_ZIP_PATH),
    ("ABSTRACT_TEST_ZIP_PATH", ABSTRACT_TEST_ZIP_PATH), ("FULLTEXT_TEST_ZIP_PATH", FULLTEXT_TEST_ZIP_PATH)
]:
    if not os.path.exists(zip_file_path): print(f"CRITICAL WARNING: {zip_path_var_name} not found: {zip_file_path}")

# NLTK Resources
nltk_resources_ready = False
try:
    stopwords.words('english'); WordNetLemmatizer().lemmatize('cats'); nltk.word_tokenize("test")
    nltk_resources_ready = True
except:
    print("Downloading NLTK (stopwords, wordnet, punkt)...")
    try: nltk.download('stopwords',quiet=True); nltk.download('wordnet',quiet=True); nltk.download('punkt',quiet=True); nltk_resources_ready=True
    except Exception as e: print(f"NLTK download error: {e}")

if nltk_resources_ready:
    STOP = set(stopwords.words("english")); LEM = WordNetLemmatizer(); _token_pat = re.compile(r"[A-Za-z]{2,}")
    print("Text cleaning utilities ready.")
else:
    print("ERROR: NLTK resources not ready. Cleaning will be basic."); STOP,LEM,_token_pat = set(),None,re.compile(r"[A-Za-z]{2,}")

def clean_text_for_bm25(text: str) -> list:
    if not isinstance(text, str): return []
    if not LEM or not nltk_resources_ready: return text.lower().strip().split()
    try: toks = nltk.word_tokenize(text.lower())
    except Exception: toks = text.lower().split()
    toks = [t for t in toks if t.isalpha() and t not in STOP and len(t) > 1]
    return [LEM.lemmatize(t) for t in toks]

def extract_text_from_abstract_json(record_dict):
    title = record_dict.get("title", "") or ""; abstract_content = record_dict.get("abstract", "") or ""
    return f"{title} {abstract_content}".strip()

MAX_WORDS_FOR_BM25_FULLTEXT = 400
def extract_and_truncate_full_text_for_bm25(record_dict):
    title = record_dict.get("title", "") or ""; abstract = record_dict.get("abstract", "") or ""; main_text = ""
    possible_fields = ["text", "full_text", "body", "body_text", "content", "fulltext"]
    for field in possible_fields:
        content = record_dict.get(field)
        if content and isinstance(content, str): main_text = content; break
    combined = f"{title} {abstract} {main_text}".strip().split()
    return " ".join(combined[:MAX_WORDS_FOR_BM25_FULLTEXT])

print("Setup and helper functions defined.")
BM25_K1_PARAM = 1.5; BM25_B_PARAM = 0.75; RM3_FB_DOCS_PARAM = 5; RM3_FB_TERMS_PARAM = 5
TOP_K_EVAL_PARAM = 5; TOP_K_PREDS_PARAM = 100
print(f"Global BM25/RM3 Parameters: K1={BM25_K1_PARAM}, B={BM25_B_PARAM}, FB_DOCS={RM3_FB_DOCS_PARAM}, FB_TERMS={RM3_FB_TERMS_PARAM}, EVAL_K={TOP_K_EVAL_PARAM}, PREDS_K={TOP_K_PREDS_PARAM}")

# ==============================================================================
# SECTION 2: GENERIC DATA LOADING & TOKENIZING FUNCTION FOR BM25
# ==============================================================================
print("\n--- SECTION 2: Defining Generic Data Loading Helper for BM25 (RAM Optimized Attempt) ---")
def load_and_tokenize_data_for_bm25(
    data_zip_path, text_extraction_func, pipeline_cache_dir,
    data_type_prefix, is_test_set=False
):
    all_ids, tokenized_corpus, queries, qrels = [], [], OrderedDict(), defaultdict(set)
    data_fully_loaded, corpus_from_cache, meta_from_cache = False, False, False
    tokenized_corpus_cache = os.path.join(pipeline_cache_dir, f'tokenized_corpus_{data_type_prefix}.pkl')
    meta_cache_file_name = f'ids_docs_meta_{data_type_prefix}.pkl' if is_test_set else f'ids_queries_qrels_{data_type_prefix}.pkl'
    ids_meta_cache = os.path.join(pipeline_cache_dir, meta_cache_file_name)

    if os.path.exists(tokenized_corpus_cache):
        print(f"  Attempting to load cached tokenized corpus for {data_type_prefix} from: {tokenized_corpus_cache}")
        try:
            with open(tokenized_corpus_cache, 'rb') as f: tokenized_corpus = pickle.load(f)
            if isinstance(tokenized_corpus, list) and (not tokenized_corpus or isinstance(tokenized_corpus[0], list)):
                corpus_from_cache = True; print(f"    Successfully loaded tokenized corpus for {data_type_prefix} ({len(tokenized_corpus)} docs).")
            else: tokenized_corpus = []
        except Exception as e: print(f"    Error loading tokenized corpus cache for {data_type_prefix}: {e}"); tokenized_corpus = []
    if os.path.exists(ids_meta_cache):
        print(f"  Attempting to load cached IDs/meta for {data_type_prefix} from: {ids_meta_cache}")
        try:
            with open(ids_meta_cache, 'rb') as f: meta = pickle.load(f)
            all_ids = meta.get('all_ids', [])
            if not is_test_set:
                queries = meta.get('queries', OrderedDict()); qrels_temp = meta.get('qrels', defaultdict(set)); qrels = defaultdict(set, qrels_temp)
            if all_ids : meta_from_cache = True; print(f"    Successfully loaded IDs ({len(all_ids)}) for {data_type_prefix}. Queries: {len(queries)}, Qrels: {len(qrels)}.")
            else: all_ids, queries, qrels = [], OrderedDict(), defaultdict(set)
        except Exception as e: print(f"    Error loading IDs/meta cache for {data_type_prefix}: {e}"); all_ids, queries, qrels = [], OrderedDict(), defaultdict(set)

    if corpus_from_cache and meta_from_cache and (len(all_ids) == len(tokenized_corpus) if tokenized_corpus else False):
        data_fully_loaded = True; print(f"  All {data_type_prefix} data (corpus & meta) successfully loaded from cache.")
    elif corpus_from_cache and is_test_set and all_ids and (len(all_ids) == len(tokenized_corpus) if tokenized_corpus else False):
        data_fully_loaded = True; print(f"  {data_type_prefix} test documents (corpus & IDs) successfully loaded from cache. Queries loaded separately.")
    else:
        print(f"  Cache miss or mismatch for {data_type_prefix}. Processing from ZIP: {data_zip_path}")
        if not meta_from_cache: all_ids, queries, qrels = [], OrderedDict(), defaultdict(set)
        if not os.path.exists(data_zip_path): print(f"  ERROR: Data ZIP not found: {data_zip_path}"); return all_ids, tokenized_corpus, queries, qrels, False
        print(f"  --- Loading & Tokenizing {data_type_prefix} Data from: {os.path.basename(data_zip_path)} ---")
        raw_texts_to_tokenize, ids_to_tokenize_local = [], []
        with zipfile.ZipFile(data_zip_path, 'r') as z:
            entries = z.namelist(); zip_root = entries[0].split('/')[0] if entries and '/' in entries[0] else ""
            jsonl_path_prefix = os.path.join(zip_root, 'documents/') if zip_root else 'documents/'
            jsonl_files_in_zip = [info for info in z.infolist() if info.filename.lower().startswith(jsonl_path_prefix.lower()) and info.filename.lower().endswith(".jsonl") and not info.is_dir()]
            print(f"    Found {len(jsonl_files_in_zip)} JSONL files in '{jsonl_path_prefix}'.")
            if not corpus_from_cache or not meta_from_cache or (all_ids and tokenized_corpus and len(all_ids) != len(tokenized_corpus)):
                for info in tqdm(jsonl_files_in_zip, desc=f"Reading {data_type_prefix} JSONLs", unit="file"):
                    with z.open(info) as f_jsonl:
                        for line_byte in f_jsonl:
                            try:
                                record = json.loads(line_byte.decode('utf-8')); doc_id_str = str(record.get("id")); raw_text = text_extraction_func(record)
                                if doc_id_str and raw_text:
                                    if not meta_from_cache: ids_to_tokenize_local.append(doc_id_str)
                                    if not corpus_from_cache: raw_texts_to_tokenize.append(raw_text)
                            except: pass
                if not meta_from_cache: all_ids = ids_to_tokenize_local
                if not corpus_from_cache :
                    print(f"    Tokenizing {len(raw_texts_to_tokenize)} docs for {data_type_prefix}...")
                    tokenized_corpus = [clean_text_for_bm25(text) for text in tqdm(raw_texts_to_tokenize, desc=f"Tokenizing {data_type_prefix}")]
                    del raw_texts_to_tokenize; gc.collect() # Free memory
                    if tokenized_corpus:
                        with open(tokenized_corpus_cache, 'wb') as f: pickle.dump(tokenized_corpus, f, protocol=pickle.HIGHEST_PROTOCOL); print(f"    Saved tokenized corpus for {data_type_prefix}.")
            if not meta_from_cache:
                if not is_test_set:
                    query_file_path = os.path.join(zip_root, 'queries.txt') if zip_root else 'queries.txt'
                    qrels_file_path = os.path.join(zip_root, 'qrels.txt') if zip_root else 'qrels.txt'
                    if query_file_path in z.namelist():
                        with z.open(query_file_path) as f_q: queries.update(OrderedDict(l.decode('utf-8',errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8',errors='ignore').strip()))
                        print(f"    Loaded {len(queries)} queries for {data_type_prefix}.")
                    if qrels_file_path in z.namelist():
                        lines_read, qrels_added = 0,0
                        with z.open(qrels_file_path) as f_r:
                            for i, l_bytes in enumerate(f_r):
                                lines_read+=1; line_str = l_bytes.decode('utf-8', errors='ignore').strip(); parts = line_str.split()
                                qid_r, did_r, rel_r_str = (None, None, None)
                                if len(parts) == 4: qid_r, _, did_r, rel_r_str = parts[0], parts[1], parts[2], parts[3]
                                elif len(parts) == 3: qid_r, did_r, rel_r_str = parts[0], parts[1], parts[2]
                                else: continue
                                try:
                                    if int(rel_r_str) > 0: qrels[str(qid_r)].add(str(did_r)); qrels_added+=1
                                except ValueError: pass
                        print(f"    Finished qrels for {data_type_prefix}. Lines: {lines_read}. Added: {qrels_added}. QIDs: {len(qrels)}.")
                if all_ids:
                    meta_to_save = {'all_ids': all_ids}
                    if not is_test_set: meta_to_save['queries'] = queries; meta_to_save['qrels'] = qrels
                    with open(ids_meta_cache, 'wb') as f: pickle.dump(meta_to_save, f, protocol=pickle.HIGHEST_PROTOCOL); print(f"    Saved meta data for {data_type_prefix}.")
        data_fully_loaded = bool(all_ids and tokenized_corpus and (len(all_ids) == len(tokenized_corpus)) and (queries if not is_test_set or not queries else True) )
    if not data_fully_loaded: print(f"  ERROR: {data_type_prefix} data loading or tokenization partially failed or mismatched.")
    return all_ids, tokenized_corpus, queries, qrels, data_fully_loaded

# ==============================================================================
# SECTION 3: GENERIC BM25+RM3 CORE EVALUATION LOGIC
# ==============================================================================
print("\n--- SECTION 3: Defining BM25+RM3 Core Evaluation Logic ---")
def evaluate_bm25_rm3_core_logic(
    bm25_model, tokenized_corpus_docs, all_doc_ids, queries_dict, qrels_dict,
    desc_prefix="BM25_Eval", fb_docs_param=5, fb_terms_param=5,
    top_k_eval_param=5, top_k_preds_to_return_param=100,
    QIDs_TO_DEBUG = None
):
    results = {}; predictions = OrderedDict(); temp_p, temp_r, temp_ap = [], [], []
    if not all_doc_ids or not tokenized_corpus_docs or bm25_model is None:
        print(f"Error in evaluate_bm25_rm3_core_logic ({desc_prefix}): Missing critical inputs."); return results, predictions
    has_qrels_for_eval = qrels_dict is not None and len(qrels_dict) > 0
    print(f"Starting BM25+RM3 ({desc_prefix}) for {len(queries_dict)} queries. Qrels available for eval: {has_qrels_for_eval}")

    for qid_idx, (qid, query_text_raw) in enumerate(tqdm(queries_dict.items(), desc=f"BM25+RM3 ({desc_prefix})")):
        IS_DEBUG_QID = (QIDs_TO_DEBUG is not None and qid in QIDs_TO_DEBUG)
        if IS_DEBUG_QID: print(f"\n\n==================== DEBUGGING QID: {qid} ({desc_prefix}) ===================="); print(f"  Raw Query: '{query_text_raw}'")
        tokenized_query = clean_text_for_bm25(query_text_raw)
        if IS_DEBUG_QID: print(f"  Tokenized Query ({len(tokenized_query)} terms): {tokenized_query[:20]}")
        if not tokenized_query:
            if IS_DEBUG_QID: print("  Query is empty after cleaning. Skipping."); predictions[qid] = []
            if has_qrels_for_eval and qid in qrels_dict and qrels_dict[qid]: temp_p.append(0.0); temp_r.append(0.0); temp_ap.append(0.0)
            continue
        initial_scores = bm25_model.get_scores(tokenized_query)
        valid_initial_indices = np.argsort(initial_scores)[::-1]
        feedback_doc_indices = [idx for idx in valid_initial_indices if idx < len(tokenized_corpus_docs)][:fb_docs_param]
        if IS_DEBUG_QID and feedback_doc_indices: print(f"  Initial BM25 Scores (Top {len(feedback_doc_indices)} for feedback): {initial_scores[feedback_doc_indices]}")
        term_scores_expansion = Counter()
        if feedback_doc_indices:
            feedback_doc_actual_scores = initial_scores[feedback_doc_indices]; scores_sum = np.sum(feedback_doc_actual_scores)
            doc_weights = feedback_doc_actual_scores / scores_sum if scores_sum > 1e-9 else np.ones(len(feedback_doc_actual_scores)) / (len(feedback_doc_actual_scores) if len(feedback_doc_actual_scores)>0 else 1.0)
            for i, doc_idx in enumerate(feedback_doc_indices):
                doc_tokens = tokenized_corpus_docs[doc_idx]; term_freq_in_doc = Counter(doc_tokens); doc_len = len(doc_tokens)
                if doc_len > 0:
                    current_doc_weight = doc_weights[i] if i < len(doc_weights) else (1.0/len(feedback_doc_indices) if len(feedback_doc_indices)>0 else 0)
                    for term, freq in term_freq_in_doc.items(): term_scores_expansion[term] += current_doc_weight * (freq / doc_len)
        if IS_DEBUG_QID: print(f"  Term Scores for Expansion (Top 10 Pre-Filter): {term_scores_expansion.most_common(10)}")
        for t_original in tokenized_query: term_scores_expansion.pop(t_original, None)
        expansion_terms = [t for t, _ in term_scores_expansion.most_common(fb_terms_param)]
        if IS_DEBUG_QID: print(f"  Expansion Terms: {expansion_terms}")
        final_query_tokens = tokenized_query + expansion_terms
        if not final_query_tokens and tokenized_query: final_query_tokens = tokenized_query
        if IS_DEBUG_QID: print(f"  Final Query Tokens ({len(final_query_tokens)} terms): {final_query_tokens[:20]}")
        if not final_query_tokens: predictions[qid] = []; continue
        final_scores = bm25_model.get_scores(final_query_tokens)
        top_all_indices = [i for i in np.argsort(final_scores)[::-1] if i < len(all_doc_ids)]
        current_preds_doc_ids_full = [str(all_doc_ids[i]) for i in top_all_indices[:top_k_preds_to_return_param]]
        predictions[qid] = current_preds_doc_ids_full
        if IS_DEBUG_QID: print(f"  Top {min(10, top_k_preds_to_return_param)} Predicted Doc IDs: {current_preds_doc_ids_full[:10]}")
        if has_qrels_for_eval:
            current_preds_ids_eval = current_preds_doc_ids_full[:top_k_eval_param]
            gold_relevant_docs = qrels_dict.get(qid)
            n_gold = len(gold_relevant_docs) if gold_relevant_docs is not None else 0
            hits = [1 if doc_id in gold_relevant_docs else 0 for doc_id in current_preds_ids_eval] if gold_relevant_docs is not None else [0]*len(current_preds_ids_eval)
            n_hits = sum(hits)
            if IS_DEBUG_QID: print(f"  Gold: {gold_relevant_docs}, Hits: {hits}, N_Hits: {n_hits}")
            temp_p.append(n_hits / top_k_eval_param if top_k_eval_param > 0 else 0.0)
            temp_r.append(n_hits / n_gold if n_gold > 0 else (0.0 if n_hits == 0 else 1.0))
            if n_gold > 0:
                ap_q, h_q_count = 0.0, 0
                for rank_idx, hit_flag_val in enumerate(hits):
                    if hit_flag_val: h_q_count += 1; ap_q += h_q_count / (rank_idx + 1)
                den_ap = min(n_gold, top_k_eval_param); current_ap = ap_q / den_ap if den_ap > 0 else 0.0
                temp_ap.append(current_ap)
                if IS_DEBUG_QID: print(f"    P@{top_k_eval_param}: {temp_p[-1]:.4f}, R@{top_k_eval_param}: {temp_r[-1]:.4f}, AP: {current_ap:.4f}")
            else:
                temp_ap.append(0.0)
                if IS_DEBUG_QID: print(f"    P@{top_k_eval_param}: {temp_p[-1]:.4f}, R@{top_k_eval_param}: {temp_r[-1]:.4f}, AP: 0.0 (no gold docs)")
        if IS_DEBUG_QID: print(f"================== END DEBUG QID: {qid} ({desc_prefix}) ==================\n")
    if has_qrels_for_eval and temp_p:
        results[f'P@{top_k_eval_param}'] = np.mean(temp_p) if temp_p else 0.0
        results[f'R@{top_k_eval_param}'] = np.mean(temp_r) if temp_r else 0.0
        results[f'MAP@{top_k_eval_param}'] = np.mean(temp_ap) if temp_ap else 0.0
    return results, predictions

# ==============================================================================
# SECTION 3B: WRAPPER FOR BM25+RM3 TRAINING & EVALUATION
# (Corrected sample printing loop)
# ==============================================================================
print("\n--- SECTION 3B: Defining BM25+RM3 Training Pipeline Wrapper ---")
def run_bm25_rm3_training_pipeline(
    data_zip_path, text_extraction_func, pipeline_cache_dir, data_type_prefix,
    bm25_model_cache_path, results_cache_path, predictions_cache_path,
    k1,b,fb_docs,fb_terms,top_k_eval,top_k_preds, QIDs_TO_DEBUG_LIST=None
):
    all_doc_ids, tokenized_corpus_docs, queries_dict, qrels_dict, data_loaded = load_and_tokenize_data_for_bm25(
        data_zip_path, text_extraction_func, pipeline_cache_dir, data_type_prefix, is_test_set=False
    )
    bm25_model, results, predictions = None, {}, OrderedDict()
    fully_processed = False # Renamed for clarity

    if data_loaded:
        if os.path.exists(bm25_model_cache_path) and os.path.exists(results_cache_path) and os.path.exists(predictions_cache_path):
            print(f"  Attempting to load all cached components for {data_type_prefix}...")
            try:
                with open(bm25_model_cache_path, 'rb') as f: bm25_model = pickle.load(f)
                with open(results_cache_path, 'rb') as f: results = pickle.load(f)
                with open(predictions_cache_path, 'rb') as f: predictions = pickle.load(f)
                if bm25_model and results and predictions and (len(predictions) == len(queries_dict) if queries_dict else True):
                    print(f"    Successfully loaded all components for {data_type_prefix} from cache.")
                    fully_processed = True
            except Exception as e: print(f"    Error loading cached components for {data_type_prefix}: {e}. Will regenerate.")

        if not fully_processed:
            if bm25_model is None:
                if os.path.exists(bm25_model_cache_path):
                    try:
                        with open(bm25_model_cache_path, 'rb') as f: bm25_model = pickle.load(f); print(f"    Loaded BM25 model for {data_type_prefix} from cache.")
                    except: bm25_model = None
                if bm25_model is None:
                    print(f"  Building BM25 index for {data_type_prefix}...")
                    if not tokenized_corpus_docs: print(f"ERROR: Tokenized corpus for {data_type_prefix} is empty."); return None, {}, OrderedDict()
                    if not all_doc_ids or len(tokenized_corpus_docs) != len(all_doc_ids): print(f"ERROR: Mismatch tokenized_corpus/all_doc_ids for {data_type_prefix}."); return None, {}, OrderedDict()
                    bm25_model = BM25Okapi(tokenized_corpus_docs, k1=k1, b=b); print(f"    BM25 model for {data_type_prefix} built.")
                    try:
                        with open(bm25_model_cache_path, 'wb') as f: pickle.dump(bm25_model, f, protocol=pickle.HIGHEST_PROTOCOL); print(f"    BM25 model for {data_type_prefix} saved.")
                    except Exception as e_save: print(f"    Error saving BM25 model for {data_type_prefix}: {e_save}")

            if bm25_model:
                results, predictions = evaluate_bm25_rm3_core_logic(
                    bm25_model, tokenized_corpus_docs, all_doc_ids, queries_dict, qrels_dict, data_type_prefix,
                    fb_docs, fb_terms, top_k_eval, top_k_preds, QIDs_TO_DEBUG_LIST
                )
                if results:
                    with open(results_cache_path, 'wb') as f: pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL); print(f"    Saved {data_type_prefix} results to {results_cache_path}")
                if predictions:
                    with open(predictions_cache_path, 'wb') as f: pickle.dump(predictions, f, protocol=pickle.HIGHEST_PROTOCOL); print(f"    Saved {data_type_prefix} predictions to {predictions_cache_path}")

        if results: print(f"\n{data_type_prefix} Final Results (Top {top_k_eval}):"); [print(f"  {m}: {v:.4f}") for m,v in results.items()]

        # CORRECTED sample printing loop
        if predictions:
            count_dp = 0
            print(f"\n--- Sample {data_type_prefix} Predictions (Top {top_k_preds}, showing up to 3) ---")
            for q_id, preds_list in predictions.items():
                print(f"QID {q_id} Top 3 (from {len(preds_list)} total preds): {preds_list[:3]}")
                count_dp += 1
                if count_dp >= 3:
                    break
    else: print(f"Skipping {data_type_prefix}: data loading failed.")
    return bm25_model, results, predictions

# ==============================================================================
# SECTION 4: GENERIC BM25+RM3 TEST PREDICTION GENERATION FUNCTION
# ==============================================================================
print("\n--- SECTION 4: Defining Generic BM25+RM3 Test Prediction Helper ---")
def load_test_queries_from_zip_corrected(zip_path, query_file_name_in_zip): # Copied from A1-A4 cell
    queries_dict = OrderedDict()
    if not os.path.exists(zip_path): print(f"ERROR: Query ZIP {zip_path} not found"); return queries_dict
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            actual_query_path = next((name for name in z.namelist() if name.endswith(query_file_name_in_zip)), None)
            if not actual_query_path:
                 potential_root = z.namelist()[0].split('/')[0] if z.namelist() and '/' in z.namelist()[0] else ""
                 if potential_root: actual_query_path = next((name for name in z.namelist() if name == f"{potential_root}/{query_file_name_in_zip}" or name == query_file_name_in_zip), None)
                 if not actual_query_path and query_file_name_in_zip in z.namelist(): actual_query_path = query_file_name_in_zip
            if actual_query_path and actual_query_path in z.namelist():
                print(f"  Found query file: '{actual_query_path}' in {os.path.basename(zip_path)}")
                with z.open(actual_query_path) as f_q:
                    queries_dict.update(OrderedDict(l.decode('utf-8', errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8', errors='ignore').strip()))
                print(f"    → Loaded {len(queries_dict)} queries from {query_file_name_in_zip}.")
            else: print(f"  ERROR: '{query_file_name_in_zip}' not found in {os.path.basename(zip_path)}. Available: {z.namelist()[:10]}")
    except Exception as e: print(f"  Error loading queries from {query_file_name_in_zip} in {zip_path}: {e}")
    return queries_dict

def run_bm25_rm3_test_predictions(
    trained_bm25_model_path, test_data_zip_path, text_extraction_func,
    pipeline_cache_dir, data_type_prefix, # Test specific cache and prefix
    k1=1.5, b=0.75, fb_docs=5, fb_terms=5, top_k_preds=100,
    QIDs_TO_DEBUG_LIST=None
):
    print(f"\n--- Starting Test Prediction Generation for {data_type_prefix} ---")
    if not os.path.exists(trained_bm25_model_path):
        print(f"ERROR: Trained BM25 model not found at {trained_bm25_model_path}. Cannot proceed."); return {}
    try:
        with open(trained_bm25_model_path, 'rb') as f: bm25_model = pickle.load(f)
        print(f"  Loaded trained BM25 model from {trained_bm25_model_path} for {data_type_prefix}.")
    except Exception as e_load_model: print(f"ERROR: Could not load BM25 model: {e_load_model}"); return {}

    test_doc_ids, test_tokenized_corpus, _, _, test_docs_loaded = load_and_tokenize_data_for_bm25(
        test_data_zip_path, text_extraction_func, pipeline_cache_dir, # Use pipeline_cache_dir for test docs too
        data_type_prefix + "_Docs", is_test_set=True
    )
    if not test_docs_loaded or not test_tokenized_corpus: print(f"ERROR: Failed to load/tokenize test docs for {data_type_prefix}."); return {}
    if len(test_doc_ids) != len(test_tokenized_corpus): print(f"ERROR: Mismatch test_doc_ids/corpus for {data_type_prefix}."); return {}

    all_query_predictions = OrderedDict()
    TEST_QUERY_FILES_LIST = ["queries_2024-11_test.txt", "queries_2025-01_test.txt"]

    for query_file_name in TEST_QUERY_FILES_LIST:
        predictions_cache_path_specific = os.path.join(pipeline_cache_dir, f'predictions_{data_type_prefix}_{os.path.splitext(query_file_name)[0]}.pkl')
        current_query_file_predictions = OrderedDict()
        if os.path.exists(predictions_cache_path_specific):
            print(f"  Loading cached predictions for {data_type_prefix} - {query_file_name} from {predictions_cache_path_specific}")
            try:
                with open(predictions_cache_path_specific, 'rb') as f: current_query_file_predictions = pickle.load(f)
                print(f"    Loaded {len(current_query_file_predictions)} predictions.")
                all_query_predictions.update(current_query_file_predictions)
                if current_query_file_predictions: # Corrected sample printing loop
                    count_sample = 0; print(f"    Sample Preds ({query_file_name}):")
                    for q_id_s, preds_s_list in current_query_file_predictions.items():
                        print(f"      QID {q_id_s} Top 3: {preds_s_list[:3]}"); count_sample += 1
                        if count_sample >= 2: break
                continue
            except Exception as e_load_test_preds: print(f"    Error loading cached predictions: {e_load_test_preds}. Regenerating.")

        print(f"  Processing query file: {query_file_name} for {data_type_prefix}")
        test_queries = load_test_queries_from_zip_corrected(test_data_zip_path, query_file_name)
        if not test_queries: print(f"    No queries loaded from {query_file_name}. Skipping."); continue

        # Call core logic without qrels for test prediction generation
        _, current_query_file_predictions = evaluate_bm25_rm3_core_logic(
            bm25_model, test_tokenized_corpus, test_doc_ids, test_queries, None, # qrels_dict is None
            f"{data_type_prefix}_{query_file_name}", fb_docs, fb_terms,
            0, top_k_preds, QIDs_TO_DEBUG_LIST # top_k_eval_param is 0 as no qrels
        )

        if current_query_file_predictions:
            with open(predictions_cache_path_specific, 'wb') as f: pickle.dump(current_query_file_predictions, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"  Saved predictions for {data_type_prefix} - {query_file_name} to {predictions_cache_path_specific}")
            all_query_predictions.update(current_query_file_predictions)
            # Corrected sample printing loop
            count_sample_test = 0; print(f"    Sample Preds ({query_file_name}):")
            for q_id_s_test, preds_s_list_test in current_query_file_predictions.items():
                print(f"      QID {q_id_s_test} Top 3: {preds_s_list_test[:3]}"); count_sample_test += 1
                if count_sample_test >= 2: break
    return all_query_predictions

print("Generic BM25+RM3 helper functions defined.")


--- SECTION 1: Initializing Common Setup & Helper Functions ---
Mounted at /content/drive
Google Drive mounted successfully.
Path definitions complete.
Text cleaning utilities ready.
Setup and helper functions defined.
Global BM25/RM3 Parameters: K1=1.5, B=0.75, FB_DOCS=5, FB_TERMS=5, EVAL_K=5, PREDS_K=100

--- SECTION 2: Defining Generic Data Loading Helper for BM25 (RAM Optimized Attempt) ---

--- SECTION 3: Defining BM25+RM3 Core Evaluation Logic ---

--- SECTION 3B: Defining BM25+RM3 Training Pipeline Wrapper ---

--- SECTION 4: Defining Generic BM25+RM3 Test Prediction Helper ---
Generic BM25+RM3 helper functions defined.


In [ ]:
# ==============================================================================
# CELL 2: TASK A6 - BM25+RM3 ON TRAINING FULL-TEXT
# ==============================================================================
print("\n\n--- SECTION A6: BM25+RM3 on Training Full-Text ---")
# Cache paths for A6
tokenized_corpus_A6_cache = os.path.join(PIPELINE_A6_CACHE_DIR, 'tokenized_corpus_A6_fulltext.pkl')
ids_queries_qrels_A6_cache = os.path.join(PIPELINE_A6_CACHE_DIR, 'ids_queries_qrels_A6_fulltext.pkl')
bm25_model_A6_cache_path = os.path.join(PIPELINE_A6_CACHE_DIR, 'bm25_model_A6_fulltext.pkl') # Corrected variable name
bm25_results_A6_cache_path = os.path.join(PIPELINE_A6_CACHE_DIR, f'bm25_rm3_results_A6_fulltext_eval{TOP_K_EVAL_PARAM}.pkl') # Corrected variable name
bm25_predictions_A6_cache_path = os.path.join(PIPELINE_A6_CACHE_DIR, f'bm25_rm3_predictions_A6_fulltext_top{TOP_K_PREDS_PARAM}.pkl') # Corrected variable name

# For debugging A6 results if they are zero, set list of QIDs from fulltext qrels
# QIDs_TO_DEBUG_A6_LIST = ['your_fulltext_training_qid1', 'your_fulltext_training_qid2']
QIDs_TO_DEBUG_A6_LIST = None

bm25_A6_model, bm25_rm3_results_A6, bm25_rm3_predictions_A6 = run_bm25_rm3_training_pipeline(
    FULLTEXT_TRAIN_ZIP_PATH, extract_and_truncate_full_text_for_bm25, # Use full-text extraction
    PIPELINE_A6_CACHE_DIR, "A6_FullText_Train",
    bm25_model_A6_cache_path, # Pass correct variable
    bm25_results_A6_cache_path, # Pass correct variable
    bm25_predictions_A6_cache_path, # Pass correct variable
    BM25_K1_PARAM, BM25_B_PARAM, RM3_FB_DOCS_PARAM, RM3_FB_TERMS_PARAM, TOP_K_EVAL_PARAM, TOP_K_PREDS_PARAM,
    QIDs_TO_DEBUG_A6_LIST
)
gc.collect()
print("\n\n--- Task A6 Processing Complete ---")



--- SECTION A6: BM25+RM3 on Training Full-Text ---
  Attempting to load cached tokenized corpus for A6_FullText_Train from: /content/drive/MyDrive/colab-4/cache_A6_fulltext_bm25_train/tokenized_corpus_A6_FullText_Train.pkl
    Error loading tokenized corpus cache for A6_FullText_Train: Ran out of input
  Cache miss or mismatch for A6_FullText_Train. Processing from ZIP: /content/drive/MyDrive/colab-4/longeval_sci_training_2025_fulltext.zip
  --- Loading & Tokenizing A6_FullText_Train Data from: longeval_sci_training_2025_fulltext.zip ---
    Found 21 JSONL files in 'longeval_sci_training_2025_fulltext/documents/'.


Reading A6_FullText_Train JSONLs:   0%|          | 0/21 [00:00<?, ?file/s]

    Tokenizing 2014257 docs for A6_FullText_Train...


Tokenizing A6_FullText_Train:   0%|          | 0/2014257 [00:00<?, ?it/s]

    Saved tokenized corpus for A6_FullText_Train.
    Loaded 393 queries for A6_FullText_Train.
    Finished qrels for A6_FullText_Train. Lines: 4262. Added: 3573. QIDs: 393.
    Saved meta data for A6_FullText_Train.
  Building BM25 index for A6_FullText_Train...
    BM25 model for A6_FullText_Train built.
    BM25 model for A6_FullText_Train saved.
Starting BM25+RM3 (A6_FullText_Train) for 393 queries. Qrels available for eval: True


BM25+RM3 (A6_FullText_Train):   0%|          | 0/393 [00:00<?, ?it/s]

    Saved A6_FullText_Train results to /content/drive/MyDrive/colab-4/cache_A6_fulltext_bm25_train/bm25_rm3_results_A6_fulltext_eval5.pkl
    Saved A6_FullText_Train predictions to /content/drive/MyDrive/colab-4/cache_A6_fulltext_bm25_train/bm25_rm3_predictions_A6_fulltext_top100.pkl

A6_FullText_Train Final Results (Top 5):
  P@5: 0.1313
  R@5: 0.0799
  MAP@5: 0.0939

--- Sample A6_FullText_Train Predictions (Top 100, showing up to 3) ---
QID ce5bfacf-8652-4bc1-a5b0-6144a917fb1c Top 3 (from 100 total preds): ['120154901', '24162972', '112515106']
QID 71657c3b-4112-49d5-86cb-705325aeb06e Top 3 (from 100 total preds): ['143423861', '10999452', '10999655']
QID f60d94de-b903-48da-b13f-a8334c70f949 Top 3 (from 100 total preds): ['74798737', '9362554', '156394500']


--- Task A6 Processing Complete ---


In [ ]:
# ==============================================================================
# CELL 3: TASK A7 - BM25+RM3 ON TESTING ABSTRACTS (using A5 model)
# ==============================================================================
print("\n\n--- SECTION A7: BM25+RM3 on Testing Abstracts (using A5 model) ---")
bm25_model_A5_path = os.path.join(PIPELINE_A5_CACHE_DIR, 'bm25_model_A5_abstract.pkl') # Model from A5

# For debugging A7 retrieval steps for specific QIDs
# QIDs_TO_DEBUG_A7_LIST = ['your_abstract_test_qid1'] # Replace with actual test QIDs if needed
QIDs_TO_DEBUG_A7_LIST = None # Set to a list of QIDs to debug, or None to run all

# Ensure TOP_K_PREDS_PARAM from Cell 1 (Common Setup) is used for consistency.
# This TOP_K_PREDS_PARAM determines how many candidates are generated by the run_bm25_rm3_test_predictions function.
# It's important that this matches the expected number of candidates if these predictions
# are to be used by a subsequent re-ranking stage that expects a certain number (e.g., top 100).

print(f"DEBUG A7: Using TOP_K_PREDS_PARAM = {TOP_K_PREDS_PARAM} for generating predictions.")

predictions_A7 = run_bm25_rm3_test_predictions(
    trained_bm25_model_path=bm25_model_A5_path,
    test_data_zip_path=ABSTRACT_TEST_ZIP_PATH,
    text_extraction_func=extract_text_from_abstract_json, # text_extraction_func for abstracts
    pipeline_cache_dir=PIPELINE_A7_CACHE_DIR,
    data_type_prefix="A7_Abstract_Test",
    k1=BM25_K1_PARAM, # Pass global K1
    b=BM25_B_PARAM,    # Pass global B
    fb_docs=RM3_FB_DOCS_PARAM, # Pass global RM3 feedback docs
    fb_terms=RM3_FB_TERMS_PARAM, # Pass global RM3 feedback terms
    top_k_preds=TOP_K_PREDS_PARAM, # Number of predictions to generate per query
    QIDs_TO_DEBUG_LIST=QIDs_TO_DEBUG_A7_LIST
)

if predictions_A7:
    # Calculate total number of prediction instances (qid -> list_of_doc_ids)
    # This means sum of lengths of all lists of predicted doc_ids
    total_prediction_instances = 0
    # Check if predictions_A7 is not empty and its values are lists
    if isinstance(predictions_A7, dict) and all(isinstance(preds, list) for preds in predictions_A7.values()):
        for qid_preds_list in predictions_A7.values():
            total_prediction_instances += len(qid_preds_list)
    print(f"  Generated/loaded {total_prediction_instances} total prediction doc instances across {len(predictions_A7)} queries for A7.")

    # Display sample predictions
    count_sample_a7 = 0
    print(f"\n  Sample A7 Predictions (Top 3 shown from {TOP_K_PREDS_PARAM} generated per query):")
    for q_id_a7, preds_a7_list in predictions_A7.items():
        print(f"    QID {q_id_a7} Top 3: {preds_a7_list[:3]}")
        count_sample_a7 += 1
        if count_sample_a7 >= 3:
            break
else:
    print("  No predictions generated or loaded for A7.")

gc.collect()
print("\n\n--- Task A7 Processing Complete ---")



--- SECTION A7: BM25+RM3 on Testing Abstracts (using A5 model) ---
DEBUG A7: Using TOP_K_PREDS_PARAM = 100 for generating predictions.

--- Starting Test Prediction Generation for A7_Abstract_Test ---
  Loaded trained BM25 model from /content/drive/MyDrive/colab-4/cache_A5_abstract_bm25_train/bm25_model_A5_abstract.pkl for A7_Abstract_Test.
  Attempting to load cached tokenized corpus for A7_Abstract_Test_Docs from: /content/drive/MyDrive/colab-4/cache_A7_abstract_bm25_test/tokenized_corpus_A7_Abstract_Test_Docs.pkl
    Successfully loaded tokenized corpus for A7_Abstract_Test_Docs (1524039 docs).
  Attempting to load cached IDs/meta for A7_Abstract_Test_Docs from: /content/drive/MyDrive/colab-4/cache_A7_abstract_bm25_test/ids_docs_meta_A7_Abstract_Test_Docs.pkl
    Successfully loaded IDs (1524039) for A7_Abstract_Test_Docs. Queries: 0, Qrels: 0.
  All A7_Abstract_Test_Docs data (corpus & meta) successfully loaded from cache.
  Processing query file: queries_2024-11_test.txt for A7_

BM25+RM3 (A7_Abstract_Test_queries_2024-11_test.txt):   0%|          | 0/99 [00:00<?, ?it/s]

  Saved predictions for A7_Abstract_Test - queries_2024-11_test.txt to /content/drive/MyDrive/colab-4/cache_A7_abstract_bm25_test/predictions_A7_Abstract_Test_queries_2024-11_test.pkl
    Sample Preds (queries_2024-11_test.txt):
      QID 254ecdb5-45e5-45ce-9b3f-492d1e1c085e Top 3: ['111593917', '42897108', '12514859']
      QID eaa374bf-cd61-4d9f-9b99-87f3d81f2636 Top 3: ['208393', '124264082', '136927730']
  Processing query file: queries_2025-01_test.txt for A7_Abstract_Test
  Found query file: 'longeval_sci_testing_2025_abstract/queries_2025-01_test.txt' in longeval_sci_testing_2025_abstract.zip
    → Loaded 492 queries from queries_2025-01_test.txt.
Starting BM25+RM3 (A7_Abstract_Test_queries_2025-01_test.txt) for 492 queries. Qrels available for eval: False


BM25+RM3 (A7_Abstract_Test_queries_2025-01_test.txt):   0%|          | 0/492 [00:00<?, ?it/s]

  Saved predictions for A7_Abstract_Test - queries_2025-01_test.txt to /content/drive/MyDrive/colab-4/cache_A7_abstract_bm25_test/predictions_A7_Abstract_Test_queries_2025-01_test.pkl
    Sample Preds (queries_2025-01_test.txt):
      QID a3915e30-a219-4be5-9681-3f0e3f9a3449 Top 3: ['18138135', '23331082', '40089441']
      QID 93974b6a-fe0a-4637-9192-70c031d184c0 Top 3: ['52435205', '40175133', '74196610']
  Generated/loaded 53900 total prediction doc instances across 539 queries for A7.

  Sample A7 Predictions (Top 3 shown from 100 generated per query):
    QID 254ecdb5-45e5-45ce-9b3f-492d1e1c085e Top 3: ['111593917', '42897108', '12514859']
    QID eaa374bf-cd61-4d9f-9b99-87f3d81f2636 Top 3: ['208393', '124264082', '136927730']
    QID 6de22390-e9d2-4966-bdad-061f596d1766 Top 3: ['17186361', '59092509', '5487913']


--- Task A7 Processing Complete ---


In [ ]:
# ==============================================================================
# CELL 4: TASK A8 - BM25+RM3 ON TESTING FULL-TEXT (using A6 model)
# ==============================================================================
print("\n\n--- SECTION A8: BM25+RM3 on Testing Full-Text (using A6 model) ---")
bm25_model_A6_path = os.path.join(PIPELINE_A6_CACHE_DIR, 'bm25_model_A6_fulltext.pkl') # Model from A6

# For debugging A8 retrieval steps for specific QIDs
# QIDs_TO_DEBUG_A8_LIST = ['your_fulltext_test_qid1'] # Replace with actual test QIDs if needed
QIDs_TO_DEBUG_A8_LIST = None # Set to a list of QIDs to debug, or None to run all

print(f"DEBUG A8: Using TOP_K_PREDS_PARAM = {TOP_K_PREDS_PARAM} for generating predictions.")
print(f"DEBUG A8: Using BM25 model from A6: {bm25_model_A6_path}")
print(f"DEBUG A8: Using test data ZIP: {FULLTEXT_TEST_ZIP_PATH}")
print(f"DEBUG A8: Using cache directory: {PIPELINE_A8_CACHE_DIR}")

predictions_A8 = run_bm25_rm3_test_predictions(
    trained_bm25_model_path=bm25_model_A6_path,
    test_data_zip_path=FULLTEXT_TEST_ZIP_PATH, # Use the full-text test ZIP
    text_extraction_func=extract_and_truncate_full_text_for_bm25, # Use full-text extraction & truncation for BM25
    pipeline_cache_dir=PIPELINE_A8_CACHE_DIR,
    data_type_prefix="A8_FullText_Test",
    k1=BM25_K1_PARAM,
    b=BM25_B_PARAM,
    fb_docs=RM3_FB_DOCS_PARAM,
    fb_terms=RM3_FB_TERMS_PARAM,
    top_k_preds=TOP_K_PREDS_PARAM,
    QIDs_TO_DEBUG_LIST=QIDs_TO_DEBUG_A8_LIST
)

if predictions_A8:
    total_prediction_instances = 0
    if isinstance(predictions_A8, dict) and all(isinstance(preds, list) for preds in predictions_A8.values()):
        for qid_preds_list in predictions_A8.values():
            total_prediction_instances += len(qid_preds_list)
    print(f"  Generated/loaded {total_prediction_instances} total prediction doc instances across {len(predictions_A8)} queries for A8.")

    count_sample_a8 = 0
    print(f"\n  Sample A8 Predictions (Top 3 shown from {TOP_K_PREDS_PARAM} generated per query):")
    for q_id_a8, preds_a8_list in predictions_A8.items():
        print(f"    QID {q_id_a8} Top 3: {preds_a8_list[:3]}")
        count_sample_a8 += 1
        if count_sample_a8 >= 3:
            break
else:
    print("  No predictions generated or loaded for A8.")

gc.collect()
print("\n\n--- Task A8 Processing Complete ---")



--- SECTION A8: BM25+RM3 on Testing Full-Text (using A6 model) ---
DEBUG A8: Using TOP_K_PREDS_PARAM = 100 for generating predictions.
DEBUG A8: Using BM25 model from A6: /content/drive/MyDrive/colab-4/cache_A6_fulltext_bm25_train/bm25_model_A6_fulltext.pkl
DEBUG A8: Using test data ZIP: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_fulltext.zip
DEBUG A8: Using cache directory: /content/drive/MyDrive/colab-4/cache_A8_fulltext_bm25_test/

--- Starting Test Prediction Generation for A8_FullText_Test ---
  Loaded trained BM25 model from /content/drive/MyDrive/colab-4/cache_A6_fulltext_bm25_train/bm25_model_A6_fulltext.pkl for A8_FullText_Test.
  Attempting to load cached tokenized corpus for A8_FullText_Test_Docs from: /content/drive/MyDrive/colab-4/cache_A8_fulltext_bm25_test/tokenized_corpus_A8_FullText_Test_Docs.pkl
    Successfully loaded tokenized corpus for A8_FullText_Test_Docs (1524039 docs).
  Attempting to load cached IDs/meta for A8_FullText_Test_Docs from: /conten

BM25+RM3 (A8_FullText_Test_queries_2024-11_test.txt):   0%|          | 0/99 [00:00<?, ?it/s]

  Saved predictions for A8_FullText_Test - queries_2024-11_test.txt to /content/drive/MyDrive/colab-4/cache_A8_fulltext_bm25_test/predictions_A8_FullText_Test_queries_2024-11_test.pkl
    Sample Preds (queries_2024-11_test.txt):
      QID 254ecdb5-45e5-45ce-9b3f-492d1e1c085e Top 3: ['74472793', '106754341', '136338855']
      QID eaa374bf-cd61-4d9f-9b99-87f3d81f2636 Top 3: ['18159801', '122107348', '46137175']
  Processing query file: queries_2025-01_test.txt for A8_FullText_Test
  Found query file: 'longeval_sci_testing_2025_fulltext/queries_2025-01_test.txt' in longeval_sci_testing_2025_fulltext.zip
    → Loaded 492 queries from queries_2025-01_test.txt.
Starting BM25+RM3 (A8_FullText_Test_queries_2025-01_test.txt) for 492 queries. Qrels available for eval: False


BM25+RM3 (A8_FullText_Test_queries_2025-01_test.txt):   0%|          | 0/492 [00:00<?, ?it/s]

  Saved predictions for A8_FullText_Test - queries_2025-01_test.txt to /content/drive/MyDrive/colab-4/cache_A8_fulltext_bm25_test/predictions_A8_FullText_Test_queries_2025-01_test.pkl
    Sample Preds (queries_2025-01_test.txt):
      QID a3915e30-a219-4be5-9681-3f0e3f9a3449 Top 3: ['453544', '3817549', '160828408']
      QID 93974b6a-fe0a-4637-9192-70c031d184c0 Top 3: ['8254638', '38460182', '53033868']
  Generated/loaded 53900 total prediction doc instances across 539 queries for A8.

  Sample A8 Predictions (Top 3 shown from 100 generated per query):
    QID 254ecdb5-45e5-45ce-9b3f-492d1e1c085e Top 3: ['74472793', '106754341', '136338855']
    QID eaa374bf-cd61-4d9f-9b99-87f3d81f2636 Top 3: ['18159801', '122107348', '46137175']
    QID 6de22390-e9d2-4966-bdad-061f596d1766 Top 3: ['84886604', '125148905', '75338402']


--- Task A8 Processing Complete ---


In [ ]:
# ==============================================================================
# CELL 0: INSTALLATIONS FOR SBERT TASKS
# (Run this section, THEN STOP, MANUALLY RESTART RUNTIME, then run Cell 1 and subsequent task cells)
# ==============================================================================
print("Starting installations for SBERT tasks (if this is the first run of the cell in a new session)...\n")
# Step 1: Install specific PyTorch ecosystem versions
print("Installing PyTorch, Torchvision, Torchaudio...")
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
print("PyTorch ecosystem installation attempt finished.")

# Step 2: Install sentence-transformers and other libraries
print("\nInstalling sentence-transformers and other libraries...")
!pip install sentence-transformers==2.7.0 nltk tqdm scikit-learn pandas -q
# Ensure torch.cuda.amp is available if using older PyTorch, but with 2.3.1 it's built-in.
# No separate install needed for contextlib.
print("Other packages installation attempt finished.")

print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
print("CELL 0 (INSTALLATIONS) HAS COMPLETED BOTH INSTALLATION STEPS.")
print("YOU **MUST** NOW MANUALLY RESTART THE COLAB RUNTIME FOR ALL CHANGES TO TAKE EFFECT.")
print("GO TO THE MENU: Runtime -> Restart runtime...")
print("AFTER RESTARTING, PROCEED TO RUN CELL 1 (Common Setup & Helper Functions for SBERT).")
print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")

Starting installations for SBERT tasks (if this is the first run of the cell in a new session)...

Installing PyTorch, Torchvision, Torchaudio...
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 111.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 78.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 57.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 120.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 44.5 MB/s 

In [ ]:
# ==============================================================================
# CELL 1: COMMON SETUP & HELPER FUNCTIONS FOR SBERT TASKS
# ==============================================================================
print("\n--- SECTION 1: Initializing Common Setup & Helper Functions for SBERT Tasks ---")
from google.colab import drive
import os
import gc
from collections import OrderedDict, defaultdict
import zipfile
import json
import re
import nltk
# from nltk.corpus import stopwords # Less critical for SBERT
# from nltk.stem import WordNetLemmatizer # Less critical for SBERT
from tqdm.notebook import tqdm
import numpy as np
import pickle
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity # Can be used, but util.semantic_search is often more direct
from torch.cuda.amp import autocast # For Mixed Precision
import contextlib # For autocast with CPU fallback

try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Google Drive mounting error: {e}. Please ensure Drive is accessible and authorized.")

# --- Path Definitions ---
DRIVE_BASE_FOLDER = '/content/drive/MyDrive/colab-4/'
ABSTRACT_TRAIN_ZIP_FILENAME = 'longeval_sci_training_2025_abstract.zip'
FULLTEXT_TRAIN_ZIP_FILENAME = 'longeval_sci_training_2025_fulltext.zip'
ABSTRACT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, ABSTRACT_TRAIN_ZIP_FILENAME)
FULLTEXT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_FOLDER, FULLTEXT_TRAIN_ZIP_FILENAME)

# Cache directories for SBERT embeddings (separate from BM25/TF-IDF)
SBERT_EMBEDDINGS_ABSTRACT_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_B1_abstract_sbert_train/')
SBERT_EMBEDDINGS_FULLTEXT_CACHE_DIR = os.path.join(DRIVE_BASE_FOLDER, 'cache_B2_fulltext_sbert_train/')
os.makedirs(SBERT_EMBEDDINGS_ABSTRACT_CACHE_DIR, exist_ok=True)
os.makedirs(SBERT_EMBEDDINGS_FULLTEXT_CACHE_DIR, exist_ok=True)

print("Path definitions complete.")
if not os.path.exists(ABSTRACT_TRAIN_ZIP_PATH): print(f"WARNING: ABSTRACT_TRAIN_ZIP_PATH not found: {ABSTRACT_TRAIN_ZIP_PATH}")
if not os.path.exists(FULLTEXT_TRAIN_ZIP_PATH): print(f"WARNING: FULLTEXT_TRAIN_ZIP_PATH not found: {FULLTEXT_TRAIN_ZIP_PATH}")

# NLTK (Optional for SBERT, but good to have for consistency if other methods use it)
# nltk_resources_ready_sbert = False
# try:
#     stopwords.words('english'); WordNetLemmatizer().lemmatize('cats') # Example check
#     nltk_resources_ready_sbert = True; print("NLTK resources seem available.")
# except:
#     print("Downloading NLTK (stopwords, wordnet) for SBERT text functions (if needed)...")
#     try: nltk.download('stopwords',quiet=True); nltk.download('wordnet',quiet=True); nltk_resources_ready_sbert=True
#     except Exception as e: print(f"NLTK download error: {e}")
# if nltk_resources_ready_sbert: STOP_SBERT = set(stopwords.words("english")); LEM_SBERT = WordNetLemmatizer()
# else: STOP_SBERT, LEM_SBERT = set(), None; print("Warning: NLTK not fully ready for SBERT text functions.")
# _token_pat_sbert = re.compile(r"[A-Za-z]{2,}")

def clean_text_for_sbert(text: str) -> str: # SBERT generally prefers raw or minimally processed text
    if not isinstance(text, str): return ""
    # Basic cleaning: lowercase and strip whitespace. Some models are case-sensitive, check model card.
    # For models like all-MiniLM-L6-v2, lowercasing is generally fine.
    return text.lower().strip()

def extract_text_from_abstract_json_for_sbert(record_dict):
    title = record_dict.get("title", "") or ""
    abstract_content = record_dict.get("abstract", "") or ""
    # SBERT often benefits from meaningful concatenations
    return f"{title}. {abstract_content}".strip() # Adding a period for sentence separation

MAX_WORDS_FOR_SBERT_FULLTEXT = 400 # Or another value based on model's max sequence length and your strategy
def extract_and_truncate_full_text_for_sbert(record_dict):
    title = record_dict.get("title", "") or ""
    abstract = record_dict.get("abstract", "") or ""
    main_text = ""; possible_fields = ["text", "full_text", "body", "body_text", "content", "fulltext"]
    for field in possible_fields:
        content = record_dict.get(field)
        if content and isinstance(content, str): main_text = content; break
    # Meaningful concatenation for SBERT
    combined_text = f"{title}. {abstract}. {main_text}".strip() # Add periods for sentence boundary hints
    words = combined_text.split() # Simple word split for truncation
    truncated_text = " ".join(words[:MAX_WORDS_FOR_SBERT_FULLTEXT])
    return clean_text_for_sbert(truncated_text) # Apply minimal SBERT cleaning

def load_sbert_data(data_zip_path, text_extraction_func, pipeline_cache_dir, data_type_prefix):
    ids, texts, queries, qrels = [], [], OrderedDict(), defaultdict(set)
    loaded_from_cache = False

    # Define cache path for this specific dataset (e.g., B1_abstract_loaded_data.pkl)
    loaded_data_cache = os.path.join(pipeline_cache_dir, f'loaded_data_{data_type_prefix}.pkl')

    if os.path.exists(loaded_data_cache):
        print(f"  Attempting to load cached data for {data_type_prefix} from: {loaded_data_cache}")
        try:
            with open(loaded_data_cache, 'rb') as f: cached = pickle.load(f)
            ids, texts, queries, qrels = cached['ids'], cached['texts'], cached['queries'], cached['qrels']
            if ids and texts and queries and (len(ids) == len(texts)): # Basic check
                print(f"    Successfully loaded {len(ids)} docs & associated data for {data_type_prefix} from cache.")
                loaded_from_cache = True
            else: ids, texts, queries, qrels = [], [], OrderedDict(), defaultdict(set) # Reset
        except Exception as e: print(f"    Error loading data cache for {data_type_prefix}: {e}"); ids, texts, queries, qrels = [], [], OrderedDict(), defaultdict(set)

    if not loaded_from_cache:
        if not os.path.exists(data_zip_path): print(f"  ERROR: Data ZIP not found for {data_type_prefix}: {data_zip_path}"); return [],[],OrderedDict(),defaultdict(set),False
        print(f"  --- Loading & Processing {data_type_prefix} Data from: {os.path.basename(data_zip_path)} ---")
        with zipfile.ZipFile(data_zip_path, 'r') as z:
            entries = z.namelist(); zip_root = entries[0].split('/')[0] if entries and '/' in entries[0] else ""
            jsonl_path_prefix = os.path.join(zip_root, 'documents/') if zip_root else 'documents/'
            query_file_path = os.path.join(zip_root, 'queries.txt') if zip_root else 'queries.txt'
            qrels_file_path = os.path.join(zip_root, 'qrels.txt') if zip_root else 'qrels.txt'

            jsonl_files = [info for info in z.infolist() if info.filename.lower().startswith(jsonl_path_prefix.lower()) and info.filename.lower().endswith(".jsonl") and not info.is_dir()]
            print(f"    Found {len(jsonl_files)} JSONL files in '{jsonl_path_prefix}'.")
            for info in tqdm(jsonl_files, desc=f"Reading {data_type_prefix} JSONLs", unit="file"):
                with z.open(info) as f_jsonl:
                    for line_byte in f_jsonl:
                        try:
                            record = json.loads(line_byte.decode('utf-8')); doc_id = str(record.get("id")); raw_text = text_extraction_func(record)
                            if doc_id and raw_text: ids.append(doc_id); texts.append(raw_text) # raw_text is already cleaned by text_extraction_func
                        except: pass # Optionally log errors
            print(f"    Total {data_type_prefix} docs processed: {len(texts)}")
            if query_file_path in z.namelist():
                with z.open(query_file_path) as f_q: queries.update(OrderedDict(l.decode('utf-8',errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8',errors='ignore').strip()))
                print(f"    Loaded {len(queries)} queries.")
            if qrels_file_path in z.namelist():
                with z.open(qrels_file_path) as f_r:
                    for l in f_r: parts=l.decode('utf-8',errors='ignore').strip().split(); qrels[str(parts[0])].add(str(parts[2])) if len(parts)>=3 and parts[-1].isdigit() and int(parts[-1])>0 else None
                print(f"    Loaded {len(qrels)} queries with qrels.")
            if ids and texts and queries:
                with open(loaded_data_cache, 'wb') as f: pickle.dump({'ids':ids, 'texts':texts, 'queries':queries, 'qrels':qrels}, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved loaded data for {data_type_prefix} to cache.")
                loaded_from_cache = True # Mark as successfully loaded and cached

    data_valid = bool(ids and texts and queries and (len(ids) == len(texts)))
    if not data_valid: print(f"  ERROR: {data_type_prefix} data loading or processing resulted in inconsistent or empty lists.")
    return ids, texts, queries, qrels, data_valid

def evaluate_sbert_retrieval(doc_ids_list, query_embeddings_dict, doc_embeddings_matrix, qrels_dict, top_k=5, similarity_fn=util.semantic_search):
    all_preds = OrderedDict(); eval_metrics = {'P':[], 'R':[], 'AP':[]}
    print(f"Evaluating SBERT retrieval for {len(query_embeddings_dict)} queries at Top-{top_k}...")

    # Prepare doc_embeddings_matrix for util.semantic_search if it's not already a tensor
    if not torch.is_tensor(doc_embeddings_matrix) and isinstance(doc_embeddings_matrix, np.ndarray):
        doc_embeddings_tensor = torch.tensor(doc_embeddings_matrix, dtype=torch.float32)
        if torch.cuda.is_available():
            doc_embeddings_tensor = doc_embeddings_tensor.to('cuda')
    else:
        doc_embeddings_tensor = doc_embeddings_matrix


    for qid, query_embedding in tqdm(query_embeddings_dict.items(), desc="SBERT Evaluating"):
        if not torch.is_tensor(query_embedding) and isinstance(query_embedding, np.ndarray):
            query_tensor = torch.tensor(query_embedding, dtype=torch.float32).unsqueeze(0) # Add batch dim
            if torch.cuda.is_available():
                query_tensor = query_tensor.to('cuda')
        else: # Assuming it's already a tensor (potentially on GPU)
            query_tensor = query_embedding.unsqueeze(0) if query_embedding.ndim == 1 else query_embedding


        hits_semantic = similarity_fn(query_tensor, doc_embeddings_tensor, top_k=top_k)

        pred_ids_for_qid = []
        if hits_semantic and len(hits_semantic) > 0 and len(hits_semantic[0]) > 0:
            pred_ids_for_qid = [doc_ids_list[hit['corpus_id']] for hit in hits_semantic[0]]
        all_preds[qid] = pred_ids_for_qid

        gold_docs = qrels_dict.get(qid, set())
        n_gold = len(gold_docs)

        num_hits = 0
        if gold_docs: # Only calculate hits if there are gold documents
            for doc_id_pred in pred_ids_for_qid:
                if doc_id_pred in gold_docs:
                    num_hits += 1

        eval_metrics['P'].append(num_hits / top_k if top_k > 0 else 0.0)
        eval_metrics['R'].append(num_hits / n_gold if n_gold > 0 else (0.0 if num_hits == 0 else 1.0))

        if n_gold > 0:
            ap_score_q, current_hits_count = 0.0, 0
            for i, doc_id_pred in enumerate(pred_ids_for_qid):
                if doc_id_pred in gold_docs:
                    current_hits_count += 1
                    ap_score_q += current_hits_count / (i + 1)
            eval_metrics['AP'].append(ap_score_q / min(n_gold, top_k) if min(n_gold, top_k) > 0 else 0.0)
        else:
            eval_metrics['AP'].append(0.0)

    final_results = {
        f'P@{top_k}': np.mean(eval_metrics['P']) if eval_metrics['P'] else 0.0,
        f'R@{top_k}': np.mean(eval_metrics['R']) if eval_metrics['R'] else 0.0,
        f'MAP@{top_k}': np.mean(eval_metrics['AP']) if eval_metrics['AP'] else 0.0
    }
    return final_results, all_preds

print("Common Setup for SBERT tasks complete.")


--- SECTION 1: Initializing Common Setup & Helper Functions for SBERT Tasks ---
Mounted at /content/drive
Google Drive mounted successfully.
Path definitions complete.
Common Setup for SBERT tasks complete.


In [ ]:
# ==============================================================================
# CELL 2: TASK B1 - SBERT on Training Abstracts
# ==============================================================================
print("\n\n--- TASK B1: SBERT on Training Abstracts ---")

# --- Configuration for B1 ---
SBERT_MODEL_NAME_B1 = 'all-MiniLM-L6-v2'  # You can change this
ENCODE_BATCH_SIZE_B1 = 64 # Adjust based on GPU memory (e.g., 32, 64, 128)
TOP_K_EVAL_B1 = 5

# --- Load Data for B1 ---
ids_B1, texts_B1, queries_B1_dict, qrels_B1_dict, data_loaded_B1 = load_sbert_data(
    ABSTRACT_TRAIN_ZIP_PATH,
    extract_text_from_abstract_json_for_sbert, # Uses SBERT specific text extraction
    SBERT_EMBEDDINGS_ABSTRACT_CACHE_DIR, # Cache for loaded data
    "B1_Abstracts"
)

doc_embeddings_B1 = np.array([])
model_sbert_B1 = None

if data_loaded_B1:
    print(f"Data loaded for B1: {len(ids_B1)} documents.")
    doc_embeddings_file_B1 = os.path.join(SBERT_EMBEDDINGS_ABSTRACT_CACHE_DIR, f'doc_embeddings_{SBERT_MODEL_NAME_B1.replace("/","_")}.pkl')

    try:
        print(f"Loading SBERT model: {SBERT_MODEL_NAME_B1}...")
        model_sbert_B1 = SentenceTransformer(SBERT_MODEL_NAME_B1)
        if torch.cuda.is_available(): model_sbert_B1.to(torch.device("cuda")); print("  SBERT Model B1 moved to GPU.")
        else: print("  WARNING: GPU not available for SBERT Model B1. CPU will be slow.")
    except Exception as e: print(f"Error loading SBERT model {SBERT_MODEL_NAME_B1}: {e}"); model_sbert_B1 = None

    if model_sbert_B1:
        if os.path.exists(doc_embeddings_file_B1):
            print(f"  Loading cached document embeddings for B1 from: {doc_embeddings_file_B1}")
            try:
                with open(doc_embeddings_file_B1, 'rb') as f: doc_embeddings_B1 = pickle.load(f)
                if not (isinstance(doc_embeddings_B1, np.ndarray) and \
                        doc_embeddings_B1.shape[0] == len(ids_B1) and \
                        doc_embeddings_B1.shape[1] == model_sbert_B1.get_sentence_embedding_dimension()):
                    print("  Cached embeddings mismatch or invalid. Regenerating."); doc_embeddings_B1 = np.array([])
                else: print(f"    Loaded {doc_embeddings_B1.shape[0]} embeddings, shape {doc_embeddings_B1.shape}.")
            except Exception as e: print(f"  Error loading cached embeddings: {e}. Regenerating."); doc_embeddings_B1 = np.array([])

        if doc_embeddings_B1.size == 0:
            print(f"  Generating document embeddings for {len(texts_B1)} abstracts using {SBERT_MODEL_NAME_B1}...")
            with autocast() if torch.cuda.is_available() else contextlib.nullcontext():
                doc_embeddings_B1 = model_sbert_B1.encode(texts_B1, show_progress_bar=True, batch_size=ENCODE_BATCH_SIZE_B1, convert_to_numpy=True)
            print(f"    Generated abstract embeddings. Shape: {doc_embeddings_B1.shape}")
            try:
                with open(doc_embeddings_file_B1, 'wb') as f: pickle.dump(doc_embeddings_B1, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved abstract embeddings to: {doc_embeddings_file_B1}")
            except Exception as e: print(f"    Error saving abstract embeddings: {e}")

        # Encode Queries for B1
        if queries_B1_dict and doc_embeddings_B1.size > 0:
            print(f"  Encoding {len(queries_B1_dict)} queries for B1...")
            query_texts_B1 = [clean_text_for_sbert(q_text) for q_text in queries_B1_dict.values()]
            with autocast() if torch.cuda.is_available() else contextlib.nullcontext():
                query_embeddings_B1_all = model_sbert_B1.encode(query_texts_B1, show_progress_bar=True, batch_size=ENCODE_BATCH_SIZE_B1, convert_to_numpy=True)

            query_embeddings_B1_dict = {qid: emb for qid, emb in zip(queries_B1_dict.keys(), query_embeddings_B1_all)}
            print(f"    Query embeddings generated. Shape of one: {query_embeddings_B1_all[0].shape if len(query_embeddings_B1_all)>0 else 'N/A'}")

            # Perform Retrieval and Evaluation for B1
            results_B1, predictions_B1 = evaluate_sbert_retrieval(ids_B1, query_embeddings_B1_dict, doc_embeddings_B1, qrels_B1_dict, top_k=TOP_K_EVAL_B1)
            print(f"\n--- B1 SBERT ({SBERT_MODEL_NAME_B1}) on Abstracts - Results ---")
            for metric, value in results_B1.items(): print(f"  {metric}: {value:.4f}")

            # Save predictions for B1
            predictions_B1_path = os.path.join(SBERT_EMBEDDINGS_ABSTRACT_CACHE_DIR, f'predictions_B1_{SBERT_MODEL_NAME_B1.replace("/","_")}.pkl')
            with open(predictions_B1_path, 'wb') as f: pickle.dump(predictions_B1, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"  Saved B1 predictions to {predictions_B1_path}")

            count_sample_b1 = 0
            print(f"\n  Sample B1 Predictions (Top {TOP_K_EVAL_B1}):")
            for q_id_b1, preds_b1_list in predictions_B1.items():
                print(f"    QID {q_id_b1} Top {min(3, TOP_K_EVAL_B1)}: {preds_b1_list[:3]}")
                count_sample_b1 += 1
                if count_sample_b1 >= 3: break
        else:
            print("  Skipping B1 retrieval/evaluation: Queries or document embeddings not ready.")
    else:
        print("  Skipping B1: SBERT model could not be loaded.")
else:
    print("Skipping B1: Data loading failed.")
gc.collect()
print("\n--- Task B1 Complete ---")



--- TASK B1: SBERT on Training Abstracts ---
  --- Loading & Processing B1_Abstracts Data from: longeval_sci_training_2025_abstract.zip ---
    Found 21 JSONL files in 'longeval_sci_training_2025_abstract/documents/'.


Reading B1_Abstracts JSONLs:   0%|          | 0/21 [00:00<?, ?file/s]

    Total B1_Abstracts docs processed: 2014265
    Loaded 393 queries.
    Loaded 393 queries with qrels.
    Saved loaded data for B1_Abstracts to cache.
Data loaded for B1: 2014265 documents.
Loading SBERT model: all-MiniLM-L6-v2...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

    Generated abstract embeddings. Shape: (2014265, 384)
    Saved abstract embeddings to: /content/drive/MyDrive/colab-4/cache_B1_abstract_sbert_train/doc_embeddings_all-MiniLM-L6-v2.pkl
  Encoding 393 queries for B1...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

    Query embeddings generated. Shape of one: (384,)
Evaluating SBERT retrieval for 393 queries at Top-5...


SBERT Evaluating:   0%|          | 0/393 [00:00<?, ?it/s]


--- B1 SBERT (all-MiniLM-L6-v2) on Abstracts - Results ---
  P@5: 0.0529
  R@5: 0.0308
  MAP@5: 0.0354
  Saved B1 predictions to /content/drive/MyDrive/colab-4/cache_B1_abstract_sbert_train/predictions_B1_all-MiniLM-L6-v2.pkl

  Sample B1 Predictions (Top 5):
    QID ce5bfacf-8652-4bc1-a5b0-6144a917fb1c Top 3: ['1251432', '134912405', '77362495']
    QID 71657c3b-4112-49d5-86cb-705325aeb06e Top 3: ['6046613', '46771511', '78227636']
    QID f60d94de-b903-48da-b13f-a8334c70f949 Top 3: ['63395091', '128137052', '70372761']

--- Task B1 Complete ---


In [ ]:
# ==============================================================================
# CELL 3: TASK B2 - SBERT on Training Full-Text
# ==============================================================================
print("\n\n--- TASK B2: SBERT on Training Full-Text (Truncated) ---")

# --- Configuration for B2 ---
SBERT_MODEL_NAME_B2 = 'all-MiniLM-L6-v2'  # Can be same as B1 or different
ENCODE_BATCH_SIZE_B2 = 64 # Adjust based on GPU memory
TOP_K_EVAL_B2 = 5
# MAX_WORDS_FOR_SBERT_FULLTEXT is defined in Cell 1 helper functions

# --- Load Data for B2 ---
ids_B2, texts_B2, queries_B2_dict, qrels_B2_dict, data_loaded_B2 = load_sbert_data(
    FULLTEXT_TRAIN_ZIP_PATH, # Use the full-text ZIP
    extract_and_truncate_full_text_for_sbert, # Uses SBERT specific full-text extraction & truncation
    SBERT_EMBEDDINGS_FULLTEXT_CACHE_DIR, # Cache for loaded data for B2
    "B2_FullText"
)

doc_embeddings_B2 = np.array([])
model_sbert_B2 = None

if data_loaded_B2:
    print(f"Data loaded for B2: {len(ids_B2)} documents (truncated full-text).")
    doc_embeddings_file_B2 = os.path.join(SBERT_EMBEDDINGS_FULLTEXT_CACHE_DIR, f'doc_embeddings_{SBERT_MODEL_NAME_B2.replace("/","_")}_max{MAX_WORDS_FOR_SBERT_FULLTEXT}w.pkl')

    try:
        print(f"Loading SBERT model: {SBERT_MODEL_NAME_B2} for B2...")
        model_sbert_B2 = SentenceTransformer(SBERT_MODEL_NAME_B2)
        if torch.cuda.is_available(): model_sbert_B2.to(torch.device("cuda")); print("  SBERT Model B2 moved to GPU.")
        else: print("  WARNING: GPU not available for SBERT Model B2. CPU will be slow.")
    except Exception as e: print(f"Error loading SBERT model {SBERT_MODEL_NAME_B2} for B2: {e}"); model_sbert_B2 = None

    if model_sbert_B2:
        if os.path.exists(doc_embeddings_file_B2):
            print(f"  Loading cached document embeddings for B2 from: {doc_embeddings_file_B2}")
            try:
                with open(doc_embeddings_file_B2, 'rb') as f: doc_embeddings_B2 = pickle.load(f)
                if not (isinstance(doc_embeddings_B2, np.ndarray) and \
                        doc_embeddings_B2.shape[0] == len(ids_B2) and \
                        doc_embeddings_B2.shape[1] == model_sbert_B2.get_sentence_embedding_dimension()):
                    print("  Cached embeddings mismatch or invalid for B2. Regenerating."); doc_embeddings_B2 = np.array([])
                else: print(f"    Loaded {doc_embeddings_B2.shape[0]} embeddings for B2, shape {doc_embeddings_B2.shape}.")
            except Exception as e: print(f"  Error loading cached embeddings for B2: {e}. Regenerating."); doc_embeddings_B2 = np.array([])

        if doc_embeddings_B2.size == 0:
            print(f"  Generating document embeddings for {len(texts_B2)} (truncated) full-texts using {SBERT_MODEL_NAME_B2}...")
            with autocast() if torch.cuda.is_available() else contextlib.nullcontext():
                doc_embeddings_B2 = model_sbert_B2.encode(texts_B2, show_progress_bar=True, batch_size=ENCODE_BATCH_SIZE_B2, convert_to_numpy=True)
            print(f"    Generated full-text embeddings. Shape: {doc_embeddings_B2.shape}")
            try:
                with open(doc_embeddings_file_B2, 'wb') as f: pickle.dump(doc_embeddings_B2, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved full-text embeddings to: {doc_embeddings_file_B2}")
            except Exception as e: print(f"    Error saving full-text embeddings: {e}")

        # Encode Queries for B2 (assuming same queries as B1 for this example, adjust if needed)
        # If queries for full-text are different, load them specifically
        if queries_B2_dict and doc_embeddings_B2.size > 0:
            print(f"  Encoding {len(queries_B2_dict)} queries for B2...")
            query_texts_B2 = [clean_text_for_sbert(q_text) for q_text in queries_B2_dict.values()]
            with autocast() if torch.cuda.is_available() else contextlib.nullcontext():
                query_embeddings_B2_all = model_sbert_B2.encode(query_texts_B2, show_progress_bar=True, batch_size=ENCODE_BATCH_SIZE_B2, convert_to_numpy=True)

            query_embeddings_B2_dict = {qid: emb for qid, emb in zip(queries_B2_dict.keys(), query_embeddings_B2_all)}
            print(f"    Query embeddings generated for B2. Shape of one: {query_embeddings_B2_all[0].shape if len(query_embeddings_B2_all)>0 else 'N/A'}")

            # Perform Retrieval and Evaluation for B2
            results_B2, predictions_B2 = evaluate_sbert_retrieval(ids_B2, query_embeddings_B2_dict, doc_embeddings_B2, qrels_B2_dict, top_k=TOP_K_EVAL_B2)
            print(f"\n--- B2 SBERT ({SBERT_MODEL_NAME_B2}) on Full-Text (Truncated) - Results ---")
            for metric, value in results_B2.items(): print(f"  {metric}: {value:.4f}")

            # Save predictions for B2
            predictions_B2_path = os.path.join(SBERT_EMBEDDINGS_FULLTEXT_CACHE_DIR, f'predictions_B2_{SBERT_MODEL_NAME_B2.replace("/","_")}_max{MAX_WORDS_FOR_SBERT_FULLTEXT}w.pkl')
            with open(predictions_B2_path, 'wb') as f: pickle.dump(predictions_B2, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"  Saved B2 predictions to {predictions_B2_path}")

            count_sample_b2 = 0
            print(f"\n  Sample B2 Predictions (Top {TOP_K_EVAL_B2}):")
            for q_id_b2, preds_b2_list in predictions_B2.items():
                print(f"    QID {q_id_b2} Top {min(3, TOP_K_EVAL_B2)}: {preds_b2_list[:3]}")
                count_sample_b2 += 1
                if count_sample_b2 >= 3: break
        else:
            print("  Skipping B2 retrieval/evaluation: Queries or document embeddings not ready.")
    else:
        print("  Skipping B2: SBERT model could not be loaded.")
else:
    print("Skipping B2: Data loading failed.")
gc.collect()
print("\n--- Task B2 Complete ---")



--- TASK B2: SBERT on Training Full-Text (Truncated) ---
  --- Loading & Processing B2_FullText Data from: longeval_sci_training_2025_fulltext.zip ---
    Found 21 JSONL files in 'longeval_sci_training_2025_fulltext/documents/'.


Reading B2_FullText JSONLs:   0%|          | 0/21 [00:00<?, ?file/s]

    Total B2_FullText docs processed: 2014265
    Loaded 393 queries.
    Loaded 393 queries with qrels.
    Saved loaded data for B2_FullText to cache.
Data loaded for B2: 2014265 documents (truncated full-text).
Loading SBERT model: all-MiniLM-L6-v2 for B2...
  SBERT Model B2 moved to GPU.
  Generating document embeddings for 2014265 (truncated) full-texts using all-MiniLM-L6-v2...


Batches:   0%|          | 0/31473 [00:00<?, ?it/s]

    Generated full-text embeddings. Shape: (2014265, 384)
    Saved full-text embeddings to: /content/drive/MyDrive/colab-4/cache_B2_fulltext_sbert_train/doc_embeddings_all-MiniLM-L6-v2_max400w.pkl
  Encoding 393 queries for B2...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

    Query embeddings generated for B2. Shape of one: (384,)
Evaluating SBERT retrieval for 393 queries at Top-5...


SBERT Evaluating:   0%|          | 0/393 [00:00<?, ?it/s]


--- B2 SBERT (all-MiniLM-L6-v2) on Full-Text (Truncated) - Results ---
  P@5: 0.0580
  R@5: 0.0337
  MAP@5: 0.0365
  Saved B2 predictions to /content/drive/MyDrive/colab-4/cache_B2_fulltext_sbert_train/predictions_B2_all-MiniLM-L6-v2_max400w.pkl

  Sample B2 Predictions (Top 5):
    QID ce5bfacf-8652-4bc1-a5b0-6144a917fb1c Top 3: ['1251432', '134912405', '77362495']
    QID 71657c3b-4112-49d5-86cb-705325aeb06e Top 3: ['6046613', '46771511', '155492206']
    QID f60d94de-b903-48da-b13f-a8334c70f949 Top 3: ['63395091', '51408020', '126710811']

--- Task B2 Complete ---


In [ ]:
# ==============================================================================
# CELL: TASK B3 - SBERT on Testing Abstracts
# ==============================================================================
print("\n\n--- TASK B3: SBERT on Testing Abstracts ---")

# --- Configuration for B3 ---
SBERT_MODEL_NAME_B3 = 'all-MiniLM-L6-v2' # Or your chosen SBERT model
ABSTRACT_TEST_ZIP_PATH_B3 = ABSTRACT_TEST_ZIP_PATH # Defined in your global config
PIPELINE_B3_CACHE_DIR = "/content/drive/MyDrive/colab-4/cache_B3_abstract_sbert_test/" # NEW cache directory
os.makedirs(PIPELINE_B3_CACHE_DIR, exist_ok=True)

SBERT_EMBEDDING_BATCH_SIZE_B3 = 128 # Adjust based on GPU memory
SBERT_TOP_K_PREDS_B3 = 100 # Number of predictions to save per query

# --- Load SBERT Model ---
print(f"  Loading SBERT model: {SBERT_MODEL_NAME_B3}")
# Ensure sbert_model_global is defined from your SBERT common setup, or load it here
if 'sbert_model_global' not in globals() or sbert_model_global.model_name_or_path != SBERT_MODEL_NAME_B3:
    sbert_model_b3 = SentenceTransformer(SBERT_MODEL_NAME_B3)
    if torch.cuda.is_available():
        sbert_model_b3 = sbert_model_b3.to(torch.device("cuda"))
    print(f"    SBERT model '{SBERT_MODEL_NAME_B3}' loaded. Device: {sbert_model_b3.device}")
else:
    sbert_model_b3 = sbert_model_global
    print(f"    Using pre-loaded SBERT model '{sbert_model_b3.model_name_or_path}'. Device: {sbert_model_b3.device}")


# --- Load Test Documents (Abstracts) ---
# Using extract_text_from_abstract_json_for_sbert from your SBERT common setup
test_doc_ids_b3, test_doc_texts_b3 = load_test_documents_for_sbert(
    ABSTRACT_TEST_ZIP_PATH_B3,
    extract_text_from_abstract_json_for_sbert, # Specific for abstracts
    PIPELINE_B3_CACHE_DIR,
    "B3_Abstract_Test_Docs"
)

# --- Load Test Queries ---
# Assuming TEST_QUERY_FILES_LIST = ["queries_2024-11_test.txt", "queries_2025-01_test.txt"] is defined globally
all_test_queries_b3 = OrderedDict()
for query_file_name in TEST_QUERY_FILES_LIST:
    queries_for_file = load_test_queries_from_zip_corrected(ABSTRACT_TEST_ZIP_PATH_B3, query_file_name)
    all_test_queries_b3.update(queries_for_file)
print(f"  Loaded a total of {len(all_test_queries_b3)} test queries for B3.")

# --- Encode Test Documents (Abstracts) ---
corpus_embeddings_b3_path = os.path.join(PIPELINE_B3_CACHE_DIR, f'corpus_embeddings_b3_{SBERT_MODEL_NAME_B3.replace("/","_")}.pkl')
if os.path.exists(corpus_embeddings_b3_path) and test_doc_texts_b3:
    print(f"  Loading cached corpus embeddings for B3 from {corpus_embeddings_b3_path}")
    with open(corpus_embeddings_b3_path, 'rb') as f: test_corpus_embeddings_b3 = pickle.load(f)
    # Sanity check
    if len(test_corpus_embeddings_b3) != len(test_doc_texts_b3):
        print(f"    Cache mismatch for B3 corpus embeddings ({len(test_corpus_embeddings_b3)}) vs docs ({len(test_doc_texts_b3)}). Re-encoding.")
        test_corpus_embeddings_b3 = None
    else:
        print(f"    Loaded {len(test_corpus_embeddings_b3)} embeddings.")
else:
    test_corpus_embeddings_b3 = None

if test_corpus_embeddings_b3 is None and test_doc_texts_b3:
    print(f"  Encoding {len(test_doc_texts_b3)} test abstract documents for B3 using {SBERT_MODEL_NAME_B3}...")
    # Clean texts before encoding if necessary (your sbert helper might do this)
    cleaned_test_doc_texts_b3 = [clean_text_for_sbert(text) for text in test_doc_texts_b3]
    test_corpus_embeddings_b3 = sbert_model_b3.encode(
        cleaned_test_doc_texts_b3,
        batch_size=SBERT_EMBEDDING_BATCH_SIZE_B3,
        show_progress_bar=True,
        convert_to_tensor=True
    )
    if test_corpus_embeddings_b3 is not None: # Check if encoding was successful
        with open(corpus_embeddings_b3_path, 'wb') as f: pickle.dump(test_corpus_embeddings_b3, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"    Saved B3 corpus embeddings to {corpus_embeddings_b3_path}")
elif not test_doc_texts_b3:
    print("ERROR: No test documents loaded for B3. Cannot encode.")

# --- Generate Predictions for Test Queries ---
sbert_predictions_b3 = OrderedDict()
predictions_b3_cache_path = os.path.join(PIPELINE_B3_CACHE_DIR, f'predictions_b3_{SBERT_MODEL_NAME_B3.replace("/","_")}_top{SBERT_TOP_K_PREDS_B3}.pkl')

if os.path.exists(predictions_b3_cache_path):
    print(f"  Loading cached B3 predictions from: {predictions_b3_cache_path}")
    try:
        with open(predictions_b3_cache_path, 'rb') as f: sbert_predictions_b3 = pickle.load(f)
        print(f"    Loaded {len(sbert_predictions_b3)} query predictions for B3.")
    except Exception as e_load_pred:
        print(f"    Error loading cached B3 predictions: {e_load_pred}. Regenerating.")
        sbert_predictions_b3 = OrderedDict()

if not sbert_predictions_b3 and all_test_queries_b3 and test_corpus_embeddings_b3 is not None and test_doc_ids_b3:
    print(f"  Generating SBERT predictions for {len(all_test_queries_b3)} test queries (B3)...")

    # Prepare query texts and encode them
    query_texts_b3 = [clean_text_for_sbert(q_text) for q_text in all_test_queries_b3.values()]
    query_embeddings_b3 = sbert_model_b3.encode(
        query_texts_b3,
        batch_size=SBERT_EMBEDDING_BATCH_SIZE_B3,
        show_progress_bar=True,
        convert_to_tensor=True
    )

    # Perform semantic search for all queries at once
    # util.semantic_search returns a list of lists of hits for each query
    all_hits = util.semantic_search(query_embeddings_b3, test_corpus_embeddings_b3, top_k=SBERT_TOP_K_PREDS_B3)

    query_ids_list_b3 = list(all_test_queries_b3.keys())
    for i, qid in enumerate(tqdm(query_ids_list_b3, desc="Processing B3 SBERT Results")):
        hits = all_hits[i]
        sbert_predictions_b3[qid] = [test_doc_ids_b3[hit['corpus_id']] for hit in hits]
        # Optionally, can also store scores: # [ (test_doc_ids_b3[hit['corpus_id']], hit['score']) for hit in hits]

    if sbert_predictions_b3:
        with open(predictions_b3_cache_path, 'wb') as f: pickle.dump(sbert_predictions_b3, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  Saved B3 SBERT predictions to {predictions_b3_cache_path}")

elif sbert_predictions_b3:
     print(f"  B3 SBERT predictions already loaded from cache.")
else:
    print("ERROR: Cannot generate B3 predictions. Missing queries, document embeddings, or document IDs.")

# Display sample predictions for B3
if sbert_predictions_b3:
    print(f"\n  Sample B3 SBERT Predictions (Top 3 of {SBERT_TOP_K_PREDS_B3} shown):")
    count_sample_b3 = 0
    for q_id, preds_list in sbert_predictions_b3.items():
        print(f"    QID {q_id} Top 3: {preds_list[:3]}")
        count_sample_b3 += 1
        if count_sample_b3 >= 3: break
else:
    print("  No B3 SBERT predictions available to display.")

gc.collect()
print("\n--- Task B3 Complete ---")



--- TASK B3: SBERT on Testing Abstracts ---
  Loading SBERT model: all-MiniLM-L6-v2


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

Batches:   0%|          | 0/11907 [00:00<?, ?it/s]

    Saved B3 corpus embeddings to /content/drive/MyDrive/colab-4/cache_B3_abstract_sbert_test/corpus_embeddings_b3_all-MiniLM-L6-v2.pkl
  Generating SBERT predictions for 552 test queries (B3)...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Processing B3 SBERT Results:   0%|          | 0/552 [00:00<?, ?it/s]

  Saved B3 SBERT predictions to /content/drive/MyDrive/colab-4/cache_B3_abstract_sbert_test/predictions_b3_all-MiniLM-L6-v2_top100.pkl

  Sample B3 SBERT Predictions (Top 3 of 100 shown):
    QID d585f080-4519-4952-8278-0d13fcd03fec Top 3: ['70339957', '70339957', '117725589']
    QID 254ecdb5-45e5-45ce-9b3f-492d1e1c085e Top 3: ['20487814', '72181450', '25027878']
    QID eaa374bf-cd61-4d9f-9b99-87f3d81f2636 Top 3: ['122525857', '103917394', '126134288']

--- Task B3 Complete ---


In [ ]:
# ==============================================================================
# CELL: TASK B4 - SBERT on Testing Full-Text
# ==============================================================================
print("\n\n--- TASK B4: SBERT on Testing Full-Text ---")

# --- Configuration for B4 ---
# Using global configurations from Step 0 where possible
SBERT_MODEL_NAME_B4 = SBERT_MODEL_NAME_DEFAULT
FULLTEXT_TEST_ZIP_PATH_B4 = FULLTEXT_TEST_ZIP_PATH
# PIPELINE_B4_CACHE_DIR is already defined in Step 0 and directory created
# os.makedirs(PIPELINE_B4_CACHE_DIR, exist_ok=True) # Already done in Step 0

SBERT_EMBEDDING_BATCH_SIZE_B4 = SBERT_EMBEDDING_BATCH_SIZE # From Step 0
SBERT_TOP_K_PREDS_B4 = TOP_K_PREDS_PARAM # From Step 0 (e.g., 100)
# MAX_WORDS_FOR_SBERT_FULLTEXT is defined in Step 0

# --- Load SBERT Model ---
print(f"  Loading SBERT model: {SBERT_MODEL_NAME_B4}")
# Attempt to reuse a globally loaded model if available and matches, otherwise load fresh.
# This assumes 'sbert_model_global' might be loaded in an SBERT common setup cell.
# If not, it will always load here.
if 'sbert_model_global' in globals() and hasattr(sbert_model_global, 'model_name_or_path') and sbert_model_global.model_name_or_path == SBERT_MODEL_NAME_B4:
    sbert_model_b4 = sbert_model_global
    print(f"    Using pre-loaded SBERT model '{sbert_model_b4.model_name_or_path}'. Device: {sbert_model_b4.device}")
else:
    sbert_model_b4 = SentenceTransformer(SBERT_MODEL_NAME_B4)
    if torch.cuda.is_available():
        sbert_model_b4 = sbert_model_b4.to(torch.device("cuda"))
    print(f"    SBERT model '{SBERT_MODEL_NAME_B4}' loaded. Device: {sbert_model_b4.device}")

# --- Load Test Documents (Full-Text) ---
# Ensure `load_test_documents_for_sbert` is defined (see prerequisite note above)
# `extract_and_truncate_full_text_for_sbert` is defined in Step 0
print(f"  Attempting to load test documents for B4 from: {FULLTEXT_TEST_ZIP_PATH_B4}")
test_doc_ids_b4, test_doc_texts_b4 = load_test_documents_for_sbert(
    FULLTEXT_TEST_ZIP_PATH_B4,
    extract_and_truncate_full_text_for_sbert, # Specific for full-text & SBERT truncation
    PIPELINE_B4_CACHE_DIR, # Cache directory for B4's raw docs
    "B4_FullText_Test_Docs" # Unique prefix for B4 raw doc cache
)

# --- Load Test Queries ---
# `load_test_queries_from_zip_corrected` is defined in Step 0
# `TEST_QUERY_FILES_LIST` is defined in Step 0
all_test_queries_b4 = OrderedDict()
print(f"  Loading test queries for B4 from: {FULLTEXT_TEST_ZIP_PATH_B4}")
for query_file_name in TEST_QUERY_FILES_LIST:
    queries_for_file = load_test_queries_from_zip_corrected(FULLTEXT_TEST_ZIP_PATH_B4, query_file_name)
    all_test_queries_b4.update(queries_for_file)
print(f"  Loaded a total of {len(all_test_queries_b4)} test queries for B4.")

# --- Encode Test Documents (Full-Text) ---
corpus_embeddings_b4_path = os.path.join(PIPELINE_B4_CACHE_DIR, f'corpus_embeddings_b4_{SBERT_MODEL_NAME_B4.replace("/","_")}.pkl')
test_corpus_embeddings_b4 = None # Initialize

if os.path.exists(corpus_embeddings_b4_path) and test_doc_texts_b4:
    print(f"  Loading cached corpus embeddings for B4 from {corpus_embeddings_b4_path}")
    try:
        with open(corpus_embeddings_b4_path, 'rb') as f: test_corpus_embeddings_b4 = pickle.load(f)
        # Sanity check
        if len(test_corpus_embeddings_b4) != len(test_doc_texts_b4):
            print(f"    Cache mismatch for B4 corpus embeddings ({len(test_corpus_embeddings_b4)}) vs docs ({len(test_doc_texts_b4)}). Re-encoding.")
            test_corpus_embeddings_b4 = None
        else:
            print(f"    Loaded {len(test_corpus_embeddings_b4)} embeddings.")
    except Exception as e_load_emb:
        print(f"    Error loading cached B4 corpus embeddings: {e_load_emb}. Re-encoding.")
        test_corpus_embeddings_b4 = None

if test_corpus_embeddings_b4 is None and test_doc_texts_b4: # If not loaded from cache or cache was invalid
    print(f"  Encoding {len(test_doc_texts_b4)} test full-text documents for B4 using {SBERT_MODEL_NAME_B4}...")
    # `clean_text_for_sbert` is defined in Step 0.
    # `extract_and_truncate_full_text_for_sbert` already applies cleaning and truncation.
    # So, `test_doc_texts_b4` should already be cleaned and truncated.
    test_corpus_embeddings_b4 = sbert_model_b4.encode(
        test_doc_texts_b4, # Texts are already processed by extract_and_truncate_full_text_for_sbert
        batch_size=SBERT_EMBEDDING_BATCH_SIZE_B4,
        show_progress_bar=True,
        convert_to_tensor=True
    )
    if test_corpus_embeddings_b4 is not None: # Check if encoding was successful
        with open(corpus_embeddings_b4_path, 'wb') as f: pickle.dump(test_corpus_embeddings_b4, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"    Saved B4 corpus embeddings to {corpus_embeddings_b4_path}")
elif not test_doc_texts_b4:
    print("ERROR: No test documents loaded for B4. Cannot encode.")

# --- Generate Predictions for Test Queries ---
sbert_predictions_b4 = OrderedDict()
predictions_b4_cache_path = os.path.join(PIPELINE_B4_CACHE_DIR, f'predictions_b4_{SBERT_MODEL_NAME_B4.replace("/","_")}_top{SBERT_TOP_K_PREDS_B4}.pkl')

if os.path.exists(predictions_b4_cache_path):
    print(f"  Loading cached B4 predictions from: {predictions_b4_cache_path}")
    try:
        with open(predictions_b4_cache_path, 'rb') as f: sbert_predictions_b4 = pickle.load(f)
        print(f"    Loaded {len(sbert_predictions_b4)} query predictions for B4.")
    except Exception as e_load_pred_b4:
        print(f"    Error loading cached B4 predictions: {e_load_pred_b4}. Regenerating.")
        sbert_predictions_b4 = OrderedDict()

if not sbert_predictions_b4 and all_test_queries_b4 and test_corpus_embeddings_b4 is not None and test_doc_ids_b4:
    print(f"  Generating SBERT predictions for {len(all_test_queries_b4)} test queries (B4)...")

    # Prepare query texts and encode them
    # `clean_text_for_sbert` is defined in Step 0
    query_texts_b4 = [clean_text_for_sbert(q_text) for q_text in all_test_queries_b4.values()]
    query_embeddings_b4 = sbert_model_b4.encode(
        query_texts_b4,
        batch_size=SBERT_EMBEDDING_BATCH_SIZE_B4,
        show_progress_bar=True,
        convert_to_tensor=True
    )

    # Perform semantic search for all queries at once
    # `util.semantic_search` is from sentence_transformers
    all_hits_b4 = util.semantic_search(query_embeddings_b4, test_corpus_embeddings_b4, top_k=SBERT_TOP_K_PREDS_B4)

    query_ids_list_b4 = list(all_test_queries_b4.keys())
    for i, qid in enumerate(tqdm(query_ids_list_b4, desc="Processing B4 SBERT Results")):
        hits = all_hits_b4[i]
        sbert_predictions_b4[qid] = [test_doc_ids_b4[hit['corpus_id']] for hit in hits]
        # Optionally, can also store scores:
        # sbert_predictions_b4[qid] = [(test_doc_ids_b4[hit['corpus_id']], hit['score']) for hit in hits]

    if sbert_predictions_b4:
        with open(predictions_b4_cache_path, 'wb') as f: pickle.dump(sbert_predictions_b4, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  Saved B4 SBERT predictions to {predictions_b4_cache_path}")

elif sbert_predictions_b4: # If loaded from cache
     print(f"  B4 SBERT predictions already loaded from cache.")
else:
    print("ERROR: Cannot generate B4 predictions. Missing queries, document embeddings, or document IDs.")

# Display sample predictions for B4
if sbert_predictions_b4:
    print(f"\n  Sample B4 SBERT Predictions (Top 3 of {SBERT_TOP_K_PREDS_B4} shown):")
    count_sample_b4 = 0
    for q_id, preds_list in sbert_predictions_b4.items():
        print(f"    QID {q_id} Top 3: {preds_list[:3]}") # Assumes preds_list contains only IDs
        # If preds_list contains (doc_id, score) tuples:
        # print(f"    QID {q_id} Top 3: {[item[0] for item in preds_list[:3]]}")
        count_sample_b4 += 1
        if count_sample_b4 >= 3: break
else:
    print("  No B4 SBERT predictions available to display.")

gc.collect()
print("\n--- Task B4 Complete ---")




--- TASK B4: SBERT on Testing Full-Text ---
  Loading SBERT model: all-MiniLM-L6-v2
    SBERT model 'all-MiniLM-L6-v2' loaded. Device: cuda:0
  Attempting to load test documents for B4 from: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_fulltext.zip
  Processing test documents for B4_FullText_Test_Docs from ZIP: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_fulltext.zip
    Found 16 JSONL files with prefix: 'longeval_sci_testing_2025_fulltext/documents/'


Reading B4_FullText_Test_Docs JSONLs:   0%|          | 0/16 [00:00<?, ?file/s]

    Processed 1524045 documents for B4_FullText_Test_Docs.
    Saved document IDs and texts for B4_FullText_Test_Docs to /content/drive/MyDrive/colab-4/cache_B4_fulltext_sbert_test/
  Loading test queries for B4 from: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_fulltext.zip
  Loaded a total of 552 test queries for B4.
  Encoding 1524045 test full-text documents for B4 using all-MiniLM-L6-v2...


Batches:   0%|          | 0/11907 [00:00<?, ?it/s]

    Saved B4 corpus embeddings to /content/drive/MyDrive/colab-4/cache_B4_fulltext_sbert_test/corpus_embeddings_b4_all-MiniLM-L6-v2.pkl
  Generating SBERT predictions for 552 test queries (B4)...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Processing B4 SBERT Results:   0%|          | 0/552 [00:00<?, ?it/s]

  Saved B4 SBERT predictions to /content/drive/MyDrive/colab-4/cache_B4_fulltext_sbert_test/predictions_b4_all-MiniLM-L6-v2_top100.pkl

  Sample B4 SBERT Predictions (Top 3 of 100 shown):
    QID d585f080-4519-4952-8278-0d13fcd03fec Top 3: ['2635344', '2501592', '8964702']
    QID 254ecdb5-45e5-45ce-9b3f-492d1e1c085e Top 3: ['57282207', '96733820', '6255751']
    QID eaa374bf-cd61-4d9f-9b99-87f3d81f2636 Top 3: ['57282207', '96733820', '6255751']

--- Task B4 Complete ---


In [ ]:
# ==============================================================================
# CELL 0: INSTALLATIONS FOR CROSSENCODER TASKS (C1, C2)
# ==============================================================================
print("Starting installations for CrossEncoder tasks (if this is the first run or after a factory reset)...\n")
# Step 1: Install specific PyTorch ecosystem versions
print("Installing PyTorch, Torchvision, Torchaudio...")
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
print("PyTorch ecosystem installation attempt finished.")

# Step 2: Install sentence-transformers (for CrossEncoder) and other libraries
print("\nInstalling sentence-transformers and other libraries...")
# rank_bm25 might be needed if you re-run A5/A6 stages before C1/C2 in the same notebook flow
!pip install sentence-transformers==2.7.0 nltk tqdm scikit-learn pandas rank_bm25 -q
print("Other packages installation attempt finished.")

print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
print("CELL 0 (INSTALLATIONS) HAS COMPLETED BOTH INSTALLATION STEPS.")
print("IF THIS WAS THE FIRST TIME THESE PACKAGES WERE INSTALLED/UPDATED IN THIS SESSION,")
print("YOU **MUST** NOW MANUALLY RESTART THE COLAB RUNTIME FOR ALL CHANGES TO TAKE EFFECT.")
print("GO TO THE MENU: Runtime -> Restart runtime...")
print("AFTER RESTARTING, PROCEED TO RUN CELL 1 (Common Setup & Helper Functions).")
print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")

Starting installations for CrossEncoder tasks (if this is the first run or after a factory reset)...

Installing PyTorch, Torchvision, Torchaudio...
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 98.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 101.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 131.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 43.9 MB/

In [ ]:
# ==============================================================================
# CELL 2: TASK C1 - Neural Re-ranking Pipeline (Training Abstracts)
# ==============================================================================
print("\n\n--- TASK C1: Neural Re-ranking Pipeline (Abstracts) ---")

# --- Configuration for C1 ---
CROSS_ENCODER_MODEL_NAME_C1 = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
# TOP_K_PREDS_PARAM is defined in Cell 1 (Common Setup for CrossEncoder)
# It should be 100 based on your A5 setup for bm25_rm3_predictions_A5_abstract_top100.pkl
BM25_CANDIDATES_A5_PATH = os.path.join(PIPELINE_A5_CACHE_DIR, f'bm25_rm3_predictions_A5_abstract_top{TOP_K_PREDS_PARAM}.pkl')
TOP_K_EVAL_C1 = 5  # Evaluate final re-ranked list at Top-5
RERANK_BATCH_SIZE_C1 = 128 # Adjust based on GPU memory

# --- Load Raw Document Data and Qrels for C1 (Abstracts) ---
# This data is used to get the actual text of documents for re-ranking
# and qrels for final evaluation.
# The `pipeline_cache_dir_for_raw_data` for C1 is `PIPELINE_C1_CACHE_DIR`
ids_C1_docs, raw_texts_C1_docs, queries_C1, qrels_C1, data_loaded_C1_raw = load_raw_data_for_reranking(
    ABSTRACT_TRAIN_ZIP_PATH,
    extract_text_from_abstract_json_for_ce, # Use CrossEncoder specific text extraction
    PIPELINE_C1_CACHE_DIR,                 # Cache directory for C1's raw data
    "C1_Abstracts_Raw"
)

reranked_preds_C1 = OrderedDict()
reranker_results_C1 = {}
cross_encoder_C1 = None
loaded_C1_from_cache = False

if data_loaded_C1_raw:
    print(f"Raw data loaded for C1: {len(ids_C1_docs)} documents, {len(queries_C1)} queries.")
    # Create a mapping from doc_id to its raw text for quick lookup
    doc_id_to_text_map_C1 = dict(zip(ids_C1_docs, raw_texts_C1_docs))
    del ids_C1_docs # Free up memory if docs are numerous, map is primary need now
    del raw_texts_C1_docs
    gc.collect()

    # --- Load First-Stage Candidates (from A5) ---
    first_stage_candidates_C1 = OrderedDict()
    if os.path.exists(BM25_CANDIDATES_A5_PATH):
        print(f"  Loading BM25+RM3 candidates for C1 from: {BM25_CANDIDATES_A5_PATH}")
        try:
            with open(BM25_CANDIDATES_A5_PATH, 'rb') as f:
                first_stage_candidates_C1 = pickle.load(f)
            print(f"    Loaded {len(first_stage_candidates_C1)} query candidate lists for C1.")
            # Basic check: Ensure it matches the number of queries loaded for C1 raw data
            if len(first_stage_candidates_C1) != len(queries_C1):
                print(f"    WARNING: Mismatch in number of queries between A5 candidates ({len(first_stage_candidates_C1)}) and C1 queries ({len(queries_C1)}). Using intersection of QIDs.")
                common_qids_c1 = set(first_stage_candidates_C1.keys()) & set(queries_C1.keys())
                first_stage_candidates_C1 = OrderedDict((qid, first_stage_candidates_C1[qid]) for qid in common_qids_c1)
                queries_C1 = OrderedDict((qid, queries_C1[qid]) for qid in common_qids_c1) # Align queries_C1
                print(f"    Adjusted to {len(common_qids_c1)} common QIDs for C1 processing.")

        except Exception as e:
            print(f"    Error loading BM25+RM3 candidates for C1: {e}")
            first_stage_candidates_C1 = None # Ensure it's None if loading failed
    else:
        print(f"ERROR: BM25+RM3 candidate file from A5 not found at {BM25_CANDIDATES_A5_PATH}. C1 cannot proceed.")
        first_stage_candidates_C1 = None

    if first_stage_candidates_C1 and queries_C1: # Proceed only if candidates and queries are available
        # Define cache paths for C1 re-ranked results
        model_name_sanitized_c1 = CROSS_ENCODER_MODEL_NAME_C1.replace("/", "_")
        reranked_preds_C1_path = os.path.join(PIPELINE_C1_CACHE_DIR, f'reranked_preds_C1_{model_name_sanitized_c1}.pkl')
        reranker_results_C1_path = os.path.join(PIPELINE_C1_CACHE_DIR, f'reranker_results_C1_{model_name_sanitized_c1}.pkl')

        if os.path.exists(reranked_preds_C1_path) and os.path.exists(reranker_results_C1_path):
            print(f"  Loading cached C1 re-ranked predictions and results...")
            try:
                with open(reranked_preds_C1_path, 'rb') as f: reranked_preds_C1 = pickle.load(f)
                with open(reranker_results_C1_path, 'rb') as f: reranker_results_C1 = pickle.load(f)
                # Check if loaded data corresponds to current queries
                if reranked_preds_C1 and reranker_results_C1 and (set(reranked_preds_C1.keys()) == set(queries_C1.keys())):
                    print("    Successfully loaded C1 re-ranked outputs from cache.")
                    loaded_C1_from_cache = True
                else:
                    print("    C1 cached data incomplete or QID mismatch. Regenerating.")
                    reranked_preds_C1 = OrderedDict() # Reset if mismatch
                    reranker_results_C1 = {}
            except Exception as e:
                print(f"    Error loading C1 cache: {e}. Regenerating.")
                reranked_preds_C1 = OrderedDict()
                reranker_results_C1 = {}

        if not loaded_C1_from_cache:
            try:
                print(f"  Loading Cross-Encoder model: {CROSS_ENCODER_MODEL_NAME_C1} for C1...")
                # CORRECTED INSTANTIATION:
                cross_encoder_C1 = CrossEncoder(CROSS_ENCODER_MODEL_NAME_C1)
                # The device is typically handled by PyTorch internally.
                # If GPU is available, model.to(device) is often called inside the library.
                # You can check the device it's on, if needed, after loading:
                actual_device_c1 = next(cross_encoder_C1.model.parameters()).device
                print(f"    Cross-Encoder for C1 loaded. Model is on device: {actual_device_c1}")

            except Exception as e:
                print(f"    Error loading Cross-Encoder {CROSS_ENCODER_MODEL_NAME_C1}: {e}")
                cross_encoder_C1 = None

            if cross_encoder_C1 and queries_C1: # Ensure model and queries are available
                print(f"  Re-ranking candidates for {len(queries_C1)} queries using {CROSS_ENCODER_MODEL_NAME_C1} (Abstracts)...")
                for qid, query_text in tqdm(queries_C1.items(), desc="C1 Re-ranking Abstracts"):
                    candidate_doc_ids_for_qid = first_stage_candidates_C1.get(qid) # Get candidates for the current QID
                    if not candidate_doc_ids_for_qid:
                        reranked_preds_C1[qid] = []
                        continue

                    sentence_pairs_c1 = []
                    valid_candidate_ids_for_reranking_c1 = [] # Store doc IDs that had text and were included in pairs

                    # Prepare pairs for the current query
                    cleaned_query_text_c1 = clean_text_for_sbert(query_text) # Minimal cleaning for query
                    for doc_id in candidate_doc_ids_for_qid:
                        doc_text_content = doc_id_to_text_map_C1.get(str(doc_id)) # Ensure doc_id is string for lookup
                        if doc_text_content: # If document text exists in our map
                            sentence_pairs_c1.append([cleaned_query_text_c1, doc_text_content])
                            valid_candidate_ids_for_reranking_c1.append(str(doc_id))
                        # else:
                        #     print(f"      Warning QID {qid}: Doc ID {doc_id} from BM25 candidates not found in raw text map for C1.")

                    if not sentence_pairs_c1: # No valid pairs found
                        reranked_preds_C1[qid] = []
                        continue

                    # Predict scores using CrossEncoder
                    # Autocast for mixed precision if on GPU
                    amp_context_manager_c1 = autocast() if next(cross_encoder_C1.model.parameters()).device.type == 'cuda' else contextlib.nullcontext()
                    with amp_context_manager_c1:
                        ce_scores_c1 = cross_encoder_C1.predict(sentence_pairs_c1, show_progress_bar=False, batch_size=RERANK_BATCH_SIZE_C1)

                    if len(ce_scores_c1) != len(valid_candidate_ids_for_reranking_c1):
                        print(f"    Warning: Score/ID mismatch for QID {qid} in C1. Number of scores: {len(ce_scores_c1)}, number of valid candidate IDs: {len(valid_candidate_ids_for_reranking_c1)}. Fallback to original candidate order (truncated).")
                        reranked_preds_C1[qid] = [str(d_id) for d_id in valid_candidate_ids_for_reranking_c1[:TOP_K_PREDS_PARAM]] # Use original list if mismatch
                        continue

                    # Combine scores with document IDs and sort
                    scored_candidates_c1 = sorted(zip(ce_scores_c1, valid_candidate_ids_for_reranking_c1), key=lambda x: x[0], reverse=True)
                    reranked_preds_C1[qid] = [doc_id for score, doc_id in scored_candidates_c1] # Store all re-ranked, will be truncated by evaluation

                # Evaluate re-ranked results
                reranker_results_C1 = evaluate_reranked_results(reranked_preds_C1, qrels_C1, top_k=TOP_K_EVAL_C1)

                # Save results and predictions
                with open(reranked_preds_C1_path, 'wb') as f: pickle.dump(reranked_preds_C1, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved C1 re-ranked predictions to {reranked_preds_C1_path}")
                with open(reranker_results_C1_path, 'wb') as f: pickle.dump(reranker_results_C1, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved C1 re-ranker results to {reranker_results_C1_path}")
            else:
                print("  Skipping C1 re-ranking: CrossEncoder model could not be loaded or no queries available.")

        # Display results (either loaded or newly computed)
        if reranker_results_C1:
            print(f"\n--- C1 CrossEncoder ({CROSS_ENCODER_MODEL_NAME_C1}) on Abstracts - Final Results ---")
            for metric, value in reranker_results_C1.items():
                print(f"  {metric}: {value:.4f}")
        if reranked_preds_C1:
            count_sample_c1 = 0
            print(f"\n  Sample C1 Re-ranked Predictions (Top {TOP_K_EVAL_C1} shown, from {TOP_K_PREDS_PARAM} candidates):")
            for q_id_c1, preds_c1_list in reranked_preds_C1.items():
                print(f"    QID {q_id_c1} Top {min(3, len(preds_c1_list))}: {preds_c1_list[:min(3, len(preds_c1_list))]}")
                count_sample_c1 += 1
                if count_sample_c1 >= 3:
                    break
    else:
        print("Skipping C1: BM25+RM3 candidates from A5 (Abstracts) could not be loaded.")
else:
    print("Skipping C1: Raw abstract data loading for re-ranking failed.")

gc.collect()
print("\n--- Task C1 Complete ---")



--- TASK C1: Neural Re-ranking Pipeline (Abstracts) ---
  Attempting to load cached raw data for C1_Abstracts_Raw from: /content/drive/MyDrive/colab-4/cache_C1_abstract_crossencoder_train/raw_data_cache/loaded_raw_data_for_reranking_C1_Abstracts_Raw.pkl
    Successfully loaded 2014265 raw docs & data for C1_Abstracts_Raw from cache.
Raw data loaded for C1: 2014265 documents, 393 queries.
  Loading BM25+RM3 candidates for C1 from: /content/drive/MyDrive/colab-4/cache_A5_abstract_bm25_train/bm25_rm3_predictions_A5_abstract_top100.pkl
    Loaded 393 query candidate lists for C1.
  Loading Cross-Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2 for C1...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

...

    Cross-Encoder for C1 loaded. Model is on device: cpu
  Re-ranking candidates for 393 queries using cross-encoder/ms-marco-MiniLM-L-6-v2 (Abstracts)...


C1 Re-ranking Abstracts:   0%|          | 0/393 [00:00<?, ?it/s]

Evaluating re-ranked results for 393 queries at Top-5...


Evaluating Re-ranked:   0%|          | 0/393 [00:00<?, ?it/s]

    Saved C1 re-ranked predictions to /content/drive/MyDrive/colab-4/cache_C1_abstract_crossencoder_train/reranked_preds_C1_cross-encoder_ms-marco-MiniLM-L-6-v2.pkl
    Saved C1 re-ranker results to /content/drive/MyDrive/colab-4/cache_C1_abstract_crossencoder_train/reranker_results_C1_cross-encoder_ms-marco-MiniLM-L-6-v2.pkl

--- C1 CrossEncoder (cross-encoder/ms-marco-MiniLM-L-6-v2) on Abstracts - Final Results ---
  P@5: 0.1466
  R@5: 0.0914
  MAP@5: 0.0994

  Sample C1 Re-ranked Predictions (Top 5 shown, from 100 candidates):
    QID ce5bfacf-8652-4bc1-a5b0-6144a917fb1c Top 3: ['1589737', '85972941', '126513738']
    QID 71657c3b-4112-49d5-86cb-705325aeb06e Top 3: ['43096829', '19524046', '132588042']
    QID f60d94de-b903-48da-b13f-a8334c70f949 Top 3: ['89687169', '89686569', '147660304']

--- Task C1 Complete ---


In [ ]:
# ==============================================================================
# CELL 3: TASK C2 - Neural Re-ranking Pipeline (Training Full-Text)
# ==============================================================================
print("\n\n--- TASK C2: Neural Re-ranking Pipeline (Full-Text) ---")

# --- Configuration for C2 ---
CROSS_ENCODER_MODEL_NAME_C2 = 'cross-encoder/ms-marco-MiniLM-L-6-v2' # Can be the same or different from C1
# TOP_K_PREDS_PARAM is defined in Cell 1 (Common Setup for CrossEncoder)
# Ensure it was set to 100 (matching the A6 output file name format if A6 used this for saving predictions)
BM25_CANDIDATES_A6_PATH = os.path.join(PIPELINE_A6_CACHE_DIR, f'bm25_rm3_predictions_A6_fulltext_top{TOP_K_PREDS_PARAM}.pkl')
TOP_K_EVAL_C2 = 5  # Evaluate final re-ranked list at Top-5
RERANK_BATCH_SIZE_C2 = 128 # Adjust based on GPU memory
# MAX_WORDS_FOR_CE_FULLTEXT is defined in Cell 1 (Common Setup for CrossEncoder), e.g., 510

# --- Load Raw Document Data and Qrels for C2 (Full-Text) ---
ids_C2_docs, raw_texts_C2_docs, queries_C2, qrels_C2, data_loaded_C2_raw = load_raw_data_for_reranking(
    FULLTEXT_TRAIN_ZIP_PATH,                        # Use the full-text ZIP
    extract_and_truncate_full_text_for_ce,          # Use CrossEncoder specific full-text extraction & truncation
    PIPELINE_C2_CACHE_DIR,                          # Cache directory for C2's raw data
    "C2_FullText_Raw"
)

reranked_preds_C2 = OrderedDict()
reranker_results_C2 = {}
cross_encoder_C2 = None
loaded_C2_from_cache = False

if data_loaded_C2_raw:
    print(f"Raw data loaded for C2: {len(ids_C2_docs)} documents (truncated full-text for CE), {len(queries_C2)} queries.")
    doc_id_to_text_map_C2 = dict(zip(ids_C2_docs, raw_texts_C2_docs))
    del ids_C2_docs # Free up memory
    del raw_texts_C2_docs
    gc.collect()

    # --- Load First-Stage Candidates (from A6) ---
    first_stage_candidates_C2 = OrderedDict()
    if os.path.exists(BM25_CANDIDATES_A6_PATH):
        print(f"  Loading BM25+RM3 candidates for C2 from: {BM25_CANDIDATES_A6_PATH}")
        try:
            with open(BM25_CANDIDATES_A6_PATH, 'rb') as f:
                first_stage_candidates_C2 = pickle.load(f)
            print(f"    Loaded {len(first_stage_candidates_C2)} query candidate lists for C2.")
            if len(first_stage_candidates_C2) != len(queries_C2):
                print(f"    WARNING: Mismatch in number of queries between A6 candidates ({len(first_stage_candidates_C2)}) and C2 queries ({len(queries_C2)}). Using intersection.")
                common_qids_c2 = set(first_stage_candidates_C2.keys()) & set(queries_C2.keys())
                first_stage_candidates_C2 = OrderedDict((qid, first_stage_candidates_C2[qid]) for qid in common_qids_c2)
                queries_C2 = OrderedDict((qid, queries_C2[qid]) for qid in common_qids_c2) # Align queries_C2
                print(f"    Adjusted to {len(common_qids_c2)} common QIDs for C2 processing.")
        except Exception as e:
            print(f"    Error loading BM25+RM3 candidates for C2: {e}")
            first_stage_candidates_C2 = None
    else:
        print(f"ERROR: BM25+RM3 candidate file from A6 not found at {BM25_CANDIDATES_A6_PATH}. C2 cannot proceed.")
        first_stage_candidates_C2 = None

    if first_stage_candidates_C2 and queries_C2:
        model_name_sanitized_c2 = CROSS_ENCODER_MODEL_NAME_C2.replace("/", "_")
        # Include truncation info in cache filename for clarity if MAX_WORDS_FOR_CE_FULLTEXT is configurable
        reranked_preds_C2_path = os.path.join(PIPELINE_C2_CACHE_DIR, f'reranked_preds_C2_{model_name_sanitized_c2}_max{MAX_WORDS_FOR_CE_FULLTEXT}w.pkl')
        reranker_results_C2_path = os.path.join(PIPELINE_C2_CACHE_DIR, f'reranker_results_C2_{model_name_sanitized_c2}_max{MAX_WORDS_FOR_CE_FULLTEXT}w.pkl')

        if os.path.exists(reranked_preds_C2_path) and os.path.exists(reranker_results_C2_path):
            print(f"  Loading cached C2 re-ranked predictions and results...")
            try:
                with open(reranked_preds_C2_path, 'rb') as f: reranked_preds_C2 = pickle.load(f)
                with open(reranker_results_C2_path, 'rb') as f: reranker_results_C2 = pickle.load(f)
                if reranked_preds_C2 and reranker_results_C2 and (set(reranked_preds_C2.keys()) == set(queries_C2.keys())): # Check QID consistency
                    print("    Successfully loaded C2 re-ranked outputs from cache.")
                    loaded_C2_from_cache = True
                else:
                    print("    C2 cached data incomplete or QID mismatch. Regenerating.")
                    reranked_preds_C2 = OrderedDict(); reranker_results_C2 = {} # Reset
            except Exception as e:
                print(f"    Error loading C2 cache: {e}. Regenerating.")
                reranked_preds_C2 = OrderedDict(); reranker_results_C2 = {} # Reset

        if not loaded_C2_from_cache:
            # ---- Debug GPU Availability ----
            print("\n--- Checking PyTorch CUDA Availability for C2 ---")
            if torch.cuda.is_available():
                print(f"PyTorch CUDA is available.")
                print(f"  Number of GPUs: {torch.cuda.device_count()}")
                current_gpu_id_c2 = torch.cuda.current_device()
                print(f"  Current GPU ID: {current_gpu_id_c2}")
                print(f"  Current GPU Name: {torch.cuda.get_device_name(current_gpu_id_c2)}")
                device_to_use_c2 = torch.device("cuda")
            else:
                print("PyTorch CUDA is NOT available. Model will use CPU.")
                device_to_use_c2 = torch.device("cpu")
            print("--------------------------------------------------")
            # ---- End Debug GPU ----

            try:
                print(f"  Loading Cross-Encoder model: {CROSS_ENCODER_MODEL_NAME_C2} for C2...")
                # CORRECTED: Initialize CrossEncoder without the 'device' argument
                cross_encoder_C2 = CrossEncoder(CROSS_ENCODER_MODEL_NAME_C2)

                # Now, explicitly try to move the underlying model to the desired device
                if device_to_use_c2.type == 'cuda':
                    try:
                        cross_encoder_C2.model.to(device_to_use_c2)
                        print(f"    Cross-Encoder's underlying model for C2 explicitly moved to: {device_to_use_c2}")
                    except Exception as e_move_c2:
                        print(f"    Error moving Cross-Encoder's model for C2 to {device_to_use_c2}: {e_move_c2}. Will use CPU.")
                        cross_encoder_C2.model.to(torch.device("cpu")) # Fallback to CPU for the model

                actual_device_c2 = next(cross_encoder_C2.model.parameters()).device
                print(f"    Cross-Encoder for C2 loaded. Underlying model is actually on device: {actual_device_c2}")

            except Exception as e:
                print(f"    Error loading Cross-Encoder {CROSS_ENCODER_MODEL_NAME_C2} for C2: {e}")
                cross_encoder_C2 = None

            if cross_encoder_C2 and queries_C2:
                print(f"  Re-ranking candidates for {len(queries_C2)} queries using {CROSS_ENCODER_MODEL_NAME_C2} (Full-Text, Truncated ~{MAX_WORDS_FOR_CE_FULLTEXT}w)...")
                for qid, query_text in tqdm(queries_C2.items(), desc="C2 Re-ranking Full-Text"):
                    candidate_doc_ids_for_qid = first_stage_candidates_C2.get(qid)
                    if not candidate_doc_ids_for_qid:
                        reranked_preds_C2[qid] = []
                        continue

                    sentence_pairs_c2 = []
                    valid_candidate_ids_for_reranking_c2 = []
                    cleaned_query_text_c2 = clean_text_for_sbert(query_text) # Minimal cleaning for query

                    for doc_id in candidate_doc_ids_for_qid:
                        doc_text_content = doc_id_to_text_map_C2.get(str(doc_id))
                        if doc_text_content:
                            sentence_pairs_c2.append([cleaned_query_text_c2, doc_text_content])
                            valid_candidate_ids_for_reranking_c2.append(str(doc_id))

                    if not sentence_pairs_c2:
                        reranked_preds_C2[qid] = []
                        continue

                    amp_context_manager_c2 = autocast() if next(cross_encoder_C2.model.parameters()).device.type == 'cuda' else contextlib.nullcontext()
                    with amp_context_manager_c2:
                        ce_scores_c2 = cross_encoder_C2.predict(sentence_pairs_c2, show_progress_bar=False, batch_size=RERANK_BATCH_SIZE_C2)

                    if len(ce_scores_c2) != len(valid_candidate_ids_for_reranking_c2):
                        print(f"    Warning: Score/ID mismatch for QID {qid} in C2. Scores: {len(ce_scores_c2)}, Valid IDs: {len(valid_candidate_ids_for_reranking_c2)}. Fallback.")
                        reranked_preds_C2[qid] = [str(d_id) for d_id in valid_candidate_ids_for_reranking_c2[:TOP_K_PREDS_PARAM]]
                        continue

                    scored_candidates_c2 = sorted(zip(ce_scores_c2, valid_candidate_ids_for_reranking_c2), key=lambda x: x[0], reverse=True)
                    reranked_preds_C2[qid] = [doc_id for score, doc_id in scored_candidates_c2]

                reranker_results_C2 = evaluate_reranked_results(reranked_preds_C2, qrels_C2, top_k=TOP_K_EVAL_C2)
                with open(reranked_preds_C2_path, 'wb') as f: pickle.dump(reranked_preds_C2, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved C2 re-ranked predictions to {reranked_preds_C2_path}")
                with open(reranker_results_C2_path, 'wb') as f: pickle.dump(reranker_results_C2, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved C2 re-ranker results to {reranker_results_C2_path}")
            else:
                print("  Skipping C2 re-ranking: CrossEncoder model could not be loaded or no queries available.")

        if reranker_results_C2:
            print(f"\n--- C2 CrossEncoder ({CROSS_ENCODER_MODEL_NAME_C2}) on Full-Text (Truncated ~{MAX_WORDS_FOR_CE_FULLTEXT}w) - Final Results ---")
            for metric, value in reranker_results_C2.items():
                print(f"  {metric}: {value:.4f}")
        if reranked_preds_C2:
            count_sample_c2 = 0
            # Check if reranked_preds_C2 is not empty before trying to access its values
            num_candidates_display_c2 = len(next(iter(reranked_preds_C2.values()),[])) if reranked_preds_C2 else 0
            print(f"\n  Sample C2 Re-ranked Predictions (Top {TOP_K_EVAL_C2} shown, from {num_candidates_display_c2} re-ranked candidates per query):")
            for q_id_c2, preds_c2_list in reranked_preds_C2.items():
                print(f"    QID {q_id_c2} Top {min(3, len(preds_c2_list))}: {preds_c2_list[:min(3, len(preds_c2_list))]}")
                count_sample_c2 += 1
                if count_sample_c2 >= 3:
                    break
    else:
        print("Skipping C2: BM25+RM3 candidates from A6 (Full-Text) could not be loaded or no queries for C2.")
else:
    print("Skipping C2: Raw full-text data loading for re-ranking failed.")

gc.collect()
print("\n--- Task C2 Complete ---")



--- TASK C2: Neural Re-ranking Pipeline (Full-Text) ---
  Attempting to load cached raw data for C2_FullText_Raw from: /content/drive/MyDrive/colab-4/cache_C2_fulltext_crossencoder_train/raw_data_cache/loaded_raw_data_for_reranking_C2_FullText_Raw.pkl
    Successfully loaded 2014265 raw docs & data for C2_FullText_Raw from cache.
Raw data loaded for C2: 2014265 documents (truncated full-text for CE), 393 queries.
  Loading BM25+RM3 candidates for C2 from: /content/drive/MyDrive/colab-4/cache_A6_fulltext_bm25_train/bm25_rm3_predictions_A6_fulltext_top100.pkl
    Loaded 375 query candidate lists for C2.
    Adjusted to 375 common QIDs for C2 processing.

--- Checking PyTorch CUDA Availability for C2 ---
PyTorch CUDA is available.
  Number of GPUs: 1
  Current GPU ID: 0
  Current GPU Name: Tesla T4
--------------------------------------------------
  Loading Cross-Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2 for C2...
    Cross-Encoder's underlying model for C2 explicitly moved t

C2 Re-ranking Full-Text:   0%|          | 0/375 [00:00<?, ?it/s]

Evaluating re-ranked results for 375 queries at Top-5...


Evaluating Re-ranked:   0%|          | 0/375 [00:00<?, ?it/s]

    Saved C2 re-ranked predictions to /content/drive/MyDrive/colab-4/cache_C2_fulltext_crossencoder_train/reranked_preds_C2_cross-encoder_ms-marco-MiniLM-L-6-v2_max510w.pkl
    Saved C2 re-ranker results to /content/drive/MyDrive/colab-4/cache_C2_fulltext_crossencoder_train/reranker_results_C2_cross-encoder_ms-marco-MiniLM-L-6-v2_max510w.pkl

--- C2 CrossEncoder (cross-encoder/ms-marco-MiniLM-L-6-v2) on Full-Text (Truncated ~510w) - Final Results ---
  P@5: 0.1531
  R@5: 0.0955
  MAP@5: 0.1043

  Sample C2 Re-ranked Predictions (Top 5 shown, from 100 re-ranked candidates per query):
    QID 5db67c65-d274-4190-8ae2-db6088a1a671 Top 3: ['155747269', '29096546', '296823']
    QID f224f213-b184-4619-894c-317297ad6190 Top 3: ['40844104', '3549146', '485204']
    QID 798c08fc-b1c1-481d-a27c-39c76d4114ae Top 3: ['4194063', '75198976', '140687965']

--- Task C2 Complete ---


In [ ]:
# ==============================================================================
# STEP 0: NOTEBOOK SETUP, INSTALLATIONS, AND GLOBAL CONFIGURATIONS
# ==============================================================================
# Run this cell first to set up the environment.
# A runtime restart might be required after installations.

# ------------------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------------------------
print("--- Mounting Google Drive ---")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted successfully.\n")

# ------------------------------------------------------------------------------
# 2. INSTALL LIBRARIES
# ------------------------------------------------------------------------------
print("--- Installing Required Libraries ---")
try:
    # Sentence-Transformers
    get_ipython().system('pip install -U sentence-transformers -q')
    print("sentence-transformers installed/updated.")

    # For BM25
    get_ipython().system('pip install rank_bm25 -q')
    print("rank_bm25 installed.")

    # For NLTK (stopwords, tokenization)
    get_ipython().system('pip install nltk -q')
    print("nltk installed.")

    # For progress bars
    get_ipython().system('pip install tqdm -q')
    print("tqdm installed.")

except Exception as e:
    print(f"Error during library installation: {e}")
print("Library installations complete. A runtime restart might be needed if new versions were installed.\n")

# ------------------------------------------------------------------------------
# 3. IMPORT LIBRARIES
# ------------------------------------------------------------------------------
print("--- Importing Libraries ---")
import os
import gc # Garbage Collector
import json
import zipfile
import pickle
import re
import string
from collections import OrderedDict, defaultdict
from tqdm.notebook import tqdm # For progress bars in notebooks
import numpy as np
import random # For sampling if needed

# NLTK
import nltk
# Corrected NLTK resource download
try:
    print("Checking for NLTK 'stopwords' resource...")
    stopwords_path = nltk.data.find('corpora/stopwords')
    print("'stopwords' resource found.")
except LookupError:
    print("'stopwords' resource not found. Downloading...")
    nltk.download('stopwords', quiet=True)
    print("'stopwords' resource downloaded.")

try:
    print("Checking for NLTK 'punkt' resource...")
    punkt_path = nltk.data.find('tokenizers/punkt')
    print("'punkt' resource found.")
except LookupError:
    print("'punkt' resource not found. Downloading...")
    nltk.download('punkt', quiet=True)
    print("'punkt' resource downloaded.")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# PyTorch
import torch
from torch.cuda.amp import autocast
import contextlib

# Sentence-Transformers
from sentence_transformers import SentenceTransformer, CrossEncoder, util

# BM25
from rank_bm25 import BM25Okapi, BM25Plus

print("Libraries imported successfully.\n")

# ------------------------------------------------------------------------------
# 4. GLOBAL CONFIGURATIONS AND PATHS
# ------------------------------------------------------------------------------
print("--- Setting Global Configurations and Paths ---")

# Base path for your project in Google Drive
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/" # ADJUST THIS TO YOUR DRIVE PATH

# --- Data File Paths ---
ABSTRACT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_training_2025_abstract.zip')
FULLTEXT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_training_2025_fulltext.zip')
ABSTRACT_TEST_ZIP_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_abstract.zip')
FULLTEXT_TEST_ZIP_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_fulltext.zip')

TEST_QUERY_FILES_LIST = ["queries_2024-11_test.txt", "queries_2025-01_test.txt"]

# --- Cache Directories ---
PIPELINE_A1_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A1_abstract_tfidf_train/")
PIPELINE_A2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A2_fulltext_tfidf_train/")
PIPELINE_A3_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A3_abstract_tfidf_test/")
PIPELINE_A4_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A4_fulltext_tfidf_test/")
PIPELINE_A5_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A5_abstract_bm25_train/")
PIPELINE_A6_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A6_fulltext_bm25_train/")
PIPELINE_A7_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A7_abstract_bm25_test/")
PIPELINE_A8_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A8_fulltext_bm25_test/")
PIPELINE_B1_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B1_abstract_sbert_train/")
PIPELINE_B2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B2_fulltext_sbert_train/")
PIPELINE_B3_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B3_abstract_sbert_test/")
PIPELINE_B4_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B4_fulltext_sbert_test/")
PIPELINE_C1_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C1_abstract_crossencoder_train/")
PIPELINE_C2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C2_fulltext_crossencoder_train/")
PIPELINE_C3_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C3_abstract_crossencoder_test/")
PIPELINE_C4_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C4_fulltext_crossencoder_test/")

for path_val in list(globals().values()):
    if isinstance(path_val, str) and "CACHE_DIR" in path_val and path_val.startswith(DRIVE_BASE_PATH):
        os.makedirs(path_val, exist_ok=True)
        if "crossencoder" in path_val or "sbert" in path_val:
             os.makedirs(os.path.join(path_val, "raw_data_cache"), exist_ok=True)

# --- Model Names ---
SBERT_MODEL_NAME_DEFAULT = 'all-MiniLM-L6-v2'
CROSS_ENCODER_MODEL_NAME_DEFAULT = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

# --- Key Parameters ---
BM25_K1_PARAM = 1.5
BM25_B_PARAM = 0.75
RM3_FB_DOCS_PARAM = 5
RM3_FB_TERMS_PARAM = 5
SBERT_EMBEDDING_BATCH_SIZE = 128
MAX_WORDS_FOR_SBERT_FULLTEXT = 256
MAX_WORDS_FOR_CE_FULLTEXT = 510
RERANK_BATCH_SIZE_CE = 128
TOP_K_PREDS_PARAM = 100
TOP_K_EVAL_PARAM = 5

STOPWORDS = set(stopwords.words('english'))
print("Global configurations set.\n")

# ------------------------------------------------------------------------------
# 5. CORE UTILITY FUNCTIONS
# ------------------------------------------------------------------------------
print("--- Defining Core Utility Functions ---")

def clean_text_for_bm25(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha() and word not in STOPWORDS and len(word) > 1]
    return tokens

def clean_text_for_sbert(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_for_tfidf(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha() and word not in STOPWORDS and len(word) > 1]
    return " ".join(tokens)

def extract_text_from_abstract_json(record):
    title = record.get('title', '')
    abstract = record.get('abstract', '')
    title = str(title) if title is not None else ''
    abstract = str(abstract) if abstract is not None else ''
    return (title + " " + abstract).strip()

def extract_and_truncate_full_text_for_bm25(record, max_words=400):
    full_text = record.get('full_text', '')
    full_text = str(full_text) if full_text is not None else ''
    words = full_text.split()
    truncated_text = " ".join(words[:max_words])
    return truncated_text

def extract_text_from_abstract_json_for_sbert(record):
    return clean_text_for_sbert(extract_text_from_abstract_json(record))

def extract_and_truncate_full_text_for_sbert(record, max_words=MAX_WORDS_FOR_SBERT_FULLTEXT):
    full_text = record.get('full_text', '')
    full_text = str(full_text) if full_text is not None else ''
    cleaned_full_text = clean_text_for_sbert(full_text)
    words = cleaned_full_text.split()
    truncated_text = " ".join(words[:max_words])
    return truncated_text

def extract_text_from_abstract_json_for_ce(record):
    return clean_text_for_sbert(extract_text_from_abstract_json(record))

def extract_and_truncate_full_text_for_ce(record, max_words=MAX_WORDS_FOR_CE_FULLTEXT):
    full_text = record.get('full_text', '')
    full_text = str(full_text) if full_text is not None else ''
    cleaned_full_text = clean_text_for_sbert(full_text)
    words = cleaned_full_text.split()
    truncated_text = " ".join(words[:max_words])
    return truncated_text

def calculate_precision_recall_map_at_k(predictions, qrels, k=5):
    num_queries = len(predictions)
    if num_queries == 0: return 0.0, 0.0, 0.0
    total_precision_at_k = 0
    total_recall_at_k = 0
    mean_ap_at_k = 0
    for qid, ranked_docs in predictions.items():
        if qid not in qrels or not qrels[qid]: continue
        relevant_docs_for_query = qrels[qid]
        retrieved_k_docs = ranked_docs[:k]
        num_relevant_retrieved_at_k = len(set(retrieved_k_docs) & relevant_docs_for_query)
        total_precision_at_k += num_relevant_retrieved_at_k / len(retrieved_k_docs) if retrieved_k_docs else 0
        total_recall_at_k += num_relevant_retrieved_at_k / len(relevant_docs_for_query) if relevant_docs_for_query else 0
        ap_at_k = 0
        relevant_retrieved_count = 0
        for i, doc_id in enumerate(retrieved_k_docs):
            if doc_id in relevant_docs_for_query:
                relevant_retrieved_count += 1
                ap_at_k += relevant_retrieved_count / (i + 1)
        mean_ap_at_k += (ap_at_k / min(len(relevant_docs_for_query), k) if relevant_docs_for_query and min(len(relevant_docs_for_query), k) > 0 else 0)
    avg_p_at_k = total_precision_at_k / num_queries
    avg_r_at_k = total_recall_at_k / num_queries
    map_at_k = mean_ap_at_k / num_queries
    return avg_p_at_k, avg_r_at_k, map_at_k

def evaluate_predictions(predictions, qrels, top_k_eval=TOP_K_EVAL_PARAM, desc="Evaluating"):
    print(f"Evaluating predictions for {len(predictions)} queries at Top-{top_k_eval} ({desc})...")
    if not predictions:
        print("  No predictions to evaluate.")
        return {"P@"+str(top_k_eval): 0, "R@"+str(top_k_eval): 0, "MAP@"+str(top_k_eval): 0}
    p_at_k, r_at_k, map_at_k = calculate_precision_recall_map_at_k(predictions, qrels, k=top_k_eval)
    results = {
        f"P@{top_k_eval}": p_at_k,
        f"R@{top_k_eval}": r_at_k,
        f"MAP@{top_k_eval}": map_at_k
    }
    return results

def format_predictions_for_trec(predictions_dict, run_id, output_file_path, max_docs_per_query=100):
    print(f"Formatting predictions for TREC run file: {output_file_path} with run_id: {run_id}")
    lines_written = 0
    with open(output_file_path, 'w') as f_out:
        for qid, docs_or_docs_with_scores in predictions_dict.items():
            for rank_minus_1, item in enumerate(docs_or_docs_with_scores[:max_docs_per_query]):
                rank = rank_minus_1 + 1
                doc_id = None
                score = 0.0
                if isinstance(item, tuple) and len(item) == 2:
                    doc_id, score = str(item[0]), float(item[1])
                elif isinstance(item, str):
                    doc_id = str(item)
                    score = max_docs_per_query - rank + 1
                else:
                    print(f"  Warning: Unexpected item format for QID {qid}: {item}. Skipping.")
                    continue
                f_out.write(f"{qid}\tQ0\t{doc_id}\t{rank}\t{score:.6f}\t{run_id}\n")
                lines_written +=1
    print(f"  TREC run file saved with {lines_written} lines for {len(predictions_dict)} queries.")

def load_test_queries_from_zip_corrected(zip_path, query_file_name_in_zip):
    queries_dict = OrderedDict()
    if not os.path.exists(zip_path): print(f"ERROR: Query ZIP {zip_path} not found"); return queries_dict
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            actual_query_path = None
            possible_paths = [
                query_file_name_in_zip,
                os.path.join(z.namelist()[0].split('/')[0] if z.namelist() and '/' in z.namelist()[0] else "", query_file_name_in_zip)
            ]
            for p_path in possible_paths:
                if p_path in z.namelist():
                    actual_query_path = p_path
                    break
            if actual_query_path:
                with z.open(actual_query_path) as f_q:
                    queries_dict.update(OrderedDict(l.decode('utf-8', errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8', errors='ignore').strip()))
            else: print(f"  ERROR: '{query_file_name_in_zip}' not found in {os.path.basename(zip_path)}. Top entries: {z.namelist()[:5]}")
    except Exception as e: print(f"  Error loading queries from {query_file_name_in_zip} in {zip_path}: {e}")
    return queries_dict

print("Core utility functions defined.\n")
print("--- STEP 0: SETUP COMPLETE ---")
# REMEMBER TO RESTART RUNTIME IF PIP INSTALLS WERE SIGNIFICANT


--- Mounting Google Drive ---
Mounted at /content/drive
Google Drive mounted successfully.

--- Installing Required Libraries ---
sentence-transformers installed/updated.
rank_bm25 installed.
nltk installed.
tqdm installed.
Library installations complete. A runtime restart might be needed if new versions were installed.

--- Importing Libraries ---
Checking for NLTK 'stopwords' resource...
'stopwords' resource not found. Downloading...
'stopwords' resource downloaded.
Checking for NLTK 'punkt' resource...
'punkt' resource not found. Downloading...
'punkt' resource downloaded.
Libraries imported successfully.

--- Setting Global Configurations and Paths ---
Global configurations set.

--- Defining Core Utility Functions ---
Core utility functions defined.

--- STEP 0: SETUP COMPLETE ---


In [ ]:
# This function should be defined in your notebook before running C3/C4.
# For example, add it to your "Step 0" or a "Common Setup for CrossEncoder Tasks" cell.

def load_test_documents_for_ce(data_zip_path, text_extraction_func, cache_dir, data_type_prefix):
    """
    Loads document IDs and raw texts from a ZIP file for CrossEncoder test set processing.
    Caches the results. (Very similar to load_test_documents_for_sbert)
    """
    os.makedirs(cache_dir, exist_ok=True)
    # For consistency with Step 0, cache sub-folder for raw data could be used,
    # but direct caching in cache_dir is also fine as done for SBERT.
    # Let's use a subfolder for raw data for CE tasks if pipeline_cache_dir is the main task dir.
    raw_data_specific_cache_dir = os.path.join(cache_dir, "raw_data_cache") # As per Step 0 convention
    os.makedirs(raw_data_specific_cache_dir, exist_ok=True)

    doc_ids_cache_path = os.path.join(raw_data_specific_cache_dir, f'doc_ids_{data_type_prefix}.pkl')
    doc_texts_cache_path = os.path.join(raw_data_specific_cache_dir, f'doc_texts_{data_type_prefix}.pkl')

    if os.path.exists(doc_ids_cache_path) and os.path.exists(doc_texts_cache_path):
        print(f"  Loading cached document IDs and texts for {data_type_prefix} from {raw_data_specific_cache_dir}")
        with open(doc_ids_cache_path, 'rb') as f: doc_ids = pickle.load(f)
        with open(doc_texts_cache_path, 'rb') as f: doc_texts = pickle.load(f)
        print(f"    Loaded {len(doc_ids)} documents.")
        return doc_ids, doc_texts

    print(f"  Processing test documents for {data_type_prefix} from ZIP: {data_zip_path}")
    doc_ids = []
    doc_texts = []

    if not os.path.exists(data_zip_path):
        print(f"ERROR: Data ZIP not found: {data_zip_path}"); return [], []

    with zipfile.ZipFile(data_zip_path, 'r') as z:
        entries = z.namelist()
        zip_root = entries[0].split('/')[0] if entries and '/' in entries[0] else ""

        potential_prefixes = [
            os.path.join(zip_root, 'documents/') if zip_root else 'documents/',
            zip_root + "/" if zip_root else "",
            ""
        ]

        jsonl_files_in_zip = []
        for prefix_attempt in potential_prefixes:
            current_attempt_files = [info for info in z.infolist() if info.filename.lower().startswith(prefix_attempt.lower()) and info.filename.lower().endswith(".jsonl") and not info.is_dir()]
            if prefix_attempt == "" and not current_attempt_files:
                 current_attempt_files = [info for info in z.infolist() if '/' not in info.filename and info.filename.lower().endswith(".jsonl") and not info.is_dir()]
            if current_attempt_files:
                jsonl_files_in_zip = current_attempt_files
                print(f"    Found {len(jsonl_files_in_zip)} JSONL files with prefix: '{prefix_attempt}'")
                break

        if not jsonl_files_in_zip:
            print(f"    ERROR: Could not find any .jsonl files in expected locations within {data_zip_path}.")
            return [], []

        for info in tqdm(jsonl_files_in_zip, desc=f"Reading {data_type_prefix} JSONLs", unit="file"):
            with z.open(info) as f_jsonl:
                for line_byte in f_jsonl:
                    try:
                        record = json.loads(line_byte.decode('utf-8'))
                        doc_id_str = str(record.get("id"))
                        # Use the specific text extraction function passed (e.g., for abstracts for CE)
                        raw_text = text_extraction_func(record)
                        if doc_id_str and raw_text is not None:
                            doc_ids.append(doc_id_str)
                            doc_texts.append(raw_text)
                    except Exception:
                        pass

    print(f"    Processed {len(doc_ids)} documents for {data_type_prefix}.")
    if doc_ids and doc_texts:
        with open(doc_ids_cache_path, 'wb') as f: pickle.dump(doc_ids, f, protocol=pickle.HIGHEST_PROTOCOL)
        with open(doc_texts_cache_path, 'wb') as f: pickle.dump(doc_texts, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"    Saved document IDs and texts for {data_type_prefix} to {raw_data_specific_cache_dir}")

    return doc_ids, doc_texts


In [ ]:
# ==============================================================================
# CELL: TASK C3 - Neural Re-ranking on Testing Abstracts
# ==============================================================================
print("\n\n--- TASK C3: Neural Re-ranking on Testing Abstracts ---")

# --- Configuration for C3 ---
# Using global configurations from Step 0 where possible
CROSS_ENCODER_MODEL_NAME_C3 = CROSS_ENCODER_MODEL_NAME_DEFAULT
ABSTRACT_TEST_ZIP_PATH_C3 = ABSTRACT_TEST_ZIP_PATH
# PIPELINE_C3_CACHE_DIR is defined in Step 0
# PIPELINE_A7_CACHE_DIR is defined in Step 0 (for loading A7 predictions)

RERANK_BATCH_SIZE_C3 = RERANK_BATCH_SIZE_CE # From Step 0
# TOP_K_PREDS_PARAM from Step 0 was used by A7 to generate candidates (e.g., 100)
# C3 will re-rank these top N candidates.

# --- Load Raw Test Document Data (Abstracts) for Re-ranking ---
# `extract_text_from_abstract_json_for_ce` is defined in Step 0
# Ensure `load_test_documents_for_ce` (provided above) is defined in your notebook
print(f"  Attempting to load raw test abstract documents for C3 from: {ABSTRACT_TEST_ZIP_PATH_C3}")
test_doc_ids_c3, test_doc_texts_c3 = load_test_documents_for_ce(
    ABSTRACT_TEST_ZIP_PATH_C3,
    extract_text_from_abstract_json_for_ce, # Specific for abstracts and CE
    PIPELINE_C3_CACHE_DIR,                  # Cache directory for C3's raw data
    "C3_Abstract_Test_RawDocs"              # Unique prefix for C3's raw doc cache
)

doc_id_to_text_map_c3 = {}
if test_doc_ids_c3 and test_doc_texts_c3:
    doc_id_to_text_map_c3 = dict(zip(test_doc_ids_c3, test_doc_texts_c3))
    print(f"  Created doc_id_to_text_map for {len(doc_id_to_text_map_c3)} test abstract documents (C3).")
    del test_doc_ids_c3, test_doc_texts_c3 # Free up memory
    gc.collect()
else:
    print("ERROR: Failed to load raw test abstract documents for C3. Cannot proceed.")
    # Add a dummy assignment to prevent further errors if you run parts of the cell
    doc_id_to_text_map_c3 = {}
    all_test_queries_c3 = {}
    first_stage_candidates_c3 = {}


# --- Load Test Queries ---
# `load_test_queries_from_zip_corrected` and `TEST_QUERY_FILES_LIST` are from Step 0
all_test_queries_c3 = OrderedDict()
if doc_id_to_text_map_c3: # Proceed only if docs were loaded
    print(f"  Loading test queries for C3 from: {ABSTRACT_TEST_ZIP_PATH_C3}")
    for query_file_name in TEST_QUERY_FILES_LIST:
        queries_for_file = load_test_queries_from_zip_corrected(ABSTRACT_TEST_ZIP_PATH_C3, query_file_name)
        all_test_queries_c3.update(queries_for_file)
    print(f"  Loaded a total of {len(all_test_queries_c3)} test queries for C3.")

# --- Load First-Stage Candidate Predictions (from A7) ---
first_stage_candidates_c3 = OrderedDict()
if doc_id_to_text_map_c3 and all_test_queries_c3: # Proceed only if docs and queries loaded
    print(f"  Loading first-stage BM25+RM3 candidates from A7 (Abstracts Test)...")
    for query_file_name in TEST_QUERY_FILES_LIST:
        a7_preds_file_name = f'predictions_A7_Abstract_Test_{os.path.splitext(query_file_name)[0]}.pkl'
        a7_preds_path = os.path.join(PIPELINE_A7_CACHE_DIR, a7_preds_file_name)
        if os.path.exists(a7_preds_path):
            try:
                with open(a7_preds_path, 'rb') as f:
                    a7_preds_for_file = pickle.load(f)
                first_stage_candidates_c3.update(a7_preds_for_file)
                print(f"    Loaded {len(a7_preds_for_file)} candidates from {a7_preds_file_name}")
            except Exception as e_load_a7:
                print(f"    Error loading A7 predictions from {a7_preds_path}: {e_load_a7}")
        else:
            print(f"    WARNING: A7 prediction file not found: {a7_preds_path}")
    print(f"  Total first-stage candidates loaded for C3: {len(first_stage_candidates_c3)} queries.")
    # Align with loaded queries if there's any mismatch (e.g., if a pred file was missing)
    common_qids_c3 = set(first_stage_candidates_c3.keys()) & set(all_test_queries_c3.keys())
    first_stage_candidates_c3 = OrderedDict((qid, first_stage_candidates_c3[qid]) for qid in common_qids_c3 if qid in first_stage_candidates_c3) # Ensure qid exists in first_stage_candidates_c3
    all_test_queries_c3 = OrderedDict((qid, all_test_queries_c3[qid]) for qid in common_qids_c3 if qid in all_test_queries_c3) # Ensure qid exists in all_test_queries_c3
    print(f"    Adjusted to {len(common_qids_c3)} common QIDs for C3 processing.")


# --- Re-ranking Logic ---
reranked_preds_c3 = OrderedDict()
cross_encoder_c3 = None
loaded_c3_from_cache = False

# Define cache path for C3 re-ranked results
# Include model name for clarity if you experiment with different CEs
reranked_preds_c3_path = os.path.join(PIPELINE_C3_CACHE_DIR, f'reranked_preds_C3_{CROSS_ENCODER_MODEL_NAME_C3.replace("/", "_")}.pkl')

if os.path.exists(reranked_preds_c3_path):
    print(f"  Loading cached C3 re-ranked predictions from: {reranked_preds_c3_path}")
    try:
        with open(reranked_preds_c3_path, 'rb') as f: reranked_preds_c3 = pickle.load(f)
        # Basic check: ensure loaded predictions cover the current set of queries
        if reranked_preds_c3 and (set(reranked_preds_c3.keys()) == set(all_test_queries_c3.keys())):
            print(f"    Successfully loaded {len(reranked_preds_c3)} C3 re-ranked predictions from cache.")
            loaded_c3_from_cache = True
        else:
            print(f"    C3 cached data incomplete or QID mismatch with current queries. Regenerating.")
            reranked_preds_c3 = OrderedDict() # Reset
    except Exception as e_load_c3_cache:
        print(f"    Error loading C3 cache: {e_load_c3_cache}. Regenerating.")
        reranked_preds_c3 = OrderedDict()

if not loaded_c3_from_cache and doc_id_to_text_map_c3 and all_test_queries_c3 and first_stage_candidates_c3:
    # --- Load CrossEncoder Model ---
    print(f"\n--- Checking PyTorch CUDA Availability for C3 ---")
    if torch.cuda.is_available():
        print(f"PyTorch CUDA is available. Device: {torch.cuda.get_device_name(0)}")
        device_to_use_c3 = torch.device("cuda")
    else:
        print("PyTorch CUDA is NOT available. Model will use CPU.")
        device_to_use_c3 = torch.device("cpu")
    print("--------------------------------------------------")

    try:
        print(f"  Loading Cross-Encoder model: {CROSS_ENCODER_MODEL_NAME_C3} for C3...")
        cross_encoder_c3 = CrossEncoder(CROSS_ENCODER_MODEL_NAME_C3)
        if device_to_use_c3.type == 'cuda':
            try:
                cross_encoder_c3.model.to(device_to_use_c3)
                print(f"    Cross-Encoder's underlying model for C3 moved to: {device_to_use_c3}")
            except Exception as e_move_c3:
                print(f"    Error moving model to {device_to_use_c3}: {e_move_c3}. Using CPU.")
                cross_encoder_c3.model.to(torch.device("cpu"))
        actual_device_c3 = next(cross_encoder_c3.model.parameters()).device
        print(f"    Cross-Encoder for C3 loaded. Model is on device: {actual_device_c3}")
    except Exception as e_load_ce:
        print(f"    Error loading Cross-Encoder {CROSS_ENCODER_MODEL_NAME_C3}: {e_load_ce}")
        cross_encoder_c3 = None

    if cross_encoder_c3:
        print(f"  Re-ranking candidates for {len(all_test_queries_c3)} test queries using {CROSS_ENCODER_MODEL_NAME_C3} (Abstracts - C3)...")
        for qid, query_text in tqdm(all_test_queries_c3.items(), desc="C3 Re-ranking Test Abstracts"):
            candidate_doc_ids_for_qid = first_stage_candidates_c3.get(qid)
            if not candidate_doc_ids_for_qid:
                reranked_preds_c3[qid] = [] # No candidates from A7 for this qid
                continue

            sentence_pairs_c3 = []
            valid_candidate_ids_for_reranking_c3 = []
            # `clean_text_for_sbert` is suitable for CE query cleaning (from Step 0)
            cleaned_query_text_c3 = clean_text_for_sbert(query_text)

            for doc_id in candidate_doc_ids_for_qid: # These are from A7, should be strings
                doc_text_content = doc_id_to_text_map_c3.get(str(doc_id)) # Ensure lookup with string ID
                if doc_text_content:
                    sentence_pairs_c3.append([cleaned_query_text_c3, doc_text_content])
                    valid_candidate_ids_for_reranking_c3.append(str(doc_id))

            if not sentence_pairs_c3:
                reranked_preds_c3[qid] = [] # No valid candidates with text found
                continue

            # Mixed precision if on GPU
            amp_context_manager_c3 = autocast() if next(cross_encoder_c3.model.parameters()).device.type == 'cuda' else contextlib.nullcontext()
            with amp_context_manager_c3:
                ce_scores_c3 = cross_encoder_c3.predict(
                    sentence_pairs_c3,
                    show_progress_bar=False,
                    batch_size=RERANK_BATCH_SIZE_C3
                )

            # Ensure scores and IDs align
            if len(ce_scores_c3) != len(valid_candidate_ids_for_reranking_c3):
                print(f"    Warning: Score/ID mismatch for QID {qid} in C3. Scores: {len(ce_scores_c3)}, Valid IDs: {len(valid_candidate_ids_for_reranking_c3)}. Fallback to A7 order.")
                reranked_preds_c3[qid] = [str(d_id) for d_id in valid_candidate_ids_for_reranking_c3[:TOP_K_PREDS_PARAM]] # Use A7 order, truncated
                continue

            # Store (doc_id, score) for TREC formatting later, sorted by score
            scored_candidates_c3 = sorted(zip(valid_candidate_ids_for_reranking_c3, ce_scores_c3.tolist()), key=lambda x: x[1], reverse=True)
            reranked_preds_c3[qid] = scored_candidates_c3 # Store list of (doc_id, score) tuples

        if reranked_preds_c3:
            with open(reranked_preds_c3_path, 'wb') as f: pickle.dump(reranked_preds_c3, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"    Saved C3 re-ranked predictions to {reranked_preds_c3_path}")
    else:
        print("  Skipping C3 re-ranking: Model, queries, documents, or first-stage candidates not available.")

# Display sample predictions for C3
if reranked_preds_c3:
    print(f"\n  Sample C3 Re-ranked Predictions (Top 3 shown):")
    count_sample_c3 = 0
    for q_id, preds_list_with_scores in reranked_preds_c3.items():
        # preds_list_with_scores is [(doc_id, score), ...]
        top_3_docs = [item[0] for item in preds_list_with_scores[:3]]
        print(f"    QID {q_id} Top 3 Docs: {top_3_docs}")
        count_sample_c3 += 1
        if count_sample_c3 >= 3: break
elif not (doc_id_to_text_map_c3 and all_test_queries_c3 and first_stage_candidates_c3):
     print("  C3 re-ranking skipped due to missing initial data (docs, queries, or A7 candidates).")
else:
    print("  No C3 re-ranked predictions available to display (possibly an issue during re-ranking).")


gc.collect()
print("\n--- Task C3 Complete ---")




--- TASK C3: Neural Re-ranking on Testing Abstracts ---
  Attempting to load raw test abstract documents for C3 from: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_abstract.zip
  Processing test documents for C3_Abstract_Test_RawDocs from ZIP: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_abstract.zip
    Found 16 JSONL files with prefix: 'longeval_sci_testing_2025_abstract/documents/'


Reading C3_Abstract_Test_RawDocs JSONLs:   0%|          | 0/16 [00:00<?, ?file/s]

    Processed 1524045 documents for C3_Abstract_Test_RawDocs.
    Saved document IDs and texts for C3_Abstract_Test_RawDocs to /content/drive/MyDrive/colab-4/cache_C3_abstract_crossencoder_test/raw_data_cache
  Created doc_id_to_text_map for 1213728 test abstract documents (C3).
  Loading test queries for C3 from: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_abstract.zip
  Loaded a total of 552 test queries for C3.
  Loading first-stage BM25+RM3 candidates from A7 (Abstracts Test)...
    Loaded 97 candidates from predictions_A7_Abstract_Test_queries_2024-11_test.pkl
    Loaded 481 candidates from predictions_A7_Abstract_Test_queries_2025-01_test.pkl
  Total first-stage candidates loaded for C3: 539 queries.
    Adjusted to 539 common QIDs for C3 processing.

--- Checking PyTorch CUDA Availability for C3 ---
PyTorch CUDA is available. Device: Tesla T4
--------------------------------------------------
  Loading Cross-Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2 for C3

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

    Cross-Encoder's underlying model for C3 moved to: cuda
    Cross-Encoder for C3 loaded. Model is on device: cuda:0
  Re-ranking candidates for 539 test queries using cross-encoder/ms-marco-MiniLM-L-6-v2 (Abstracts - C3)...


C3 Re-ranking Test Abstracts:   0%|          | 0/539 [00:00<?, ?it/s]

<ipython-input-11-03f915728e28>:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_context_manager_c3 = autocast() if next(cross_encoder_c3.model.parameters()).device.type == 'cuda' else contextlib.nullcontext()


    Saved C3 re-ranked predictions to /content/drive/MyDrive/colab-4/cache_C3_abstract_crossencoder_test/reranked_preds_C3_cross-encoder_ms-marco-MiniLM-L-6-v2.pkl

  Sample C3 Re-ranked Predictions (Top 3 shown):
    QID c654fde5-81b3-4431-8177-2058231d3fd2 Top 3 Docs: ['134474676', '11776049', '10576818']
    QID 2caf6fce-0665-4434-ba07-6a4b6036232c Top 3 Docs: ['111730896', '42810370', '111733164']
    QID 962f72dc-9397-4e84-a7d5-c8bef612a512 Top 3 Docs: ['19939034', '139710397', '19706742']

--- Task C3 Complete ---


In [ ]:
import os
import zipfile
import orjson
import pickle
import gc
import contextlib
from collections import OrderedDict

import torch
from torch.cuda.amp import autocast
from sentence_transformers.cross_encoder import CrossEncoder

import pyterrier as pt
from tqdm.autonotebook import tqdm

# CELL: TASK C4 – Efficient Neural Re‐ranking on Test Full‐Texts

# --- Configuration ---
BASE_DIR                    = '/content/drive/MyDrive/colab-4'
TEST_ABS_ZIP                = os.path.join(BASE_DIR, 'longeval_sci_testing_2025_abstract.zip')
TEST_FT_ZIP                 = os.path.join(BASE_DIR, 'longeval_sci_testing_2025_fulltext.zip')
PIPELINE_A8_CACHE_DIR       = os.path.join(BASE_DIR, 'cache_A8_fulltext_bm25_test')
PIPELINE_C4_CACHE_DIR       = os.path.join(BASE_DIR, 'cache_C4_fulltext_crossencoder_test')
CROSS_ENCODER_MODEL_NAME    = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
RERANK_BATCH_SIZE           = 128
TOP_K_PREDS                 = 100  # number of A8 candidates to re‐rank
TOP_K_OUTPUT                = 5    # number of top docs to keep

os.makedirs(PIPELINE_C4_CACHE_DIR, exist_ok=True)

# --- 1. Initialize PyTerrier ---
if not pt.java.started():
    pt.init(version="snapshot", helper_version="snapshot")

# --- 2. Load test queries from abstract ZIP ---
print("Loading test queries for C4…")
all_test_queries = OrderedDict()
with zipfile.ZipFile(TEST_ABS_ZIP, 'r') as z:
    txt_files = [n for n in z.namelist() if n.lower().endswith('.txt')]
    for fname in txt_files:
        with z.open(fname) as f:
            for raw in f:
                line = raw.decode('utf-8').strip()
                if not line:
                    continue
                if '\t' in line:
                    qid, txt = line.split('\t', 1)
                else:
                    qid, txt = line.split(' ', 1)
                all_test_queries[qid] = txt
print(f"  Loaded {len(all_test_queries)} queries from {len(txt_files)} .txt files.")

# --- 3. Load first-stage A8 candidates ---
print("Loading A8 BM25+RM3 candidates for C4…")
first_stage = OrderedDict()
for pkl_name in ['predictions_A8_FullText_Test_queries_2024-11_test.pkl',
                 'predictions_A8_FullText_Test_queries_2025-01_test.pkl']:
    path = os.path.join(PIPELINE_A8_CACHE_DIR, pkl_name)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            preds = pickle.load(f)
        first_stage.update(preds)
        print(f"  {len(preds)} candidates loaded from {pkl_name}")
    else:
        print(f"  WARNING: missing {pkl_name}")
common_qids = set(all_test_queries) & set(first_stage)
all_test_queries = OrderedDict((qid, all_test_queries[qid]) for qid in common_qids)
first_stage = OrderedDict((qid, first_stage[qid]) for qid in common_qids)
print(f"  {len(common_qids)} queries with BM25 candidates.")

# --- 4. Collect unique candidate doc IDs ---
candidate_doc_ids = set()
for docs in first_stage.values():
    candidate_doc_ids.update(str(d) for d in docs[:TOP_K_PREDS])
print(f"  {len(candidate_doc_ids)} unique candidate docs to load.")

# --- 5. Load only needed full-texts from ZIP ---
print("Loading candidate full-texts from ZIP…")
test_fulltexts = {}
with zipfile.ZipFile(TEST_FT_ZIP, 'r') as z:
    jsonl_files = [info for info in z.infolist() if info.filename.endswith('.jsonl')]
    for info in tqdm(jsonl_files, desc="Files", unit="file"):
        with z.open(info) as fh:
            for raw in fh:
                try:
                    data = orjson.loads(raw)
                    doc_id = str(data.get('_id') or data.get('id') or '')
                    if doc_id in candidate_doc_ids:
                        title = data.get('title') or ''
                        conts = data.get('contents', [])
                        parts = []
                        if isinstance(conts, list):
                            for it in conts:
                                if isinstance(it, dict) and 'content' in it:
                                    parts.append(it['content'])
                                elif isinstance(it, str):
                                    parts.append(it)
                        elif isinstance(conts, str):
                            parts.append(conts)
                        tokens = (title + ' ' + ' '.join(parts)).split()[:512]
                        text = ' '.join(tokens)
                        if text:
                            test_fulltexts[doc_id] = text
                except:
                    continue
print(f"  Loaded {len(test_fulltexts)}/{len(candidate_doc_ids)} candidate full-texts.")

# --- 6. Load or compute re-ranked outputs ---
cache_path = os.path.join(
    PIPELINE_C4_CACHE_DIR,
    f'reranked_C4_{CROSS_ENCODER_MODEL_NAME.replace("/","_")}.pkl'
)
loaded = False
if os.path.exists(cache_path):
    with open(cache_path, 'rb') as f:
        reranked = pickle.load(f)
    if set(reranked) == common_qids:
        print("Loaded cached re-ranked results.")
        loaded = True
    else:
        print("Cache QIDs mismatch; recomputing.")
if not loaded:
    print("Running CrossEncoder re-ranking for C4…")
    ce = CrossEncoder(CROSS_ENCODER_MODEL_NAME)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ce.model.to(device)
    reranked = OrderedDict()
    for qid, query in tqdm(all_test_queries.items(), desc="Re-ranking Queries"):
        cands = first_stage[qid][:TOP_K_PREDS]
        pairs, ids = [], []
        for doc in cands:
            txt = test_fulltexts.get(str(doc))
            if txt:
                pairs.append([query, txt])
                ids.append(doc)
        if not pairs:
            reranked[qid] = []
            continue
        amp_ctx = autocast() if device.type == 'cuda' else contextlib.nullcontext()
        with amp_ctx:
            scores = ce.predict(pairs, batch_size=RERANK_BATCH_SIZE)
        scored = sorted(zip(ids, scores.tolist()), key=lambda x: x[1], reverse=True)
        reranked[qid] = scored[:TOP_K_OUTPUT]
    with open(cache_path, 'wb') as f:
        pickle.dump(reranked, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved re-ranked C4 results to cache: {cache_path}")

# --- 7. Sample output ---
print("\nSample C4 re-ranked top-3 doc IDs:")
for i, (qid, sc) in enumerate(reranked.items()):
    print(f"  QID {qid}: {[doc for doc, _ in sc[:3]]}")
    if i >= 2:
        break

gc.collect()
print("--- TASK C4 Complete ---")


Loading test queries for C4…
  Loaded 552 queries from 2 .txt files.
Loading A8 BM25+RM3 candidates for C4…
  97 candidates loaded from predictions_A8_FullText_Test_queries_2024-11_test.pkl
  481 candidates loaded from predictions_A8_FullText_Test_queries_2025-01_test.pkl
  539 queries with BM25 candidates.
  50498 unique candidate docs to load.
Loading candidate full-texts from ZIP…


Files:   0%|          | 0/16 [00:00<?, ?file/s]

  Loaded 50498/50498 candidate full-texts.
Running CrossEncoder re-ranking for C4…


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

Re-ranking Queries:   0%|          | 0/539 [00:00<?, ?it/s]

<ipython-input-2-934a7f572ca0>:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = autocast() if device.type == 'cuda' else contextlib.nullcontext()


Saved re-ranked C4 results to cache: /content/drive/MyDrive/colab-4/cache_C4_fulltext_crossencoder_test/reranked_C4_cross-encoder_ms-marco-MiniLM-L-6-v2.pkl

Sample C4 re-ranked top-3 doc IDs:
  QID 913be005-73ab-4b73-9fb1-4f8f4204f4f8: ['131584437', '88911751', '73016040']
  QID db22df12-adf1-4ecf-9de1-31825f16ffaa: ['149906579', '130186657', '74198257']
  QID 5e8b8fa1-ab82-4ff1-9208-9e66286fa46a: ['12243308', '63393379', '58678516']
--- TASK C4 Complete ---


In [ ]:
# ==============================================================================
# CELL: TASK B1 (v2) - SBERT (all-mpnet-base-v2) on Training Abstracts
# ==============================================================================
print("\n\n--- TASK B1 (v2): SBERT with 'all-mpnet-base-v2' on Training Abstracts ---")

# --- Configuration uses V2 globals from Step 0 (v2) ---
# These should be defined in your "Step 0 (v2)" cell:
# SBERT_MODEL_NAME_V2
# ABSTRACT_TRAIN_ZIP_PATH
# PIPELINE_B1_V2_CACHE_DIR
# SBERT_EMBEDDING_BATCH_SIZE_V2
# TOP_K_EVAL_PARAM_V2
# TOP_K_PREDS_PARAM_V2 # For saving predictions

# --- Load Data for B1 (v2) ---
# Using v2 helper functions defined in Step 0 (v2)
# Ensure `load_sbert_data_v2` and `extract_text_from_abstract_json_for_sbert_v2` are defined
ids_B1_v2, texts_B1_v2, queries_B1_dict_v2, qrels_B1_dict_v2, data_loaded_B1_v2 = load_sbert_data_v2(
    ABSTRACT_TRAIN_ZIP_PATH,
    extract_text_from_abstract_json_for_sbert_v2,
    PIPELINE_B1_V2_CACHE_DIR,
    f"B1_Abstracts_{SBERT_MODEL_NAME_V2.replace('/','_')}" # Unique prefix including model
)

doc_embeddings_B1_v2 = None
model_sbert_B1_v2 = None

if data_loaded_B1_v2:
    print(f"Data loaded for B1 (v2): {len(ids_B1_v2)} documents.")
    # Construct filename including the new model name for cache uniqueness
    doc_embeddings_file_B1_v2 = os.path.join(PIPELINE_B1_V2_CACHE_DIR, f'doc_embeddings_{SBERT_MODEL_NAME_V2.replace("/","_")}.pkl')

    try:
        print(f"Loading SBERT model: {SBERT_MODEL_NAME_V2}...")
        model_sbert_B1_v2 = SentenceTransformer(SBERT_MODEL_NAME_V2)
        if torch.cuda.is_available():
            model_sbert_B1_v2.to(torch.device("cuda"))
            print(f"  SBERT Model B1 (v2) '{SBERT_MODEL_NAME_V2}' moved to GPU.")
        else:
            print(f"  WARNING: GPU not available for SBERT Model B1 (v2). CPU will be slow.")
    except Exception as e:
        print(f"Error loading SBERT model {SBERT_MODEL_NAME_V2}: {e}")
        model_sbert_B1_v2 = None

    if model_sbert_B1_v2:
        if os.path.exists(doc_embeddings_file_B1_v2):
            print(f"  Loading cached document embeddings for B1 (v2) from: {doc_embeddings_file_B1_v2}")
            try:
                with open(doc_embeddings_file_B1_v2, 'rb') as f: doc_embeddings_B1_v2 = pickle.load(f)
                # Ensure it's a tensor and on the correct device after loading
                if not (isinstance(doc_embeddings_B1_v2, (torch.Tensor)) and \
                        doc_embeddings_B1_v2.shape[0] == len(ids_B1_v2) and \
                        doc_embeddings_B1_v2.shape[1] == model_sbert_B1_v2.get_sentence_embedding_dimension()):
                    print("  Cached embeddings B1 (v2) mismatch or invalid type. Regenerating."); doc_embeddings_B1_v2 = None
                else:
                    print(f"    Loaded {doc_embeddings_B1_v2.shape[0]} embeddings for B1 (v2).")
                    doc_embeddings_B1_v2 = doc_embeddings_B1_v2.to(model_sbert_B1_v2.device)
            except Exception as e:
                print(f"  Error loading cached B1 (v2) embeddings: {e}. Regenerating."); doc_embeddings_B1_v2 = None

        if doc_embeddings_B1_v2 is None or (torch.is_tensor(doc_embeddings_B1_v2) and doc_embeddings_B1_v2.nelement() == 0):
            print(f"  Generating document embeddings for {len(texts_B1_v2)} abstracts using {SBERT_MODEL_NAME_V2}...")
            # texts_B1_v2 should already be processed by extract_text_from_abstract_json_for_sbert_v2
            # Use torch.amp.autocast for mixed precision if available and on CUDA
            with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()) if torch.cuda.is_available() else contextlib.nullcontext():
                doc_embeddings_B1_v2 = model_sbert_B1_v2.encode(texts_B1_v2, show_progress_bar=True, batch_size=SBERT_EMBEDDING_BATCH_SIZE_V2, convert_to_tensor=True)
            print(f"    Generated abstract embeddings for B1 (v2). Shape: {doc_embeddings_B1_v2.shape}")
            try:
                with open(doc_embeddings_file_B1_v2, 'wb') as f: pickle.dump(doc_embeddings_B1_v2, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved abstract embeddings for B1 (v2) to: {doc_embeddings_file_B1_v2}")
            except Exception as e:
                print(f"    Error saving abstract embeddings for B1 (v2): {e}")

        if queries_B1_dict_v2 and doc_embeddings_B1_v2 is not None and doc_embeddings_B1_v2.nelement() > 0 :
            print(f"  Encoding {len(queries_B1_dict_v2)} queries for B1 (v2)...")
            query_texts_B1_v2 = [clean_text_for_sbert_v2(q_text) for q_text in queries_B1_dict_v2.values()] # Use v2 cleaner
            with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()) if torch.cuda.is_available() else contextlib.nullcontext():
                query_embeddings_B1_all_v2 = model_sbert_B1_v2.encode(query_texts_B1_v2, show_progress_bar=True, batch_size=SBERT_EMBEDDING_BATCH_SIZE_V2, convert_to_tensor=True)

            query_embeddings_B1_dict_v2 = {qid: emb for qid, emb in zip(queries_B1_dict_v2.keys(), query_embeddings_B1_all_v2)}

            # Ensure evaluate_sbert_retrieval_v2 is defined from Step 0 (v2)
            results_B1_v2, predictions_B1_v2 = evaluate_sbert_retrieval_v2(
                ids_B1_v2, query_embeddings_B1_dict_v2, doc_embeddings_B1_v2,
                qrels_B1_dict_v2, top_k=TOP_K_EVAL_PARAM_V2 # Uses TOP_K_EVAL_PARAM_V2 for evaluation cutoff
            )
            print(f"\n--- B1 (v2) SBERT ({SBERT_MODEL_NAME_V2}) on Abstracts - Results (Top {TOP_K_EVAL_PARAM_V2}) ---")
            for metric, value in results_B1_v2.items(): print(f"  {metric}: {value:.4f}")

            # Save predictions (top TOP_K_PREDS_PARAM_V2)
            predictions_B1_v2_path = os.path.join(PIPELINE_B1_V2_CACHE_DIR, f'predictions_B1_{SBERT_MODEL_NAME_V2.replace("/","_")}_top{TOP_K_PREDS_PARAM_V2}.pkl')
            with open(predictions_B1_v2_path, 'wb') as f: pickle.dump(predictions_B1_v2, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"  Saved B1 (v2) predictions (top {TOP_K_PREDS_PARAM_V2}) to {predictions_B1_v2_path}")
        else:
            print("  Skipping B1 (v2) retrieval/evaluation: Queries or document embeddings not ready.")
    else:
        print("  Skipping B1 (v2): SBERT model could not be loaded.")
else:
    print("Skipping B1 (v2): Data loading failed.")
gc.collect()
print("\n--- Task B1 (v2) Complete ---")




--- TASK B1 (v2): SBERT with 'all-mpnet-base-v2' on Training Abstracts ---
  Attempting to load cached raw data for B1_Abstracts_all-mpnet-base-v2 from: /content/drive/MyDrive/colab-4/cache_B1_v2_abstract_sbert_mpnet_train/raw_data_cache/loaded_data_B1_Abstracts_all-mpnet-base-v2.pkl
    Successfully loaded 2014265 docs & data for B1_Abstracts_all-mpnet-base-v2 from cache.
Data loaded for B1 (v2): 2014265 documents.
Loading SBERT model: all-mpnet-base-v2...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

...

    Generated abstract embeddings for B1 (v2). Shape: torch.Size([2014265, 768])
    Saved abstract embeddings for B1 (v2) to: /content/drive/MyDrive/colab-4/cache_B1_v2_abstract_sbert_mpnet_train/doc_embeddings_all-mpnet-base-v2.pkl
  Encoding 393 queries for B1 (v2)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating SBERT retrieval (v2) for 393 queries at Top-5...


SBERT Evaluating (v2):   0%|          | 0/393 [00:00<?, ?it/s]


--- B1 (v2) SBERT (all-mpnet-base-v2) on Abstracts - Results (Top 5) ---
  P@5: 0.0555
  R@5: 0.0309
  MAP@5: 0.0377
  Saved B1 (v2) predictions (top 100) to /content/drive/MyDrive/colab-4/cache_B1_v2_abstract_sbert_mpnet_train/predictions_B1_all-mpnet-base-v2_top100.pkl

--- Task B1 (v2) Complete ---


In [ ]:
# ==============================================================================
# CELL: TASK B2 (v2) - SBERT (all-mpnet-base-v2) on Training Full-Text
# ==============================================================================
print("\n\n--- TASK B2 (v2): SBERT with 'all-mpnet-base-v2' on Training Full-Text (Truncated) ---")

# --- Configuration uses V2 globals from Step 0 (v2) ---
# SBERT_MODEL_NAME_V2
# FULLTEXT_TRAIN_ZIP_PATH
# PIPELINE_B2_V2_CACHE_DIR
# SBERT_EMBEDDING_BATCH_SIZE_V2
# MAX_WORDS_FOR_SBERT_FULLTEXT_V2
# TOP_K_EVAL_PARAM_V2
# TOP_K_PREDS_PARAM_V2

# --- Load Data for B2 (v2) ---
# Using v2 helper functions defined in Step 0 (v2)
ids_B2_v2, texts_B2_v2, queries_B2_dict_v2, qrels_B2_dict_v2, data_loaded_B2_v2 = load_sbert_data_v2(
    FULLTEXT_TRAIN_ZIP_PATH,
    extract_and_truncate_full_text_for_sbert_v2, # Use v2 text extraction for full-text
    PIPELINE_B2_V2_CACHE_DIR,
    f"B2_FullText_{SBERT_MODEL_NAME_V2.replace('/','_')}_max{MAX_WORDS_FOR_SBERT_FULLTEXT_V2}w"
)

doc_embeddings_B2_v2 = None
model_sbert_B2_v2 = None

if data_loaded_B2_v2:
    print(f"Data loaded for B2 (v2): {len(ids_B2_v2)} documents.")
    # Construct filename including the new model name and truncation info for cache uniqueness
    doc_embeddings_file_B2_v2 = os.path.join(PIPELINE_B2_V2_CACHE_DIR, f'doc_embeddings_{SBERT_MODEL_NAME_V2.replace("/","_")}_max{MAX_WORDS_FOR_SBERT_FULLTEXT_V2}w.pkl')

    try:
        print(f"Loading SBERT model: {SBERT_MODEL_NAME_V2}...")
        model_sbert_B2_v2 = SentenceTransformer(SBERT_MODEL_NAME_V2)
        if torch.cuda.is_available():
            model_sbert_B2_v2.to(torch.device("cuda"))
            print(f"  SBERT Model B2 (v2) '{SBERT_MODEL_NAME_V2}' moved to GPU.")
        else:
            print(f"  WARNING: GPU not available for SBERT Model B2 (v2). CPU will be slow.")
    except Exception as e:
        print(f"Error loading SBERT model {SBERT_MODEL_NAME_V2}: {e}")
        model_sbert_B2_v2 = None

    if model_sbert_B2_v2:
        if os.path.exists(doc_embeddings_file_B2_v2):
            print(f"  Loading cached document embeddings for B2 (v2) from: {doc_embeddings_file_B2_v2}")
            try:
                with open(doc_embeddings_file_B2_v2, 'rb') as f: doc_embeddings_B2_v2 = pickle.load(f)
                # Ensure it's a tensor and on the correct device after loading
                if not (isinstance(doc_embeddings_B2_v2, (torch.Tensor)) and \
                        doc_embeddings_B2_v2.shape[0] == len(ids_B2_v2) and \
                        doc_embeddings_B2_v2.shape[1] == model_sbert_B2_v2.get_sentence_embedding_dimension()):
                    print("  Cached embeddings B2 (v2) mismatch or invalid type. Regenerating."); doc_embeddings_B2_v2 = None
                else:
                    print(f"    Loaded {doc_embeddings_B2_v2.shape[0]} embeddings for B2 (v2).")
                    doc_embeddings_B2_v2 = doc_embeddings_B2_v2.to(model_sbert_B2_v2.device)
            except Exception as e:
                print(f"  Error loading cached B2 (v2) embeddings: {e}. Regenerating."); doc_embeddings_B2_v2 = None

        if doc_embeddings_B2_v2 is None or (torch.is_tensor(doc_embeddings_B2_v2) and doc_embeddings_B2_v2.nelement() == 0):
            print(f"  Generating document embeddings for {len(texts_B2_v2)} (truncated) full-texts using {SBERT_MODEL_NAME_V2}...")
            # texts_B2_v2 are already processed by extract_and_truncate_full_text_for_sbert_v2
            with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()) if torch.cuda.is_available() else contextlib.nullcontext():
                doc_embeddings_B2_v2 = model_sbert_B2_v2.encode(texts_B2_v2, show_progress_bar=True, batch_size=SBERT_EMBEDDING_BATCH_SIZE_V2, convert_to_tensor=True)
            print(f"    Generated full-text embeddings for B2 (v2). Shape: {doc_embeddings_B2_v2.shape}")
            try:
                with open(doc_embeddings_file_B2_v2, 'wb') as f: pickle.dump(doc_embeddings_B2_v2, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved full-text embeddings for B2 (v2) to: {doc_embeddings_file_B2_v2}")
            except Exception as e:
                print(f"    Error saving full-text embeddings for B2 (v2): {e}")

        if queries_B2_dict_v2 and doc_embeddings_B2_v2 is not None and doc_embeddings_B2_v2.nelement() > 0:
            print(f"  Encoding {len(queries_B2_dict_v2)} queries for B2 (v2)...")
            query_texts_B2_v2 = [clean_text_for_sbert_v2(q_text) for q_text in queries_B2_dict_v2.values()]
            with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()) if torch.cuda.is_available() else contextlib.nullcontext():
                query_embeddings_B2_all_v2 = model_sbert_B2_v2.encode(query_texts_B2_v2, show_progress_bar=True, batch_size=SBERT_EMBEDDING_BATCH_SIZE_V2, convert_to_tensor=True)

            query_embeddings_B2_dict_v2 = {qid: emb for qid, emb in zip(queries_B2_dict_v2.keys(), query_embeddings_B2_all_v2)}

            # Ensure evaluate_sbert_retrieval_v2 is defined from Step 0 (v2)
            results_B2_v2, predictions_B2_v2 = evaluate_sbert_retrieval_v2(
                ids_B2_v2, query_embeddings_B2_dict_v2, doc_embeddings_B2_v2,
                qrels_B2_dict_v2, top_k=TOP_K_EVAL_PARAM_V2
            )
            print(f"\n--- B2 (v2) SBERT ({SBERT_MODEL_NAME_V2}) on Full-Text (Truncated ~{MAX_WORDS_FOR_SBERT_FULLTEXT_V2}w) - Results (Top {TOP_K_EVAL_PARAM_V2}) ---")
            for metric, value in results_B2_v2.items(): print(f"  {metric}: {value:.4f}")

            predictions_B2_v2_path = os.path.join(PIPELINE_B2_V2_CACHE_DIR, f'predictions_B2_{SBERT_MODEL_NAME_V2.replace("/","_")}_max{MAX_WORDS_FOR_SBERT_FULLTEXT_V2}w_top{TOP_K_PREDS_PARAM_V2}.pkl')
            with open(predictions_B2_v2_path, 'wb') as f: pickle.dump(predictions_B2_v2, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"  Saved B2 (v2) predictions (top {TOP_K_PREDS_PARAM_V2}) to {predictions_B2_v2_path}")
        else:
            print("  Skipping B2 (v2) retrieval/evaluation: Queries or document embeddings not ready.")
    else:
        print("  Skipping B2 (v2): SBERT model could not be loaded.")
else:
    print("Skipping B2 (v2): Data loading failed.")
gc.collect()
print("\n--- Task B2 (v2) Complete ---")



--- TASK B2 (v2): SBERT with 'all-mpnet-base-v2' on Training Full-Text (Truncated) ---
  --- Loading & Processing B2_FullText_all-mpnet-base-v2_max384w Data from: longeval_sci_training_2025_fulltext.zip ---
    Found 21 JSONL files with prefix: 'longeval_sci_training_2025_fulltext/documents/'


Reading B2_FullText_all-mpnet-base-v2_max384w JSONLs:   0%|          | 0/21 [00:00<?, ?file/s]

    Total B2_FullText_all-mpnet-base-v2_max384w docs processed: 2014265
    Loaded 393 queries.
    Loaded 393 queries with qrels.
    Saved loaded raw data for B2_FullText_all-mpnet-base-v2_max384w to cache: /content/drive/MyDrive/colab-4/cache_B2_v2_fulltext_sbert_mpnet_train/raw_data_cache/loaded_data_B2_FullText_all-mpnet-base-v2_max384w.pkl
Data loaded for B2 (v2): 2014265 documents.
Loading SBERT model: all-mpnet-base-v2...
  SBERT Model B2 (v2) 'all-mpnet-base-v2' moved to GPU.
  Generating document embeddings for 2014265 (truncated) full-texts using all-mpnet-base-v2...


Batches:   0%|          | 0/3935 [00:00<?, ?it/s]

    Generated full-text embeddings for B2 (v2). Shape: torch.Size([2014265, 768])
    Saved full-text embeddings for B2 (v2) to: /content/drive/MyDrive/colab-4/cache_B2_v2_fulltext_sbert_mpnet_train/doc_embeddings_all-mpnet-base-v2_max384w.pkl
  Encoding 393 queries for B2 (v2)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating SBERT retrieval (v2) for 393 queries at Top-5...


SBERT Evaluating (v2):   0%|          | 0/393 [00:00<?, ?it/s]


--- B2 (v2) SBERT (all-mpnet-base-v2) on Full-Text (Truncated ~384w) - Results (Top 5) ---
  P@5: 0.0601
  R@5: 0.0326
  MAP@5: 0.0396
  Saved B2 (v2) predictions (top 100) to /content/drive/MyDrive/colab-4/cache_B2_v2_fulltext_sbert_mpnet_train/predictions_B2_all-mpnet-base-v2_max384w_top100.pkl

--- Task B2 (v2) Complete ---


In [ ]:
# ==============================================================================
# CELL: TASK B3 (v2) - SBERT (all-mpnet-base-v2) on Testing Abstracts
# ==============================================================================
print("\n\n--- TASK B3 (v2): SBERT with 'all-mpnet-base-v2' on Testing Abstracts ---")

# --- Configuration uses V2 globals from Step 0 (v2) ---
# SBERT_MODEL_NAME_V2
# ABSTRACT_TEST_ZIP_PATH
# PIPELINE_B3_V2_CACHE_DIR
# SBERT_EMBEDDING_BATCH_SIZE_V2
# TOP_K_PREDS_PARAM_V2
# TEST_QUERY_FILES_LIST

# --- Load SBERT Model ---
print(f"  Loading SBERT model: {SBERT_MODEL_NAME_V2}")
try:
    model_sbert_B3_v2 = SentenceTransformer(SBERT_MODEL_NAME_V2)
    if torch.cuda.is_available():
        model_sbert_B3_v2.to(torch.device("cuda"))
        print(f"    SBERT model B3 (v2) '{SBERT_MODEL_NAME_V2}' loaded. Device: {model_sbert_B3_v2.device}")
    else:
        print(f"    SBERT model B3 (v2) '{SBERT_MODEL_NAME_V2}' loaded. Device: {model_sbert_B3_v2.device} (CPU)")
except Exception as e:
    print(f"    Error loading SBERT model {SBERT_MODEL_NAME_V2}: {e}")
    model_sbert_B3_v2 = None


# --- Load Test Documents (Abstracts) ---
print(f"  Attempting to load test abstract documents for B3 (v2) from: {ABSTRACT_TEST_ZIP_PATH}")
test_doc_ids_B3_v2, test_doc_texts_B3_v2 = load_test_documents_for_sbert_v2(
    ABSTRACT_TEST_ZIP_PATH,
    extract_text_from_abstract_json_for_sbert_v2,
    PIPELINE_B3_V2_CACHE_DIR,
    f"B3_Abstract_Test_Docs_{SBERT_MODEL_NAME_V2.replace('/','_')}"
)

# --- Load Test Queries ---
all_test_queries_B3_v2 = OrderedDict()
if test_doc_ids_B3_v2:
    print(f"  Loading test queries for B3 (v2) from: {ABSTRACT_TEST_ZIP_PATH}")
    for query_file_name in TEST_QUERY_FILES_LIST:
        queries_for_file = load_test_queries_from_zip_corrected(ABSTRACT_TEST_ZIP_PATH, query_file_name)
        all_test_queries_B3_v2.update(queries_for_file)
    print(f"  Loaded a total of {len(all_test_queries_B3_v2)} test queries for B3 (v2).")
else:
    print("  Skipping query loading for B3 (v2) as document loading failed or returned empty.")


# --- Encode Test Documents (Abstracts) ---
corpus_embeddings_B3_v2_path = os.path.join(PIPELINE_B3_V2_CACHE_DIR, f'corpus_embeddings_B3_{SBERT_MODEL_NAME_V2.replace("/","_")}.pkl')
test_corpus_embeddings_B3_v2 = None

if model_sbert_B3_v2 and test_doc_texts_B3_v2:
    if os.path.exists(corpus_embeddings_B3_v2_path):
        print(f"  Loading cached B3 (v2) corpus embeddings from {corpus_embeddings_B3_v2_path}")
        try:
            with open(corpus_embeddings_B3_v2_path, 'rb') as f: test_corpus_embeddings_B3_v2 = pickle.load(f)
            if not (isinstance(test_corpus_embeddings_B3_v2, torch.Tensor) and \
                    test_corpus_embeddings_B3_v2.shape[0] == len(test_doc_texts_B3_v2) and \
                    test_corpus_embeddings_B3_v2.shape[1] == model_sbert_B3_v2.get_sentence_embedding_dimension()):
                print(f"    Cache B3 (v2) mismatch/type or dimension error. Expected dim: {model_sbert_B3_v2.get_sentence_embedding_dimension()}. Re-encoding."); test_corpus_embeddings_B3_v2 = None
            else:
                print(f"    Loaded {test_corpus_embeddings_B3_v2.shape[0]} B3 (v2) embeddings. Moving to model device.")
                test_corpus_embeddings_B3_v2 = test_corpus_embeddings_B3_v2.to(model_sbert_B3_v2.device)
        except Exception as e_load_emb:
             print(f"    Error loading cached B3 (v2) embeddings: {e_load_emb}. Re-encoding.")
             test_corpus_embeddings_B3_v2 = None

    if test_corpus_embeddings_B3_v2 is None or (torch.is_tensor(test_corpus_embeddings_B3_v2) and test_corpus_embeddings_B3_v2.nelement() == 0) :
        print(f"  Encoding {len(test_doc_texts_B3_v2)} test abstract documents for B3 (v2)...")
        with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()) if torch.cuda.is_available() else contextlib.nullcontext(): # Corrected autocast usage
            test_corpus_embeddings_B3_v2 = model_sbert_B3_v2.encode(
                test_doc_texts_B3_v2, batch_size=SBERT_EMBEDDING_BATCH_SIZE_V2,
                show_progress_bar=True, convert_to_tensor=True
            )
        if test_corpus_embeddings_B3_v2 is not None:
            with open(corpus_embeddings_B3_v2_path, 'wb') as f: pickle.dump(test_corpus_embeddings_B3_v2, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"    Saved B3 (v2) corpus embeddings to {corpus_embeddings_B3_v2_path}")
elif not test_doc_texts_B3_v2:
    print("ERROR: No test documents loaded for B3 (v2). Cannot encode.")
elif not model_sbert_B3_v2:
    print("ERROR: SBERT model for B3 (v2) not loaded. Cannot encode.")


# --- Generate Predictions for B3 (v2) ---
sbert_predictions_B3_v2 = OrderedDict()
predictions_B3_v2_cache_path = os.path.join(PIPELINE_B3_V2_CACHE_DIR, f'predictions_B3_{SBERT_MODEL_NAME_V2.replace("/","_")}_top{TOP_K_PREDS_PARAM_V2}.pkl')

if os.path.exists(predictions_B3_v2_cache_path):
    print(f"  Loading cached B3 (v2) predictions from: {predictions_B3_v2_cache_path}")
    try:
        with open(predictions_B3_v2_cache_path, 'rb') as f: sbert_predictions_B3_v2 = pickle.load(f)
        print(f"    Loaded {len(sbert_predictions_B3_v2)} query predictions for B3 (v2).")
    except Exception as e_load_pred_b3:
        print(f"    Error loading cached B3 (v2) predictions: {e_load_pred_b3}. Will attempt to regenerate.")
        sbert_predictions_B3_v2 = OrderedDict()

if (not sbert_predictions_B3_v2 or len(sbert_predictions_B3_v2) != len(all_test_queries_B3_v2)) and \
   all_test_queries_B3_v2 and \
   test_corpus_embeddings_B3_v2 is not None and test_corpus_embeddings_B3_v2.nelement() > 0 and \
   test_doc_ids_B3_v2 and model_sbert_B3_v2:

    print(f"  Generating SBERT predictions for {len(all_test_queries_B3_v2)} test queries (B3 v2)...\"")
    query_texts_B3_v2 = [clean_text_for_sbert_v2(q_text) for q_text in all_test_queries_B3_v2.values()]
    with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()) if torch.cuda.is_available() else contextlib.nullcontext(): # Corrected autocast
        query_embeddings_B3_v2 = model_sbert_B3_v2.encode(
            query_texts_B3_v2, batch_size=SBERT_EMBEDDING_BATCH_SIZE_V2,
            show_progress_bar=True, convert_to_tensor=True
        )

    if query_embeddings_B3_v2.device != test_corpus_embeddings_B3_v2.device:
        test_corpus_embeddings_B3_v2 = test_corpus_embeddings_B3_v2.to(query_embeddings_B3_v2.device)

    all_hits_B3_v2 = util.semantic_search(query_embeddings_B3_v2, test_corpus_embeddings_B3_v2, top_k=TOP_K_PREDS_PARAM_V2)
    query_ids_list_B3_v2 = list(all_test_queries_B3_v2.keys())
    for i, qid in enumerate(tqdm(query_ids_list_B3_v2, desc="Processing B3 (v2) SBERT Results")):
        hits = all_hits_B3_v2[i]
        sbert_predictions_B3_v2[qid] = [test_doc_ids_B3_v2[hit['corpus_id']] for hit in hits] if hits else []

    if sbert_predictions_B3_v2:
        with open(predictions_B3_v2_cache_path, 'wb') as f: pickle.dump(sbert_predictions_B3_v2, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  Saved B3 (v2) SBERT predictions to {predictions_B3_v2_cache_path}")

elif sbert_predictions_B3_v2 and len(sbert_predictions_B3_v2) == len(all_test_queries_B3_v2):
     print(f"  B3 (v2) SBERT predictions already loaded from cache and match current query set.")
else:
    if not (all_test_queries_B3_v2 and test_corpus_embeddings_B3_v2 is not None and test_corpus_embeddings_B3_v2.nelement() > 0 and test_doc_ids_B3_v2 and model_sbert_B3_v2):
        print("ERROR: Cannot generate B3 (v2) predictions due to missing prerequisites (queries, doc embeddings, doc IDs, or model).")
    elif sbert_predictions_B3_v2:
        print("INFO: Loaded B3 (v2) predictions from cache, but they do not match the current query set. No re-generation triggered based on current logic.")

if sbert_predictions_B3_v2:
    print(f"\n  Sample B3 (v2) SBERT Predictions (Top 3 of {TOP_K_PREDS_PARAM_V2} shown):")
    count_sample_b3_v2 = 0
    for q_id, preds_list in sbert_predictions_B3_v2.items():
        print(f"    QID {q_id} Top 3: {preds_list[:3]}"); count_sample_b3_v2 += 1
        if count_sample_b3_v2 >= 3: break
else: print("  No B3 (v2) SBERT predictions available to display.")
gc.collect()
print("\n--- Task B3 (v2) Complete ---")



--- TASK B3 (v2): SBERT with 'all-mpnet-base-v2' on Testing Abstracts ---
  Loading SBERT model: all-mpnet-base-v2


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

...

Reading B3_Abstract_Test_Docs_all-mpnet-base-v2 JSONLs:   0%|          | 0/16 [00:00<?, ?file/s]

    Total B3_Abstract_Test_Docs_all-mpnet-base-v2 docs processed: 1524045
    Saved loaded raw data for B3_Abstract_Test_Docs_all-mpnet-base-v2 to cache: /content/drive/MyDrive/colab-4/cache_B3_v2_abstract_sbert_mpnet_test/raw_data_cache/loaded_data_B3_Abstract_Test_Docs_all-mpnet-base-v2.pkl
  Loading test queries for B3 (v2) from: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_abstract.zip
  Loaded a total of 552 test queries for B3 (v2).
  Loading cached B3 (v2) corpus embeddings from /content/drive/MyDrive/colab-4/cache_B3_v2_abstract_sbert_mpnet_test/corpus_embeddings_B3_all-mpnet-base-v2.pkl
    Loaded 1524045 B3 (v2) embeddings. Moving to model device.
  Generating SBERT predictions for 552 test queries (B3 v2)..."


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Processing B3 (v2) SBERT Results:   0%|          | 0/552 [00:00<?, ?it/s]

  Saved B3 (v2) SBERT predictions to /content/drive/MyDrive/colab-4/cache_B3_v2_abstract_sbert_mpnet_test/predictions_B3_all-mpnet-base-v2_top100.pkl

  Sample B3 (v2) SBERT Predictions (Top 3 of 100 shown):
    QID d585f080-4519-4952-8278-0d13fcd03fec Top 3: ['104463117', '70339957', '70339957']
    QID 254ecdb5-45e5-45ce-9b3f-492d1e1c085e Top 3: ['72181450', '20487814', '25027878']
    QID eaa374bf-cd61-4d9f-9b99-87f3d81f2636 Top 3: ['2978572', '103917394', '92165779']

--- Task B3 (v2) Complete ---


In [ ]:
# ==============================================================================
# CELL: STEP 0 (v2): NOTEBOOK SETUP FOR SBERT V2 EXPERIMENTS (all-mpnet-base-v2)
# ==============================================================================
# Run this cell first. A runtime restart might be required after installations.

# ------------------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------------------------
print("--- Mounting Google Drive ---")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted successfully.\n")

# ------------------------------------------------------------------------------
# 2. INSTALL LIBRARIES (will be skipped if already satisfied by previous Step 0)
# ------------------------------------------------------------------------------
print("--- Checking/Installing Required Libraries ---")
try:
    # Sentence-Transformers (ensure a version compatible with your PyTorch)
    get_ipython().system('pip install -U sentence-transformers==2.7.0 -q')
    print("sentence-transformers checked/installed.")

    # For BM25 (if you plan to use BM25 candidates later, not strictly needed for SBERT encoding itself)
    get_ipython().system('pip install rank_bm25 -q')
    print("rank_bm25 installed.")

    # For NLTK (stopwords, tokenization)
    get_ipython().system('pip install nltk -q')
    print("nltk installed.")

    # For progress bars
    get_ipython().system('pip install tqdm -q')
    print("tqdm installed.")

except Exception as e:
    print(f"Error during library installation: {e}")
print("Library checks/installations complete. If major packages were newly installed, a runtime restart might be beneficial.\n")

# ------------------------------------------------------------------------------
# 3. IMPORT LIBRARIES
# ------------------------------------------------------------------------------
print("--- Importing Libraries ---")
import os
import gc # Garbage Collector
import json
import zipfile
import pickle
import re
import string
from collections import OrderedDict, defaultdict
from tqdm.notebook import tqdm # For progress bars in notebooks
import numpy as np
import random # For sampling if needed

# NLTK
import nltk
# Corrected NLTK resource download
try:
    print("Checking for NLTK 'stopwords' resource...")
    stopwords_path = nltk.data.find('corpora/stopwords')
    print("'stopwords' resource found.")
except LookupError:
    print("'stopwords' resource not found. Downloading...")
    nltk.download('stopwords', quiet=True)
    print("'stopwords' resource downloaded.")

try:
    print("Checking for NLTK 'punkt' resource...")
    punkt_path = nltk.data.find('tokenizers/punkt')
    print("'punkt' resource found.")
except LookupError:
    print("'punkt' resource not found. Downloading...")
    nltk.download('punkt', quiet=True)
    print("'punkt' resource downloaded.")

try:
    print("Checking for NLTK 'wordnet' resource...")
    wordnet_path = nltk.data.find('corpora/wordnet')
    print("'wordnet' resource found.")
except LookupError:
    print("'wordnet' resource not found. Downloading...")
    nltk.download('wordnet', quiet=True)
    print("'wordnet' resource downloaded.")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
# from nltk.stem import WordNetLemmatizer # Not strictly needed for SBERT usually

# PyTorch
import torch
from torch.cuda.amp import autocast as autocast_torch # Keep alias if used elsewhere
import contextlib

# Sentence-Transformers
from sentence_transformers import SentenceTransformer, util

# BM25 (if needed for other parts of the notebook, not directly for SBERT encoding)
# from rank_bm25 import BM25Okapi, BM25Plus

print("Libraries imported successfully.\n")

# ------------------------------------------------------------------------------
# 4. GLOBAL CONFIGURATIONS AND PATHS (SBERT V2 FOCUS)
# ------------------------------------------------------------------------------
print("--- Setting Global Configurations and Paths for SBERT v2 ---")

DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/" # ADJUST THIS TO YOUR DRIVE PATH

# --- Data File Paths ---
ABSTRACT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_training_2025_abstract.zip')
FULLTEXT_TRAIN_ZIP_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_training_2025_fulltext.zip')
ABSTRACT_TEST_ZIP_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_abstract.zip')
FULLTEXT_TEST_ZIP_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_fulltext.zip')

TEST_QUERY_FILES_LIST = ["queries_2024-11_test.txt", "queries_2025-01_test.txt"]

# --- SBERT V2 Specific Model ---
SBERT_MODEL_NAME_V2 = 'all-mpnet-base-v2'
print(f"SBERT Model for v2 experiments: {SBERT_MODEL_NAME_V2}")

# --- Cache Directories for SBERT V2 ---
PIPELINE_B1_V2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B1_v2_abstract_sbert_mpnet_train/")
PIPELINE_B2_V2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B2_v2_fulltext_sbert_mpnet_train/")
PIPELINE_B3_V2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B3_v2_abstract_sbert_mpnet_test/")
PIPELINE_B4_V2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B4_v2_fulltext_sbert_mpnet_test/")

for path_val in [PIPELINE_B1_V2_CACHE_DIR, PIPELINE_B2_V2_CACHE_DIR, PIPELINE_B3_V2_CACHE_DIR, PIPELINE_B4_V2_CACHE_DIR]:
    os.makedirs(path_val, exist_ok=True)
    os.makedirs(os.path.join(path_val, "raw_data_cache"), exist_ok=True) # Subfolder for raw texts/ids
print("SBERT v2 Cache directories created/ensured.")

# --- Key Parameters for SBERT V2 ---
SBERT_EMBEDDING_BATCH_SIZE_V2 = 128
MAX_WORDS_FOR_SBERT_FULLTEXT_V2 = 384 # DEFINED HERE
TOP_K_PREDS_PARAM_V2 = 100 # DEFINED HERE
TOP_K_EVAL_PARAM_V2 = 5    # DEFINED HERE

STOPWORDS_SET = set(stopwords.words('english')) # Defined here for use in helper functions

print(f"SBERT v2 Parameters: Batch Size={SBERT_EMBEDDING_BATCH_SIZE_V2}, Max Words Full-text={MAX_WORDS_FOR_SBERT_FULLTEXT_V2}")
print("Global configurations for SBERT v2 set.\n")

# ------------------------------------------------------------------------------
# 5. CORE UTILITY FUNCTIONS FOR SBERT V2
# ------------------------------------------------------------------------------
print("--- Defining Core Utility Functions for SBERT v2 ---")

def clean_text_for_sbert_v2(text: str) -> str:
    if not isinstance(text, str): return ""
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def extract_text_from_abstract_json_for_sbert_v2(record_dict):
    title = record_dict.get("title", "") or ""
    abstract_content = record_dict.get("abstract", "") or ""
    return clean_text_for_sbert_v2(f"{title}. {abstract_content}".strip())

def extract_and_truncate_full_text_for_sbert_v2(record_dict, max_words=MAX_WORDS_FOR_SBERT_FULLTEXT_V2):
    title = record_dict.get("title", "") or ""
    abstract = record_dict.get("abstract", "") or ""
    main_text = "";
    possible_fields = ["text", "full_text", "body", "body_text", "content", "fulltext"]
    for field in possible_fields:
        content = record_dict.get(field)
        if content and isinstance(content, str): main_text = content; break
    combined_text = f"{title}. {abstract}. {main_text}".strip()
    cleaned_combined_text = clean_text_for_sbert_v2(combined_text)
    words = cleaned_combined_text.split()
    truncated_text = " ".join(words[:max_words])
    return truncated_text

def load_sbert_data_v2(data_zip_path, text_extraction_func, pipeline_cache_dir_v2, data_type_prefix_v2, is_test_set=False):
    ids, texts, queries, qrels = [], [], OrderedDict(), defaultdict(set)
    loaded_from_cache = False
    raw_data_cache_subdir = os.path.join(pipeline_cache_dir_v2, "raw_data_cache")
    os.makedirs(raw_data_cache_subdir, exist_ok=True)
    cache_file_suffix = data_type_prefix_v2
    if "FullText" in data_type_prefix_v2 and "sbert_v2" in text_extraction_func.__name__:
        cache_file_suffix += f"_max{MAX_WORDS_FOR_SBERT_FULLTEXT_V2}w"
    loaded_data_cache_path = os.path.join(raw_data_cache_subdir, f'loaded_data_{cache_file_suffix}.pkl')

    if os.path.exists(loaded_data_cache_path):
        print(f"  Attempting to load cached raw data for {data_type_prefix_v2} from: {loaded_data_cache_path}")
        try:
            with open(loaded_data_cache_path, 'rb') as f: cached = pickle.load(f)
            ids = cached.get('ids', [])
            texts = cached.get('texts', [])
            if not is_test_set:
                queries = cached.get('queries', OrderedDict())
                qrels_temp = cached.get('qrels', defaultdict(set)); qrels = defaultdict(set, qrels_temp)
            if ids and texts and (len(ids) == len(texts)) and (queries if not is_test_set else True):
                print(f"    Successfully loaded {len(ids)} docs & data for {data_type_prefix_v2} from cache.")
                loaded_from_cache = True
            else: ids, texts, queries, qrels = [], [], OrderedDict(), defaultdict(set); print("    Cached data invalid.")
        except Exception as e: print(f"    Error loading raw data cache for {data_type_prefix_v2}: {e}"); ids, texts, queries, qrels = [], [], OrderedDict(), defaultdict(set)

    if not loaded_from_cache:
        if not os.path.exists(data_zip_path): print(f"  ERROR: Data ZIP not found: {data_zip_path}"); return [],[],OrderedDict(),defaultdict(set),False
        print(f"  --- Loading & Processing {data_type_prefix_v2} Data from: {os.path.basename(data_zip_path)} ---")
        with zipfile.ZipFile(data_zip_path, 'r') as z:
            entries = z.namelist(); zip_root = entries[0].split('/')[0] if entries and '/' in entries[0] else ""
            jsonl_path_prefix = os.path.join(zip_root, 'documents/') if zip_root else 'documents/'
            if not is_test_set:
                query_file_path = os.path.join(zip_root, 'queries.txt') if zip_root else 'queries.txt'
                qrels_file_path = os.path.join(zip_root, 'qrels.txt') if zip_root else 'qrels.txt'
            jsonl_files = []
            potential_prefixes_docs = [jsonl_path_prefix, zip_root + "/" if zip_root else "", ""]
            for prefix_attempt_docs in potential_prefixes_docs:
                current_attempt_files_docs = [info for info in z.infolist() if info.filename.lower().startswith(prefix_attempt_docs.lower()) and info.filename.lower().endswith(".jsonl") and not info.is_dir()]
                if prefix_attempt_docs == "" and not current_attempt_files_docs: current_attempt_files_docs = [info for info in z.infolist() if '/' not in info.filename and info.filename.lower().endswith(".jsonl") and not info.is_dir()]
                if current_attempt_files_docs: jsonl_files = current_attempt_files_docs; print(f"    Found {len(jsonl_files)} JSONL files with prefix: '{prefix_attempt_docs}'"); break
            if not jsonl_files: print(f"    ERROR: No JSONL document files found in {data_zip_path}. Searched prefixes: {potential_prefixes_docs}"); return [],[],OrderedDict(),defaultdict(set),False
            for info in tqdm(jsonl_files, desc=f"Reading {data_type_prefix_v2} JSONLs", unit="file"):
                with z.open(info) as f_jsonl:
                    for line_byte in f_jsonl:
                        try:
                            record = json.loads(line_byte.decode('utf-8')); doc_id = str(record.get("id"));
                            extracted_text = text_extraction_func(record)
                            if doc_id and extracted_text is not None: ids.append(doc_id); texts.append(extracted_text)
                        except: pass
            print(f"    Total {data_type_prefix_v2} docs processed: {len(texts)}")
            if not is_test_set:
                if query_file_path in z.namelist():
                    with z.open(query_file_path) as f_q: queries.update(OrderedDict(l.decode('utf-8',errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8',errors='ignore').strip()))
                    print(f"    Loaded {len(queries)} queries.")
                else: print(f"    Warning: Query file '{query_file_path}' not found in ZIP for training set.")
                if qrels_file_path in z.namelist():
                    with z.open(qrels_file_path) as f_r:
                        for l in f_r:
                            parts=l.decode('utf-8',errors='ignore').strip().split(); qid_r, did_r, rel_r_str = (None,None,None)
                            if len(parts) == 4: qid_r, _, did_r, rel_r_str = parts[0],parts[1],parts[2],parts[3]
                            elif len(parts) == 3: qid_r, did_r, rel_r_str = parts[0],parts[1],parts[2]
                            if qid_r and did_r and rel_r_str and rel_r_str.isdigit() and int(rel_r_str)>0: qrels[str(qid_r)].add(str(did_r))
                    print(f"    Loaded {len(qrels)} queries with qrels.")
                else: print(f"    Warning: Qrels file '{qrels_file_path}' not found in ZIP for training set.")
            if ids and texts:
                data_to_cache = {'ids':ids, 'texts':texts}
                if not is_test_set: data_to_cache['queries'] = queries; data_to_cache['qrels'] = qrels
                with open(loaded_data_cache_path, 'wb') as f: pickle.dump(data_to_cache, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"    Saved loaded raw data for {data_type_prefix_v2} to cache: {loaded_data_cache_path}")
                loaded_from_cache = True
    data_valid_check = bool(ids and texts and (len(ids) == len(texts)))
    if not is_test_set: data_valid_check = data_valid_check and bool(queries)
    if not data_valid_check: print(f"  ERROR: {data_type_prefix_v2} data loading resulted in inconsistent lists.")
    return ids, texts, queries, qrels, data_valid_check

def load_test_documents_for_sbert_v2(data_zip_path, text_extraction_func, pipeline_cache_dir_v2, data_type_prefix_v2):
    ids, texts, _, _, data_valid = load_sbert_data_v2(
        data_zip_path,
        text_extraction_func,
        pipeline_cache_dir_v2,
        data_type_prefix_v2,
        is_test_set=True
    )
    if not data_valid:
        print(f"Error loading test documents for {data_type_prefix_v2}. IDs or Texts might be missing or mismatched.")
    return ids, texts

def load_test_queries_from_zip_corrected(zip_path, query_file_name_in_zip):
    queries_dict = OrderedDict()
    if not os.path.exists(zip_path): print(f"ERROR: Query ZIP {zip_path} not found"); return queries_dict
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            actual_query_path = None; entries = z.namelist()
            zip_root = entries[0].split('/')[0] if entries and '/' in entries[0] else ""
            possible_paths = [query_file_name_in_zip, os.path.join(zip_root, query_file_name_in_zip)]
            for p_path in possible_paths:
                if p_path in z.namelist(): actual_query_path = p_path; break
            if actual_query_path:
                with z.open(actual_query_path) as f_q: queries_dict.update(OrderedDict(l.decode('utf-8', errors='ignore').strip().split("\t",1) for l in f_q if "\t" in l.decode('utf-8', errors='ignore').strip()))
            else: print(f"  ERROR: '{query_file_name_in_zip}' not found in expected locations within {os.path.basename(zip_path)}. Top entries: {z.namelist()[:5]}")
    except Exception as e: print(f"  Error loading queries from {query_file_name_in_zip} in {zip_path}: {e}")
    return queries_dict

def evaluate_sbert_retrieval_v2(doc_ids_list, query_embeddings_dict, doc_embeddings_matrix, qrels_dict, top_k=TOP_K_EVAL_PARAM_V2, top_k_preds_to_return=TOP_K_PREDS_PARAM_V2, similarity_fn=util.semantic_search):
    all_preds = OrderedDict();
    p_at_k_list, r_at_k_list, ap_at_k_list = [], [], []
    print(f"Evaluating SBERT retrieval (v2) for {len(query_embeddings_dict)} queries. Eval@Top-{top_k}. Preds@Top-{top_k_preds_to_return}...")
    doc_embeddings_tensor = doc_embeddings_matrix
    target_device = 'cpu'
    if query_embeddings_dict: # Check if dict is not empty
        first_query_id = list(query_embeddings_dict.keys())[0]
        if torch.is_tensor(query_embeddings_dict[first_query_id]):
            target_device = query_embeddings_dict[first_query_id].device
    elif torch.is_tensor(doc_embeddings_tensor):
        target_device = doc_embeddings_tensor.device

    if not torch.is_tensor(doc_embeddings_tensor):
        doc_embeddings_tensor = torch.tensor(doc_embeddings_matrix, dtype=torch.float32).to(target_device)
    elif doc_embeddings_tensor.device != target_device:
        doc_embeddings_tensor = doc_embeddings_tensor.to(target_device)

    ordered_query_ids = list(query_embeddings_dict.keys())
    ordered_query_embeddings_list = []
    for qid_oe in ordered_query_ids:
        emb_oe = query_embeddings_dict[qid_oe]
        if not torch.is_tensor(emb_oe):
            emb_oe = torch.tensor(emb_oe, dtype=torch.float32)
        ordered_query_embeddings_list.append(emb_oe.to(target_device))
    if not ordered_query_embeddings_list: print("No query embeddings to process."); return {}, OrderedDict()
    ordered_query_embeddings = torch.stack(ordered_query_embeddings_list)

    amp_device_type = target_device.type if target_device else 'cpu' # Ensure target_device is not None
    with torch.amp.autocast(device_type=amp_device_type, enabled=(amp_device_type == 'cuda')):
         all_query_hits = similarity_fn(ordered_query_embeddings, doc_embeddings_tensor, top_k=top_k_preds_to_return)

    for i, qid in enumerate(tqdm(ordered_query_ids, desc="SBERT Evaluating (v2)")):
        query_hits_for_qid = all_query_hits[i]
        pred_ids_for_submission = [doc_ids_list[hit['corpus_id']] for hit in query_hits_for_qid[:top_k_preds_to_return]] if query_hits_for_qid else []
        all_preds[qid] = pred_ids_for_submission
        pred_ids_for_eval = [doc_ids_list[hit['corpus_id']] for hit in query_hits_for_qid[:top_k]] if query_hits_for_qid else []
        gold_docs = qrels_dict.get(qid, set()); n_gold = len(gold_docs); num_hits_at_k = 0
        if gold_docs: num_hits_at_k = sum(1 for doc_id_pred in pred_ids_for_eval if doc_id_pred in gold_docs)
        p_at_k_list.append(num_hits_at_k / top_k if top_k > 0 else 0.0)
        if n_gold > 0: r_at_k_list.append(num_hits_at_k / n_gold)
        elif num_hits_at_k == 0: r_at_k_list.append(1.0)
        else: r_at_k_list.append(0.0)
        if n_gold > 0:
            ap_score_q, current_hits_count = 0.0, 0
            for rank_idx, hit_entry in enumerate(query_hits_for_qid[:top_k]):
                doc_id_pred_eval = doc_ids_list[hit_entry['corpus_id']]
                if doc_id_pred_eval in gold_docs: current_hits_count += 1; ap_score_q += current_hits_count / (rank_idx + 1)
            ap_at_k_list.append(ap_score_q / min(n_gold, top_k) if min(n_gold, top_k) > 0 else 0.0)
        else: ap_at_k_list.append(0.0)
    final_results = {
        f'P@{top_k}': np.mean(p_at_k_list) if p_at_k_list else 0.0,
        f'R@{top_k}': np.mean(r_at_k_list) if r_at_k_list else 0.0,
        f'MAP@{top_k}': np.mean(ap_at_k_list) if ap_at_k_list else 0.0
    }
    return final_results, all_preds

print("Core utility functions for SBERT v2 defined.\n")
print("--- STEP 0 (v2) FOR SBERT EXPERIMENTS COMPLETE ---")


# ==============================================================================
# CELL: TASK B4 (v2) - SBERT (all-mpnet-base-v2) on Testing Full-Text
# ==============================================================================
print("\n\n--- TASK B4 (v2): SBERT with 'all-mpnet-base-v2' on Testing Full-Text ---")

# --- Configuration uses V2 globals from Step 0 (v2) ---
# SBERT_MODEL_NAME_V2
# FULLTEXT_TEST_ZIP_PATH
# PIPELINE_B4_V2_CACHE_DIR
# SBERT_EMBEDDING_BATCH_SIZE_V2
# MAX_WORDS_FOR_SBERT_FULLTEXT_V2
# TOP_K_PREDS_PARAM_V2
# TEST_QUERY_FILES_LIST

# --- Load SBERT Model ---
print(f"  Loading SBERT model: {SBERT_MODEL_NAME_V2}")
try:
    model_sbert_B4_v2 = SentenceTransformer(SBERT_MODEL_NAME_V2)
    if torch.cuda.is_available():
        model_sbert_B4_v2.to(torch.device("cuda"))
        print(f"    SBERT model B4 (v2) '{SBERT_MODEL_NAME_V2}' loaded. Device: {model_sbert_B4_v2.device}")
    else:
        print(f"    SBERT model B4 (v2) '{SBERT_MODEL_NAME_V2}' loaded. Device: {model_sbert_B4_v2.device} (CPU)")
except Exception as e:
    print(f"    Error loading SBERT model {SBERT_MODEL_NAME_V2}: {e}")
    model_sbert_B4_v2 = None


# --- Load Test Documents (Full-Text) ---
print(f"  Attempting to load test full-text documents for B4 (v2) from: {FULLTEXT_TEST_ZIP_PATH}")
test_doc_ids_B4_v2, test_doc_texts_B4_v2 = load_test_documents_for_sbert_v2(
    FULLTEXT_TEST_ZIP_PATH,
    extract_and_truncate_full_text_for_sbert_v2,
    PIPELINE_B4_V2_CACHE_DIR,
    f"B4_FullText_Test_Docs_{SBERT_MODEL_NAME_V2.replace('/','_')}_max{MAX_WORDS_FOR_SBERT_FULLTEXT_V2}w"
)

# --- Load Test Queries ---
all_test_queries_B4_v2 = OrderedDict()
if test_doc_ids_B4_v2:
    print(f"  Loading test queries for B4 (v2) from: {FULLTEXT_TEST_ZIP_PATH}")
    for query_file_name in TEST_QUERY_FILES_LIST:
        queries_for_file = load_test_queries_from_zip_corrected(FULLTEXT_TEST_ZIP_PATH, query_file_name)
        all_test_queries_B4_v2.update(queries_for_file)
    print(f"  Loaded a total of {len(all_test_queries_B4_v2)} test queries for B4 (v2).")
else:
    print("  Skipping query loading for B4 (v2) as document loading failed or returned empty.")

# --- Encode Test Documents (Full-Text) ---
corpus_embeddings_B4_v2_path = os.path.join(PIPELINE_B4_V2_CACHE_DIR, f'corpus_embeddings_B4_{SBERT_MODEL_NAME_V2.replace("/","_")}_max{MAX_WORDS_FOR_SBERT_FULLTEXT_V2}w.pkl')
test_corpus_embeddings_B4_v2 = None
if model_sbert_B4_v2 and test_doc_texts_B4_v2:
    if os.path.exists(corpus_embeddings_B4_v2_path):
        print(f"  Loading cached B4 (v2) corpus embeddings from {corpus_embeddings_B4_v2_path}")
        try:
            with open(corpus_embeddings_B4_v2_path, 'rb') as f: test_corpus_embeddings_B4_v2 = pickle.load(f)
            if not (isinstance(test_corpus_embeddings_B4_v2, torch.Tensor) and \
                    test_corpus_embeddings_B4_v2.shape[0] == len(test_doc_texts_B4_v2) and \
                    test_corpus_embeddings_B4_v2.shape[1] == model_sbert_B4_v2.get_sentence_embedding_dimension()):
                print(f"    Cache B4 (v2) mismatch/type or dimension error. Expected dim: {model_sbert_B4_v2.get_sentence_embedding_dimension()}. Re-encoding."); test_corpus_embeddings_B4_v2 = None
            else:
                print(f"    Loaded {test_corpus_embeddings_B4_v2.shape[0]} B4 (v2) embeddings.")
                test_corpus_embeddings_B4_v2 = test_corpus_embeddings_B4_v2.to(model_sbert_B4_v2.device)
        except Exception as e_load_emb_b4:
             print(f"    Error loading cached B4 (v2) embeddings: {e_load_emb_b4}. Re-encoding.")
             test_corpus_embeddings_B4_v2 = None

    if (test_corpus_embeddings_B4_v2 is None or (torch.is_tensor(test_corpus_embeddings_B4_v2) and test_corpus_embeddings_B4_v2.nelement() == 0)) and test_doc_texts_B4_v2:
        print(f"  Encoding {len(test_doc_texts_B4_v2)} test full-text documents for B4 (v2)...")
        with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()) if torch.cuda.is_available() else contextlib.nullcontext():
            test_corpus_embeddings_B4_v2 = model_sbert_B4_v2.encode(
                test_doc_texts_B4_v2, batch_size=SBERT_EMBEDDING_BATCH_SIZE_V2,
                show_progress_bar=True, convert_to_tensor=True
            )
        if test_corpus_embeddings_B4_v2 is not None:
            with open(corpus_embeddings_B4_v2_path, 'wb') as f: pickle.dump(test_corpus_embeddings_B4_v2, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"    Saved B4 (v2) corpus embeddings to {corpus_embeddings_B4_v2_path}")
elif not test_doc_texts_B4_v2:
    print("ERROR: No test documents loaded for B4 (v2). Cannot encode.")
elif not model_sbert_B4_v2:
    print("ERROR: SBERT model for B4 (v2) not loaded. Cannot encode.")

# --- Generate Predictions for B4 (v2) ---
sbert_predictions_B4_v2 = OrderedDict()
predictions_B4_v2_cache_path = os.path.join(PIPELINE_B4_V2_CACHE_DIR, f'predictions_B4_{SBERT_MODEL_NAME_V2.replace("/","_")}_top{TOP_K_PREDS_PARAM_V2}_max{MAX_WORDS_FOR_SBERT_FULLTEXT_V2}w.pkl')

if os.path.exists(predictions_B4_v2_cache_path):
    print(f"  Loading cached B4 (v2) predictions from: {predictions_B4_v2_cache_path}")
    try:
        with open(predictions_B4_v2_cache_path, 'rb') as f: sbert_predictions_B4_v2 = pickle.load(f)
        print(f"    Loaded {len(sbert_predictions_B4_v2)} query predictions for B4 (v2).")
    except Exception as e_load_pred_b4:
        print(f"    Error loading cached B4 (v2) predictions: {e_load_pred_b4}. Will attempt to regenerate.")
        sbert_predictions_B4_v2 = OrderedDict()

if (not sbert_predictions_B4_v2 or len(sbert_predictions_B4_v2) != len(all_test_queries_B4_v2)) and \
   all_test_queries_B4_v2 and \
   test_corpus_embeddings_B4_v2 is not None and test_corpus_embeddings_B4_v2.nelement() > 0 and \
   test_doc_ids_B4_v2 and model_sbert_B4_v2:

    print(f"  Generating SBERT predictions for {len(all_test_queries_B4_v2)} test queries (B4 v2)...")
    query_texts_B4_v2 = [clean_text_for_sbert_v2(q_text) for q_text in all_test_queries_B4_v2.values()]
    with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()) if torch.cuda.is_available() else contextlib.nullcontext():
        query_embeddings_B4_v2 = model_sbert_B4_v2.encode(
            query_texts_B4_v2, batch_size=SBERT_EMBEDDING_BATCH_SIZE_V2,
            show_progress_bar=True, convert_to_tensor=True
        )

    if query_embeddings_B4_v2.device != test_corpus_embeddings_B4_v2.device:
        test_corpus_embeddings_B4_v2 = test_corpus_embeddings_B4_v2.to(query_embeddings_B4_v2.device)

    all_hits_B4_v2 = util.semantic_search(query_embeddings_B4_v2, test_corpus_embeddings_B4_v2, top_k=TOP_K_PREDS_PARAM_V2)
    query_ids_list_B4_v2 = list(all_test_queries_B4_v2.keys())
    for i, qid in enumerate(tqdm(query_ids_list_B4_v2, desc="Processing B4 (v2) SBERT Results")):
        hits = all_hits_B4_v2[i]
        sbert_predictions_B4_v2[qid] = [test_doc_ids_B4_v2[hit['corpus_id']] for hit in hits] if hits else []

    if sbert_predictions_B4_v2:
        with open(predictions_B4_v2_cache_path, 'wb') as f: pickle.dump(sbert_predictions_B4_v2, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  Saved B4 (v2) SBERT predictions to {predictions_B4_v2_cache_path}")
elif sbert_predictions_B4_v2 and len(sbert_predictions_B4_v2) == len(all_test_queries_B4_v2):
     print(f"  B4 (v2) SBERT predictions already loaded from cache and match current query set.")
else:
    if not (all_test_queries_B4_v2 and test_corpus_embeddings_B4_v2 is not None and test_corpus_embeddings_B4_v2.nelement() > 0 and test_doc_ids_B4_v2 and model_sbert_B4_v2):
        print("ERROR: Cannot generate B4 (v2) predictions due to missing prerequisites (queries, doc embeddings, doc IDs, or model).")
    elif sbert_predictions_B4_v2:
        print("INFO: Loaded B4 (v2) predictions from cache, but they do not match the current query set. No re-generation triggered based on current logic.")

if sbert_predictions_B4_v2:
    print(f"\n  Sample B4 (v2) SBERT Predictions (Top 3 of {TOP_K_PREDS_PARAM_V2} shown):")
    count_sample_b4_v2 = 0
    for q_id, preds_list in sbert_predictions_B4_v2.items():
        print(f"    QID {q_id} Top 3: {preds_list[:3]}"); count_sample_b4_v2 += 1
        if count_sample_b4_v2 >= 3: break
else: print("  No B4 (v2) SBERT predictions available to display.")
gc.collect()
print("\n--- Task B4 (v2) Complete ---")



--- Mounting Google Drive ---
Mounted at /content/drive
Google Drive mounted successfully.

--- Checking/Installing Required Libraries ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

    SBERT model B4 (v2) 'all-mpnet-base-v2' loaded. Device: cuda:0
  Attempting to load test full-text documents for B4 (v2) from: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_fulltext.zip
  Attempting to load cached raw data for B4_FullText_Test_Docs_all-mpnet-base-v2_max384w from: /content/drive/MyDrive/colab-4/cache_B4_v2_fulltext_sbert_mpnet_test/raw_data_cache/loaded_data_B4_FullText_Test_Docs_all-mpnet-base-v2_max384w_max384w.pkl
    Successfully loaded 1524045 docs & data for B4_FullText_Test_Docs_all-mpnet-base-v2_max384w from cache.
  Loading test queries for B4 (v2) from: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_fulltext.zip
  Loaded a total of 552 test queries for B4 (v2).
  Loading cached B4 (v2) corpus embeddings from /content/drive/MyDrive/colab-4/cache_B4_v2_fulltext_sbert_mpnet_test/corpus_embeddings_B4_all-mpnet-base-v2_max384w.pkl
    Loaded 1524045 B4 (v2) embeddings.
  Generating SBERT predictions for 552 test queries (B4 v2)...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Processing B4 (v2) SBERT Results:   0%|          | 0/552 [00:00<?, ?it/s]

  Saved B4 (v2) SBERT predictions to /content/drive/MyDrive/colab-4/cache_B4_v2_fulltext_sbert_mpnet_test/predictions_B4_all-mpnet-base-v2_top100_max384w.pkl

  Sample B4 (v2) SBERT Predictions (Top 3 of 100 shown):
    QID d585f080-4519-4952-8278-0d13fcd03fec Top 3: ['104463117', '150160329', '148660071']
    QID 254ecdb5-45e5-45ce-9b3f-492d1e1c085e Top 3: ['25027878', '72181450', '25027878']
    QID eaa374bf-cd61-4d9f-9b99-87f3d81f2636 Top 3: ['103917394', '54268947', '2978572']

--- Task B4 (v2) Complete ---


In [ ]:
# CELL 4: SECTION 1 - TASK A5 (BM25 Abstracts) & C1 (CrossEncoder on Abstracts)
# ==================================================================================================================
print("\n--- CELL 4: SECTION 1 - TASK A5 & C1 on Abstracts ---")
preds_A5, preds_C1 = OrderedDict(), OrderedDict()
metrics_A5, metrics_C1 = {}, {}
ids_A5, texts_A5_ce, queries_A5, qrels_A5, ok_A5, texts_A5_bm25 = load_data_for_tasks(
    ABSTRACT_TRAIN_ZIP_PATH, extract_abstract_text_for_ce, MAX_WORDS_FOR_CE_ABSTRACT,
    extract_abstract_text_for_bm25, None, PIPELINE_A5_CACHE_DIR, "A5_Abstract_Train"
)
if ok_A5:
    tokenized_corpus_A5 = [tokenize_for_bm25(t) for t in tqdm(texts_A5_bm25, desc="Tokenizing A5")]
    del texts_A5_bm25; gc.collect()
    bm25_A5 = initialize_bm25_retriever(tokenized_corpus_A5)
    if bm25_A5:
        preds_A5 = run_bm25_pipeline(bm25_A5, queries_A5, ids_A5, PIPELINE_A5_CACHE_DIR, "A5")
        if preds_A5 :
            print("\n--- BM25 Abstracts (A5) Metrics ---")
            metrics_A5 = calculate_metrics_custom(preds_A5, qrels_A5, k=TOP_K_EVAL_PARAM)
            for n, v in metrics_A5.items(): print(f"  {n}: {v:.4f}")
            metrics_C1_tuple = setup_cross_encoder_pipeline_combined(
                CROSS_ENCODER_MODEL_NAME_FOR_C_TASKS, preds_A5, queries_A5, texts_A5_ce, qrels_A5,
                PIPELINE_C1_CACHE_DIR, "C1_Abstract_Rerank"
            )
            if metrics_C1_tuple and isinstance(metrics_C1_tuple, tuple) and len(metrics_C1_tuple) == 2:
                 metrics_C1, preds_C1 = metrics_C1_tuple
            else: print("Warning: C1 setup_cross_encoder_pipeline_combined did not return expected tuple or failed."); preds_C1 = OrderedDict()
        else: print("Skipping C1: BM25 (A5) predictions are empty or failed.")
    else: print("Skipping A5 pipeline: BM25 model initialization failed.")
else: print("Skipping A5/C1: data load failed.")
gc.collect()
print("--- CELL 4: SECTION 1 Complete ---\n")



--- CELL 4: SECTION 1 - TASK A5 & C1 on Abstracts ---
Loaded from cache: /content/drive/MyDrive/colab-4/cache_A5_abstract_bm25_train/raw_data_cache/loaded_data_A5_Abstract_Train_CE_510w_BM25_Fullw.pkl


Tokenizing A5:   0%|          | 0/2014265 [00:00<?, ?it/s]

Initializing BM25Plus on 2014265 docs.
Loaded BM25 predictions from cache: /content/drive/MyDrive/colab-4/cache_A5_abstract_bm25_train/predictions_A5_bm25_top100.pkl

--- BM25 Abstracts (A5) Metrics ---
  P@5: 0.2087
  R@5: 0.1367
  MAP@5: 0.1486
  MRR: 0.3669
--- Running new CE pipeline for C1_Abstract_Rerank ---


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

CE C1_Abstract_Rerank:   0%|          | 0/393 [00:00<?, ?it/s]

Saved new CE predictions to: /content/drive/MyDrive/colab-4/cache_C1_abstract_crossencoder_A5_rerank_v2/predictions_CE_C1_Abstract_Rerank_cross-encoder_ms-marco-MiniLM-L-12-v2.pkl
--- New CE Metrics for C1_Abstract_Rerank (Top 5) ---
  P@5: 0.1573
  R@5: 0.0998
  MAP@5: 0.1048
  MRR: 0.3209
--- CELL 4: SECTION 1 Complete ---



In [ ]:
print("\n--- CELL 5: SECTION 2 - TASK A6 & C2 on Full-Text ---")
preds_A6, preds_C2 = OrderedDict(), OrderedDict()
metrics_A6, metrics_C2 = {}, {}
ids_A6, texts_A6_ce, queries_A6, qrels_A6, ok_A6, texts_A6_bm25 = load_data_for_tasks(
    FULLTEXT_TRAIN_ZIP_PATH, extract_and_truncate_full_text_for_ce, MAX_WORDS_FOR_CE_FULLTEXT,
    extract_and_truncate_full_text_for_bm25, MAX_WORDS_FOR_BM25_FULLTEXT, PIPELINE_A6_CACHE_DIR, "A6_FullText_Train"
)
if ok_A6:
    tokenized_corpus_A6 = [tokenize_for_bm25(t) for t in tqdm(texts_A6_bm25, desc="Tokenizing A6")]
    del texts_A6_bm25; gc.collect()
    bm25_A6 = initialize_bm25_retriever(tokenized_corpus_A6)
    if bm25_A6:
        preds_A6 = run_bm25_pipeline(bm25_A6, queries_A6, ids_A6, PIPELINE_A6_CACHE_DIR, "A6")
        if preds_A6:
            print("\n--- BM25 Full-Text (A6) Metrics ---")
            metrics_A6 = calculate_metrics_custom(preds_A6, qrels_A6, k=TOP_K_EVAL_PARAM)
            for n, v in metrics_A6.items(): print(f"  {n}: {v:.4f}")
            metrics_C2_tuple = setup_cross_encoder_pipeline_combined(
                CROSS_ENCODER_MODEL_NAME_FOR_C_TASKS, preds_A6, queries_A6, texts_A6_ce, qrels_A6,
                PIPELINE_C2_CACHE_DIR, "C2_FullText_Rerank"
            )
            if metrics_C2_tuple and isinstance(metrics_C2_tuple, tuple) and len(metrics_C2_tuple) == 2:
                metrics_C2, preds_C2 = metrics_C2_tuple
            else: print("Warning: C2 setup_cross_encoder_pipeline_combined did not return expected tuple or failed."); preds_C2 = OrderedDict()
        else: print("Skipping C2: BM25 (A6) predictions are empty or failed.")
    else: print("Skipping A6 pipeline: BM25 model initialization failed.")
else: print("Skipping A6/C2: data load failed.")
gc.collect()
print("--- CELL 5: SECTION 2 Complete ---\n")



--- CELL 5: SECTION 2 - TASK A6 & C2 on Full-Text ---
Loading data from longeval_sci_training_2025_fulltext.zip


Reading JSONLs:   0%|          | 0/21 [00:00<?, ?it/s]

Processed 2014265 docs, 393 queries, 393 qrels entries.
Saved cache to /content/drive/MyDrive/colab-4/cache_A6_fulltext_bm25_train/raw_data_cache/loaded_data_A6_FullText_Train_CE_510w_BM25_400w.pkl


Tokenizing A6:   0%|          | 0/2014265 [00:00<?, ?it/s]

Initializing BM25Plus on 2014265 docs.


BM25 A6:   0%|          | 0/393 [00:00<?, ?it/s]

Saved new BM25 predictions to: /content/drive/MyDrive/colab-4/cache_A6_fulltext_bm25_train/predictions_A6_bm25_top100.pkl

--- BM25 Full-Text (A6) Metrics ---
  P@5: 0.2051
  R@5: 0.1340
  MAP@5: 0.1483
  MRR: 0.3738
--- Running new CE pipeline for C2_FullText_Rerank ---


CE C2_FullText_Rerank:   0%|          | 0/393 [00:00<?, ?it/s]

Saved new CE predictions to: /content/drive/MyDrive/colab-4/cache_C2_fulltext_crossencoder_A6_rerank/predictions_CE_C2_FullText_Rerank_cross-encoder_ms-marco-MiniLM-L-12-v2.pkl
--- New CE Metrics for C2_FullText_Rerank (Top 5) ---
  P@5: 0.1613
  R@5: 0.1030
  MAP@5: 0.1069
  MRR: 0.3217
--- CELL 5: SECTION 2 Complete ---



In [ ]:
# CELL: TASK C3 – Neural Re‐ranking on Test Abstracts (USING L-12-v2 MODEL - AUTOCAST FIX)

import os
import zipfile
import orjson
import pickle
import gc
import contextlib # Make sure contextlib is imported
from collections import OrderedDict

import torch
from torch.amp import autocast # Corrected import for modern PyTorch
from sentence_transformers.cross_encoder import CrossEncoder

# --- 0. Mount Google Drive ---
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
print("-" * 50)


# --- Configuration ---
BASE_DIR                               = '/content/drive/MyDrive/colab-4'
TEST_ABS_ZIP                           = os.path.join(BASE_DIR, 'longeval_sci_testing_2025_abstract.zip')
PIPELINE_A7_CACHE_DIR                  = os.path.join(BASE_DIR, 'cache_A7_abstract_bm25_test')

PIPELINE_C3_L12_CACHE_DIR             = os.path.join(BASE_DIR, 'cache_C3_abstract_crossencoder_L12_test')
os.makedirs(PIPELINE_C3_L12_CACHE_DIR, exist_ok=True)
os.makedirs(os.path.join(PIPELINE_C3_L12_CACHE_DIR, "raw_data_cache"), exist_ok=True)

CROSS_ENCODER_MODEL_NAME_C3_L12        = 'cross-encoder/ms-marco-MiniLM-L-12-v2'

RERANK_BATCH_SIZE_C3                   = 128
TOP_K_PREDS_PARAM                      = 100
TOP_K_OUTPUT                           = 5

# --- 2. Load test abstracts from ZIP ---
print("Loading test abstracts for C3 (L-12-v2 experiment)…")
test_abstracts_cache_path = os.path.join(PIPELINE_C3_L12_CACHE_DIR, "raw_data_cache", "test_abstracts_map_c3_l12.pkl")
if os.path.exists(test_abstracts_cache_path):
    with open(test_abstracts_cache_path, 'rb') as f:
        test_abstracts = pickle.load(f)
    print(f"  Loaded {len(test_abstracts)} abstracts from L-12 cache.")
else:
    test_abstracts = {}
    with zipfile.ZipFile(TEST_ABS_ZIP, 'r') as z:
        for info in z.infolist():
            if not info.filename.startswith('longeval_sci_testing_2025_abstract/documents/') or not info.filename.endswith('.jsonl'):
                continue
            with z.open(info) as f:
                for raw in f:
                    try:
                        data   = orjson.loads(raw)
                        doc_id = data.get('_id') or data.get('id')
                        if not doc_id: continue
                        title = data.get('title') or ''
                        abst  = data.get('abstractText') or data.get('abstract') or ''
                        text  = (title + ' ' + abst).strip()
                        if text:
                            test_abstracts[str(doc_id)] = text
                    except:
                        continue
    with open(test_abstracts_cache_path, 'wb') as f:
        pickle.dump(test_abstracts, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Loaded and cached {len(test_abstracts)} abstracts for L-12 experiment.")

# --- 3. Load test queries from ZIP ---
print("Loading test queries for C3 from ZIP…")
all_test_queries_c3 = OrderedDict()
with zipfile.ZipFile(TEST_ABS_ZIP, 'r') as z:
    txt_files = [
        n for n in z.namelist()
        if n.startswith('longeval_sci_testing_2025_abstract/queries_') and n.lower().endswith('.txt')
    ]
    if not txt_files:
        print("  ⚠️ WARNING: No query .txt files found with the specified path pattern inside the ZIP.")

    for fname in txt_files:
        with z.open(fname) as f:
            for raw in f:
                line = raw.decode('utf-8').strip()
                if not line: continue
                if '\t' in line:
                    qid, txt = line.split('\t', 1)
                else:
                    qid, txt = line.split(' ', 1)
                all_test_queries_c3[qid] = txt
print(f"  Loaded {len(all_test_queries_c3)} queries from {len(txt_files)} .txt files.")

# --- 4. Load first-stage A7 candidates ---
print("Loading A7 candidates for C3…")
first_stage_candidates_c3 = OrderedDict()
a7_candidate_files = [
    'predictions_A7_Abstract_Test_queries_2024-11_test.pkl',
    'predictions_A7_Abstract_Test_queries_2025-01_test.pkl'
]
for fname in a7_candidate_files:
    path = os.path.join(PIPELINE_A7_CACHE_DIR, fname)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            preds = pickle.load(f)
        first_stage_candidates_c3.update(preds)
        print(f"  Loaded {len(preds)} candidates from {fname}")
    else:
        print(f"  WARNING: missing A7 candidate file {path}")
print(f"  Total queries with candidates: {len(first_stage_candidates_c3)}")

common_qids_list = []
if all_test_queries_c3 and first_stage_candidates_c3:
    common_qids = set(all_test_queries_c3.keys()) & set(first_stage_candidates_c3.keys())
    all_test_queries_c3     = OrderedDict((qid, all_test_queries_c3[qid]) for qid in common_qids if qid in all_test_queries_c3)
    first_stage_candidates_c3 = OrderedDict((qid, first_stage_candidates_c3[qid]) for qid in common_qids if qid in first_stage_candidates_c3)
    common_qids_list = list(all_test_queries_c3.keys())
    print(f"  {len(common_qids_list)} common QIDs for re‐ranking.")
else:
    print("  No common QIDs found or candidates/queries missing. Re-ranking will be skipped.")

# --- 5. Load or compute re‐ranked outputs ---
cache_path = os.path.join(
    PIPELINE_C3_L12_CACHE_DIR,
    f'reranked_C3_{CROSS_ENCODER_MODEL_NAME_C3_L12.replace("/","_")}.pkl'
)

loaded = False
# Always recompute for clarity
if not loaded and common_qids_list:
    print(f"Loading CrossEncoder model: {CROSS_ENCODER_MODEL_NAME_C3_L12} for C3 (L-12)...")
    ce = CrossEncoder(CROSS_ENCODER_MODEL_NAME_C3_L12)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ce.model.to(device)
    print(f"  Cross-Encoder for C3 (L-12) loaded. Model is on device: {device}")
    reranked_preds_c3 = OrderedDict()

    print(f"Re-ranking candidates for {len(all_test_queries_c3)} queries using {CROSS_ENCODER_MODEL_NAME_C3_L12} (Abstracts)...")

    query_count_l12 = 0
    for qid, query_text in all_test_queries_c3.items():
        query_count_l12 +=1
        if query_count_l12 % 50 == 0:
            print(f"  Processing C3 (L-12) query {query_count_l12}/{len(all_test_queries_c3)}: {qid}")

        candidates = first_stage_candidates_c3.get(qid, [])[:TOP_K_PREDS_PARAM]
        pairs, ids = [], []
        for doc_id in candidates:
            text = test_abstracts.get(str(doc_id))
            if text:
                pairs.append([query_text, text])
                ids.append(doc_id)
        if not pairs:
            reranked_preds_c3[qid] = []
            continue

        # === THIS IS THE CORRECTED AUTOCAST SECTION ===
        if device.type == 'cuda':
            # For PyTorch 1.6+, autocast is part of torch.cuda.amp or torch.amp
            # The FutureWarning suggested torch.amp.autocast('cuda', args...)
            # Using dtype=torch.float16 is common for performance with autocast on CUDA
            amp_ctx = autocast(device_type='cuda', dtype=torch.float16)
        else:
            amp_ctx = contextlib.nullcontext()
        # ==============================================

        with amp_ctx:
            scores = ce.predict(pairs, batch_size=RERANK_BATCH_SIZE_C3, show_progress_bar=False)

        scored = sorted(zip(ids, scores.tolist()), key=lambda x: x[1], reverse=True)
        reranked_preds_c3[qid] = scored[:TOP_K_OUTPUT]

    with open(cache_path, 'wb') as f:
        pickle.dump(reranked_preds_c3, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved re-ranked results (L-12 model) to cache: {cache_path}")
elif not common_qids_list:
    print("Skipping re-ranking as no common QIDs were found between test queries and A7 candidates.")
    reranked_preds_c3 = OrderedDict()

# --- 6. Sample output ---
print("\nSample C3 (L-12 model) re-ranked top-3 for first 3 queries:")
if reranked_preds_c3:
    for i, (qid, scored) in enumerate(reranked_preds_c3.items()):
        print(f"  QID {qid}: {[doc for doc, _ in scored[:3]]}")
        if i >= 2:
            break
else:
    print("  No L-12 predictions to display for C3.")

gc.collect()
print("\n--- TASK C3 (using L-12-v2 model) Complete ---")

Mounted at /content/drive
Google Drive mounted successfully.
--------------------------------------------------
Loading test abstracts for C3 (L-12-v2 experiment)…
  Loaded 1213723 abstracts from L-12 cache.
Loading test queries for C3 from ZIP…
  Loaded 552 queries from 2 .txt files.
Loading A7 candidates for C3…
  Loaded 97 candidates from predictions_A7_Abstract_Test_queries_2024-11_test.pkl
  Loaded 481 candidates from predictions_A7_Abstract_Test_queries_2025-01_test.pkl
  Total queries with candidates: 539
  539 common QIDs for re‐ranking.
Loading CrossEncoder model: cross-encoder/ms-marco-MiniLM-L-12-v2 for C3 (L-12)...
  Cross-Encoder for C3 (L-12) loaded. Model is on device: cuda
Re-ranking candidates for 539 queries using cross-encoder/ms-marco-MiniLM-L-12-v2 (Abstracts)...
  Processing C3 (L-12) query 50/539: fc03ea0c-ee25-448f-b720-f7a49f47e641
  Processing C3 (L-12) query 100/539: 801735ce-f8a2-46e0-89d6-c643d1cad85d
  Processing C3 (L-12) query 150/539: 7c6a1ca0-d936-4ac6

In [ ]:
# CELL: TASK C4 – Neural Re‐ranking on Test Full-Text (USING L-12-v2 MODEL - FULL SCRIPT)

import os
import zipfile
import orjson
import pickle
import gc
import contextlib
from collections import OrderedDict
import time # For timestamps

import torch
from torch.amp import autocast # Corrected import for modern PyTorch
from sentence_transformers.cross_encoder import CrossEncoder

# --- 0. Mount Google Drive ---
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
print("-" * 50)

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# --- Configuration ---
BASE_DIR                               = '/content/drive/MyDrive/colab-4'
TEST_FULLTEXT_ZIP                      = os.path.join(BASE_DIR, 'longeval_sci_testing_2025_fulltext.zip')
PIPELINE_A8_CACHE_DIR                  = os.path.join(BASE_DIR, 'cache_A8_fulltext_bm25_test')

# === Cache directory for THIS L-12-v2 C4 experiment ===
PIPELINE_C4_L12_CACHE_DIR             = os.path.join(BASE_DIR, 'cache_C4_fulltext_crossencoder_L12_test')
os.makedirs(PIPELINE_C4_L12_CACHE_DIR, exist_ok=True)
os.makedirs(os.path.join(PIPELINE_C4_L12_CACHE_DIR, "raw_data_cache"), exist_ok=True)

# === Cross-Encoder model for THIS C4 experiment ===
CROSS_ENCODER_MODEL_NAME_C4_L12        = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
# =========================================================

RERANK_BATCH_SIZE_C4                   = 128 # Adjust if OOM on GPU with full-text
TOP_K_PREDS_PARAM                      = 100  # number of A8 candidates to re‐rank
TOP_K_OUTPUT                           = 100   # Save all re-ranked candidates up to TOP_K_PREDS_PARAM for C4
MAX_WORDS_FOR_CE_FULLTEXT              = 510  # Max words for truncating full-text

# --- 2. Load test full-texts from ZIP ---
print(f"{get_timestamp()} Loading test full-texts for C4 (L-12-v2 experiment)…")
test_fulltexts_cache_path = os.path.join(PIPELINE_C4_L12_CACHE_DIR, "raw_data_cache", "test_fulltexts_map_c4_l12.pkl")

test_full_texts = {} # Initialize
if os.path.exists(test_fulltexts_cache_path):
    try:
        with open(test_fulltexts_cache_path, 'rb') as f:
            test_full_texts = pickle.load(f)
        print(f"{get_timestamp()}   SUCCESS: Loaded {len(test_full_texts)} full-texts from L-12 cache: {test_fulltexts_cache_path}")
    except Exception as e_cache:
        print(f"{get_timestamp()}   WARNING: Found L-12 cache file, but failed to load it: {e_cache}. Will try to rebuild.")
        test_full_texts = {}
else:
    print(f"{get_timestamp()}   NOTE: L-12 full-text cache not found at {test_fulltexts_cache_path}.")

if not test_full_texts: # If cache was not loaded or failed to load
    print(f"{get_timestamp()} Building full-text map from ZIP...")
    if not os.path.exists(TEST_FULLTEXT_ZIP):
        print(f"{get_timestamp()}   FATAL ERROR: Full-text ZIP file not found at {TEST_FULLTEXT_ZIP}")
    else:
        print(f"{get_timestamp()}   Opening ZIP file: {TEST_FULLTEXT_ZIP}")
        with zipfile.ZipFile(TEST_FULLTEXT_ZIP, 'r') as z:
            jsonl_files_in_zip = [
                info for info in z.infolist()
                if info.filename.startswith('longeval_sci_testing_2025_fulltext/documents/') and \
                   info.filename.endswith('.jsonl') and \
                   not info.is_dir()
            ]
            total_jsonl_files = len(jsonl_files_in_zip)
            print(f"{get_timestamp()}   Found {total_jsonl_files} JSONL files to process in ZIP.")

            docs_processed_total = 0
            for i, file_info in enumerate(jsonl_files_in_zip):
                print(f"{get_timestamp()}   Processing JSONL file {i+1}/{total_jsonl_files}: {file_info.filename}")
                docs_in_current_file = 0
                with z.open(file_info) as f:
                    for line_number, raw in enumerate(f):
                        try:
                            data   = orjson.loads(raw)
                            doc_id = data.get('_id') or data.get('id')
                            if not doc_id: continue

                            title = data.get('title', '') or ''
                            abstract = data.get('abstractText', data.get('abstract', '')) or ''
                            body_text = ""
                            if 'sections' in data and data['sections']:
                                body_text = " ".join(section.get('text', '') for section in data['sections'] if section.get('text'))
                            elif 'text' in data:
                                body_text = data.get('text', '') or ''

                            full_content = (title + " " + abstract + " " + body_text).strip()

                            if full_content:
                                test_full_texts[str(doc_id)] = full_content
                                docs_processed_total += 1
                                docs_in_current_file += 1
                                if docs_in_current_file % 20000 == 0: # Log every 20000 docs
                                     print(f"{get_timestamp()}     ... processed {docs_in_current_file} docs from {file_info.filename} (total map size: {len(test_full_texts)})")
                        except Exception as e_doc:
                            continue
                print(f"{get_timestamp()}     Finished {file_info.filename}, loaded {docs_in_current_file} docs. (Total map size: {len(test_full_texts)})")

        if test_full_texts:
            with open(test_fulltexts_cache_path, 'wb') as f:
                pickle.dump(test_full_texts, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"{get_timestamp()}   Loaded and cached a total of {len(test_full_texts)} full-texts to new L-12 cache: {test_fulltexts_cache_path}.")
        else:
            print(f"{get_timestamp()}   No documents were loaded from ZIP. Cache not saved.")

if not test_full_texts:
    print(f"{get_timestamp()} FATAL: No full-text documents were loaded. C4 re-ranking cannot proceed.")
else:
    # --- 3. Load test queries from ZIP ---
    print(f"{get_timestamp()} Loading test queries for C4 from ZIP…")
    all_test_queries_c4 = OrderedDict()
    if not os.path.exists(TEST_FULLTEXT_ZIP):
        print(f"{get_timestamp()}   FATAL ERROR: Query ZIP file not found at {TEST_FULLTEXT_ZIP}")
    else:
        with zipfile.ZipFile(TEST_FULLTEXT_ZIP, 'r') as z:
            txt_files = [
                n for n in z.namelist()
                if n.startswith('longeval_sci_testing_2025_fulltext/queries_') and n.lower().endswith('.txt')
            ]
            if not txt_files:
                print(f"{get_timestamp()}   ⚠️ WARNING: No query .txt files found with the specified path pattern inside the ZIP.")

            for fname in txt_files:
                with z.open(fname) as f:
                    for raw in f:
                        line = raw.decode('utf-8').strip()
                        if not line: continue
                        if '\t' in line:
                            qid, txt = line.split('\t', 1)
                        else:
                            qid, txt = line.split(' ', 1)
                        all_test_queries_c4[qid] = txt
    print(f"{get_timestamp()}   Loaded {len(all_test_queries_c4)} queries from {len(txt_files)} .txt files.")

    # --- 4. Load first-stage A8 candidates ---
    print(f"{get_timestamp()} Loading A8 candidates for C4…")
    first_stage_candidates_c4 = OrderedDict()
    a8_candidate_files = [
        'predictions_A8_FullText_Test_queries_2024-11_test.pkl',
        'predictions_A8_FullText_Test_queries_2025-01_test.pkl'
    ]
    for fname in a8_candidate_files:
        path = os.path.join(PIPELINE_A8_CACHE_DIR, fname)
        if os.path.exists(path):
            with open(path, 'rb') as f:
                preds = pickle.load(f)
            first_stage_candidates_c4.update(preds)
            print(f"{get_timestamp()}   Loaded {len(preds)} candidates from {fname}")
        else:
            print(f"{get_timestamp()}   WARNING: missing A8 candidate file {path}")
    print(f"{get_timestamp()}   Total queries with candidates: {len(first_stage_candidates_c4)}")

    common_qids_list_c4 = []
    if all_test_queries_c4 and first_stage_candidates_c4:
        common_qids_c4 = set(all_test_queries_c4.keys()) & set(first_stage_candidates_c4.keys())
        all_test_queries_c4     = OrderedDict((qid, all_test_queries_c4[qid]) for qid in common_qids_c4 if qid in all_test_queries_c4)
        first_stage_candidates_c4 = OrderedDict((qid, first_stage_candidates_c4[qid]) for qid in common_qids_c4 if qid in first_stage_candidates_c4)
        common_qids_list_c4 = list(all_test_queries_c4.keys())
        print(f"{get_timestamp()}   {len(common_qids_list_c4)} common QIDs for re‐ranking.")
    else:
        print(f"{get_timestamp()}   No common QIDs found or candidates/queries missing. Re-ranking will be skipped for C4.")

    # --- 5. Load or compute re‐ranked outputs ---
    cache_path_c4 = os.path.join(
        PIPELINE_C4_L12_CACHE_DIR,
        f'reranked_C4_{CROSS_ENCODER_MODEL_NAME_C4_L12.replace("/","_")}_max{MAX_WORDS_FOR_CE_FULLTEXT}w.pkl'
    )

    reranked_preds_c4 = OrderedDict() # Initialize
    loaded_c4 = False
    # Always recompute for this script unless you re-enable cache loading specifically for reranked_preds_c4
    if not loaded_c4 and common_qids_list_c4 and test_full_texts:
        print(f"{get_timestamp()} Loading CrossEncoder model: {CROSS_ENCODER_MODEL_NAME_C4_L12} for C4 (L-12)...")
        ce_c4 = CrossEncoder(CROSS_ENCODER_MODEL_NAME_C4_L12)
        device_c4 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        ce_c4.model.to(device_c4)
        print(f"{get_timestamp()}   Cross-Encoder for C4 (L-12) loaded. Model is on device: {device_c4}")

        print(f"{get_timestamp()} Re-ranking candidates for {len(all_test_queries_c4)} queries using {CROSS_ENCODER_MODEL_NAME_C4_L12} (Full-Text, Truncated)...")

        query_count_c4_l12 = 0
        for qid, query_text in all_test_queries_c4.items():
            query_count_c4_l12 +=1
            if query_count_c4_l12 % 10 == 0:
                print(f"{get_timestamp()}   Processing C4 (L-12) query {query_count_c4_l12}/{len(all_test_queries_c4)}: {qid}")

            candidates = first_stage_candidates_c4.get(qid, [])[:TOP_K_PREDS_PARAM]
            pairs, ids = [], []
            for doc_id_candidate in candidates: # Renamed to avoid clash
                doc_text_full = test_full_texts.get(str(doc_id_candidate)) # Ensure key is string
                if doc_text_full:
                    doc_text_truncated = " ".join(doc_text_full.split()[:MAX_WORDS_FOR_CE_FULLTEXT])
                    pairs.append([query_text, doc_text_truncated])
                    ids.append(doc_id_candidate) # Use original doc_id from candidates
                # else:
                #     print(f"    DEBUG: Text for doc_id {doc_id_candidate} not found in test_full_texts map for QID {qid}")


            if not pairs:
                # print(f"    INFO: No valid (text found) candidates for QID {qid} to re-rank.")
                reranked_preds_c4[qid] = []
                continue

            if device_c4.type == 'cuda':
                amp_ctx_c4 = autocast(device_type='cuda', dtype=torch.float16)
            else:
                amp_ctx_c4 = contextlib.nullcontext()

            with amp_ctx_c4:
                scores = ce_c4.predict(pairs, batch_size=RERANK_BATCH_SIZE_C4, show_progress_bar=False)

            scored = sorted(zip(ids, scores.tolist()), key=lambda x: x[1], reverse=True)
            reranked_preds_c4[qid] = scored[:TOP_K_OUTPUT]

        with open(cache_path_c4, 'wb') as f:
            pickle.dump(reranked_preds_c4, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"{get_timestamp()} Saved re-ranked results (L-12 model, Full-Text) to cache: {cache_path_c4}")

    elif not common_qids_list_c4:
        print(f"{get_timestamp()} Skipping C4 re-ranking as no common QIDs were found.")
    elif not test_full_texts: # This condition should ideally be caught earlier
        print(f"{get_timestamp()} Skipping C4 re-ranking as no full-text documents were loaded.")


    # --- 6. Sample output ---
    print(f"\n{get_timestamp()} Sample C4 (L-12 model) re-ranked top-3 for first 3 queries:")
    if reranked_preds_c4:
        for i, (qid, scored_docs) in enumerate(reranked_preds_c4.items()): # Renamed to avoid clash
            print(f"  QID {qid}: {[doc_id_out for doc_id_out, _ in scored_docs[:3]]}") # Renamed to avoid clash
            if i >= 2:
                break
    else:
        print("  No L-12 predictions to display for C4.")

gc.collect()
print(f"\n{get_timestamp()} --- TASK C4 (using L-12-v2 model) Complete ---")

Mounted at /content/drive
Google Drive mounted successfully.
--------------------------------------------------
2025-05-25 11:59:10 Loading test full-texts for C4 (L-12-v2 experiment)…
2025-05-25 11:59:10   NOTE: L-12 full-text cache not found at /content/drive/MyDrive/colab-4/cache_C4_fulltext_crossencoder_L12_test/raw_data_cache/test_fulltexts_map_c4_l12.pkl.
2025-05-25 11:59:10 Building full-text map from ZIP...
2025-05-25 11:59:10   Opening ZIP file: /content/drive/MyDrive/colab-4/longeval_sci_testing_2025_fulltext.zip
2025-05-25 11:59:10   Found 16 JSONL files to process in ZIP.
2025-05-25 11:59:10   Processing JSONL file 1/16: longeval_sci_testing_2025_fulltext/documents/documents_000001.jsonl
2025-05-25 11:59:40     ... processed 20000 docs from longeval_sci_testing_2025_fulltext/documents/documents_000001.jsonl (total map size: 20000)
2025-05-25 12:00:07     ... processed 40000 docs from longeval_sci_testing_2025_fulltext/documents/documents_000001.jsonl (total map size: 40000)

In [ ]:
import os
import pickle
from collections import OrderedDict

from google.colab import drive
import os

try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully at /content/drive")

    # You can also immediately check if your base project path exists
    DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"
    if os.path.exists(DRIVE_BASE_PATH):
        print(f"Your project base path exists: {DRIVE_BASE_PATH}")
    else:
        print(f"⚠️ WARNING: Your project base path was NOT found: {DRIVE_BASE_PATH}")
        print("Please double-check the DRIVE_BASE_PATH variable in your scripts.")

except Exception as e:
    print(f"Error mounting Google Drive: {e}")

print("\n--- End of Mount Script ---")

# ==============================================================================
# 1. SETUP - CONFIGURE YOUR VARIABLES HERE
# ==============================================================================
# Define your team name
GROUP_ID = "clef25-sambs"

# Define the base path for your project on Google Drive
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"

# Define the directory where your final TREC run files will be saved
# A new folder will be created if it doesn't exist.
SUBMISSION_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_runs_A_only")

# Define the top-k value used when generating predictions (especially for BM25)
# This is used to find the correct .pkl filename for A5 and A6.
TOP_K_PREDS_PARAM = 100

# Define the list of test query files. This is used for A3, A4, A7, A8.
TEST_QUERY_FILES_LIST = ["queries_2024-11_test.txt", "queries_2025-01_test.txt"]

# --- Define cache directories for each 'A' pipeline ---
# (I have filled these in with the correct folder names from your output)
PIPELINE_A1_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A1_abstract_tfidf_train/")
PIPELINE_A2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A2_fulltext_tfidf_train/")
PIPELINE_A3_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A3_abstract_tfidf_test/")
PIPELINE_A4_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A4_fulltext_tfidf_test/")
PIPELINE_A5_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A5_abstract_bm25_train/")
PIPELINE_A6_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A6_fulltext_bm25_train/")
PIPELINE_A7_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A7_abstract_bm25_test/")
PIPELINE_A8_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_A8_fulltext_bm25_test/")


# ==============================================================================
# 2. HELPER FUNCTIONS (FROM YOUR NOTEBOOK)
# ==============================================================================

def format_and_save_trec_run(
    predictions_dict,
    run_id,
    output_dir,
    output_filename,
    max_docs_per_query=100,
    generate_pseudo_scores=False
    ):
    """Formats predictions into TREC run file format and saves it."""
    os.makedirs(output_dir, exist_ok=True)
    output_file_path = os.path.join(output_dir, output_filename)

    print(f"Formatting predictions for TREC run file: {output_file_path}")
    lines_written = 0
    queries_processed = 0

    with open(output_file_path, 'w') as f_out:
        if not isinstance(predictions_dict, OrderedDict):
            predictions_dict_ordered = OrderedDict(sorted(predictions_dict.items()))
        else:
            predictions_dict_ordered = predictions_dict

        for qid, ranked_items in predictions_dict_ordered.items():
            queries_processed += 1
            if not ranked_items:
                continue

            for rank, item in enumerate(ranked_items[:max_docs_per_query], 1):
                doc_id = None
                score = 0.0

                if generate_pseudo_scores:
                    doc_id = str(item)
                    score = float(max_docs_per_query - rank + 1)
                else: # Scores are assumed to be provided as (doc_id, score) tuples
                    if isinstance(item, tuple) and len(item) == 2:
                        doc_id = str(item[0])
                        score = float(item[1])
                    else: # Fallback if format is unexpected
                        print(f"  Warning (QID {qid}): Expected (doc_id, score) tuple, got {type(item)}. Skipping.")
                        continue

                if doc_id:
                    f_out.write(f"{qid}\tQ0\t{doc_id}\t{rank}\t{score:.6f}\t{run_id}\n")
                    lines_written += 1

    print(f"  ✅ TREC run file saved: {output_file_path} ({lines_written} lines for {queries_processed} queries)")

def load_predictions_from_pkl(path, task_name):
    """Helper to load a single .pkl file."""
    if os.path.exists(path):
        with open(path, 'rb') as f:
            preds = pickle.load(f)
        print(f"  Loaded {task_name} predictions from: {path} ({len(preds)} queries)")
        return preds
    else:
        print(f"  ❌ WARNING: {task_name} prediction file not found: {path}")
        return None

def load_and_combine_test_predictions(cache_dir, base_filename_pattern, task_name):
    """Helper to load and combine predictions for test sets (split by query file)."""
    combined_preds = OrderedDict()
    query_file_bases = [os.path.splitext(qf)[0] for qf in TEST_QUERY_FILES_LIST]

    for query_file_base in query_file_bases:
        pred_path = os.path.join(cache_dir, base_filename_pattern.format(query_file_base))
        if os.path.exists(pred_path):
            with open(pred_path, 'rb') as f:
                preds_for_file = pickle.load(f)
                combined_preds.update(preds_for_file)
            print(f"  Loaded and combined {task_name} predictions from: {pred_path}")
        else:
            print(f"  ❌ WARNING: {task_name} prediction file not found: {pred_path}")
    return combined_preds

# ==============================================================================
# 3. EXECUTION FOR PART 'A'
# ==============================================================================
print(f"All TREC run files will be saved in: {SUBMISSION_DIR}")

# --- A: Traditional IR Models ---
# A1: TF-IDF Training Abstracts
print(f"\n--- Formatting A1: TF-IDF Training Abstracts ---")
try:
    a1_preds = load_predictions_from_pkl(os.path.join(PIPELINE_A1_CACHE_DIR, 'tfidf_predictions_A1.pkl'), "A1")
    if a1_preds:
        # NOTE: Your function call used pseudo scores. This assumes the .pkl file contains a list of doc_ids, not (doc_id, score) tuples.
        format_and_save_trec_run(a1_preds, f"{GROUP_ID}_A1_TFIDF_TrainAbs", SUBMISSION_DIR, f"{GROUP_ID}_A1_TFIDF_TrainAbs.txt", generate_pseudo_scores=True)
except Exception as e: print(f"Error for A1: {e}")

# A2: TF-IDF Training Full-Text
print(f"\n--- Formatting A2: TF-IDF Training Full-Text ---")
try:
    a2_preds = load_predictions_from_pkl(os.path.join(PIPELINE_A2_CACHE_DIR, 'tfidf_predictions_A2.pkl'), "A2")
    if a2_preds:
        format_and_save_trec_run(a2_preds, f"{GROUP_ID}_A2_TFIDF_TrainFull", SUBMISSION_DIR, f"{GROUP_ID}_A2_TFIDF_TrainFull.txt", generate_pseudo_scores=True)
except Exception as e: print(f"Error for A2: {e}")

# A3: TF-IDF Testing Abstracts
print(f"\n--- Formatting A3: TF-IDF Testing Abstracts ---")
try:
    a3_preds = load_and_combine_test_predictions(PIPELINE_A3_CACHE_DIR, 'tfidf_predictions_A3_{}.pkl', "A3")
    if a3_preds:
        format_and_save_trec_run(a3_preds, f"{GROUP_ID}_A3_TFIDF_TestAbs", SUBMISSION_DIR, f"{GROUP_ID}_A3_TFIDF_TestAbs.txt", generate_pseudo_scores=True)
except Exception as e: print(f"Error for A3: {e}")

# A4: TF-IDF Testing Full-Text
print(f"\n--- Formatting A4: TF-IDF Testing Full-Text ---")
try:
    a4_preds = load_and_combine_test_predictions(PIPELINE_A4_CACHE_DIR, 'tfidf_predictions_A4_{}.pkl', "A4")
    if a4_preds:
        format_and_save_trec_run(a4_preds, f"{GROUP_ID}_A4_TFIDF_TestFull", SUBMISSION_DIR, f"{GROUP_ID}_A4_TFIDF_TestFull.txt", generate_pseudo_scores=True)
except Exception as e: print(f"Error for A4: {e}")

# A5: BM25+RM3 Training Abstracts
print(f"\n--- Formatting A5: BM25+RM3 Training Abstracts ---")
try:
    # Note: Using the filename `bm25_rm3_predictions_A5_abstract_top100.pkl` based on our findings
    a5_preds = load_predictions_from_pkl(os.path.join(PIPELINE_A5_CACHE_DIR, f'bm25_rm3_predictions_A5_abstract_top{TOP_K_PREDS_PARAM}.pkl'), "A5")
    if a5_preds:
        # NOTE: Your function call used pseudo scores. If A5 saves (doc_id, score) tuples, change this to generate_pseudo_scores=False
        format_and_save_trec_run(a5_preds, f"{GROUP_ID}_A5_BM25_TrainAbs", SUBMISSION_DIR, f"{GROUP_ID}_A5_BM25_TrainAbs.txt", generate_pseudo_scores=True)
except Exception as e: print(f"Error for A5: {e}")

# A6: BM25+RM3 Training Full-Text
print(f"\n--- Formatting A6: BM25+RM3 Training Full-Text ---")
try:
    a6_preds = load_predictions_from_pkl(os.path.join(PIPELINE_A6_CACHE_DIR, f'bm25_rm3_predictions_A6_fulltext_top{TOP_K_PREDS_PARAM}.pkl'), "A6")
    if a6_preds:
        format_and_save_trec_run(a6_preds, f"{GROUP_ID}_A6_BM25_TrainFull", SUBMISSION_DIR, f"{GROUP_ID}_A6_BM25_TrainFull.txt", generate_pseudo_scores=True)
except Exception as e: print(f"Error for A6: {e}")

# A7: BM25+RM3 Testing Abstracts
print(f"\n--- Formatting A7: BM25+RM3 Testing Abstracts ---")
try:
    a7_preds = load_and_combine_test_predictions(PIPELINE_A7_CACHE_DIR, 'predictions_A7_Abstract_Test_{}.pkl', "A7")
    if a7_preds:
        format_and_save_trec_run(a7_preds, f"{GROUP_ID}_A7_BM25_TestAbs", SUBMISSION_DIR, f"{GROUP_ID}_A7_BM25_TestAbs.txt", generate_pseudo_scores=True)
except Exception as e: print(f"Error for A7: {e}")

# A8: BM25+RM3 Testing Full-Text
print(f"\n--- Formatting A8: BM25+RM3 Testing Full-Text ---")
try:
    a8_preds = load_and_combine_test_predictions(PIPELINE_A8_CACHE_DIR, 'predictions_A8_FullText_Test_{}.pkl', "A8")
    if a8_preds:
        format_and_save_trec_run(a8_preds, f"{GROUP_ID}_A8_BM25_TestFull", SUBMISSION_DIR, f"{GROUP_ID}_A8_BM25_TestFull.txt", generate_pseudo_scores=True)
except Exception as e: print(f"Error for A8: {e}")

print("\n--- Part 'A' TREC formatting complete. ---")

Mounted at /content/drive
Google Drive mounted successfully at /content/drive
Your project base path exists: /content/drive/MyDrive/colab-4/

--- End of Mount Script ---
All TREC run files will be saved in: /content/drive/MyDrive/colab-4/final_submission_runs_A_only

--- Formatting A1: TF-IDF Training Abstracts ---
  Loaded A1 predictions from: /content/drive/MyDrive/colab-4/cache_A1_abstract_tfidf_train/tfidf_predictions_A1.pkl (393 queries)
Formatting predictions for TREC run file: /content/drive/MyDrive/colab-4/final_submission_runs_A_only/clef25-sambs_A1_TFIDF_TrainAbs.txt
  ✅ TREC run file saved: /content/drive/MyDrive/colab-4/final_submission_runs_A_only/clef25-sambs_A1_TFIDF_TrainAbs.txt (1965 lines for 393 queries)

--- Formatting A2: TF-IDF Training Full-Text ---
  Loaded A2 predictions from: /content/drive/MyDrive/colab-4/cache_A2_fulltext_tfidf_train/tfidf_predictions_A2.pkl (393 queries)
Formatting predictions for TREC run file: /content/drive/MyDrive/colab-4/final_submissi

In [ ]:
import os
import pickle
from collections import OrderedDict

# ==============================================================================
# 1. SETUP - CONFIGURED FOR PART B
# ==============================================================================
GROUP_ID = "clef25-sambs"
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"
SUBMISSION_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_runs_B_only")

# --- Define cache directories for BOTH versions of your B pipelines ---
# (These paths are confirmed correct from your previous outputs)
PIPELINE_B1_V1_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B1_abstract_sbert_train/")
PIPELINE_B1_V2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B1_v2_abstract_sbert_mpnet_train/")
PIPELINE_B2_V1_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B2_fulltext_sbert_train/")
PIPELINE_B2_V2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B2_v2_fulltext_sbert_mpnet_train/")
PIPELINE_B3_V1_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B3_abstract_sbert_test/")
PIPELINE_B3_V2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B3_v2_abstract_sbert_mpnet_test/")
PIPELINE_B4_V1_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B4_fulltext_sbert_test/")
PIPELINE_B4_V2_CACHE_DIR = os.path.join(DRIVE_BASE_PATH, "cache_B4_v2_fulltext_sbert_mpnet_test/")

# ==============================================================================
# 2. HELPER FUNCTIONS (FROM YOUR NOTEBOOK)
# ==============================================================================
def format_and_save_trec_run(
    predictions_dict, run_id, output_dir, output_filename,
    max_docs_per_query=100, generate_pseudo_scores=False
    ):
    os.makedirs(output_dir, exist_ok=True)
    output_file_path = os.path.join(output_dir, output_filename)
    print(f"Formatting predictions for TREC run file: {output_file_path}")
    lines_written = 0
    with open(output_file_path, 'w') as f_out:
        predictions_dict_ordered = OrderedDict(sorted(predictions_dict.items()))
        for qid, ranked_items in predictions_dict_ordered.items():
            if not ranked_items: continue
            for rank, item in enumerate(ranked_items[:max_docs_per_query], 1):
                doc_id, score = str(item), float(max_docs_per_query - rank + 1)
                f_out.write(f"{qid}\tQ0\t{doc_id}\t{rank}\t{score:.6f}\t{run_id}\n")
                lines_written += 1
    print(f"  ✅ TREC run file saved: {output_file_path} ({lines_written} lines)")

def load_predictions_from_pkl(path, task_name):
    if os.path.exists(path):
        with open(path, 'rb') as f:
            preds = pickle.load(f)
        print(f"  Loaded {task_name} predictions from: {path} ({len(preds)} queries)")
        return preds
    else:
        print(f"  ❌ WARNING: {task_name} prediction file not found: {path}")
        return None

# ==============================================================================
# 3. EXECUTION FOR PART B (USING CORRECT FILENAMES)
# ==============================================================================
print(f"All TREC run files for Part B will be saved in: {SUBMISSION_DIR}")

# --- B1: SBERT Training Abstracts (V1 & V2) ---
print("\n--- Formatting B1 (V1): SBERT (MiniLM) ---")
b1_v1_preds = load_predictions_from_pkl(os.path.join(PIPELINE_B1_V1_CACHE_DIR, 'predictions_B1_all-MiniLM-L6-v2.pkl'), "B1 (V1)")
if b1_v1_preds:
    format_and_save_trec_run(b1_v1_preds, f"{GROUP_ID}_B1_SBERT_V1_TrainAbs", SUBMISSION_DIR, f"{GROUP_ID}_B1_SBERT_V1_TrainAbs.txt", generate_pseudo_scores=True)

print("\n--- Formatting B1 (V2): SBERT (MPNet) ---")
b1_v2_preds = load_predictions_from_pkl(os.path.join(PIPELINE_B1_V2_CACHE_DIR, 'predictions_B1_all-mpnet-base-v2_top100.pkl'), "B1 (V2)")
if b1_v2_preds:
    format_and_save_trec_run(b1_v2_preds, f"{GROUP_ID}_B1_SBERT_V2_MPNet_TrainAbs", SUBMISSION_DIR, f"{GROUP_ID}_B1_SBERT_V2_MPNet_TrainAbs.txt", generate_pseudo_scores=True)

# --- B2: SBERT Training Full-Text (V1 & V2) ---
print("\n--- Formatting B2 (V1): SBERT (MiniLM) ---")
b2_v1_preds = load_predictions_from_pkl(os.path.join(PIPELINE_B2_V1_CACHE_DIR, 'predictions_B2_all-MiniLM-L6-v2_max400w.pkl'), "B2 (V1)")
if b2_v1_preds:
    format_and_save_trec_run(b2_v1_preds, f"{GROUP_ID}_B2_SBERT_V1_TrainFull", SUBMISSION_DIR, f"{GROUP_ID}_B2_SBERT_V1_TrainFull.txt", generate_pseudo_scores=True)

print("\n--- Formatting B2 (V2): SBERT (MPNet) ---")
b2_v2_preds = load_predictions_from_pkl(os.path.join(PIPELINE_B2_V2_CACHE_DIR, 'predictions_B2_all-mpnet-base-v2_max384w_top100.pkl'), "B2 (V2)")
if b2_v2_preds:
    format_and_save_trec_run(b2_v2_preds, f"{GROUP_ID}_B2_SBERT_V2_MPNet_TrainFull", SUBMISSION_DIR, f"{GROUP_ID}_B2_SBERT_V2_MPNet_TrainFull.txt", generate_pseudo_scores=True)

# --- B3: SBERT Testing Abstracts (V1 & V2) ---
print("\n--- Formatting B3 (V1): SBERT (MiniLM) ---")
b3_v1_preds = load_predictions_from_pkl(os.path.join(PIPELINE_B3_V1_CACHE_DIR, 'predictions_b3_all-MiniLM-L6-v2_top100.pkl'), "B3 (V1)")
if b3_v1_preds:
    format_and_save_trec_run(b3_v1_preds, f"{GROUP_ID}_B3_SBERT_V1_TestAbs", SUBMISSION_DIR, f"{GROUP_ID}_B3_SBERT_V1_TestAbs.txt", generate_pseudo_scores=True)

print("\n--- Formatting B3 (V2): SBERT (MPNet) ---")
b3_v2_preds = load_predictions_from_pkl(os.path.join(PIPELINE_B3_V2_CACHE_DIR, 'predictions_B3_all-mpnet-base-v2_top100.pkl'), "B3 (V2)")
if b3_v2_preds:
    format_and_save_trec_run(b3_v2_preds, f"{GROUP_ID}_B3_SBERT_V2_MPNet_TestAbs", SUBMISSION_DIR, f"{GROUP_ID}_B3_SBERT_V2_MPNet_TestAbs.txt", generate_pseudo_scores=True)

# --- B4: SBERT Testing Full-Text (V1 & V2) ---
print("\n--- Formatting B4 (V1): SBERT (MiniLM) ---")
b4_v1_preds = load_predictions_from_pkl(os.path.join(PIPELINE_B4_V1_CACHE_DIR, 'predictions_b4_all-MiniLM-L6-v2_top100.pkl'), "B4 (V1)")
if b4_v1_preds:
    format_and_save_trec_run(b4_v1_preds, f"{GROUP_ID}_B4_SBERT_V1_TestFull", SUBMISSION_DIR, f"{GROUP_ID}_B4_SBERT_V1_TestFull.txt", generate_pseudo_scores=True)

print("\n--- Formatting B4 (V2): SBERT (MPNet) ---")
b4_v2_preds = load_predictions_from_pkl(os.path.join(PIPELINE_B4_V2_CACHE_DIR, 'predictions_B4_all-mpnet-base-v2_top100_max384w.pkl'), "B4 (V2)")
if b4_v2_preds:
    format_and_save_trec_run(b4_v2_preds, f"{GROUP_ID}_B4_SBERT_V2_MPNet_TestFull", SUBMISSION_DIR, f"{GROUP_ID}_B4_SBERT_V2_MPNet_TestFull.txt", generate_pseudo_scores=True)

print("\n\n--- Part B TREC formatting complete. ---")

All TREC run files for Part B will be saved in: /content/drive/MyDrive/colab-4/final_submission_runs_B_only

--- Formatting B1 (V1): SBERT (MiniLM) ---
  Loaded B1 (V1) predictions from: /content/drive/MyDrive/colab-4/cache_B1_abstract_sbert_train/predictions_B1_all-MiniLM-L6-v2.pkl (393 queries)
Formatting predictions for TREC run file: /content/drive/MyDrive/colab-4/final_submission_runs_B_only/clef25-sambs_B1_SBERT_V1_TrainAbs.txt
  ✅ TREC run file saved: /content/drive/MyDrive/colab-4/final_submission_runs_B_only/clef25-sambs_B1_SBERT_V1_TrainAbs.txt (1965 lines)

--- Formatting B1 (V2): SBERT (MPNet) ---
  Loaded B1 (V2) predictions from: /content/drive/MyDrive/colab-4/cache_B1_v2_abstract_sbert_mpnet_train/predictions_B1_all-mpnet-base-v2_top100.pkl (393 queries)
Formatting predictions for TREC run file: /content/drive/MyDrive/colab-4/final_submission_runs_B_only/clef25-sambs_B1_SBERT_V2_MPNet_TrainAbs.txt
  ✅ TREC run file saved: /content/drive/MyDrive/colab-4/final_submission_r

In [ ]:
import os
import pickle
from collections import OrderedDict
import time # For timestamps

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# ==============================================================================
# 1. SETUP - RELEVANT FOR PART C
# ==============================================================================
GROUP_ID = "clef25-sambs"
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"
# Create a new, comprehensive submission directory
SUBMISSION_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_runs_ALL_PARTS_COMPLETE_v1") # Or your preferred C-specific output
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# --- Model Names (ensure these match what's in your .pkl filenames for C runs) ---
CROSS_ENCODER_MODEL_L6 = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
CROSS_ENCODER_MODEL_L12 = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
MAX_WORDS_FOR_CE_FULLTEXT = 510 # Used in some C run filenames

# --- Cache Directories for Part C (from your directory listings and successful run logs) ---
CACHE_C1_STD_L6_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C1_abstract_crossencoder_train/")
CACHE_C1_V2_L12_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C1_abstract_crossencoder_A5_rerank_v2/")
CACHE_C2_STD_L6_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C2_fulltext_crossencoder_train/")
CACHE_C2_V2_L12_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C2_fulltext_crossencoder_A6_rerank/")
CACHE_C3_STD_L6_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C3_abstract_crossencoder_test/")
CACHE_C3_L12_DIR = os.path.join(DRIVE_BASE_PATH, 'cache_C3_abstract_crossencoder_L12_test/')
CACHE_C4_STD_L6_SUCCESSFUL_DIR = os.path.join(DRIVE_BASE_PATH, "cache_C4_fulltext_crossencoder_test/") # For the successful L-6 C4 run
CACHE_C4_L12_DIR = os.path.join(DRIVE_BASE_PATH, 'cache_C4_fulltext_crossencoder_L12_test/')


# ==============================================================================
# 2. HELPER FUNCTIONS (These are general and used by Part C)
# ==============================================================================
def format_and_save_trec_run(
    predictions_dict, run_id, output_dir, output_filename,
    max_docs_per_query=100, generate_pseudo_scores=False
    ):
    os.makedirs(output_dir, exist_ok=True)
    output_file_path = os.path.join(output_dir, output_filename)
    print(f"{get_timestamp()} Formatting predictions for TREC run file: {output_file_path}")
    lines_written = 0
    queries_processed = 0
    with open(output_file_path, 'w') as f_out:
        if not predictions_dict:
            print(f"{get_timestamp()}   ⚠️ Warning: The prediction dictionary for run '{run_id}' is empty. 0 lines will be written.")
            return

        if not isinstance(predictions_dict, OrderedDict):
            predictions_dict_ordered = OrderedDict(sorted(predictions_dict.items()))
        else:
            predictions_dict_ordered = predictions_dict

        for qid, ranked_items in predictions_dict_ordered.items():
            queries_processed +=1
            if not ranked_items:
                continue

            for rank, item in enumerate(ranked_items[:max_docs_per_query], 1):
                doc_id, score = None, 0.0
                if generate_pseudo_scores:
                    if isinstance(item, str):
                        doc_id = str(item)
                        score = float(max_docs_per_query - rank + 1)
                    else:
                        # print(f"{get_timestamp()}     Warning (QID {qid}, Run {run_id}): Expected doc_id string for pseudo-score, got {type(item)}. Skipping item.")
                        continue
                else:
                    if isinstance(item, tuple) and len(item) == 2:
                        doc_id = str(item[0])
                        score = float(item[1])
                    elif isinstance(item, str) and not generate_pseudo_scores: # Fallback
                        # print(f"{get_timestamp()}     Warning (QID {qid}, Run {run_id}): Expected (doc_id, score) tuple, but got string '{item}'. Using pseudo-score.")
                        doc_id = str(item)
                        score = float(max_docs_per_query - rank + 1)
                    else:
                        # print(f"{get_timestamp()}     Warning (QID {qid}, Run {run_id}): Expected (doc_id, score) tuple or string, got {type(item)}. Skipping item.")
                        continue

                if doc_id:
                    f_out.write(f"{qid}\tQ0\t{doc_id}\t{rank}\t{score:.6f}\t{run_id}\n")
                    lines_written += 1
    print(f"{get_timestamp()}   ✅ TREC run file saved: {output_file_path} ({lines_written} lines for {queries_processed} queries)")

def load_predictions_from_pkl(path, task_name):
    if os.path.exists(path):
        with open(path, 'rb') as f:
            preds = pickle.load(f)
        print(f"{get_timestamp()}   Loaded {task_name} predictions from: {path} (found {len(preds)} QIDs)")
        return preds
    else:
        print(f"{get_timestamp()}   ❌ WARNING: {task_name} prediction file not found: {path}")
        return None

# ==============================================================================
# 3. EXECUTION FOR PART C RUNS
# ==============================================================================
print(f"\n{get_timestamp()} Starting TREC file generation for Part C experiments.")
# This SUBMISSION_DIR is defined in the main setup block.
# If you want a C-specific one, define it here or pass it.
# For this extraction, we assume SUBMISSION_DIR is already set.
# print(f"{get_timestamp()} All Part C TREC run files will be saved in: {SUBMISSION_DIR}")


ce_model_l6_sanitized = CROSS_ENCODER_MODEL_L6.replace("/", "_")
ce_model_l12_sanitized = CROSS_ENCODER_MODEL_L12.replace("/", "_")

# --- C1 RUNS (Training Abstracts) ---
# C1 Standard (re-ranks A5 with L-6 model)
print(f"\n{get_timestamp()} --- Formatting C1 (Standard L-6, Re-ranking A5) ---")
c1_std_l6_path = os.path.join(CACHE_C1_STD_L6_DIR, f'reranked_preds_C1_{ce_model_l6_sanitized}.pkl')
c1_std_l6_preds = load_predictions_from_pkl(c1_std_l6_path, "C1 (Standard L-6)")
if c1_std_l6_preds:
    # Your log for C1 (L-6) "Sample C1 Re-ranked Predictions... Top 3: ['1589737', ...]"
    # This indicates lists of doc_ids.
    format_and_save_trec_run(c1_std_l6_preds, f"{GROUP_ID}_C1_CE_L6_TrainAbs", SUBMISSION_DIR, f"{GROUP_ID}_C1_CE_L6_TrainAbs.txt", generate_pseudo_scores=True)

# C1 V2 (re-ranks A5 with L-12 model)
print(f"\n{get_timestamp()} --- Formatting C1 (V2 L-12, Re-ranking A5) ---")
c1_v2_l12_path = os.path.join(CACHE_C1_V2_L12_DIR, f'predictions_CE_C1_Abstract_Rerank_{ce_model_l12_sanitized}.pkl')
c1_v2_l12_preds = load_predictions_from_pkl(c1_v2_l12_path, "C1 (V2 L-12)")
if c1_v2_l12_preds:
    # Your CELL 4 log (C1 L-12) showed metrics, implying scores.
    format_and_save_trec_run(c1_v2_l12_preds, f"{GROUP_ID}_C1_CE_L12_TrainAbs", SUBMISSION_DIR, f"{GROUP_ID}_C1_CE_L12_TrainAbs.txt", generate_pseudo_scores=False)

# --- C2 RUNS (Training Full-Text) ---
# C2 Standard (re-ranks A6 with L-6 model)
print(f"\n{get_timestamp()} --- Formatting C2 (Standard L-6, Re-ranking A6) ---")
c2_std_l6_path = os.path.join(CACHE_C2_STD_L6_DIR, f'reranked_preds_C2_{ce_model_l6_sanitized}_max{MAX_WORDS_FOR_CE_FULLTEXT}w.pkl')
c2_std_l6_preds = load_predictions_from_pkl(c2_std_l6_path, "C2 (Standard L-6)")
if c2_std_l6_preds:
    # Your log for C2 (L-6) "Sample C2 Re-ranked Predictions... Top 3: ['155747269', ...]"
    # This indicates lists of doc_ids.
    format_and_save_trec_run(c2_std_l6_preds, f"{GROUP_ID}_C2_CE_L6_TrainFull", SUBMISSION_DIR, f"{GROUP_ID}_C2_CE_L6_TrainFull.txt", generate_pseudo_scores=True)

# C2 V2 (re-ranks A6 with L-12 model)
print(f"\n{get_timestamp()} --- Formatting C2 (V2 L-12, Re-ranking A6) ---")
c2_v2_l12_path = os.path.join(CACHE_C2_V2_L12_DIR, f'predictions_CE_C2_FullText_Rerank_{ce_model_l12_sanitized}.pkl')
c2_v2_l12_preds = load_predictions_from_pkl(c2_v2_l12_path, "C2 (V2 L-12)")
if c2_v2_l12_preds:
    # Your CELL 5 log (C2 L-12) showed metrics, implying scores.
    format_and_save_trec_run(c2_v2_l12_preds, f"{GROUP_ID}_C2_CE_L12_TrainFull", SUBMISSION_DIR, f"{GROUP_ID}_C2_CE_L12_TrainFull.txt", generate_pseudo_scores=False)

# --- C3 RUNS (Test Abstracts) ---
# C3 Standard (re-ranks A7 with L-6 model)
print(f"\n{get_timestamp()} --- Formatting C3 (Standard L-6, Re-ranking A7) ---")
c3_std_l6_path = os.path.join(CACHE_C3_STD_L6_DIR, f'reranked_preds_C3_{ce_model_l6_sanitized}.pkl')
c3_std_l6_preds = load_predictions_from_pkl(c3_std_l6_path, "C3 (Standard L-6)")
if c3_std_l6_preds:
    # This run successfully produced a TREC file with scores previously.
    format_and_save_trec_run(c3_std_l6_preds, f"{GROUP_ID}_C3_CE_L6_TestAbs", SUBMISSION_DIR, f"{GROUP_ID}_C3_CE_L6_TestAbs.txt", generate_pseudo_scores=False)

# C3 V2 (re-ranks A7 with L-12 model)
print(f"\n{get_timestamp()} --- Formatting C3 (V2 L-12, Re-ranking A7) ---")
c3_v2_l12_path = os.path.join(CACHE_C3_L12_DIR, f'reranked_C3_{ce_model_l12_sanitized}.pkl')
c3_v2_l12_preds = load_predictions_from_pkl(c3_v2_l12_path, "C3 (V2 L-12)")
if c3_v2_l12_preds:
    # Your C3 L-12 script saved (doc_id, score) tuples
    format_and_save_trec_run(c3_v2_l12_preds, f"{GROUP_ID}_C3_CE_L12_TestAbs", SUBMISSION_DIR, f"{GROUP_ID}_C3_CE_L12_TestAbs.txt", generate_pseudo_scores=False)

# --- C4 RUNS (Test Full-Text) ---
# C4 Standard L-6 (The successful run from your logs)
print(f"\n{get_timestamp()} --- Formatting C4 (Standard L-6, Re-ranking A8 - Successful Log Version) ---")
c4_std_l6_path = os.path.join(CACHE_C4_STD_L6_DIR, f'reranked_C4_{ce_model_l6_sanitized}.pkl')
c4_std_l6_preds = load_predictions_from_pkl(c4_std_l6_path, "C4 (Standard L-6 Successful Log)")
if c4_std_l6_preds:
    # Your inspection of this .pkl file confirmed it contains (doc_id, score) tuples.
    format_and_save_trec_run(c4_std_l6_preds, f"{GROUP_ID}_C4_CE_L6_TestFull", SUBMISSION_DIR, f"{GROUP_ID}_C4_CE_L6_TestFull.txt", generate_pseudo_scores=False)

# C4 V2 L-12 (The one you successfully ran)
print(f"\n{get_timestamp()} --- Formatting C4 (V2 L-12, Re-ranking A8) ---")
c4_v2_l12_path = os.path.join(CACHE_C4_L12_DIR, f'reranked_C4_{ce_model_l12_sanitized}_max{MAX_WORDS_FOR_CE_FULLTEXT}w.pkl')
c4_v2_l12_preds = load_predictions_from_pkl(c4_v2_l12_path, "C4 (V2 L-12)")
if c4_v2_l12_preds:
    # Your C4 L-12 script saved (doc_id, score) tuples
    format_and_save_trec_run(c4_v2_l12_preds, f"{GROUP_ID}_C4_CE_L12_TestFull", SUBMISSION_DIR, f"{GROUP_ID}_C4_CE_L12_TestFull.txt", generate_pseudo_scores=False)

print(f"\n\n{get_timestamp()} --- Part C TREC formatting complete. ---")



2025-05-25 13:34:53 Starting TREC file generation for Part C experiments.

2025-05-25 13:34:53 --- Formatting C1 (Standard L-6, Re-ranking A5) ---
2025-05-25 13:34:53   Loaded C1 (Standard L-6) predictions from: /content/drive/MyDrive/colab-4/cache_C1_abstract_crossencoder_train/reranked_preds_C1_cross-encoder_ms-marco-MiniLM-L-6-v2.pkl (found 393 QIDs)
2025-05-25 13:34:53 Formatting predictions for TREC run file: /content/drive/MyDrive/colab-4/final_submission_runs_ALL_PARTS_COMPLETE_v1/clef25-sambs_C1_CE_L6_TrainAbs.txt
2025-05-25 13:34:53   ✅ TREC run file saved: /content/drive/MyDrive/colab-4/final_submission_runs_ALL_PARTS_COMPLETE_v1/clef25-sambs_C1_CE_L6_TrainAbs.txt (37500 lines for 393 queries)

2025-05-25 13:34:53 --- Formatting C1 (V2 L-12, Re-ranking A5) ---
2025-05-25 13:34:53   Loaded C1 (V2 L-12) predictions from: /content/drive/MyDrive/colab-4/cache_C1_abstract_crossencoder_A5_rerank_v2/predictions_CE_C1_Abstract_Rerank_cross-encoder_ms-marco-MiniLM-L-12-v2.pkl (found 

In [ ]:
import os
from google.colab import drive
import time

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# ==============================================================================
# 1. MOUNT GOOGLE DRIVE
# ==============================================================================
try:
    drive.mount('/content/drive', force_remount=True)
    print(f"{get_timestamp()} Google Drive mounted successfully.")
except Exception as e:
    print(f"{get_timestamp()} Error mounting Google Drive: {e}")
    # Stop execution if Drive doesn't mount, as the rest of the script depends on it.
    raise SystemExit("Google Drive mount failed. Cannot proceed.")

print(f"{get_timestamp()} " + "-" * 50)

# ==============================================================================
# 2. DEFINE THE DIRECTORY TO SCAN
# ==============================================================================
# The user specified G:\Meine Ablage\colab-4\final_submission_trec
# This translates to the following path in Colab after mounting:
TREC_FILES_DIRECTORY = "/content/drive/MyDrive/colab-4/final_submission_trec"

print(f"{get_timestamp()} Attempting to list TREC files from: {TREC_FILES_DIRECTORY}")

# ==============================================================================
# 3. LIST .TXT FILES (ASSUMED TREC RUN FILES) IN THE DIRECTORY
# ==============================================================================
if not os.path.exists(TREC_FILES_DIRECTORY):
    print(f"{get_timestamp()}   ❌ ERROR: The directory was NOT found: {TREC_FILES_DIRECTORY}")
    print(f"{get_timestamp()}   Please ensure the path is correct and the folder exists in your Google Drive.")
elif not os.path.isdir(TREC_FILES_DIRECTORY):
    print(f"{get_timestamp()}   ❌ ERROR: The path exists, but it is NOT a directory: {TREC_FILES_DIRECTORY}")
else:
    trec_run_files_found = []
    try:
        all_items_in_dir = os.listdir(TREC_FILES_DIRECTORY)
        for item_name in all_items_in_dir:
            # TREC run files are typically .txt files
            if item_name.lower().endswith('.txt'):
                # Optionally, you can add more checks here if your TREC files have a specific naming pattern
                # For example, if they all start with "clef25-sambs_"
                # if item_name.startswith("clef25-sambs_") and item_name.lower().endswith('.txt'):
                trec_run_files_found.append(item_name)

        if trec_run_files_found:
            print(f"\n{get_timestamp()} Found the following TREC run files (.txt) in the directory:")
            for trec_file in sorted(trec_run_files_found):
                print(f"  -> {trec_file}")
            print(f"\n{get_timestamp()} Total TREC files found: {len(trec_run_files_found)}")
        else:
            print(f"{get_timestamp()}   ⚠️ No .txt files (assumed TREC runs) found in the directory: {TREC_FILES_DIRECTORY}")
            print(f"{get_timestamp()}     Contents of the directory are:")
            if not all_items_in_dir:
                print(f"{get_timestamp()}       Directory is empty.")
            else:
                for item_name in sorted(all_items_in_dir):
                     item_path_temp = os.path.join(TREC_FILES_DIRECTORY, item_name)
                     item_type_temp = "[Folder]" if os.path.isdir(item_path_temp) else "[File]"
                     print(f"{get_timestamp()}       {item_type_temp} {item_name}")


    except Exception as e:
        print(f"{get_timestamp()}   ❌ ERROR: Could not list directory contents: {e}")

print(f"\n{get_timestamp()} --- End of TREC File Listing Script ---")


Mounted at /content/drive
2025-05-25 17:53:19 Google Drive mounted successfully.
2025-05-25 17:53:19 --------------------------------------------------
2025-05-25 17:53:19 Attempting to list TREC files from: /content/drive/MyDrive/colab-4/final_submission_trec

2025-05-25 17:53:20 Found the following TREC run files (.txt) in the directory:
  -> clef25-sambs_A1_TFIDF_TrainAbs.txt
  -> clef25-sambs_A2_TFIDF_TrainFull.txt
  -> clef25-sambs_A3_TFIDF_TestAbs.txt
  -> clef25-sambs_A4_TFIDF_TestFull.txt
  -> clef25-sambs_A5_BM25_TrainAbs.txt
  -> clef25-sambs_A6_BM25_TrainFull.txt
  -> clef25-sambs_A7_BM25_TestAbs.txt
  -> clef25-sambs_A8_BM25_TestFull.txt
  -> clef25-sambs_B1_SBERT_V1_TrainAbs.txt
  -> clef25-sambs_B1_SBERT_V2_MPNet_TrainAbs.txt
  -> clef25-sambs_B2_SBERT_V1_TrainFull.txt
  -> clef25-sambs_B2_SBERT_V2_MPNet_TrainFull.txt
  -> clef25-sambs_B3_SBERT_V1_TestAbs.txt
  -> clef25-sambs_B3_SBERT_V2_MPNet_TestAbs.txt
  -> clef25-sambs_B4_SBERT_V1_TestFull.txt
  -> clef25-sambs_B4_SB

In [ ]:
import os
import shutil
import zipfile
from collections import OrderedDict
import time
from google.colab import drive

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# ==============================================================================
# 0. MOUNT GOOGLE DRIVE
# ==============================================================================
try:
    drive.mount('/content/drive', force_remount=True)
    print(f"{get_timestamp()} Google Drive mounted successfully.")
except Exception as e:
    print(f"{get_timestamp()} Error mounting Google Drive: {e}")
    raise SystemExit("Google Drive mount failed. Cannot proceed.")

print(f"{get_timestamp()} " + "-" * 50)

# ==============================================================================
# 1. SETUP - CONFIGURE FOR THE SPECIFIC RUN TO PREPARE
# ==============================================================================
TEAM_NAME = "clef25-sambs"
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"

# Directory where your combined TREC files are currently stored
SOURCE_TREC_FILE_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_runs_ALL_PARTS_COMPLETE_v1") # From your last TREC generation

# Parent directory for TIRA-formatted submissions
TIRA_SUBMISSIONS_PARENT_DIR = os.path.join(DRIVE_BASE_PATH, "tira_official_submissions_prepared")
os.makedirs(TIRA_SUBMISSIONS_PARENT_DIR, exist_ok=True)

# --- Details for the run we are preparing (Example: C3 L-12 Test Abstracts) ---
# Original combined TREC filename (from the list you provided)
ORIGINAL_COMBINED_TREC_FILENAME = "clef25-sambs_C3_CE_L12_TestAbs.txt"

# Short, unique identifier for this run for TIRA (no spaces or special chars other than '-' or '_')
RUN_ID_FOR_TIRA = "C3-CE-L12-TestAbs"
RUN_DESCRIPTION = "Cross-Encoder (ms-marco-MiniLM-L-12-v2) re-ranking A7 (BM25) candidates on Test Abstracts."
RUN_APPROACH_KEYWORDS = "Cross-Encoder, L-12, BM25-rerank, Abstracts, Test"

# ZIP file containing the original query files (needed to identify QIDs per snapshot)
# For C3 (Abstracts), this is likely the abstracts test ZIP. For C4, it would be the full-text test ZIP.
QUERY_ZIP_FILE_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_abstract.zip')

# Exact paths of the query files *inside* the ZIP archive
NOV_QUERY_FILE_IN_ZIP = 'longeval_sci_testing_2025_abstract/queries_2024-11_test.txt'
JAN_QUERY_FILE_IN_ZIP = 'longeval_sci_testing_2025_abstract/queries_2025-01_test.txt'
# ==============================================================================

# Path to the source combined TREC file
source_trec_file_path = os.path.join(SOURCE_TREC_FILE_DIR, ORIGINAL_COMBINED_TREC_FILENAME)

# Create the specific submission folder for this run
current_run_submission_dir = os.path.join(TIRA_SUBMISSIONS_PARENT_DIR, f"submission_{RUN_ID_FOR_TIRA}")

# --- Helper function to load QIDs from a query file within the ZIP ---
def load_qids_from_snapshot_zip(zip_path, query_file_in_zip):
    qids = set()
    if not os.path.exists(zip_path):
        print(f"{get_timestamp()}   ❌ ERROR: Query ZIP file not found at {zip_path}")
        return qids
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            with z.open(query_file_in_zip) as f:
                for raw_line in f:
                    line = raw_line.decode('utf-8').strip()
                    if not line: continue
                    if '\t' in line:
                        qid, _ = line.split('\t', 1)
                    else:
                        qid, _ = line.split(' ', 1) # Assuming space as fallback
                    qids.add(qid)
        print(f"{get_timestamp()}   Loaded {len(qids)} QIDs from {query_file_in_zip}")
    except KeyError:
        print(f"{get_timestamp()}   ❌ ERROR: Query file '{query_file_in_zip}' not found inside '{zip_path}'.")
    except Exception as e:
        print(f"{get_timestamp()}   ❌ ERROR: Could not load QIDs from {query_file_in_zip}: {e}")
    return qids

# ==============================================================================
# 2. CREATE TIRA SUBMISSION DIRECTORY STRUCTURE
# ==============================================================================
print(f"\n{get_timestamp()} --- Preparing TIRA submission folder for: {RUN_ID_FOR_TIRA} ---")
print(f"{get_timestamp()} Submission will be prepared in: {current_run_submission_dir}")

snapshot_folders_map = {
    "2024-11": os.path.join(current_run_submission_dir, "2024-11"),
    "2025-01": os.path.join(current_run_submission_dir, "2025-01")
}
for folder_path in snapshot_folders_map.values():
    os.makedirs(folder_path, exist_ok=True)
print(f"{get_timestamp()}   Created snapshot sub-folders.")

# ==============================================================================
# 3. LOAD QIDS FOR EACH SNAPSHOT
# ==============================================================================
print(f"\n{get_timestamp()} --- Loading QIDs for each snapshot ---")
nov_qids = load_qids_from_snapshot_zip(QUERY_ZIP_FILE_PATH, NOV_QUERY_FILE_IN_ZIP)
jan_qids = load_qids_from_snapshot_zip(QUERY_ZIP_FILE_PATH, JAN_QUERY_FILE_IN_ZIP)

if not nov_qids or not jan_qids:
    print(f"{get_timestamp()}   ❌ ERROR: Could not load QIDs for one or both snapshots. Cannot proceed with splitting.")
else:
    # ==============================================================================
    # 4. SPLIT THE COMBINED TREC FILE AND CREATE run.txt FOR EACH SNAPSHOT
    # ==============================================================================
    print(f"\n{get_timestamp()} --- Splitting combined TREC file: {ORIGINAL_COMBINED_TREC_FILENAME} ---")

    if not os.path.exists(source_trec_file_path):
        print(f"{get_timestamp()}   ❌ ERROR: Source TREC file not found: {source_trec_file_path}")
    else:
        lines_written_nov = 0
        lines_written_jan = 0

        with open(source_trec_file_path, 'r') as f_in, \
             open(os.path.join(snapshot_folders_map["2024-11"], "run.txt"), 'w') as f_nov, \
             open(os.path.join(snapshot_folders_map["2025-01"], "run.txt"), 'w') as f_jan:

            for line in f_in:
                parts = line.strip().split()
                if not parts: continue
                qid_from_line = parts[0]

                if qid_from_line in nov_qids:
                    f_nov.write(line)
                    lines_written_nov += 1
                elif qid_from_line in jan_qids:
                    f_jan.write(line)
                    lines_written_jan += 1
                # else:
                #     print(f"{get_timestamp()}     Warning: QID {qid_from_line} not found in either snapshot QID list.")

        print(f"{get_timestamp()}   ✅ Finished splitting. {lines_written_nov} lines written to 2024-11/run.txt")
        print(f"{get_timestamp()}   ✅ Finished splitting. {lines_written_jan} lines written to 2025-01/run.txt")

        # ==============================================================================
        # 5. CREATE ir-metadata.yml FILE
        # ==============================================================================
        print(f"\n{get_timestamp()} --- Creating ir-metadata.yml file ---")
        yaml_content = f"""
team: '{TEAM_NAME}'
model:
  name: '{RUN_ID_FOR_TIRA}'
  description: "{RUN_DESCRIPTION}"
  approach: '{RUN_APPROACH_KEYWORDS}'
"""
        # Print the content that will be written to the YAML file
        print(f"{get_timestamp()}   Content of ir-metadata.yml will be:\n{yaml_content.strip()}")

        yaml_path = os.path.join(current_run_submission_dir, "ir-metadata.yml")
        try:
            with open(yaml_path, 'w') as f:
                f.write(yaml_content.strip())
            print(f"{get_timestamp()}   ✅ Created metadata file: {yaml_path}")
        except Exception as e:
            print(f"{get_timestamp()}   ❌ ERROR: Could not write the metadata file: {e}")

        # ==============================================================================
        # 6. VERIFY WITH TIRA-CLI (Optional but Recommended)
        # ==============================================================================
        print(f"\n{get_timestamp()} --- Verifying submission with tira-cli (Dry Run) ---")
        print(f"{get_timestamp()} Ensure tira-cli is installed: !pip3 install --upgrade -q tira")
        print(f"{get_timestamp()} Then run: !tira-cli login-token YOUR_TOKEN_HERE")
        print(f"{get_timestamp()} And then verify with:")
        print(f"!tira-cli upload --dataset sci-20250430-test --dry-run --directory {current_run_submission_dir}")

print(f"\n{get_timestamp()} --- Preparation for {RUN_ID_FOR_TIRA} complete. ---")



Mounted at /content/drive
2025-05-25 17:56:57 Google Drive mounted successfully.
2025-05-25 17:56:57 --------------------------------------------------

2025-05-25 17:56:57 --- Preparing TIRA submission folder for: C3-CE-L12-TestAbs ---
2025-05-25 17:56:57 Submission will be prepared in: /content/drive/MyDrive/colab-4/tira_official_submissions_prepared/submission_C3-CE-L12-TestAbs
2025-05-25 17:56:57   Created snapshot sub-folders.

2025-05-25 17:56:57 --- Loading QIDs for each snapshot ---
2025-05-25 17:56:58   Loaded 99 QIDs from longeval_sci_testing_2025_abstract/queries_2024-11_test.txt
2025-05-25 17:56:58   Loaded 492 QIDs from longeval_sci_testing_2025_abstract/queries_2025-01_test.txt

2025-05-25 17:56:58 --- Splitting combined TREC file: clef25-sambs_C3_CE_L12_TestAbs.txt ---
2025-05-25 17:56:59   ✅ Finished splitting. 485 lines written to 2024-11/run.txt
2025-05-25 17:56:59   ✅ Finished splitting. 2210 lines written to 2025-01/run.txt

2025-05-25 17:56:59 --- Creating ir-metad

In [ ]:
!pip3 install --upgrade -q tira
!tira-cli login --token 46a92f03572bd0b6968f720c08d6527c2bbabb83a1406b91445bd56fb0632f5b
!tira-cli verify-installation

✓ You are authenticated against www.tira.io.
✓ TIRA home is writable.
✖ Docker/Podman is not installed. You can not run dockerized TIRA submissions.
✓ The tirex-tracker works and will track experimental metadata.

Result:
✖ Your installation is not valid, you might not use all features.


In [ ]:
!tira-cli upload --dataset sci-20250430-test --dry-run --directory /content/drive/MyDrive/colab-4/tira_official_submissions_prepared/submission_C3-CE-L12-TestAbs

✖ Please specify the name of your system. Either:

	Incorporate the tag into your ir-metadata (see https://ir-metadata.org),

	or, pass --system to tira-cli upload


In [ ]:
# Get the RUN_ID_FOR_TIRA from the preparation script for the run you are submitting
# For this example, we used C3-CE-L12-TestAbs
# Ensure this matches the run whose folder you are pointing to.
run_id_for_system_flag = "C3-CE-L12-TestAbs"
submission_folder_path = "/content/drive/MyDrive/colab-4/tira_official_submissions_prepared/submission_C3-CE-L12-TestAbs"

!tira-cli upload --dataset sci-20250430-test --system {run_id_for_system_flag} --dry-run --directory {submission_folder_path}

I check that the submission in directory '/content/drive/MyDrive/colab-4/tira_official_submissions_prepared/submission_C3-CE-L12-TestAbs' is valid...


	✖ The file /content/drive/MyDrive/colab-4/tira_official_submissions_prepared/submission_C3-CE-L12-TestAbs/ir-metadata.yml is not valid. Errors: 
		- The required field tag is missing.
		- The required field actor is missing.
		- The required field research goal is missing.
		- The required field platform is missing.
		- The required field implementation is missing.
		- The required field data is missing.
		- The required field method is missing.

Result:
	✖ Could not upload to TIRA as the directory /content/drive/MyDrive/colab-4/tira_official_submissions_prepared/submission_C3-CE-L12-TestAbs is not a valid submission.


In [ ]:
import os
import shutil
import zipfile
import time
from google.colab import drive
import torch
import platform # To get Python version dynamically

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# ==============================================================================
# 0. MOUNT GOOGLE DRIVE
# ==============================================================================
try:
    drive.mount('/content/drive', force_remount=True)
    print(f"{get_timestamp()} Google Drive mounted successfully.")
except Exception as e:
    print(f"{get_timestamp()} Error mounting Google Drive: {e}")
    raise SystemExit("Google Drive mount failed. Cannot proceed.")

print(f"{get_timestamp()} " + "-" * 50)

# ==============================================================================
# 1. SETUP - CONFIGURE FOR THE SPECIFIC RUN TO PREPARE
# ==============================================================================
# --- Core Configuration ---
TEAM_NAME = "clef25-sambs"
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/" # Base path on your Google Drive

# --- Source and Target Directories ---
SOURCE_TREC_FILE_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_runs_ALL_PARTS_COMPLETE_v1")
TIRA_SUBMISSIONS_PARENT_DIR = os.path.join(DRIVE_BASE_PATH, "tira_official_submissions_prepared_v4") # Using v4 to avoid conflicts
os.makedirs(TIRA_SUBMISSIONS_PARENT_DIR, exist_ok=True)

# --- Details for the SPECIFIC run you are preparing ---
ORIGINAL_COMBINED_TREC_FILENAME = "clef25-sambs_C3_CE_L12_TestAbs.txt"
RUN_ID_FOR_TIRA = "C3-CE-L12-TestAbs"
RUN_TAG_KEYWORDS = ['longeval', 'sci-retrieval', 'clef2025', 'cross-encoder', 'L-12', 'BM25-rerank', 'abstracts', 'test']

# --- Query File Details ---
QUERY_ZIP_FILE_PATH = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_abstract.zip')
NOV_QUERY_FILE_IN_ZIP = 'longeval_sci_testing_2025_abstract/queries_2024-11_test.txt'
JAN_QUERY_FILE_IN_ZIP = 'longeval_sci_testing_2025_abstract/queries_2025-01_test.txt'

# --- Metadata Details ---
LINK_TO_CODE_REPOSITORY = "https://github.com/your-username/your-repo" # IMPORTANT: Update with your real link
PYTHON_VERSION = platform.python_version()
PYTORCH_VERSION = torch.__version__
SENTENCE_TRANSFORMERS_VERSION = "2.2.2" # Example, use !pip show to verify
PYTERRIER_VERSION = "0.9.2" # Example, use !pip show to verify

# ==============================================================================
# PATH DEFINITIONS
# ==============================================================================
source_trec_file_path = os.path.join(SOURCE_TREC_FILE_DIR, ORIGINAL_COMBINED_TREC_FILENAME)
current_run_submission_dir = os.path.join(TIRA_SUBMISSIONS_PARENT_DIR, f"submission_{RUN_ID_FOR_TIRA}")

# ==============================================================================
# HELPER FUNCTION: Load QIDs from ZIP
# ==============================================================================
def load_qids_from_snapshot_zip(zip_path, query_file_in_zip):
    qids = set()
    if not os.path.exists(zip_path):
        print(f"{get_timestamp()}   ❌ ERROR: Query ZIP file not found at {zip_path}")
        return qids
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            if query_file_in_zip not in z.namelist():
                print(f"{get_timestamp()}   ❌ ERROR: Query file '{query_file_in_zip}' not found inside '{zip_path}'.")
                return qids
            with z.open(query_file_in_zip) as f:
                for raw_line in f:
                    line = raw_line.decode('utf-8').strip()
                    if line: qids.add(line.split()[0])
        print(f"{get_timestamp()}   Loaded {len(qids)} QIDs from {query_file_in_zip}")
    except Exception as e:
        print(f"{get_timestamp()}   ❌ ERROR: Could not load QIDs from {query_file_in_zip}: {e}")
    return qids

# ==============================================================================
# 2. CREATE DIRECTORY STRUCTURE
# ==============================================================================
print(f"\n{get_timestamp()} --- Preparing TIRA submission folder for: {RUN_ID_FOR_TIRA} ---")
os.makedirs(os.path.join(current_run_submission_dir, "2024-11"), exist_ok=True)
os.makedirs(os.path.join(current_run_submission_dir, "2025-01"), exist_ok=True)
print(f"{get_timestamp()}   Submission will be prepared in: {current_run_submission_dir}")

# ==============================================================================
# 3. LOAD QIDS & 4. SPLIT TREC FILE
# ==============================================================================
print(f"\n{get_timestamp()} --- Loading QIDs and splitting TREC file ---")
nov_qids = load_qids_from_snapshot_zip(QUERY_ZIP_FILE_PATH, NOV_QUERY_FILE_IN_ZIP)
jan_qids = load_qids_from_snapshot_zip(QUERY_ZIP_FILE_PATH, JAN_QUERY_FILE_IN_ZIP)

if not nov_qids or not jan_qids or not os.path.exists(source_trec_file_path):
    print(f"{get_timestamp()}   ❌ ERROR: Prerequisite files missing. Cannot proceed.")
else:
    lines_written_nov, lines_written_jan = 0, 0
    with open(source_trec_file_path, 'r') as f_in, \
         open(os.path.join(current_run_submission_dir, "2024-11", "run.txt"), 'w') as f_nov, \
         open(os.path.join(current_run_submission_dir, "2025-01", "run.txt"), 'w') as f_jan:
        for line in f_in:
            qid_from_line = line.strip().split()[0]
            if qid_from_line in nov_qids:
                f_nov.write(line); lines_written_nov += 1
            elif qid_from_line in jan_qids:
                f_jan.write(line); lines_written_jan += 1
    print(f"{get_timestamp()}   ✅ Finished splitting. {lines_written_nov} lines for Nov, {lines_written_jan} for Jan.")

    # ==============================================================================
    # 5. CREATE ir-metadata.yml FILE USING THE OFFICIAL SKELETON
    # ==============================================================================
    print(f"\n{get_timestamp()} --- Creating ir-metadata.yml file ---")

    tag_yaml_string = ", ".join(RUN_TAG_KEYWORDS)

    # This YAML string is meticulously filled out according to the provided skeleton.
    # The key for research goal has been changed to 'research_goal' (with an underscore).
    yaml_content = f"""
tag: '{tag_yaml_string}'

actor:
  team: '{TEAM_NAME}'

research_goal: # Changed from 'research goal' (with space) to research_goal (with underscore)
  description: |
    This submission evaluates a two-stage retrieval model for the LongEval-2025 SciRetrieval task.
    The first stage uses a traditional lexical model (BM25+RM3 from an external run, A7) to generate candidates.
    The second stage uses a deep neural network (cross-encoder/ms-marco-MiniLM-L-12-v2) to re-rank these candidates.
    The goal is to assess the effectiveness of this hybrid approach on the test abstracts across the November 2024 and January 2025 snapshots.

platform:
  software:
    libraries:
      - Python {PYTHON_VERSION}
      - PyTorch {PYTORCH_VERSION}
      - sentence-transformers {SENTENCE_TRANSFORMERS_VERSION}
      - pyterrier {PYTERRIER_VERSION}

implementation:
  source:
    repository: '{LINK_TO_CODE_REPOSITORY}'

data:
  'training data':
    - name: LongEval Training Set (implicitly via the A7 BM25 model)
    - name: MS MARCO (implicitly via the ms-marco-MiniLM-L-12-v2 cross-encoder)

method:
  automatic: true

  indexing:
    tokenizer: 'PyTerrier default (for BM25 stage)'
    stemmer: 'Porter (for BM25 stage)'
    stopwords: 'Terrier default stopword list (for BM25 stage)'

  retrieval:
    - name: 'Two-Stage: Lexical (BM25) + Neural Re-ranking (Cross-Encoder)'
      lexical: 'yes'
      deep_neural_model: 'yes'
      sparse_neural_model: 'no'
      dense_neural_model: 'yes'
      single_stage_retrieval: 'no'
"""
    yaml_path = os.path.join(current_run_submission_dir, "ir-metadata.yml")
    with open(yaml_path, 'w') as f:
        f.write(yaml_content.strip())
    print(f"{get_timestamp()}   ✅ Created definitive metadata file: {yaml_path}")

    # ==============================================================================
    # 6. VERIFY WITH TIRA-CLI
    # ==============================================================================
    print(f"\n{get_timestamp()} --- Instructions for TIRA CLI Verification & Upload ---")
    print(f"{get_timestamp()} 1. Verify (Dry Run). This should now succeed:")
    print(f"   !tira-cli upload --dataset sci-20250430-test --system {RUN_ID_FOR_TIRA} --dry-run --directory {current_run_submission_dir}")
    print(f"\n{get_timestamp()} 2. If the dry run is successful, upload for real:")
    print(f"   !tira-cli upload --dataset sci-20250430-test --system {RUN_ID_FOR_TIRA} --directory {current_run_submission_dir}")

print(f"\n{get_timestamp()} --- Preparation script for {RUN_ID_FOR_TIRA} complete. Check logs for errors. --")

Mounted at /content/drive
2025-05-25 18:34:58 Google Drive mounted successfully.
2025-05-25 18:34:58 --------------------------------------------------

2025-05-25 18:34:58 --- Preparing TIRA submission folder for: C3-CE-L12-TestAbs ---
2025-05-25 18:34:58   Submission will be prepared in: /content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v4/submission_C3-CE-L12-TestAbs

2025-05-25 18:34:58 --- Loading QIDs and splitting TREC file ---
2025-05-25 18:34:58   Loaded 99 QIDs from longeval_sci_testing_2025_abstract/queries_2024-11_test.txt
2025-05-25 18:34:58   Loaded 492 QIDs from longeval_sci_testing_2025_abstract/queries_2025-01_test.txt
2025-05-25 18:34:58   ✅ Finished splitting. 485 lines for Nov, 2210 for Jan.

2025-05-25 18:34:58 --- Creating ir-metadata.yml file ---
2025-05-25 18:34:58   ✅ Created definitive metadata file: /content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v4/submission_C3-CE-L12-TestAbs/ir-metadata.yml

2025-05-25 18:34:58 --- Instruc

In [ ]:
!pip3 install --upgrade -q tira
!tira-cli login --token 46a92f03572bd0b6968f720c08d6527c2bbabb83a1406b91445bd56fb0632f5b

!tira-cli verify-installation

✓ You are authenticated against www.tira.io.
✓ TIRA home is writable.
✖ Docker/Podman is not installed. You can not run dockerized TIRA submissions.
✓ The tirex-tracker works and will track experimental metadata.

Result:
✖ Your installation is not valid, you might not use all features.


In [ ]:
!tira-cli upload --dataset sci-20250430-test --system C3-CE-L12-TestAbs --dry-run --directory /content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v2/submission_C3-CE-L12-TestAbs

I check that the submission in directory '/content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v2/submission_C3-CE-L12-TestAbs' is valid...
	✓ I will check that the data in /content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v2/submission_C3-CE-L12-TestAbs is valid ...
	✓ The run in subdirectory 2024-11 is valid.
	✓ The run in subdirectory 2025-01 is valid.
	✓ The file ir-metadata.yml is valid.

Result:
	✓ The run is valid. I skip upload to TIRA as --dry-run was passed.


In [ ]:
!tira-cli upload --dataset sci-20250430-test --system C3-CE-L12-TestAbs --directory /content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v2/submission_C3-CE-L12-TestAbs

I check that the submission in directory '/content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v2/submission_C3-CE-L12-TestAbs' is valid...
	✓ I will check that the data in /content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v2/submission_C3-CE-L12-TestAbs is valid ...
	✓ The run in subdirectory 2024-11 is valid.
	✓ The run in subdirectory 2025-01 is valid.
	✓ The file ir-metadata.yml is valid.


Upload /content/drive/MyDrive/colab-4/tira_official_submissions_prepared_v2/submission_C3-CE-L12-TestAbs to TIRA: 100% 47.1k/47.1k [00:00<00:00, 50.7kB/s]


	✓ The data is uploaded.

I upload the metadata for the submission...
	✓ Done. Your run is available as C3-CE-L12-TestAbs at:
	https://www.tira.io/submit/longeval-2025/user/clef25-sambs/upload-submission


In [ ]:
import os
from google.colab import drive

# Mount your Google Drive
# You will be asked to authorize this script to access your drive.
drive.mount('/content/drive')

# --- Configuration ---
# The path to your folder on Google Drive.
# "G:\Meine Ablage\" is equivalent to "/content/drive/MyDrive/" in Colab.
directory_path = "/content/drive/MyDrive/colab-4/final_submission_trec"
# ---

print(f"\n## Searching for .txt files in: {directory_path}\n")

# Check if the directory actually exists
if not os.path.isdir(directory_path):
    print(f"❌ Error: The directory was not found.")
    print("Please make sure the path is correct and the folder exists on your Google Drive.")
else:
    # Find all files in the directory that end with .txt
    trec_files = [f for f in os.listdir(directory_path) if f.endswith('.txt')]

    if trec_files:
        print("✅ Found the following TREC run files:")
        # Sort the list alphabetically for easier reading
        for filename in sorted(trec_files):
            print(f"  - {filename}")
    else:
        print("ℹ️ No `.txt` files were found in this directory.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

## Searching for .txt files in: /content/drive/MyDrive/colab-4/final_submission_trec

✅ Found the following TREC run files:
  - clef25-sambs_A1_TFIDF_TrainAbs.txt
  - clef25-sambs_A2_TFIDF_TrainFull.txt
  - clef25-sambs_A3_TFIDF_TestAbs.txt
  - clef25-sambs_A4_TFIDF_TestFull.txt
  - clef25-sambs_A5_BM25_TrainAbs.txt
  - clef25-sambs_A6_BM25_TrainFull.txt
  - clef25-sambs_A7_BM25_TestAbs.txt
  - clef25-sambs_A8_BM25_TestFull.txt
  - clef25-sambs_B1_SBERT_V1_TrainAbs.txt
  - clef25-sambs_B1_SBERT_V2_MPNet_TrainAbs.txt
  - clef25-sambs_B2_SBERT_V1_TrainFull.txt
  - clef25-sambs_B2_SBERT_V2_MPNet_TrainFull.txt
  - clef25-sambs_B3_SBERT_V1_TestAbs.txt
  - clef25-sambs_B3_SBERT_V2_MPNet_TestAbs.txt
  - clef25-sambs_B4_SBERT_V1_TestFull.txt
  - clef25-sambs_B4_SBERT_V2_MPNet_TestFull.txt
  - clef25-sambs_C1_CE_L12_TrainAbs.txt
  - clef25-sambs_C1_CE_L6_TrainAbs.txt

In [ ]:
import os
import shutil
import zipfile
import time
from google.colab import drive
import platform

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# ==============================================================================
# 0. MOUNT GOOGLE DRIVE
# ==============================================================================
try:
    drive.mount('/content/drive', force_remount=True)
    print(f"{get_timestamp()} Google Drive mounted successfully.")
except Exception as e:
    print(f"{get_timestamp()} Error mounting Google Drive: {e}")
    raise SystemExit("Google Drive mount failed. Cannot proceed.")

print(f"{get_timestamp()} " + "-" * 50)

# ==============================================================================
# 1. SETUP - CONFIGURE FOR ALL RUNS
# ==============================================================================
# --- List of all TEST run files you want to submit ---
test_run_files = [
    "clef25-sambs_A3_TFIDF_TestAbs.txt",
    "clef25-sambs_A4_TFIDF_TestFull.txt",
    "clef25-sambs_A7_BM25_TestAbs.txt",
    "clef25-sambs_A8_BM25_TestFull.txt",
    "clef25-sambs_B3_SBERT_V1_TestAbs.txt",
    "clef25-sambs_B3_SBERT_V2_MPNet_TestAbs.txt",
    "clef25-sambs_B4_SBERT_V1_TestFull.txt",
    "clef25-sambs_B4_SBERT_V2_MPNet_TestFull.txt",
    "clef25-sambs_C3_CE_L12_TestAbs.txt",
    "clef25-sambs_C3_CE_L6_TestAbs.txt",
    "clef25-sambs_C4_CE_L12_TestFull.txt",
    "clef25-sambs_C4_CE_L6_TestFull.txt",
]

# --- Core Configuration ---
TEAM_NAME = "clef25-sambs"
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"
SOURCE_TREC_FILE_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_trec") # The folder from your screenshot
TIRA_SUBMISSIONS_PARENT_DIR = os.path.join(DRIVE_BASE_PATH, "tira_official_submissions_ALL") # A new parent folder for all runs
os.makedirs(TIRA_SUBMISSIONS_PARENT_DIR, exist_ok=True)

# --- Paths to Query ZIP files ---
QUERY_ZIP_ABSTRACTS = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_abstract.zip')
NOV_Q_ABSTRACTS = 'longeval_sci_testing_2025_abstract/queries_2024-11_test.txt'
JAN_Q_ABSTRACTS = 'longeval_sci_testing_2025_abstract/queries_2025-01_test.txt'

QUERY_ZIP_FULLTEXT = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_fulltext.zip')
NOV_Q_FULLTEXT = 'longeval_sci_testing_2025_fulltext/queries_2024-11_test.txt'
JAN_Q_FULLTEXT = 'longeval_sci_testing_2025_fulltext/queries_2025-01_test.txt'

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def load_qids_from_snapshot_zip(zip_path, query_file_in_zip):
    qids = set()
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            with z.open(query_file_in_zip) as f:
                for raw_line in f:
                    line = raw_line.decode('utf-8').strip()
                    if line: qids.add(line.split()[0])
        return qids
    except Exception as e:
        print(f"   ❌ ERROR loading QIDs from {zip_path}/{query_file_in_zip}: {e}")
        return None

def generate_metadata_yaml(filename, run_id_for_tira, team_name):
    """Dynamically generates the YAML content based on the filename."""

    # --- Basic Info ---
    tag = run_id_for_tira.replace("-", ", ")

    # --- Research Goal Description ---
    description = f"This run evaluates the '{run_id_for_tira}' approach for the LongEval-2025 SciRetrieval task."
    if "BM25" in filename or "TFIDF" in filename:
        description += " It uses a traditional lexical retrieval model."
    elif "SBERT" in filename:
        description += " It uses a dense retrieval model based on SBERT."
    elif "CE" in filename:
        description += " It uses a neural re-ranking model (Cross-Encoder) on top of a lexical first stage."

    # --- Method Specifics ---
    is_lexical = 'no'
    is_deep = 'no'
    is_sparse = 'no'
    is_dense = 'no'
    is_single_stage = 'yes'

    if "BM25" in filename or "TFIDF" in filename:
        is_lexical = 'yes'
    elif "SBERT" in filename:
        is_deep = 'yes'
        is_dense = 'yes'
    elif "CE" in filename:
        is_lexical = 'yes' # First stage is lexical
        is_deep = 'yes'
        is_dense = 'yes' # Cross-Encoders are dense models
        is_single_stage = 'no'

    yaml_template = f"""
tag: '{tag}'
actor:
  team: '{team_name}'
'research goal':
  description: |
    {description}
platform:
  software:
    libraries:
      - Python {platform.python_version()}
implementation:
  source:
    repository: 'https://github.com/your-username/your-repo' # Update with your link
data:
  'training data':
    - name: 'LongEval Training Set and/or public models like SBERT/MSMARCO'
method:
  automatic: true
  indexing:
    tokenizer: 'Varies by model (e.g., PyTerrier default or WordPiece)'
    stemmer: 'Varies by model (e.g., Porter or none)'
    stopwords: 'Varies by model (e.g., Terrier default or none)'
  retrieval:
    - name: '{run_id_for_tira}'
      lexical: '{is_lexical}'
      deep_neural_model: '{is_deep}'
      sparse_neural_model: '{is_sparse}'
      dense_neural_model: '{is_dense}'
      single_stage_retrieval: '{is_single_stage}'
"""
    return yaml_template.strip()

# ==============================================================================
# 3. MAIN PROCESSING LOOP
# ==============================================================================
all_cli_commands = []

for filename in test_run_files:
    print(f"\n" + "="*60)
    print(f"{get_timestamp()} Processing run: {filename}")

    # --- Parse filename to create a unique run ID for TIRA ---
    # e.g., "clef25-sambs_A3_TFIDF_TestAbs.txt" -> "A3-TFIDF-TestAbs"
    run_id_for_tira = "-".join(filename.replace(".txt", "").split("_")[1:])
    print(f"{get_timestamp()}   Run ID for TIRA: {run_id_for_tira}")

    # --- Create submission directory ---
    current_run_submission_dir = os.path.join(TIRA_SUBMISSIONS_PARENT_DIR, f"submission_{run_id_for_tira}")
    os.makedirs(os.path.join(current_run_submission_dir, "2024-11"), exist_ok=True)
    os.makedirs(os.path.join(current_run_submission_dir, "2025-01"), exist_ok=True)

    # --- Select correct query files based on filename ---
    if "TestAbs" in filename:
        print(f"{get_timestamp()}   Run type: Abstracts")
        qids_nov = load_qids_from_snapshot_zip(QUERY_ZIP_ABSTRACTS, NOV_Q_ABSTRACTS)
        qids_jan = load_qids_from_snapshot_zip(QUERY_ZIP_ABSTRACTS, JAN_Q_ABSTRACTS)
    elif "TestFull" in filename:
        print(f"{get_timestamp()}   Run type: Full-text")
        qids_nov = load_qids_from_snapshot_zip(QUERY_ZIP_FULLTEXT, NOV_Q_FULLTEXT)
        qids_jan = load_qids_from_snapshot_zip(QUERY_ZIP_FULLTEXT, JAN_Q_FULLTEXT)
    else:
        print(f"{get_timestamp()}   ❌ ERROR: Cannot determine if run is for Abstracts or Full-text.")
        continue

    if not qids_nov or not qids_jan:
        print(f"{get_timestamp()}   ❌ ERROR: Skipping run due to missing QIDs.")
        continue

    # --- Split the TREC file into snapshots ---
    source_file_path = os.path.join(SOURCE_TREC_FILE_DIR, filename)
    if not os.path.exists(source_file_path):
        print(f"{get_timestamp()}   ❌ ERROR: Source file not found: {source_file_path}")
        continue

    lines_nov, lines_jan = 0, 0
    with open(source_file_path, 'r') as f_in, \
         open(os.path.join(current_run_submission_dir, "2024-11", "run.txt"), 'w') as f_nov, \
         open(os.path.join(current_run_submission_dir, "2025-01", "run.txt"), 'w') as f_jan:
        for line in f_in:
            qid = line.strip().split()[0]
            if qid in qids_nov: f_nov.write(line); lines_nov += 1
            elif qid in qids_jan: f_jan.write(line); lines_jan += 1
    print(f"{get_timestamp()}   ✅ Split file: {lines_nov} lines for Nov, {lines_jan} for Jan.")

    # --- Generate and save metadata ---
    yaml_content = generate_metadata_yaml(filename, run_id_for_tira, TEAM_NAME)
    yaml_path = os.path.join(current_run_submission_dir, "ir-metadata.yml")
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    print(f"{get_timestamp()}   ✅ Generated ir-metadata.yml")

    # --- Store the commands for later ---
    cmd_dry_run = f"!tira-cli upload --dataset sci-20250430-test --system {run_id_for_tira} --dry-run --directory {current_run_submission_dir}"
    cmd_upload = f"!tira-cli upload --dataset sci-20250430-test --system {run_id_for_tira} --directory {current_run_submission_dir}"
    all_cli_commands.append((run_id_for_tira, cmd_dry_run, cmd_upload))

print("\n\n" + "#"*60)
print(f"{get_timestamp()} SCRIPT COMPLETE")
print("#"*60)
print("\nAll submission folders have been prepared. Now, you can run the commands below one by one in a new cell.")
print("First, run the '--dry-run' command to validate. If it's successful, run the final upload command.\n")

for run_id, cmd_dry_run, cmd_upload in all_cli_commands:
    print(f"--- Commands for run: {run_id} ---")
    print("# 1. Validate with dry run:")
    print(cmd_dry_run)
    print("\n# 2. If validation is successful, upload for real:")
    print(cmd_upload)
    print("-"*(26 + len(run_id)) + "\n")


Mounted at /content/drive
2025-05-25 18:55:24 Google Drive mounted successfully.
2025-05-25 18:55:24 --------------------------------------------------

2025-05-25 18:55:25 Processing run: clef25-sambs_A3_TFIDF_TestAbs.txt
2025-05-25 18:55:25   Run ID for TIRA: A3-TFIDF-TestAbs
2025-05-25 18:55:25   Run type: Abstracts
2025-05-25 18:55:25   ✅ Split file: 9900 lines for Nov, 45300 for Jan.
2025-05-25 18:55:25   ✅ Generated ir-metadata.yml

2025-05-25 18:55:25 Processing run: clef25-sambs_A4_TFIDF_TestFull.txt
2025-05-25 18:55:25   Run ID for TIRA: A4-TFIDF-TestFull
2025-05-25 18:55:25   Run type: Full-text
2025-05-25 18:55:26   ✅ Split file: 9900 lines for Nov, 45300 for Jan.
2025-05-25 18:55:26   ✅ Generated ir-metadata.yml

2025-05-25 18:55:26 Processing run: clef25-sambs_A7_BM25_TestAbs.txt
2025-05-25 18:55:26   Run ID for TIRA: A7-BM25-TestAbs
2025-05-25 18:55:26   Run type: Abstracts
2025-05-25 18:55:27   ✅ Split file: 9700 lines for Nov, 44200 for Jan.
2025-05-25 18:55:27   ✅ Gene

In [ ]:
!pip3 install --upgrade tira
!tira-cli login --token 46a92f03572bd0b6968f720c08d6527c2bbabb83a1406b91445bd56fb0632f5b

!tira-cli verify-installation

✓ You are authenticated against www.tira.io.
✓ TIRA home is writable.
✖ Docker/Podman is not installed. You can not run dockerized TIRA submissions.
✓ The tirex-tracker works and will track experimental metadata.

Result:
✖ Your installation is not valid, you might not use all features.


In [ ]:
!tira-cli upload --dataset sci-20250430-test --system A3-TFIDF-TestAbs --directory /content/drive/MyDrive/colab-4/tira_official_submissions_ALL/submission_A3-TFIDF-TestAbs --dry-run

I check that the submission in directory '/content/drive/MyDrive/colab-4/tira_official_submissions_ALL/submission_A3-TFIDF-TestAbs' is valid...

I will check that the data in /content/drive/MyDrive/colab-4/tira_official_submissions_ALL/submission_A3-TFIDF-TestAbs is valid ...
	✖ I expected a run file in the subdirectory 2024-11. Error: The run file has duplicate documents: the document with id "70339957" appears multiple times for query "d585f080-4519-4952-8278-0d13fcd03fec".
	✖ I expected a run file in the subdirectory 2025-01. Error: The run file has duplicate documents: the document with id "8866276" appears multiple times for query "6bdc12c2-05cf-4dec-a005-8bd0b695b47f".
	✓ The file ir-metadata.yml is valid.

Result:
	✖ Could not upload to TIRA as the directory /content/drive/MyDrive/colab-4/tira_official_submissions_ALL/submission_A3-TFIDF-TestAbs is not a valid submission.


In [ ]:
import os
from google.colab import drive
import time

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# ==============================================================================
# 0. MOUNT GOOGLE DRIVE
# ==============================================================================
try:
    drive.mount('/content/drive', force_remount=True)
    print(f"{get_timestamp()} Google Drive mounted successfully.")
except Exception as e:
    print(f"{get_timestamp()} Error mounting Google Drive: {e}")
    raise SystemExit("Google Drive mount failed. Cannot proceed.")

# ==============================================================================
# 1. SETUP - CONFIGURE DIRECTORIES
# ==============================================================================
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"
# The folder where your original run files with duplicates are located
SOURCE_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_trec")
# A NEW folder where the cleaned, de-duplicated files will be saved
DESTINATION_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_trec_deduped")

print(f"\n{get_timestamp()} Source Directory:      {SOURCE_DIR}")
print(f"{get_timestamp()} Destination Directory: {DESTINATION_DIR}")

# Create the destination directory if it doesn't exist
os.makedirs(DESTINATION_DIR, exist_ok=True)

# ==============================================================================
# 2. MAIN PROCESSING LOOP
# ==============================================================================
print("\n" + "="*60)
print(f"{get_timestamp()} Starting de-duplication process...")
print("="*60)

# Get all .txt files from the source directory
try:
    source_files = [f for f in os.listdir(SOURCE_DIR) if f.endswith('.txt')]
    if not source_files:
        print(f"❌ No .txt files found in {SOURCE_DIR}. Please check the path.")
except FileNotFoundError:
    print(f"❌ Source directory not found: {SOURCE_DIR}. Please check the path.")
    source_files = []


for filename in source_files:
    source_path = os.path.join(SOURCE_DIR, filename)
    destination_path = os.path.join(DESTINATION_DIR, filename)

    print(f"\n{get_timestamp()} Processing: {filename}")

    seen_pairs = {} # To store {query_id: {doc_id_1, doc_id_2, ...}}
    lines_written = 0
    lines_read = 0
    duplicates_found = 0

    try:
        with open(source_path, 'r') as f_in, open(destination_path, 'w') as f_out:
            for line in f_in:
                lines_read += 1
                parts = line.strip().split()
                if len(parts) < 3: continue # Skip malformed lines

                query_id = parts[0]
                doc_id = parts[2]

                # Check if we've seen this query_id before
                if query_id not in seen_pairs:
                    seen_pairs[query_id] = set()

                # Check if we've seen this doc_id for this query
                if doc_id not in seen_pairs[query_id]:
                    # If not, write it to the new file and record it
                    f_out.write(line)
                    seen_pairs[query_id].add(doc_id)
                    lines_written += 1
                else:
                    # If we have seen it, it's a duplicate for this query
                    duplicates_found += 1

        print(f"{get_timestamp()}   ✅ Complete. Found and removed {duplicates_found} duplicate(s).")
        print(f"{get_timestamp()}   Original lines: {lines_read}, Cleaned lines: {lines_written}")
        print(f"{get_timestamp()}   Clean file saved to: {destination_path}")

    except Exception as e:
        print(f"{get_timestamp()}   ❌ ERROR processing {filename}: {e}")

print("\n\n" + "="*60)
print(f"{get_timestamp()} De-duplication script finished.")
print("="*60)


Mounted at /content/drive
2025-05-25 19:09:12 Google Drive mounted successfully.

2025-05-25 19:09:12 Source Directory:      /content/drive/MyDrive/colab-4/final_submission_trec
2025-05-25 19:09:12 Destination Directory: /content/drive/MyDrive/colab-4/final_submission_trec_deduped

2025-05-25 19:09:12 Starting de-duplication process...

2025-05-25 19:09:12 Processing: clef25-sambs_A7_BM25_TestAbs.txt
2025-05-25 19:09:12   ✅ Complete. Found and removed 1 duplicate(s).
2025-05-25 19:09:12   Original lines: 53900, Cleaned lines: 53899
2025-05-25 19:09:12   Clean file saved to: /content/drive/MyDrive/colab-4/final_submission_trec_deduped/clef25-sambs_A7_BM25_TestAbs.txt

2025-05-25 19:09:12 Processing: clef25-sambs_A5_BM25_TrainAbs.txt
2025-05-25 19:09:12   ✅ Complete. Found and removed 0 duplicate(s).
2025-05-25 19:09:12   Original lines: 37500, Cleaned lines: 37500
2025-05-25 19:09:12   Clean file saved to: /content/drive/MyDrive/colab-4/final_submission_trec_deduped/clef25-sambs_A5_BM25

In [ ]:
import os
import shutil
import zipfile
import time
from google.colab import drive
import platform
import subprocess

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# ==============================================================================
# 0. MOUNT GOOGLE DRIVE
# ==============================================================================
try:
    drive.mount('/content/drive', force_remount=True)
    print(f"{get_timestamp()} Google Drive mounted successfully.")
except Exception as e:
    print(f"{get_timestamp()} Error mounting Google Drive: {e}")
    raise SystemExit("Google Drive mount failed. Cannot proceed.")

print(f"{get_timestamp()} " + "-" * 50)

# ==============================================================================
# 1. SETUP - CONFIGURE FOR ALL RUNS
# ==============================================================================
# --- List of all TEST run files you want to submit ---
# This list only includes the test files, as they are the only ones for submission.
test_run_files = [
    "clef25-sambs_A3_TFIDF_TestAbs.txt",
    "clef25-sambs_A4_TFIDF_TestFull.txt",
    "clef25-sambs_A7_BM25_TestAbs.txt",
    "clef25-sambs_A8_BM25_TestFull.txt",
    "clef25-sambs_B3_SBERT_V1_TestAbs.txt",
    "clef25-sambs_B3_SBERT_V2_MPNet_TestAbs.txt",
    "clef25-sambs_B4_SBERT_V1_TestFull.txt",
    "clef25-sambs_B4_SBERT_V2_MPNet_TestFull.txt",
    "clef25-sambs_C3_CE_L12_TestAbs.txt",
    "clef25-sambs_C3_CE_L6_TestAbs.txt",
    "clef25-sambs_C4_CE_L12_TestFull.txt",
    "clef25-sambs_C4_CE_L6_TestFull.txt",
]

# --- Core Configuration ---
TEAM_NAME = "clef25-sambs"
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"

# --- SOURCE AND TARGET DIRECTORIES ---
# Updated to use the cleaned, de-duplicated files
SOURCE_TREC_FILE_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_trec_deduped")
TIRA_SUBMISSIONS_PARENT_DIR = os.path.join(DRIVE_BASE_PATH, "tira_official_submissions_ALL")
os.makedirs(TIRA_SUBMISSIONS_PARENT_DIR, exist_ok=True)

# --- Paths to Query ZIP files ---
QUERY_ZIP_ABSTRACTS = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_abstract.zip')
NOV_Q_ABSTRACTS = 'longeval_sci_testing_2025_abstract/queries_2024-11_test.txt'
JAN_Q_ABSTRACTS = 'longeval_sci_testing_2025_abstract/queries_2025-01_test.txt'

QUERY_ZIP_FULLTEXT = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_fulltext.zip')
NOV_Q_FULLTEXT = 'longeval_sci_testing_2025_fulltext/queries_2024-11_test.txt'
JAN_Q_FULLTEXT = 'longeval_sci_testing_2025_fulltext/queries_2025-01_test.txt'

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def load_qids_from_snapshot_zip(zip_path, query_file_in_zip):
    qids = set()
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            # Ensure the file exists in the zip archive
            if query_file_in_zip not in z.namelist():
                print(f"   ❌ ERROR: Query file '{query_file_in_zip}' not found in '{zip_path}'. Available: {z.namelist()}")
                return None
            with z.open(query_file_in_zip) as f:
                for raw_line in f:
                    line = raw_line.decode('utf-8').strip()
                    if line: qids.add(line.split()[0]) # Assuming QID is the first element
        return qids
    except FileNotFoundError:
        print(f"   ❌ ERROR: ZIP file not found at '{zip_path}'.")
        return None
    except Exception as e:
        print(f"   ❌ ERROR loading QIDs from {zip_path}/{query_file_in_zip}: {e}")
        return None

def generate_metadata_yaml(filename, run_id_for_tira, team_name):
    """Dynamically generates the YAML content based on the filename."""
    tag = run_id_for_tira.replace("-", ", ") # Example: A3-TFIDF-TestAbs -> A3, TFIDF, TestAbs

    # Determine description based on model type in filename
    description = f"This run evaluates the '{run_id_for_tira}' approach for the LongEval-2025 SciRetrieval task."
    if "BM25" in filename or "TFIDF" in filename:
        description += " It uses a traditional lexical retrieval model."
    elif "SBERT" in filename:
        description += " It uses a dense retrieval model based on SBERT."
    elif "CE" in filename: # Cross-Encoder
        description += " It uses a neural re-ranking model (Cross-Encoder) on top of a lexical first stage."
    else:
        description += " It uses a custom approach."

    # Determine method specifics (true/false for YAML)
    is_lexical, is_deep, is_sparse, is_dense, is_single_stage = False, False, False, False, True # Defaults
    if "BM25" in filename or "TFIDF" in filename:
        is_lexical = True
    elif "SBERT" in filename: # Dense retriever
        is_deep = True
        is_dense = True
    elif "CE" in filename: # Cross-encoder (re-ranker)
        is_lexical = True # Assumes a lexical first stage
        is_deep = True
        is_dense = True # Cross-encoders are dense
        is_single_stage = False # It's a re-ranker, so two stages

    yaml_template = f"""
tag: '{tag}'
actor:
  team: '{team_name}'
research goal:
  description: |
    {description}
platform:
  software:
    libraries:
      - Python {platform.python_version()}
implementation:
  source:
    repository: 'https://github.com/santiago-ruiz-moreno/ScientificTextRetrieval'
data:
  'training data':
    - name: 'LongEval Training Set and/or public models like SBERT/MSMARCO'
method:
  automatic: true
  indexing:
    tokenizer: 'Varies by model (e.g., PyTerrier default or WordPiece)'
    stemmer: 'Varies by model (e.g., Porter or none)'
    stopwords: 'Varies by model (e.g., Terrier default or none)'
  retrieval:
    - name: '{run_id_for_tira}'
      lexical: {str(is_lexical).lower()}
      deep_neural_model: {str(is_deep).lower()}
      sparse_neural_model: {str(is_sparse).lower()}
      dense_neural_model: {str(is_dense).lower()}
      single_stage_retrieval: {str(is_single_stage).lower()}
"""
    return yaml_template.strip()

# ==============================================================================
# 3. FILE PREPARATION LOOP
# ==============================================================================
all_cli_commands = []
print(f"{get_timestamp()} Starting file preparation for {len(test_run_files)} runs from DEDUPED directory...")

for filename in test_run_files:
    print(f"\n" + "-"*60)
    print(f"{get_timestamp()} Preparing: {filename}")

    # Generate run_id_for_tira (e.g., "A3-TFIDF-TestAbs")
    run_id_for_tira = "-".join(filename.replace(".txt", "").split("_")[1:])

    current_run_submission_dir = os.path.join(TIRA_SUBMISSIONS_PARENT_DIR, f"submission_{run_id_for_tira}")
    os.makedirs(os.path.join(current_run_submission_dir, "2024-11"), exist_ok=True)
    os.makedirs(os.path.join(current_run_submission_dir, "2025-01"), exist_ok=True)

    # Select appropriate query files
    if "TestAbs" in filename:
        qids_nov = load_qids_from_snapshot_zip(QUERY_ZIP_ABSTRACTS, NOV_Q_ABSTRACTS)
        qids_jan = load_qids_from_snapshot_zip(QUERY_ZIP_ABSTRACTS, JAN_Q_ABSTRACTS)
    elif "TestFull" in filename:
        qids_nov = load_qids_from_snapshot_zip(QUERY_ZIP_FULLTEXT, NOV_Q_FULLTEXT)
        qids_jan = load_qids_from_snapshot_zip(QUERY_ZIP_FULLTEXT, JAN_Q_FULLTEXT)
    else:
        print(f"   ⚠️ WARNING: Could not determine if '{filename}' is for Abstracts or Full-text. Skipping.")
        continue # Skip this file if type is unclear

    if not qids_nov or not qids_jan:
        print(f"   ❌ ERROR: Failed to load QIDs for '{filename}'. Skipping.")
        continue

    source_file_path = os.path.join(SOURCE_TREC_FILE_DIR, filename)
    if not os.path.exists(source_file_path):
        print(f"   ❌ ERROR: Source file not found in DEDUPED folder: {source_file_path}")
        continue

    lines_nov, lines_jan = 0, 0
    try:
        with open(source_file_path, 'r') as f_in, \
             open(os.path.join(current_run_submission_dir, "2024-11", "run.txt"), 'w') as f_nov, \
             open(os.path.join(current_run_submission_dir, "2025-01", "run.txt"), 'w') as f_jan:
            for line_num, line_content in enumerate(f_in, 1):
                parts = line_content.strip().split()
                if not parts: # Skip empty lines
                    print(f"      ⚠️ Warning: Empty line {line_num} in {filename}")
                    continue
                qid = parts[0]
                if qid in qids_nov: f_nov.write(line_content); lines_nov += 1
                elif qid in qids_jan: f_jan.write(line_content); lines_jan += 1
    except Exception as e:
        print(f"   ❌ ERROR processing/splitting file {filename}: {e}")
        continue

    yaml_content = generate_metadata_yaml(filename, run_id_for_tira, TEAM_NAME)
    try:
        with open(os.path.join(current_run_submission_dir, "ir-metadata.yml"), 'w') as f: f.write(yaml_content)
    except Exception as e:
        print(f"   ❌ ERROR writing ir-metadata.yml for {filename}: {e}")
        continue

    print(f"{get_timestamp()}   ✅ Preparation complete for {run_id_for_tira}. Nov lines: {lines_nov}, Jan lines: {lines_jan}.")
    all_cli_commands.append((run_id_for_tira, current_run_submission_dir))

# ==============================================================================
# 4. AUTOMATED EXECUTION LOOP
# ==============================================================================
print("\n\n" + "#"*60)
print(f"{get_timestamp()} FILE PREPARATION COMPLETE")
print(f"{get_timestamp()} Now starting automated execution of TIRA uploads...")
print("#"*60)

if not all_cli_commands:
    print("No runs were prepared for upload. Exiting.")
else:
    for run_id, submission_dir in all_cli_commands:
        print(f"\n--- Processing run: {run_id} ---")

        # --- 1. Validate with --dry-run ---
        print(f"   [1/2] Validating '{run_id}' with --dry-run...")
        dry_run_command = [
            "tira-cli", "upload", "--dataset", "sci-20250430-test",
            "--system", run_id, "--dry-run", "--directory", submission_dir
        ]
        dry_run_result = subprocess.run(dry_run_command, capture_output=True, text=True, check=False) # Added check=False

        # Print command output for debugging
        print("STDOUT from dry-run:")
        print(dry_run_result.stdout)
        print("STDERR from dry-run:")
        print(dry_run_result.stderr)

        # More robust check for success: check return code and a more general success message.
        # The checkmark character can be problematic.
        validation_successful = False
        if dry_run_result.returncode == 0:
            if "The run is valid." in dry_run_result.stdout: # Check without the checkmark
                 validation_successful = True
            elif not dry_run_result.stdout.strip() and not dry_run_result.stderr.strip(): # Some versions might output nothing on success
                 # This case is tricky, might need more specific success string if TIRA version changes
                 print("   ⚠️ Dry run produced no output, assuming success based on return code 0. Verify manually if uploads fail.")
                 validation_successful = True


        if validation_successful:
            print(f"   ✅ Validation successful for '{run_id}'.")

            # --- 2. If validation succeeded, upload for real ---
            print(f"\n   [2/2] Uploading '{run_id}'...")
            upload_command = [
                "tira-cli", "upload", "--dataset", "sci-20250430-test",
                "--system", run_id, "--directory", submission_dir
            ]
            upload_result = subprocess.run(upload_command, capture_output=True, text=True, check=False) # Added check=False

            print("STDOUT from upload:")
            print(upload_result.stdout)
            print("STDERR from upload:")
            print(upload_result.stderr)

            if upload_result.returncode == 0 and "✓ The data is uploaded." in upload_result.stdout : # Check for actual upload success message
                print(f"   ✅ Successfully uploaded '{run_id}'.")
            else:
                print(f"   ❌ Upload FAILED for '{run_id}'. Return code: {upload_result.returncode}")
        else:
            print(f"   ❌ Validation FAILED for '{run_id}'. Skipping upload. Return code: {dry_run_result.returncode}")
        print("-" * (21 + len(run_id)))

    print("\n\n##################################")
    print("All executions complete.")
    print("##################################")


Mounted at /content/drive
2025-05-25 19:14:02 Google Drive mounted successfully.
2025-05-25 19:14:02 --------------------------------------------------
2025-05-25 19:14:02 Starting file preparation for 12 runs from DEDUPED directory...

------------------------------------------------------------
2025-05-25 19:14:02 Preparing: clef25-sambs_A3_TFIDF_TestAbs.txt
2025-05-25 19:14:02   ✅ Preparation complete for A3-TFIDF-TestAbs. Nov lines: 7685, Jan lines: 34004.

------------------------------------------------------------
2025-05-25 19:14:02 Preparing: clef25-sambs_A4_TFIDF_TestFull.txt
2025-05-25 19:14:02   ✅ Preparation complete for A4-TFIDF-TestFull. Nov lines: 7685, Jan lines: 34058.

------------------------------------------------------------
2025-05-25 19:14:02 Preparing: clef25-sambs_A7_BM25_TestAbs.txt
2025-05-25 19:14:02   ✅ Preparation complete for A7-BM25-TestAbs. Nov lines: 9699, Jan lines: 44200.

------------------------------------------------------------
2025-05-25 19:1

In [ ]:
import os
import shutil
import zipfile
import time
from google.colab import drive
import platform
import subprocess

# --- Timestamp helper ---
def get_timestamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# ==============================================================================
# 0. MOUNT GOOGLE DRIVE
# ==============================================================================
try:
    drive.mount('/content/drive', force_remount=True)
    print(f"{get_timestamp()} Google Drive mounted successfully.")
except Exception as e:
    print(f"{get_timestamp()} Error mounting Google Drive: {e}")
    raise SystemExit("Google Drive mount failed. Cannot proceed.")

print(f"{get_timestamp()} " + "-" * 50)

# ==============================================================================
# 1. SETUP - CONFIGURE FOR ALL RUNS
# ==============================================================================
# --- List of all TEST run files you want to submit ---
# MODIFIED FOR SINGLE RUN TEST: Only processing the first file for now.
test_run_files = [
    "clef25-sambs_A3_TFIDF_TestAbs.txt",
    "clef25-sambs_A4_TFIDF_TestFull.txt",
    "clef25-sambs_A7_BM25_TestAbs.txt",
    "clef25-sambs_A8_BM25_TestFull.txt",
    "clef25-sambs_B3_SBERT_V1_TestAbs.txt",
    "clef25-sambs_B3_SBERT_V2_MPNet_TestAbs.txt",
    "clef25-sambs_B4_SBERT_V1_TestFull.txt",
    "clef25-sambs_B4_SBERT_V2_MPNet_TestFull.txt",
    "clef25-sambs_C3_CE_L12_TestAbs.txt",
    "clef25-sambs_C3_CE_L6_TestAbs.txt",
    "clef25-sambs_C4_CE_L12_TestFull.txt",
    "clef25-sambs_C4_CE_L6_TestFull.txt",
]
# The WARNING message about SINGLE RUN TEST MODE can also be removed or commented out.
# print(f"{get_timestamp()} WARNING: Script is in SINGLE RUN TEST MODE. Processing only: {test_run_files}")
print(f"{get_timestamp()} WARNING: Script is in SINGLE RUN TEST MODE. Processing only: {test_run_files}")


# --- Core Configuration ---
TEAM_NAME = "clef25-sambs"
DRIVE_BASE_PATH = "/content/drive/MyDrive/colab-4/"

# --- SOURCE AND TARGET DIRECTORIES ---
# Updated to use the cleaned, de-duplicated files
SOURCE_TREC_FILE_DIR = os.path.join(DRIVE_BASE_PATH, "final_submission_trec_deduped")
TIRA_SUBMISSIONS_PARENT_DIR = os.path.join(DRIVE_BASE_PATH, "tira_official_submissions_ALL")
os.makedirs(TIRA_SUBMISSIONS_PARENT_DIR, exist_ok=True)

# --- Paths to Query ZIP files ---
QUERY_ZIP_ABSTRACTS = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_abstract.zip')
NOV_Q_ABSTRACTS = 'longeval_sci_testing_2025_abstract/queries_2024-11_test.txt'
JAN_Q_ABSTRACTS = 'longeval_sci_testing_2025_abstract/queries_2025-01_test.txt'

QUERY_ZIP_FULLTEXT = os.path.join(DRIVE_BASE_PATH, 'longeval_sci_testing_2025_fulltext.zip')
NOV_Q_FULLTEXT = 'longeval_sci_testing_2025_fulltext/queries_2024-11_test.txt'
JAN_Q_FULLTEXT = 'longeval_sci_testing_2025_fulltext/queries_2025-01_test.txt'

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def load_qids_from_snapshot_zip(zip_path, query_file_in_zip):
    qids = set()
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            # Ensure the file exists in the zip archive
            if query_file_in_zip not in z.namelist():
                print(f"   ❌ ERROR: Query file '{query_file_in_zip}' not found in '{zip_path}'. Available: {z.namelist()}")
                return None
            with z.open(query_file_in_zip) as f:
                for raw_line in f:
                    line = raw_line.decode('utf-8').strip()
                    if line: qids.add(line.split()[0]) # Assuming QID is the first element
        return qids
    except FileNotFoundError:
        print(f"   ❌ ERROR: ZIP file not found at '{zip_path}'.")
        return None
    except Exception as e:
        print(f"   ❌ ERROR loading QIDs from {zip_path}/{query_file_in_zip}: {e}")
        return None

def generate_metadata_yaml(filename, run_id_for_tira, team_name):
    """Dynamically generates the YAML content based on the filename."""
    tag = run_id_for_tira.replace("-", ", ") # Example: A3-TFIDF-TestAbs -> A3, TFIDF, TestAbs

    # Determine description based on model type in filename
    description = f"This run evaluates the '{run_id_for_tira}' approach for the LongEval-2025 SciRetrieval task."
    if "BM25" in filename or "TFIDF" in filename:
        description += " It uses a traditional lexical retrieval model."
    elif "SBERT" in filename:
        description += " It uses a dense retrieval model based on SBERT."
    elif "CE" in filename: # Cross-Encoder
        description += " It uses a neural re-ranking model (Cross-Encoder) on top of a lexical first stage."
    else:
        description += " It uses a custom approach."

    # Determine method specifics (true/false for YAML)
    is_lexical, is_deep, is_sparse, is_dense, is_single_stage = False, False, False, False, True # Defaults
    if "BM25" in filename or "TFIDF" in filename:
        is_lexical = True
    elif "SBERT" in filename: # Dense retriever
        is_deep = True
        is_dense = True
    elif "CE" in filename: # Cross-encoder (re-ranker)
        is_lexical = True # Assumes a lexical first stage
        is_deep = True
        is_dense = True # Cross-encoders are dense
        is_single_stage = False # It's a re-ranker, so two stages

    yaml_template = f"""
tag: '{tag}'
actor:
  team: '{team_name}'
research goal:
  description: |
    {description}
platform:
  software:
    libraries:
      - Python {platform.python_version()}
implementation:
  source:
    repository: 'https://github.com/santiago-ruiz-moreno/ScientificTextRetrieval'
data:
  'training data':
    - name: 'LongEval Training Set and/or public models like SBERT/MSMARCO'
method:
  automatic: true
  indexing:
    tokenizer: 'Varies by model (e.g., PyTerrier default or WordPiece)'
    stemmer: 'Varies by model (e.g., Porter or none)'
    stopwords: 'Varies by model (e.g., Terrier default or none)'
  retrieval:
    - name: '{run_id_for_tira}'
      lexical: {str(is_lexical).lower()}
      deep_neural_model: {str(is_deep).lower()}
      sparse_neural_model: {str(is_sparse).lower()}
      dense_neural_model: {str(is_dense).lower()}
      single_stage_retrieval: {str(is_single_stage).lower()}
"""
    return yaml_template.strip()

# ==============================================================================
# 3. FILE PREPARATION LOOP
# ==============================================================================
all_cli_commands = []
print(f"{get_timestamp()} Starting file preparation for {len(test_run_files)} runs from DEDUPED directory...")

for filename in test_run_files:
    print(f"\n" + "-"*60)
    print(f"{get_timestamp()} Preparing: {filename}")

    # Generate run_id_for_tira (e.g., "A3-TFIDF-TestAbs")
    run_id_for_tira = "-".join(filename.replace(".txt", "").split("_")[1:])

    current_run_submission_dir = os.path.join(TIRA_SUBMISSIONS_PARENT_DIR, f"submission_{run_id_for_tira}")
    os.makedirs(os.path.join(current_run_submission_dir, "2024-11"), exist_ok=True)
    os.makedirs(os.path.join(current_run_submission_dir, "2025-01"), exist_ok=True)

    # Select appropriate query files
    if "TestAbs" in filename:
        qids_nov = load_qids_from_snapshot_zip(QUERY_ZIP_ABSTRACTS, NOV_Q_ABSTRACTS)
        qids_jan = load_qids_from_snapshot_zip(QUERY_ZIP_ABSTRACTS, JAN_Q_ABSTRACTS)
    elif "TestFull" in filename:
        qids_nov = load_qids_from_snapshot_zip(QUERY_ZIP_FULLTEXT, NOV_Q_FULLTEXT)
        qids_jan = load_qids_from_snapshot_zip(QUERY_ZIP_FULLTEXT, JAN_Q_FULLTEXT)
    else:
        print(f"   ⚠️ WARNING: Could not determine if '{filename}' is for Abstracts or Full-text. Skipping.")
        continue # Skip this file if type is unclear

    if not qids_nov or not qids_jan:
        print(f"   ❌ ERROR: Failed to load QIDs for '{filename}'. Skipping.")
        continue

    source_file_path = os.path.join(SOURCE_TREC_FILE_DIR, filename)
    if not os.path.exists(source_file_path):
        print(f"   ❌ ERROR: Source file not found in DEDUPED folder: {source_file_path}")
        continue

    lines_nov, lines_jan = 0, 0
    try:
        with open(source_file_path, 'r') as f_in, \
             open(os.path.join(current_run_submission_dir, "2024-11", "run.txt"), 'w') as f_nov, \
             open(os.path.join(current_run_submission_dir, "2025-01", "run.txt"), 'w') as f_jan:
            for line_num, line_content in enumerate(f_in, 1):
                parts = line_content.strip().split()
                if not parts: # Skip empty lines
                    print(f"      ⚠️ Warning: Empty line {line_num} in {filename}")
                    continue
                qid = parts[0]
                if qid in qids_nov: f_nov.write(line_content); lines_nov += 1
                elif qid in qids_jan: f_jan.write(line_content); lines_jan += 1
    except Exception as e:
        print(f"   ❌ ERROR processing/splitting file {filename}: {e}")
        continue

    yaml_content = generate_metadata_yaml(filename, run_id_for_tira, TEAM_NAME)
    try:
        with open(os.path.join(current_run_submission_dir, "ir-metadata.yml"), 'w') as f: f.write(yaml_content)
    except Exception as e:
        print(f"   ❌ ERROR writing ir-metadata.yml for {filename}: {e}")
        continue

    print(f"{get_timestamp()}   ✅ Preparation complete for {run_id_for_tira}. Nov lines: {lines_nov}, Jan lines: {lines_jan}.")
    all_cli_commands.append((run_id_for_tira, current_run_submission_dir))

# ==============================================================================
# 4. AUTOMATED EXECUTION LOOP
# ==============================================================================
print("\n\n" + "#"*60)
print(f"{get_timestamp()} FILE PREPARATION COMPLETE")
print(f"{get_timestamp()} Now starting automated execution of TIRA uploads...")
print("#"*60)

if not all_cli_commands:
    print("No runs were prepared for upload. Exiting.")
else:
    for run_id, submission_dir in all_cli_commands:
        print(f"\n--- Processing run: {run_id} ---")

        # --- 1. Validate with --dry-run ---
        print(f"   [1/2] Validating '{run_id}' with --dry-run...")
        dry_run_command = [
            "tira-cli", "upload", "--dataset", "sci-20250430-test",
            "--system", run_id, "--dry-run", "--directory", submission_dir
        ]
        # Using check=False to prevent script from stopping on non-zero exit codes for dry-run
        dry_run_result = subprocess.run(dry_run_command, capture_output=True, text=True, check=False)

        # Print command output for debugging
        print("STDOUT from dry-run:")
        print(dry_run_result.stdout)
        print("STDERR from dry-run:")
        print(dry_run_result.stderr)

        # Check for the success message in stdout, as return code might be 1 even on valid dry run
        validation_successful = "The run is valid." in dry_run_result.stdout

        if validation_successful:
            print(f"   ✅ Validation successful for '{run_id}' (based on stdout).")

            # --- 2. If validation succeeded, upload for real ---
            print(f"\n   [2/2] Uploading '{run_id}'...")
            upload_command = [
                "tira-cli", "upload", "--dataset", "sci-20250430-test",
                "--system", run_id, "--directory", submission_dir
            ]
            # For actual upload, we expect returncode 0 for success
            upload_result = subprocess.run(upload_command, capture_output=True, text=True, check=False)

            print("STDOUT from upload:")
            print(upload_result.stdout)
            print("STDERR from upload:")
            print(upload_result.stderr)

            # Check return code AND success message for actual upload
            if upload_result.returncode == 0 and "The data is uploaded." in upload_result.stdout :
                print(f"   ✅ Successfully uploaded '{run_id}'.")
            else:
                print(f"   ❌ Upload FAILED for '{run_id}'. Return code: {upload_result.returncode}")
        else:
            # If "The run is valid." was not in stdout, then it's a real validation failure.
            print(f"   ❌ Validation FAILED for '{run_id}' (based on stdout). Skipping upload. Dry run return code: {dry_run_result.returncode}")
        print("-" * (21 + len(run_id)))

    print("\n\n##################################")
    print("All executions complete.")
    print("##################################")


Mounted at /content/drive
2025-05-25 19:18:32 Google Drive mounted successfully.
2025-05-25 19:18:32 --------------------------------------------------
2025-05-25 19:18:32 WARNING: Script is in SINGLE RUN TEST MODE. Processing only: ['clef25-sambs_A3_TFIDF_TestAbs.txt', 'clef25-sambs_A4_TFIDF_TestFull.txt', 'clef25-sambs_A7_BM25_TestAbs.txt', 'clef25-sambs_A8_BM25_TestFull.txt', 'clef25-sambs_B3_SBERT_V1_TestAbs.txt', 'clef25-sambs_B3_SBERT_V2_MPNet_TestAbs.txt', 'clef25-sambs_B4_SBERT_V1_TestFull.txt', 'clef25-sambs_B4_SBERT_V2_MPNet_TestFull.txt', 'clef25-sambs_C3_CE_L12_TestAbs.txt', 'clef25-sambs_C3_CE_L6_TestAbs.txt', 'clef25-sambs_C4_CE_L12_TestFull.txt', 'clef25-sambs_C4_CE_L6_TestFull.txt']
2025-05-25 19:18:32 Starting file preparation for 12 runs from DEDUPED directory...

------------------------------------------------------------
2025-05-25 19:18:32 Preparing: clef25-sambs_A3_TFIDF_TestAbs.txt
2025-05-25 19:18:33   ✅ Preparation complete for A3-TFIDF-TestAbs. Nov lines: 768